# Day 23 — Optimized Historical Model Selection

Complete historical optimization and freeze the selected model before future evaluation.

In [ ]:
# Imports
import json
from pathlib import Path

In [ ]:
# Project paths
def find_project_root():
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "README.md").exists():
            return path
    raise RuntimeError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

def resolve_artifact_paths(value):
    """Rebase legacy project paths in loaded metadata; never rewrite artifacts.

    Keep all non-path values, model identifiers, and scientific settings intact.
    Support the original project name and the public repository name.
    """
    if isinstance(value, dict):
        return {key: resolve_artifact_paths(item) for key, item in value.items()}
    if isinstance(value, list):
        return [resolve_artifact_paths(item) for item in value]
    if isinstance(value, str):
        normalized = value.replace("\\", "/")
        for name in ("FinGPT_Project", "FinGPT-Temporal-Generalization"):
            marker = "/" + name + "/"
            if marker in normalized and "\n" not in normalized:
                # Only standalone filesystem references, never prose containing a path.
                prefix, relative = normalized.split(marker, 1)
                if prefix.startswith("/") or (len(prefix) > 1 and prefix[1] == ":"):
                    if not any(char.isspace() for char in relative):
                        return str(PROJECT_ROOT / relative)
    return value

def load_artifact_json(stream, *args, **kwargs):
    return resolve_artifact_paths(json.load(stream, *args, **kwargs))


In [ ]:
# Reproducibility settings
# Original seeds and deterministic settings remain at their original workflow
# positions below, including stage-specific RNG resets. No new seed is introduced.

In [6]:
# ==================================================================================================
# DAY 23 — CELL 1
# FORMAL DAY 22 RESUME-STATE RECONSTRUCTION
# ==================================================================================================

import json
import platform
from pathlib import Path
from datetime import datetime, timezone

import torch
import pandas as pd
import numpy as np


# --------------------------------------------------------------------------------------------------
# 1. Day 23 identity
# --------------------------------------------------------------------------------------------------

DAY23_TITLE = (
    "Day 23 — Historical Optimization Completion "
    "and Final Configuration Selection"
)


# --------------------------------------------------------------------------------------------------
# 2. Project paths
#
# IMPORTANT:
# PROJECT_ROOT is discovered in the project-path setup cell.
# --------------------------------------------------------------------------------------------------

assert "PROJECT_ROOT" in globals(), (
    "DAY 23 STOP: PROJECT_ROOT missing. Run Cell 0A first."
)

assert PROJECT_ROOT.exists(), (
    f"DAY 23 STOP: PROJECT_ROOT does not exist: {PROJECT_ROOT}"
)


RESULTS_ROOT = (
    PROJECT_ROOT
    / "results"
)

NOTEBOOK_ROOT = (
    PROJECT_ROOT
    / "notebooks"
)

DAY22_RESULTS_ROOT = (
    RESULTS_ROOT
    / "day22"
)

DAY23_RESULTS_ROOT = (
    RESULTS_ROOT
    / "day23"
)

DAY23_RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 3. Day 22 formal checkpoint paths
# --------------------------------------------------------------------------------------------------

DAY22_PART1_CHECKPOINT_PATH = (
    DAY22_RESULTS_ROOT
    / "checkpoints"
    / "day22_part1_checkpoint.json"
)

DAY22_PART1_RESUME_PATH = (
    DAY22_RESULTS_ROOT
    / "checkpoints"
    / "day22_part1_resume_manifest.json"
)

DAY22_NOTEBOOK_PATH = (
    NOTEBOOK_ROOT
    / "day22_historical_model_optimization_and_training_adequacy_audit.ipynb"
)


# --------------------------------------------------------------------------------------------------
# 4. Load frozen Day 22 state
# --------------------------------------------------------------------------------------------------

assert DAY22_PART1_CHECKPOINT_PATH.exists(), (
    f"DAY 23 STOP: missing checkpoint: {DAY22_PART1_CHECKPOINT_PATH}"
)

assert DAY22_PART1_RESUME_PATH.exists(), (
    f"DAY 23 STOP: missing resume manifest: {DAY22_PART1_RESUME_PATH}"
)


with open(
    DAY22_PART1_CHECKPOINT_PATH,
    "r",
    encoding="utf-8",
) as f:

    day22_part1_state = load_artifact_json(
        f
    )


with open(
    DAY22_PART1_RESUME_PATH,
    "r",
    encoding="utf-8",
) as f:

    day22_resume_manifest = load_artifact_json(
        f
    )


# --------------------------------------------------------------------------------------------------
# 5. Recover frozen incumbent
# --------------------------------------------------------------------------------------------------

DAY23_INCUMBENT = (
    day22_part1_state[
        "current_incumbent"
    ]
)


INCUMBENT_MODEL_NAME = (
    DAY23_INCUMBENT[
        "model"
    ]
)

INCUMBENT_ACCURACY = float(
    DAY23_INCUMBENT[
        "accuracy"
    ]
)

INCUMBENT_MACRO_F1 = float(
    DAY23_INCUMBENT[
        "macro_f1"
    ]
)

INCUMBENT_NEG_RECALL = float(
    DAY23_INCUMBENT[
        "negative_recall"
    ]
)

INCUMBENT_NEU_RECALL = float(
    DAY23_INCUMBENT[
        "neutral_recall"
    ]
)

INCUMBENT_POS_RECALL = float(
    DAY23_INCUMBENT[
        "positive_recall"
    ]
)


# --------------------------------------------------------------------------------------------------
# 6. Recover formal resume point
# --------------------------------------------------------------------------------------------------

DAY23_RESUME_POINT = (
    day22_resume_manifest[
        "resume_from"
    ]
)

DAY23_NEXT_FORMAL_TASK = (
    day22_resume_manifest[
        "next_formal_task"
    ]
)

DAY23_PENDING_SEQUENCE = (
    day22_resume_manifest[
        "pending_sequence"
    ]
)


# --------------------------------------------------------------------------------------------------
# 7. Stage C frozen protocol paths
# --------------------------------------------------------------------------------------------------

STAGE_C_DIR = (
    DAY22_RESULTS_ROOT
    / "runs"
    / "stage_c_sampling_audit"
)

STAGE_C_ARTIFACT_DIR = (
    STAGE_C_DIR
    / "artifacts"
)

STAGE_C_PROTOCOL_PATH = (
    STAGE_C_ARTIFACT_DIR
    / "stage_c_sampling_protocol.json"
)

STAGE_C_CLASS_WEIGHTS_PATH = (
    STAGE_C_ARTIFACT_DIR
    / "stage_c_sampling_class_weights.csv"
)

STAGE_C_SAMPLER_AUDIT_PATH = (
    STAGE_C_ARTIFACT_DIR
    / "stage_c_sampler_audit_summary.json"
)


# --------------------------------------------------------------------------------------------------
# 8. Load frozen Stage C protocol
# --------------------------------------------------------------------------------------------------

assert STAGE_C_PROTOCOL_PATH.exists()
assert STAGE_C_CLASS_WEIGHTS_PATH.exists()
assert STAGE_C_SAMPLER_AUDIT_PATH.exists()


with open(
    STAGE_C_PROTOCOL_PATH,
    "r",
    encoding="utf-8",
) as f:

    STAGE_C_PROTOCOL = load_artifact_json(
        f
    )


stage_c_weight_table = pd.read_csv(
    STAGE_C_CLASS_WEIGHTS_PATH
)


# --------------------------------------------------------------------------------------------------
# 9. Recover frozen class weights
# --------------------------------------------------------------------------------------------------

STAGE_C_LABELS = [
    "negative",
    "neutral",
    "positive",
]


inverse_sqrt_class_weights = {

    row["label"]:
        float(
            row[
                "inverse_sqrt_weight"
            ]
        )

    for _, row
    in stage_c_weight_table.iterrows()
}


capped_balanced_class_weights = {

    row["label"]:
        float(
            row[
                "capped_balanced_weight"
            ]
        )

    for _, row
    in stage_c_weight_table.iterrows()
}


# --------------------------------------------------------------------------------------------------
# 10. Required artifacts
# --------------------------------------------------------------------------------------------------

required_resume_artifacts = {

    "day22_notebook":
        DAY22_NOTEBOOK_PATH,

    "day22_checkpoint":
        DAY22_PART1_CHECKPOINT_PATH,

    "day22_resume_manifest":
        DAY22_PART1_RESUME_PATH,

    "stage_c_protocol":
        STAGE_C_PROTOCOL_PATH,

    "stage_c_class_weights":
        STAGE_C_CLASS_WEIGHTS_PATH,

    "stage_c_sampler_audit":
        STAGE_C_SAMPLER_AUDIT_PATH,

    "stage_b_selected_adapter":
        (
            DAY22_RESULTS_ROOT
            / "runs"
            / "stage_b_horizon_audit"
            / "model"
            / "selected_adapter"
        ),

    "stage_b_dev_metrics":
        (
            DAY22_RESULTS_ROOT
            / "runs"
            / "stage_b_horizon_audit"
            / "artifacts"
            / "stage_b_500_optimization_dev_metrics.json"
        ),

    "inverse_sqrt_metrics":
        (
            STAGE_C_DIR
            / "inverse_sqrt"
            / "artifacts"
            / "inverse_sqrt_optimization_dev_metrics.json"
        ),

    "inverse_sqrt_checkpoint_500":
        (
            STAGE_C_DIR
            / "inverse_sqrt"
            / "model"
            / "checkpoints"
            / "checkpoint-500"
        ),
}


resume_artifact_checks = {

    name:
        path.exists()

    for name, path
    in required_resume_artifacts.items()
}


# --------------------------------------------------------------------------------------------------
# 11. Formal Day 23 resume checks
# --------------------------------------------------------------------------------------------------

day23_resume_checks = {

    "project_root_exists":
        PROJECT_ROOT.exists(),

    "day22_checkpoint_complete":
        day22_part1_state[
            "status"
        ]
        == "COMPLETE",

    "resume_manifest_day22":
        day22_resume_manifest[
            "day"
        ]
        == 22,

    "resume_point_is_capped_balanced":
        DAY23_RESUME_POINT
        ==
        "Stage C capped-balanced candidate",

    "incumbent_is_stage_b_500":
        INCUMBENT_MODEL_NAME
        ==
        "Stage B-500 natural sampling",

    "incumbent_accuracy_matches":
        abs(
            INCUMBENT_ACCURACY
            -
            0.840842
        )
        < 1e-6,

    "stage_c_protocol_frozen":
        STAGE_C_PROTOCOL[
            "freeze_status"
        ]
        ==
        "STAGE_C_SAMPLING_PROTOCOL_FROZEN",

    "historical_validation_locked":
        day22_resume_manifest[
            "historical_validation_locked"
        ]
        is True,

    "future_locked":
        day22_resume_manifest[
            "future_f1_f6_locked"
        ]
        is True,

    "all_required_resume_artifacts_present":
        all(
            resume_artifact_checks.values()
        ),
}


all_day23_resume_checks_pass = bool(
    all(
        day23_resume_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. Runtime information
# --------------------------------------------------------------------------------------------------

runtime_info = {

    "python":
        platform.python_version(),

    "torch":
        torch.__version__,

    "cuda_available":
        torch.cuda.is_available(),

    "cuda_runtime":
        torch.version.cuda,

    "gpu":
        (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),

    "gpu_memory_gb":
        (
            torch.cuda.get_device_properties(
                0
            ).total_memory
            /
            (1024 ** 3)
            if torch.cuda.is_available()
            else None
        ),
}


# --------------------------------------------------------------------------------------------------
# 13. Save Day 23 bootstrap manifest
# --------------------------------------------------------------------------------------------------

DAY23_BOOTSTRAP_PATH = (
    DAY23_RESULTS_ROOT
    / "day23_bootstrap_resume_manifest.json"
)


day23_bootstrap_manifest = {

    "day":
        23,

    "title":
        DAY23_TITLE,

    "resumed_from_day22":
        True,

    "resume_point":
        DAY23_RESUME_POINT,

    "next_formal_task":
        DAY23_NEXT_FORMAL_TASK,

    "incumbent":
        DAY23_INCUMBENT,

    "pending_sequence":
        DAY23_PENDING_SEQUENCE,

    "historical_validation_locked":
        True,

    "future_f1_f6_locked":
        True,

    "runtime":
        runtime_info,

    "resume_checks": {

        key:
            bool(
                value
            )

        for key, value
        in day23_resume_checks.items()
    },

    "all_resume_checks_pass":
        all_day23_resume_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    DAY23_BOOTSTRAP_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        day23_bootstrap_manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 14. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print(DAY23_TITLE)
print("=" * 120)


print("\n[RUNTIME]")
print(
    f"Python                      : "
    f"{runtime_info['python']}"
)
print(
    f"PyTorch                     : "
    f"{runtime_info['torch']}"
)
print(
    f"CUDA available              : "
    f"{runtime_info['cuda_available']}"
)
print(
    f"CUDA runtime                : "
    f"{runtime_info['cuda_runtime']}"
)
print(
    f"GPU                         : "
    f"{runtime_info['gpu']}"
)

if runtime_info[
    "gpu_memory_gb"
] is not None:

    print(
        f"GPU memory                  : "
        f"{runtime_info['gpu_memory_gb']:.2f} GB"
    )


print("\n[DAY 22 RECOVERY]")
print(
    f"Project root                : "
    f"{PROJECT_ROOT}"
)
print(
    f"Checkpoint                  : "
    f"{DAY22_PART1_CHECKPOINT_PATH}"
)
print(
    f"Resume manifest             : "
    f"{DAY22_PART1_RESUME_PATH}"
)
print(
    f"Resume point                : "
    f"{DAY23_RESUME_POINT}"
)


print("\n[CURRENT INCUMBENT]")
print(
    f"Model                       : "
    f"{INCUMBENT_MODEL_NAME}"
)
print(
    f"Accuracy                    : "
    f"{INCUMBENT_ACCURACY:.6f}"
)
print(
    f"Macro-F1                    : "
    f"{INCUMBENT_MACRO_F1:.6f}"
)
print(
    f"Negative recall             : "
    f"{INCUMBENT_NEG_RECALL:.6f}"
)
print(
    f"Neutral recall              : "
    f"{INCUMBENT_NEU_RECALL:.6f}"
)
print(
    f"Positive recall             : "
    f"{INCUMBENT_POS_RECALL:.6f}"
)


print("\n[STAGE C CAPPED-BALANCED WEIGHTS]")

for label in STAGE_C_LABELS:

    print(
        f"{label:<27}: "
        f"{capped_balanced_class_weights[label]:.6f}"
    )


print("\n[RESUME ARTIFACTS]")

for name, passed in resume_artifact_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[DAY 23 RESUME CHECKS]")

for name, passed in day23_resume_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[NEXT FORMAL TASK]")
print(
    DAY23_NEXT_FORMAL_TASK
)


print("\n[BOOTSTRAP ARTIFACT]")
print(
    DAY23_BOOTSTRAP_PATH
)


print("\n" + "-" * 120)

if all_day23_resume_checks_pass:

    print(
        "DAY 23 CELL 1 STATUS: PASS — "
        "DAY 22 STATE RECOVERED; "
        "READY FOR STAGE C CAPPED-BALANCED CANDIDATE"
    )

else:

    print(
        "DAY 23 CELL 1 STATUS: FAIL — "
        "DO NOT INITIALIZE NEW TRAINING"
    )

print("=" * 120)

Day 23 — Historical Optimization Completion and Final Configuration Selection

[RUNTIME]
Python                      : 3.13.15
PyTorch                     : 2.11.0+cu130
CUDA available              : True
CUDA runtime                : 13.0
GPU                         : Tesla T4
GPU memory                  : 14.56 GB

[DAY 22 RECOVERY]
Project root                : /content/gdrive_day23/MyDrive/FinGPT_Project
Checkpoint                  : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day22/checkpoints/day22_part1_checkpoint.json
Resume manifest             : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day22/checkpoints/day22_part1_resume_manifest.json
Resume point                : Stage C capped-balanced candidate

[CURRENT INCUMBENT]
Model                       : Stage B-500 natural sampling
Accuracy                    : 0.840842
Macro-F1                    : 0.808201
Negative recall             : 0.721344
Neutral recall              : 0.889754
Positive recall          

In [7]:
# ==================================================================================================
# DAY 23 — CELL 2
# FROZEN OPTIMIZATION DATA + TOKENIZER + ENCODED DATASET RECONSTRUCTION
# FIXED: observation_id is reconstructed from channel_username::message_id
# ==================================================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset

from transformers import AutoTokenizer


# --------------------------------------------------------------------------------------------------
# 1. Frozen upstream paths
# --------------------------------------------------------------------------------------------------

SOURCE_DATA_PATH = (
    PROJECT_ROOT
    / "results"
    / "day07"
    / "temporal_dataset_clean.csv"
)

OPT_SPLIT_MANIFEST_PATH = (
    DAY22_RESULTS_ROOT
    / "protocol"
    / "historical_optimization_split_manifest.csv"
)

OPT_SPLIT_AUDIT_PATH = (
    DAY22_RESULTS_ROOT
    / "protocol"
    / "historical_optimization_split_audit.json"
)


assert SOURCE_DATA_PATH.exists(), (
    f"DAY 23 STOP: missing source dataset:\n{SOURCE_DATA_PATH}"
)

assert OPT_SPLIT_MANIFEST_PATH.exists(), (
    f"DAY 23 STOP: missing optimization split manifest:\n"
    f"{OPT_SPLIT_MANIFEST_PATH}"
)

assert OPT_SPLIT_AUDIT_PATH.exists(), (
    f"DAY 23 STOP: missing optimization split audit:\n"
    f"{OPT_SPLIT_AUDIT_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 2. Load frozen source population
#
# IMPORTANT:
# temporal_dataset_clean.csv does NOT contain observation_id.
# Day 22 identity rule:
#
#     observation_id = channel_username + "::" + message_id
# --------------------------------------------------------------------------------------------------

source_df = pd.read_csv(
    SOURCE_DATA_PATH
)


required_source_columns = {
    "message_id",
    "channel_username",
    "date",
    "text",
    "sentiment",
}


assert required_source_columns.issubset(
    source_df.columns
), (
    "DAY 23 STOP: required source columns missing.\n"
    f"Available columns: {source_df.columns.tolist()}"
)


# --------------------------------------------------------------------------------------------------
# 3. Reconstruct exact frozen observation_id
# --------------------------------------------------------------------------------------------------

assert source_df[
    "message_id"
].notna().all(), (
    "DAY 23 STOP: missing message_id values."
)

assert source_df[
    "channel_username"
].notna().all(), (
    "DAY 23 STOP: missing channel_username values."
)


source_df[
    "observation_id"
] = (
    source_df[
        "channel_username"
    ].astype(str)
    +
    "::"
    +
    source_df[
        "message_id"
    ].astype(str)
)


assert source_df[
    "observation_id"
].notna().all()


assert source_df[
    "observation_id"
].is_unique, (
    "DAY 23 STOP: reconstructed observation_id is not globally unique."
)


# --------------------------------------------------------------------------------------------------
# 4. Normalize frozen source fields
# --------------------------------------------------------------------------------------------------

source_df[
    "date"
] = pd.to_datetime(
    source_df[
        "date"
    ],
    utc=True,
)


source_df[
    "sentiment"
] = (
    source_df[
        "sentiment"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


source_df[
    "text"
] = source_df[
    "text"
].astype(str)


# --------------------------------------------------------------------------------------------------
# 5. Load deterministic optimization split manifest
# --------------------------------------------------------------------------------------------------

split_manifest = pd.read_csv(
    OPT_SPLIT_MANIFEST_PATH
)


assert "observation_id" in split_manifest.columns, (
    "DAY 23 STOP: observation_id missing from split manifest."
)


split_manifest[
    "observation_id"
] = split_manifest[
    "observation_id"
].astype(str)


# --------------------------------------------------------------------------------------------------
# 6. Detect frozen split-label column
#
# Day 22 manifest is expected to use day22_split.
# We still detect it safely to avoid hard failure if schema metadata varies.
# --------------------------------------------------------------------------------------------------

candidate_split_columns = [
    col
    for col
    in split_manifest.columns
    if col != "observation_id"
]


split_column = None


for col in candidate_split_columns:

    values = set(
        split_manifest[
            col
        ]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
        .tolist()
    )

    has_train = any(
        value in {
            "train",
            "optimization_train",
            "opt_train",
        }
        for value
        in values
    )

    has_dev = any(
        value in {
            "dev",
            "validation",
            "optimization_dev",
            "opt_dev",
        }
        for value
        in values
    )

    if has_train and has_dev:

        split_column = col
        break


assert split_column is not None, (
    "DAY 23 STOP: could not identify split-label column.\n"
    f"Manifest columns: {split_manifest.columns.tolist()}"
)


manifest_split_values = (
    split_manifest[
        split_column
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


train_mask = manifest_split_values.isin(
    [
        "train",
        "optimization_train",
        "opt_train",
    ]
)


dev_mask = manifest_split_values.isin(
    [
        "dev",
        "validation",
        "optimization_dev",
        "opt_dev",
    ]
)


train_ids = (
    split_manifest.loc[
        train_mask,
        "observation_id",
    ]
    .tolist()
)


dev_ids = (
    split_manifest.loc[
        dev_mask,
        "observation_id",
    ]
    .tolist()
)


# --------------------------------------------------------------------------------------------------
# 7. Reconstruct exact Day 22 optimization populations in manifest order
# --------------------------------------------------------------------------------------------------

source_by_id = (
    source_df
    .set_index(
        "observation_id",
        drop=False,
    )
)


missing_train_ids = [
    obs_id
    for obs_id
    in train_ids
    if obs_id not in source_by_id.index
]


missing_dev_ids = [
    obs_id
    for obs_id
    in dev_ids
    if obs_id not in source_by_id.index
]


assert len(
    missing_train_ids
) == 0, (
    f"DAY 23 STOP: {len(missing_train_ids)} train IDs missing from source."
)


assert len(
    missing_dev_ids
) == 0, (
    f"DAY 23 STOP: {len(missing_dev_ids)} dev IDs missing from source."
)


optimization_train = (
    source_by_id
    .loc[
        train_ids
    ]
    .reset_index(
        drop=True
    )
    .copy()
)


optimization_dev = (
    source_by_id
    .loc[
        dev_ids
    ]
    .reset_index(
        drop=True
    )
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 8. Frozen split integrity
# --------------------------------------------------------------------------------------------------

EXPECTED_TRAIN_N = 10831
EXPECTED_DEV_N = 2708


EXPECTED_TRAIN_LABELS = {
    "negative": 2276,
    "neutral": 6767,
    "positive": 1788,
}


EXPECTED_DEV_LABELS = {
    "negative": 506,
    "neutral": 1669,
    "positive": 533,
}


train_label_counts = (
    optimization_train[
        "sentiment"
    ]
    .value_counts()
    .reindex(
        STAGE_C_LABELS,
        fill_value=0,
    )
    .astype(int)
    .to_dict()
)


dev_label_counts = (
    optimization_dev[
        "sentiment"
    ]
    .value_counts()
    .reindex(
        STAGE_C_LABELS,
        fill_value=0,
    )
    .astype(int)
    .to_dict()
)


train_max_date = (
    optimization_train[
        "date"
    ].max()
)


dev_min_date = (
    optimization_dev[
        "date"
    ].min()
)


train_dev_overlap = (
    set(
        optimization_train[
            "observation_id"
        ]
    )
    &
    set(
        optimization_dev[
            "observation_id"
        ]
    )
)


# --------------------------------------------------------------------------------------------------
# 9. Exact tokenizer / prompt semantics
# --------------------------------------------------------------------------------------------------

BASE_MODEL_NAME = "meta-llama/Meta-Llama-3-8B"

MAX_SEQ_LENGTH = 512


FROZEN_INSTRUCTION = (
    "Classify the sentiment of the following financial news "
    "as negative, neutral, or positive."
)


# Literal backslash-n, NOT real newline.
FROZEN_PROMPT_TEMPLATE = (
    r"Instruction:{instruction}\nInput:{input}\nAnswer: "
)


tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=True,
)


tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


assert tokenizer.eos_token_id == 128001
assert tokenizer.pad_token_id == 128001


def build_inference_prompt(
    text,
):

    return FROZEN_PROMPT_TEMPLATE.format(
        instruction=
            FROZEN_INSTRUCTION,

        input=
            str(
                text
            ),
    )


# --------------------------------------------------------------------------------------------------
# 10. Exact training encoder
# --------------------------------------------------------------------------------------------------

def encode_training_example(
    text,
    label,
    max_seq_length=MAX_SEQ_LENGTH,
):

    prompt_text = build_inference_prompt(
        text
    )

    target_text = (
        str(
            label
        )
        .strip()
        .lower()
        +
        tokenizer.eos_token
    )


    prompt_ids = tokenizer.encode(
        prompt_text,
        add_special_tokens=False,
    )


    target_ids = tokenizer.encode(
        target_text,
        add_special_tokens=False,
    )


    if len(
        target_ids
    ) > max_seq_length:

        raise ValueError(
            "Target sequence exceeds max_seq_length."
        )


    max_prompt_length = (
        max_seq_length
        -
        len(
            target_ids
        )
    )


    prompt_truncated = (
        len(
            prompt_ids
        )
        >
        max_prompt_length
    )


    if prompt_truncated:

        prompt_ids = (
            prompt_ids[
                :max_prompt_length
            ]
        )


    input_ids = (
        prompt_ids
        +
        target_ids
    )


    labels = (
        [-100]
        *
        len(
            prompt_ids
        )
        +
        target_ids
    )


    attention_mask = [
        1
    ] * len(
        input_ids
    )


    return {

        "input_ids":
            input_ids,

        "attention_mask":
            attention_mask,

        "labels":
            labels,

        "prompt_length":
            len(
                prompt_ids
            ),

        "target_length":
            len(
                target_ids
            ),

        "sequence_length":
            len(
                input_ids
            ),

        "prompt_truncated":
            bool(
                prompt_truncated
            ),
    }


# --------------------------------------------------------------------------------------------------
# 11. Dataset wrapper
# --------------------------------------------------------------------------------------------------

class FrozenEncodedDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
    ):

        self.dataframe = (
            dataframe
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.encoded = []


        for row in self.dataframe.itertuples(
            index=False
        ):

            self.encoded.append(
                encode_training_example(
                    text=
                        row.text,

                    label=
                        row.sentiment,
                )
            )


    def __len__(
        self,
    ):

        return len(
            self.encoded
        )


    def __getitem__(
        self,
        idx,
    ):

        item = self.encoded[
            idx
        ]

        return {
            "input_ids":
                item[
                    "input_ids"
                ],

            "attention_mask":
                item[
                    "attention_mask"
                ],

            "labels":
                item[
                    "labels"
                ],
        }


hf_opt_train_model = FrozenEncodedDataset(
    optimization_train
)


hf_opt_dev_model = FrozenEncodedDataset(
    optimization_dev
)


# --------------------------------------------------------------------------------------------------
# 12. Dynamic target-aware collator
# --------------------------------------------------------------------------------------------------

class TargetAwareDataCollator:

    def __init__(
        self,
        tokenizer,
    ):

        self.tokenizer = tokenizer


    def __call__(
        self,
        features,
    ):

        max_len = max(
            len(
                feature[
                    "input_ids"
                ]
            )
            for feature
            in features
        )


        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []


        for feature in features:

            pad_len = (
                max_len
                -
                len(
                    feature[
                        "input_ids"
                    ]
                )
            )


            batch_input_ids.append(
                feature[
                    "input_ids"
                ]
                +
                [
                    self.tokenizer.pad_token_id
                ]
                *
                pad_len
            )


            batch_attention_mask.append(
                feature[
                    "attention_mask"
                ]
                +
                [0]
                *
                pad_len
            )


            batch_labels.append(
                feature[
                    "labels"
                ]
                +
                [-100]
                *
                pad_len
            )


        return {

            "input_ids":
                torch.tensor(
                    batch_input_ids,
                    dtype=torch.long,
                ),

            "attention_mask":
                torch.tensor(
                    batch_attention_mask,
                    dtype=torch.long,
                ),

            "labels":
                torch.tensor(
                    batch_labels,
                    dtype=torch.long,
                ),
        }


data_collator = TargetAwareDataCollator(
    tokenizer
)


# --------------------------------------------------------------------------------------------------
# 13. Encoding audits
# --------------------------------------------------------------------------------------------------

train_sequence_lengths = np.array(
    [
        item[
            "sequence_length"
        ]
        for item
        in hf_opt_train_model.encoded
    ]
)


dev_sequence_lengths = np.array(
    [
        item[
            "sequence_length"
        ]
        for item
        in hf_opt_dev_model.encoded
    ]
)


train_truncated_n = int(
    sum(
        item[
            "prompt_truncated"
        ]
        for item
        in hf_opt_train_model.encoded
    )
)


dev_truncated_n = int(
    sum(
        item[
            "prompt_truncated"
        ]
        for item
        in hf_opt_dev_model.encoded
    )
)


sample_encoded = (
    hf_opt_train_model.encoded[
        0
    ]
)


sample_target_ids = [
    token_id
    for token_id, label_id
    in zip(
        sample_encoded[
            "input_ids"
        ],
        sample_encoded[
            "labels"
        ],
    )
    if label_id != -100
]


sample_target_text = tokenizer.decode(
    sample_target_ids,
    skip_special_tokens=False,
)


# --------------------------------------------------------------------------------------------------
# 14. Collator dry run
# --------------------------------------------------------------------------------------------------

collator_dry_batch = data_collator(
    [
        hf_opt_train_model[
            0
        ],
        hf_opt_train_model[
            1
        ],
        hf_opt_train_model[
            2
        ],
    ]
)


# --------------------------------------------------------------------------------------------------
# 15. Formal checks
# --------------------------------------------------------------------------------------------------

cell2_checks = {

    "source_population_39965":
        len(
            source_df
        )
        == 39965,

    "observation_id_reconstructed":
        "observation_id"
        in source_df.columns,

    "observation_id_unique":
        source_df[
            "observation_id"
        ].is_unique,

    "train_n_10831":
        len(
            optimization_train
        )
        == EXPECTED_TRAIN_N,

    "dev_n_2708":
        len(
            optimization_dev
        )
        == EXPECTED_DEV_N,

    "train_label_counts_exact":
        train_label_counts
        ==
        EXPECTED_TRAIN_LABELS,

    "dev_label_counts_exact":
        dev_label_counts
        ==
        EXPECTED_DEV_LABELS,

    "zero_train_dev_overlap":
        len(
            train_dev_overlap
        )
        == 0,

    "strict_chronology":
        train_max_date
        <
        dev_min_date,

    "train_dataset_n_10831":
        len(
            hf_opt_train_model
        )
        == 10831,

    "dev_dataset_n_2708":
        len(
            hf_opt_dev_model
        )
        == 2708,

    "literal_backslash_prompt":
        repr(
            FROZEN_PROMPT_TEMPLATE
        )
        ==
        "'Instruction:{instruction}\\\\nInput:{input}\\\\nAnswer: '",

    "eos_id_128001":
        tokenizer.eos_token_id
        == 128001,

    "pad_id_128001":
        tokenizer.pad_token_id
        == 128001,

    "right_padding":
        tokenizer.padding_side
        ==
        "right",

    "train_no_truncation":
        train_truncated_n
        == 0,

    "dev_no_truncation":
        dev_truncated_n
        == 0,

    "max_train_length_le_512":
        int(
            train_sequence_lengths.max()
        )
        <= 512,

    "max_dev_length_le_512":
        int(
            dev_sequence_lengths.max()
        )
        <= 512,

    "collator_batch_size_3":
        collator_dry_batch[
            "input_ids"
        ].shape[0]
        == 3,

    "collator_pad_token_correct":
        tokenizer.pad_token_id
        == 128001,

    "historical_validation_not_loaded":
        True,

    "future_not_loaded":
        True,
}


all_cell2_checks_pass = bool(
    all(
        cell2_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 16. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — FROZEN DATA + TOKENIZER + ENCODING RECONSTRUCTION")
print("=" * 120)


print("\n[SOURCE IDENTITY]")
print(
    "Identity rule               : "
    "channel_username::message_id"
)
print(
    f"Source rows                 : "
    f"{len(source_df):,}"
)
print(
    f"Observation IDs unique      : "
    f"{source_df['observation_id'].is_unique}"
)


print("\n[SPLIT MANIFEST]")
print(
    f"Manifest                    : "
    f"{OPT_SPLIT_MANIFEST_PATH}"
)
print(
    f"Detected split column       : "
    f"{split_column}"
)


print("\n[OPTIMIZATION TRAIN]")
print(
    f"N                           : "
    f"{len(optimization_train):,}"
)
print(
    f"Date range                  : "
    f"{optimization_train['date'].min()} "
    f"→ "
    f"{optimization_train['date'].max()}"
)
print(
    f"Labels                      : "
    f"{train_label_counts}"
)


print("\n[OPTIMIZATION DEV]")
print(
    f"N                           : "
    f"{len(optimization_dev):,}"
)
print(
    f"Date range                  : "
    f"{optimization_dev['date'].min()} "
    f"→ "
    f"{optimization_dev['date'].max()}"
)
print(
    f"Labels                      : "
    f"{dev_label_counts}"
)


print("\n[TOKENIZER / PROMPT]")
print(
    f"Base tokenizer              : "
    f"{BASE_MODEL_NAME}"
)
print(
    f"EOS token                   : "
    f"{tokenizer.eos_token}"
)
print(
    f"EOS id                      : "
    f"{tokenizer.eos_token_id}"
)
print(
    f"PAD id                      : "
    f"{tokenizer.pad_token_id}"
)
print(
    f"Padding side                : "
    f"{tokenizer.padding_side}"
)
print(
    f"Prompt template repr        : "
    f"{repr(FROZEN_PROMPT_TEMPLATE)}"
)


print("\n[ENCODING — TRAIN]")
print(
    f"Min length                  : "
    f"{train_sequence_lengths.min()}"
)
print(
    f"Mean length                 : "
    f"{train_sequence_lengths.mean():.2f}"
)
print(
    f"P95 length                  : "
    f"{np.percentile(train_sequence_lengths, 95):.0f}"
)
print(
    f"Max length                  : "
    f"{train_sequence_lengths.max()}"
)
print(
    f"Truncated                   : "
    f"{train_truncated_n}"
)


print("\n[ENCODING — DEV]")
print(
    f"Min length                  : "
    f"{dev_sequence_lengths.min()}"
)
print(
    f"Mean length                 : "
    f"{dev_sequence_lengths.mean():.2f}"
)
print(
    f"P95 length                  : "
    f"{np.percentile(dev_sequence_lengths, 95):.0f}"
)
print(
    f"Max length                  : "
    f"{dev_sequence_lengths.max()}"
)
print(
    f"Truncated                   : "
    f"{dev_truncated_n}"
)


print("\n[SAMPLE TARGET]")
print(
    f"Target text                 : "
    f"{repr(sample_target_text)}"
)


print("\n[COLLATOR DRY RUN]")
print(
    f"Input shape                 : "
    f"{tuple(collator_dry_batch['input_ids'].shape)}"
)
print(
    f"Label shape                 : "
    f"{tuple(collator_dry_batch['labels'].shape)}"
)


print("\n[FORMAL CHECKS]")

for name, passed in cell2_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n" + "-" * 120)

if all_cell2_checks_pass:

    print(
        "DAY 23 CELL 2 STATUS: PASS — "
        "FROZEN OPTIMIZATION DATA AND MODEL INPUT SEMANTICS RECOVERED"
    )

else:

    print(
        "DAY 23 CELL 2 STATUS: FAIL — "
        "DO NOT INITIALIZE CAPPED-BALANCED TRAINING"
    )

print("=" * 120)

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


DAY 23 — FROZEN DATA + TOKENIZER + ENCODING RECONSTRUCTION

[SOURCE IDENTITY]
Identity rule               : channel_username::message_id
Source rows                 : 39,965
Observation IDs unique      : True

[SPLIT MANIFEST]
Manifest                    : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day22/protocol/historical_optimization_split_manifest.csv
Detected split column       : day22_split

[OPTIMIZATION TRAIN]
N                           : 10,831
Date range                  : 2025-06-03 21:06:16+00:00 → 2025-10-10 01:36:47+00:00
Labels                      : {'negative': 2276, 'neutral': 6767, 'positive': 1788}

[OPTIMIZATION DEV]
N                           : 2,708
Date range                  : 2025-10-10 01:37:04+00:00 → 2025-10-31 21:14:14+00:00
Labels                      : {'negative': 506, 'neutral': 1669, 'positive': 533}

[TOKENIZER / PROMPT]
Base tokenizer              : meta-llama/Meta-Llama-3-8B
EOS token                   : <|end_of_text|>
EOS id          

In [8]:
# ==================================================================================================
# DAY 23 — CELL 3
# CAPPED-BALANCED DETERMINISTIC SAMPLER RECOVERY
# + EXACT MATCH AGAINST DAY 22 FROZEN SAMPLER AUDIT
# ==================================================================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from torch.utils.data import WeightedRandomSampler


# --------------------------------------------------------------------------------------------------
# 1. Frozen Stage C geometry
# --------------------------------------------------------------------------------------------------

STAGE_C_MAX_STEPS = int(
    STAGE_C_PROTOCOL[
        "training_protocol"
    ][
        "max_steps"
    ]
)

STAGE_C_TRAIN_BATCH = int(
    STAGE_C_PROTOCOL[
        "training_protocol"
    ][
        "train_batch_size"
    ]
)

STAGE_C_GRAD_ACCUM = int(
    STAGE_C_PROTOCOL[
        "training_protocol"
    ][
        "gradient_accumulation_steps"
    ]
)

STAGE_C_EFFECTIVE_BATCH = (
    STAGE_C_TRAIN_BATCH
    *
    STAGE_C_GRAD_ACCUM
)

STAGE_C_TOTAL_DRAWS = (
    STAGE_C_MAX_STEPS
    *
    STAGE_C_EFFECTIVE_BATCH
)

STAGE_C_SAMPLER_SEED = 42


assert STAGE_C_MAX_STEPS == 500
assert STAGE_C_EFFECTIVE_BATCH == 8
assert STAGE_C_TOTAL_DRAWS == 4000


# --------------------------------------------------------------------------------------------------
# 2. Load Day 22 frozen sampler audit
# --------------------------------------------------------------------------------------------------

assert STAGE_C_SAMPLER_AUDIT_PATH.exists(), (
    f"DAY 23 STOP: missing Day 22 sampler audit:\n"
    f"{STAGE_C_SAMPLER_AUDIT_PATH}"
)


with open(
    STAGE_C_SAMPLER_AUDIT_PATH,
    "r",
    encoding="utf-8",
) as f:

    day22_sampler_audit = load_artifact_json(
        f
    )


assert (
    day22_sampler_audit[
        "all_checks_pass"
    ]
    is True
), (
    "DAY 23 STOP: Day 22 sampler audit was not formally PASS."
)


day22_capped_audit = (
    day22_sampler_audit[
        "capped_balanced"
    ]
)


# --------------------------------------------------------------------------------------------------
# 3. Recover exact training-label sequence
# --------------------------------------------------------------------------------------------------

stage_c_train_labels = (
    optimization_train[
        "sentiment"
    ]
    .astype(str)
    .reset_index(drop=True)
)


assert len(
    stage_c_train_labels
) == 10831


# --------------------------------------------------------------------------------------------------
# 4. Build sample-level capped-balanced weights
#
# Frozen class weights:
# negative = 2.973199...
# neutral  = 1.0
# positive = 3.0
# --------------------------------------------------------------------------------------------------

capped_balanced_sample_weights = torch.tensor(
    [
        float(
            capped_balanced_class_weights[
                label
            ]
        )
        for label
        in stage_c_train_labels
    ],
    dtype=torch.double,
)


assert len(
    capped_balanced_sample_weights
) == 10831


# --------------------------------------------------------------------------------------------------
# 5. Theoretical capped-balanced probabilities
#
# P(class c) proportional to n_c * w_c
# --------------------------------------------------------------------------------------------------

class_counts = {

    label:
        int(
            (
                stage_c_train_labels
                ==
                label
            ).sum()
        )

    for label
    in STAGE_C_LABELS
}


capped_class_mass = {

    label:
        (
            class_counts[
                label
            ]
            *
            capped_balanced_class_weights[
                label
            ]
        )

    for label
    in STAGE_C_LABELS
}


capped_total_mass = sum(
    capped_class_mass.values()
)


capped_expected_probs = {

    label:
        float(
            capped_class_mass[
                label
            ]
            /
            capped_total_mass
        )

    for label
    in STAGE_C_LABELS
}


# --------------------------------------------------------------------------------------------------
# 6. Reconstruct deterministic 4000-draw sampler
# --------------------------------------------------------------------------------------------------

capped_generator = torch.Generator()

capped_generator.manual_seed(
    STAGE_C_SAMPLER_SEED
)


capped_sampler = WeightedRandomSampler(

    weights=
        capped_balanced_sample_weights,

    num_samples=
        STAGE_C_TOTAL_DRAWS,

    replacement=
        True,

    generator=
        capped_generator,
)


capped_balanced_indices = list(
    capped_sampler
)


assert len(
    capped_balanced_indices
) == 4000


# --------------------------------------------------------------------------------------------------
# 7. Hash deterministic draw sequence
#
# This creates a compact identity for the exact 4000-index sequence.
# --------------------------------------------------------------------------------------------------

capped_draw_sequence_string = ",".join(
    str(
        index
    )
    for index
    in capped_balanced_indices
)


capped_draw_sequence_sha256 = hashlib.sha256(
    capped_draw_sequence_string.encode(
        "utf-8"
    )
).hexdigest()


# --------------------------------------------------------------------------------------------------
# 8. Recover realized class composition
# --------------------------------------------------------------------------------------------------

capped_sampled_labels = (
    stage_c_train_labels.iloc[
        capped_balanced_indices
    ]
    .reset_index(drop=True)
)


capped_realized_counts = (
    capped_sampled_labels
    .value_counts()
    .reindex(
        STAGE_C_LABELS,
        fill_value=0,
    )
    .astype(int)
)


capped_realized_probs = (
    capped_realized_counts
    /
    STAGE_C_TOTAL_DRAWS
)


capped_unique_indices = len(
    set(
        capped_balanced_indices
    )
)


capped_unique_fraction = (
    capped_unique_indices
    /
    STAGE_C_TOTAL_DRAWS
)


capped_max_probability_deviation = max(
    abs(
        float(
            capped_realized_probs[
                label
            ]
        )
        -
        float(
            capped_expected_probs[
                label
            ]
        )
    )
    for label
    in STAGE_C_LABELS
)


# --------------------------------------------------------------------------------------------------
# 9. Recover Day 22 frozen realized values
# --------------------------------------------------------------------------------------------------

day22_expected_probs = {

    label:
        float(
            day22_capped_audit[
                "expected_probabilities"
            ][
                label
            ]
        )

    for label
    in STAGE_C_LABELS
}


day22_realized_counts = {

    label:
        int(
            day22_capped_audit[
                "realized_counts"
            ][
                label
            ]
        )

    for label
    in STAGE_C_LABELS
}


day22_realized_probs = {

    label:
        float(
            day22_capped_audit[
                "realized_probabilities"
            ][
                label
            ]
        )

    for label
    in STAGE_C_LABELS
}


day22_unique_indices = int(
    day22_capped_audit[
        "unique_indices"
    ]
)


day22_unique_fraction = float(
    day22_capped_audit[
        "unique_fraction"
    ]
)


# --------------------------------------------------------------------------------------------------
# 10. Formal Day 22 vs Day 23 consistency checks
# --------------------------------------------------------------------------------------------------

expected_probability_match = all(

    abs(
        capped_expected_probs[
            label
        ]
        -
        day22_expected_probs[
            label
        ]
    )
    < 1e-12

    for label
    in STAGE_C_LABELS
)


realized_count_match = all(

    int(
        capped_realized_counts[
            label
        ]
    )
    ==
    day22_realized_counts[
        label
    ]

    for label
    in STAGE_C_LABELS
)


realized_probability_match = all(

    abs(
        float(
            capped_realized_probs[
                label
            ]
        )
        -
        day22_realized_probs[
            label
        ]
    )
    < 1e-12

    for label
    in STAGE_C_LABELS
)


unique_index_match = (
    capped_unique_indices
    ==
    day22_unique_indices
)


unique_fraction_match = (
    abs(
        capped_unique_fraction
        -
        day22_unique_fraction
    )
    < 1e-12
)


# --------------------------------------------------------------------------------------------------
# 11. Independent reproducibility check
#
# Generate the entire sampler sequence a second time from seed 42.
# It must be bit-for-bit identical.
# --------------------------------------------------------------------------------------------------

recheck_generator = torch.Generator()

recheck_generator.manual_seed(
    STAGE_C_SAMPLER_SEED
)


recheck_sampler = WeightedRandomSampler(

    weights=
        capped_balanced_sample_weights,

    num_samples=
        STAGE_C_TOTAL_DRAWS,

    replacement=
        True,

    generator=
        recheck_generator,
)


recheck_indices = list(
    recheck_sampler
)


deterministic_sequence_exact = (
    recheck_indices
    ==
    capped_balanced_indices
)


# --------------------------------------------------------------------------------------------------
# 12. Formal sampler checks
# --------------------------------------------------------------------------------------------------

cell3_checks = {

    "max_steps_500":
        STAGE_C_MAX_STEPS
        == 500,

    "effective_batch_8":
        STAGE_C_EFFECTIVE_BATCH
        == 8,

    "total_draws_4000":
        STAGE_C_TOTAL_DRAWS
        == 4000,

    "sampler_seed_42":
        STAGE_C_SAMPLER_SEED
        == 42,

    "replacement_true":
        True,

    "sample_weights_n_10831":
        len(
            capped_balanced_sample_weights
        )
        == 10831,

    "draw_sequence_n_4000":
        len(
            capped_balanced_indices
        )
        == 4000,

    "all_indices_valid":
        (
            min(
                capped_balanced_indices
            )
            >= 0
            and
            max(
                capped_balanced_indices
            )
            <
            10831
        ),

    "all_classes_present":
        all(
            capped_realized_counts[
                label
            ]
            > 0
            for label
            in STAGE_C_LABELS
        ),

    "deterministic_sequence_exact":
        deterministic_sequence_exact,

    "expected_probabilities_match_day22":
        expected_probability_match,

    "realized_counts_match_day22":
        realized_count_match,

    "realized_probabilities_match_day22":
        realized_probability_match,

    "unique_indices_match_day22":
        unique_index_match,

    "unique_fraction_match_day22":
        unique_fraction_match,

    "historical_validation_not_used":
        True,

    "future_not_used":
        True,
}


all_cell3_checks_pass = bool(
    all(
        cell3_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 13. Build audit table
# --------------------------------------------------------------------------------------------------

capped_sampler_audit_table = pd.DataFrame(
    {
        "label":
            STAGE_C_LABELS,

        "train_count":
            [
                class_counts[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],

        "class_weight":
            [
                capped_balanced_class_weights[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],

        "expected_probability":
            [
                capped_expected_probs[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],

        "realized_count_4000":
            [
                int(
                    capped_realized_counts[
                        label
                    ]
                )
                for label
                in STAGE_C_LABELS
            ],

        "realized_probability":
            [
                float(
                    capped_realized_probs[
                        label
                    ]
                )
                for label
                in STAGE_C_LABELS
            ],

        "day22_realized_count":
            [
                day22_realized_counts[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],

        "day22_realized_probability":
            [
                day22_realized_probs[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],
    }
)


# --------------------------------------------------------------------------------------------------
# 14. Save Day 23 sampler recovery artifact
# --------------------------------------------------------------------------------------------------

DAY23_STAGE_C_DIR = (
    DAY23_RESULTS_ROOT
    / "stage_c_sampling_completion"
)

DAY23_STAGE_C_ARTIFACT_DIR = (
    DAY23_STAGE_C_DIR
    / "artifacts"
)

DAY23_STAGE_C_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


DAY23_CAPPED_SAMPLER_RECOVERY_PATH = (
    DAY23_STAGE_C_ARTIFACT_DIR
    / "capped_balanced_sampler_recovery.json"
)


day23_capped_sampler_recovery = {

    "day":
        23,

    "condition":
        "capped_balanced",

    "recovered_from_day22":
        True,

    "sampler": {

        "type":
            "WeightedRandomSampler",

        "replacement":
            True,

        "seed":
            STAGE_C_SAMPLER_SEED,

        "total_draws":
            STAGE_C_TOTAL_DRAWS,

        "draw_sequence_sha256":
            capped_draw_sequence_sha256,
    },

    "class_weights":
        capped_balanced_class_weights,

    "expected_probabilities":
        capped_expected_probs,

    "realized_counts": {

        label:
            int(
                capped_realized_counts[
                    label
                ]
            )

        for label
        in STAGE_C_LABELS
    },

    "realized_probabilities": {

        label:
            float(
                capped_realized_probs[
                    label
                ]
            )

        for label
        in STAGE_C_LABELS
    },

    "unique_indices":
        capped_unique_indices,

    "unique_fraction":
        float(
            capped_unique_fraction
        ),

    "max_probability_deviation":
        float(
            capped_max_probability_deviation
        ),

    "day22_match": {

        "expected_probabilities":
            expected_probability_match,

        "realized_counts":
            realized_count_match,

        "realized_probabilities":
            realized_probability_match,

        "unique_indices":
            unique_index_match,

        "unique_fraction":
            unique_fraction_match,
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in cell3_checks.items()
    },

    "all_checks_pass":
        all_cell3_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    DAY23_CAPPED_SAMPLER_RECOVERY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        day23_capped_sampler_recovery,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 15. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — CAPPED-BALANCED DETERMINISTIC SAMPLER RECOVERY")
print("=" * 120)


print("\n[SAMPLER GEOMETRY]")
print(
    f"Max optimizer steps         : "
    f"{STAGE_C_MAX_STEPS}"
)
print(
    f"Effective batch             : "
    f"{STAGE_C_EFFECTIVE_BATCH}"
)
print(
    f"Total draws                 : "
    f"{STAGE_C_TOTAL_DRAWS:,}"
)
print(
    f"Sampler seed                : "
    f"{STAGE_C_SAMPLER_SEED}"
)
print(
    "Replacement                 : True"
)


print("\n[CAPPED-BALANCED COMPOSITION]")

display(
    capped_sampler_audit_table
)


print("\n[DRAW IDENTITY]")
print(
    f"Unique observations         : "
    f"{capped_unique_indices:,} / "
    f"{STAGE_C_TOTAL_DRAWS:,} "
    f"({capped_unique_fraction:.2%})"
)
print(
    f"Draw sequence SHA256        : "
    f"{capped_draw_sequence_sha256}"
)


print("\n[DAY 22 MATCH]")
print(
    f"Expected probabilities      : "
    f"{expected_probability_match}"
)
print(
    f"Realized counts             : "
    f"{realized_count_match}"
)
print(
    f"Realized probabilities      : "
    f"{realized_probability_match}"
)
print(
    f"Unique indices              : "
    f"{unique_index_match}"
)
print(
    f"Unique fraction             : "
    f"{unique_fraction_match}"
)


print("\n[FORMAL CHECKS]")

for name, passed in cell3_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACT]")
print(
    DAY23_CAPPED_SAMPLER_RECOVERY_PATH
)


print("\n" + "-" * 120)

if all_cell3_checks_pass:

    print(
        "DAY 23 CELL 3 STATUS: PASS — "
        "CAPPED-BALANCED SAMPLER RECOVERED EXACTLY; "
        "FRESH MODEL INITIALIZATION PERMITTED"
    )

else:

    print(
        "DAY 23 CELL 3 STATUS: FAIL — "
        "DO NOT INITIALIZE CAPPED-BALANCED MODEL"
    )

print("=" * 120)

DAY 23 — CAPPED-BALANCED DETERMINISTIC SAMPLER RECOVERY

[SAMPLER GEOMETRY]
Max optimizer steps         : 500
Effective batch             : 8
Total draws                 : 4,000
Sampler seed                : 42
Replacement                 : True

[CAPPED-BALANCED COMPOSITION]


,label,train_count,class_weight,expected_probability,realized_count_4000,realized_probability,day22_realized_count,day22_realized_probability
0,negative,2276,2.973199,0.35808,1425,0.35625,1425,0.35625
1,neutral,6767,1.000000,0.35808,1458,0.36450,1458,0.36450
2,positive,1788,3.000000,0.28384,1117,0.27925,1117,0.27925



[DRAW IDENTITY]
Unique observations         : 3,182 / 4,000 (79.55%)
Draw sequence SHA256        : 538419c7cfb8bdcba5a0110d10ac3965dbbc1f29fba6dbfc523faf3a21143928

[DAY 22 MATCH]
Expected probabilities      : True
Realized counts             : True
Realized probabilities      : True
Unique indices              : True
Unique fraction             : True

[FORMAL CHECKS]
PASS   — max_steps_500
PASS   — effective_batch_8
PASS   — total_draws_4000
PASS   — sampler_seed_42
PASS   — replacement_true
PASS   — sample_weights_n_10831
PASS   — draw_sequence_n_4000
PASS   — all_indices_valid
PASS   — all_classes_present
PASS   — deterministic_sequence_exact
PASS   — expected_probabilities_match_day22
PASS   — realized_counts_match_day22
PASS   — realized_probabilities_match_day22
PASS   — unique_indices_match_day22
PASS   — unique_fraction_match_day22
PASS   — historical_validation_not_used
PASS   — future_not_used

[ARTIFACT]
/content/gdrive_day23/MyDrive/FinGPT_Project/results/day23/stage_c_sa

In [9]:
# ==================================================================================================
# DAY 23 — CELL 3A
# BITSANDBYTES 0.50.2 INSTALLATION + CUDA 13 / T4 INTEGRITY CHECK
# ==================================================================================================

import sys
import subprocess


# --------------------------------------------------------------------------------------------------
# 1. Install the exact Day 22 validated version
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — BITSANDBYTES DEPENDENCY RECOVERY")
print("=" * 120)

print("\n[INSTALL]")
print("Installing bitsandbytes==0.50.2 ...")


subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--upgrade",
        "bitsandbytes==0.50.2",
    ]
)


# --------------------------------------------------------------------------------------------------
# 2. Import after installation
# --------------------------------------------------------------------------------------------------

import torch
import bitsandbytes as bnb


# --------------------------------------------------------------------------------------------------
# 3. Runtime identity
# --------------------------------------------------------------------------------------------------

python_version = (
    f"{sys.version_info.major}."
    f"{sys.version_info.minor}."
    f"{sys.version_info.micro}"
)

torch_version = torch.__version__

cuda_runtime = torch.version.cuda

gpu_name = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else None
)

bnb_version = getattr(
    bnb,
    "__version__",
    "UNKNOWN",
)


# --------------------------------------------------------------------------------------------------
# 4. Native backend check
# --------------------------------------------------------------------------------------------------

bnb_backend_name = None
bnb_native_loaded = False


try:

    bnb_backend_name = (
        type(
            bnb.cextension.lib
        ).__name__
    )

    bnb_native_loaded = (
        bnb.cextension.lib
        is not None
    )

except Exception as exc:

    bnb_backend_name = (
        f"ERROR: {type(exc).__name__}: {exc}"
    )

    bnb_native_loaded = False


# --------------------------------------------------------------------------------------------------
# 5. Formal integrity checks
# --------------------------------------------------------------------------------------------------

bnb_recovery_checks = {

    "cuda_available":
        torch.cuda.is_available(),

    "gpu_is_tesla_t4":
        (
            gpu_name is not None
            and
            "T4" in gpu_name
        ),

    "cuda_runtime_13":
        (
            cuda_runtime is not None
            and
            str(
                cuda_runtime
            ).startswith(
                "13."
            )
        ),

    "bitsandbytes_0_50_2":
        str(
            bnb_version
        )
        ==
        "0.50.2",

    "bnb_native_backend_loaded":
        bnb_native_loaded,

    "bnb_backend_is_cuda_native":
        (
            bnb_backend_name
            ==
            "CudaBNBNativeLibrary"
        ),
}


all_bnb_recovery_checks_pass = bool(
    all(
        bnb_recovery_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 6. Report
# --------------------------------------------------------------------------------------------------

print("\n[RUNTIME]")
print(
    f"Python                      : "
    f"{python_version}"
)
print(
    f"PyTorch                     : "
    f"{torch_version}"
)
print(
    f"CUDA runtime                : "
    f"{cuda_runtime}"
)
print(
    f"GPU                         : "
    f"{gpu_name}"
)


print("\n[BITSANDBYTES]")
print(
    f"Version                     : "
    f"{bnb_version}"
)
print(
    f"Backend                     : "
    f"{bnb_backend_name}"
)
print(
    f"Native backend loaded       : "
    f"{bnb_native_loaded}"
)


print("\n[INTEGRITY CHECKS]")

for name, passed in bnb_recovery_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n" + "-" * 120)

if all_bnb_recovery_checks_pass:

    print(
        "DAY 23 CELL 3A STATUS: PASS — "
        "BITSANDBYTES 0.50.2 RECOVERED; "
        "4-BIT MODEL INITIALIZATION PERMITTED"
    )

else:

    print(
        "DAY 23 CELL 3A STATUS: FAIL — "
        "DO NOT LOAD 4-BIT MODEL"
    )

print("=" * 120)

DAY 23 — BITSANDBYTES DEPENDENCY RECOVERY

[INSTALL]
Installing bitsandbytes==0.50.2 ...

[RUNTIME]
Python                      : 3.13.15
PyTorch                     : 2.11.0+cu130
CUDA runtime                : 13.0
GPU                         : Tesla T4

[BITSANDBYTES]
Version                     : 0.50.2
Backend                     : CudaBNBNativeLibrary
Native backend loaded       : True

[INTEGRITY CHECKS]
PASS   — cuda_available
PASS   — gpu_is_tesla_t4
PASS   — cuda_runtime_13
PASS   — bitsandbytes_0_50_2
PASS   — bnb_native_backend_loaded
PASS   — bnb_backend_is_cuda_native

------------------------------------------------------------------------------------------------------------------------
DAY 23 CELL 3A STATUS: PASS — BITSANDBYTES 0.50.2 RECOVERED; 4-BIT MODEL INITIALIZATION PERMITTED


In [15]:
# ==================================================================================================
# DAY 23 — RESTART RECOVERY CELL
# RECOVER CORE PROJECT / STAGE PATHS AFTER RUNTIME RESET
#
# PURPOSE:
#   Reconstruct path variables only.
#   NO training
#   NO inference
#   NO retuning
# ==================================================================================================

from pathlib import Path


# --------------------------------------------------------------------------------------------------
# 1. Project roots
# --------------------------------------------------------------------------------------------------

# PROJECT_ROOT is discovered in the project-path setup cell.

DAY23_RESULTS_ROOT = (
    PROJECT_ROOT
    / "results"
    / "day23"
)

assert PROJECT_ROOT.exists(), (
    f"PROJECT_ROOT missing:\n{PROJECT_ROOT}"
)

assert DAY23_RESULTS_ROOT.exists(), (
    f"DAY23_RESULTS_ROOT missing:\n{DAY23_RESULTS_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 2. Stage C paths
# --------------------------------------------------------------------------------------------------

STAGE_C_DIR = (
    DAY23_RESULTS_ROOT
    / "stage_c_sampling_completion"
)

STAGE_C_RUNS_DIR = (
    STAGE_C_DIR
    / "runs"
)


STAGE_C_CAPPED_DIR = (
    STAGE_C_RUNS_DIR
    / "capped_balanced"
)

STAGE_C_CAPPED_ARTIFACT_DIR = (
    STAGE_C_CAPPED_DIR
    / "artifacts"
)

STAGE_C_CAPPED_MODEL_DIR = (
    STAGE_C_CAPPED_DIR
    / "model"
)

STAGE_C_CAPPED_CHECKPOINT_DIR = (
    STAGE_C_CAPPED_MODEL_DIR
    / "checkpoints"
)


# Search for likely completion marker
possible_capped_completion_markers = [
    STAGE_C_CAPPED_ARTIFACT_DIR / "capped_balanced_training_complete.json",
    STAGE_C_CAPPED_ARTIFACT_DIR / "training_complete.json",
    STAGE_C_CAPPED_DIR / "capped_balanced_training_complete.json",
    STAGE_C_CAPPED_DIR / "training_complete.json",
]


STAGE_C_CAPPED_TRAINING_COMPLETE_PATH = None

for candidate in possible_capped_completion_markers:
    if candidate.exists():
        STAGE_C_CAPPED_TRAINING_COMPLETE_PATH = candidate
        break


# --------------------------------------------------------------------------------------------------
# 3. Stage D paths
# --------------------------------------------------------------------------------------------------

STAGE_D_DIR = (
    DAY23_RESULTS_ROOT
    / "stage_d_learning_rate_audit"
)

STAGE_D_RUNS_DIR = (
    STAGE_D_DIR
    / "runs"
)

STAGE_D_ARTIFACT_DIR = (
    STAGE_D_DIR
    / "artifacts"
)

STAGE_D_SELECTED_ADAPTER_DIR = (
    STAGE_D_DIR
    / "selected_adapter"
)


# --------------------------------------------------------------------------------------------------
# 4. Stage E paths
# --------------------------------------------------------------------------------------------------

STAGE_E_DIR = (
    DAY23_RESULTS_ROOT
    / "stage_e_lora_capacity_audit"
)

STAGE_E_RUNS_DIR = (
    STAGE_E_DIR
    / "runs"
)

STAGE_E_ARTIFACT_DIR = (
    STAGE_E_DIR
    / "artifacts"
)

STAGE_E_R16A64_DIR = (
    STAGE_E_RUNS_DIR
    / "r16_a64"
)

STAGE_E_R16A64_ARTIFACT_DIR = (
    STAGE_E_R16A64_DIR
    / "artifacts"
)

STAGE_E_R16A64_MODEL_DIR = (
    STAGE_E_R16A64_DIR
    / "model"
)

STAGE_E_R16A64_CHECKPOINT_DIR = (
    STAGE_E_R16A64_MODEL_DIR
    / "checkpoints"
)

STAGE_E_SELECTED_ADAPTER_DIR = (
    STAGE_E_DIR
    / "selected_adapter"
)


# --------------------------------------------------------------------------------------------------
# 5. Final evaluation paths
# --------------------------------------------------------------------------------------------------

FINAL_FREEZE_DIR = (
    DAY23_RESULTS_ROOT
    / "final_configuration_freeze"
)

FINAL_CONFIG_PATH = (
    FINAL_FREEZE_DIR
    / "final_historical_model_configuration.json"
)

FINAL_FIREWALL_PATH = (
    FINAL_FREEZE_DIR
    / "historical_validation_firewall.json"
)

FINAL_FREEZE_COMPLETE_PATH = (
    FINAL_FREEZE_DIR
    / "final_configuration_freeze_complete.json"
)


FINAL_HISTVAL_DIR = (
    DAY23_RESULTS_ROOT
    / "final_historical_validation"
)

FINAL_HISTVAL_PREDICTIONS_PATH = (
    FINAL_HISTVAL_DIR
    / "final_historical_validation_predictions.csv"
)

FINAL_HISTVAL_METRICS_PATH = (
    FINAL_HISTVAL_DIR
    / "final_historical_validation_metrics.json"
)

FINAL_HISTVAL_COMPLETE_PATH = (
    FINAL_HISTVAL_DIR
    / "one_shot_historical_validation_complete.json"
)


# --------------------------------------------------------------------------------------------------
# 6. Artifact existence audit
# --------------------------------------------------------------------------------------------------

recovery_status = {
    "project_root":
        PROJECT_ROOT.exists(),

    "day23_results_root":
        DAY23_RESULTS_ROOT.exists(),

    "stage_c_dir":
        STAGE_C_DIR.exists(),

    "stage_c_capped_dir":
        STAGE_C_CAPPED_DIR.exists(),

    "stage_c_capped_checkpoint_dir":
        STAGE_C_CAPPED_CHECKPOINT_DIR.exists(),

    "stage_d_dir":
        STAGE_D_DIR.exists(),

    "stage_d_selected_adapter":
        STAGE_D_SELECTED_ADAPTER_DIR.exists(),

    "stage_e_dir":
        STAGE_E_DIR.exists(),

    "stage_e_selected_adapter":
        STAGE_E_SELECTED_ADAPTER_DIR.exists(),

    "final_freeze_complete":
        FINAL_FREEZE_COMPLETE_PATH.exists(),

    "final_histval_predictions":
        FINAL_HISTVAL_PREDICTIONS_PATH.exists(),

    "final_histval_complete":
        FINAL_HISTVAL_COMPLETE_PATH.exists(),
}


# --------------------------------------------------------------------------------------------------
# 7. Existing capped-balanced checkpoints
# --------------------------------------------------------------------------------------------------

if STAGE_C_CAPPED_CHECKPOINT_DIR.exists():

    existing_capped_checkpoints = sorted(
        STAGE_C_CAPPED_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
    )

else:

    existing_capped_checkpoints = []


# --------------------------------------------------------------------------------------------------
# 8. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — RESTART RECOVERY AUDIT")
print("=" * 120)

print("\n[ROOTS]")
print(f"PROJECT_ROOT                : {PROJECT_ROOT}")
print(f"DAY23_RESULTS_ROOT          : {DAY23_RESULTS_ROOT}")

print("\n[RECOVERED ARTIFACT STATUS]")

for name, passed in recovery_status.items():
    print(
        f"{'PASS' if passed else 'MISS':<6} — {name}"
    )

print("\n[STAGE C CAPPED-BALANCED]")

print(
    f"Checkpoint directory        : "
    f"{STAGE_C_CAPPED_CHECKPOINT_DIR}"
)

print(
    f"Existing checkpoints        : "
    f"{[p.name for p in existing_capped_checkpoints]}"
)

print(
    f"Completion marker detected  : "
    f"{STAGE_C_CAPPED_TRAINING_COMPLETE_PATH}"
)

print("\n" + "-" * 120)
print(
    "DAY 23 RESTART RECOVERY STATUS: PASS — "
    "CORE PATH VARIABLES RECONSTRUCTED; "
    "EXISTING ARTIFACTS CAN NOW BE AUDITED SAFELY"
)
print("=" * 120)

DAY 23 — RESTART RECOVERY AUDIT

[ROOTS]
PROJECT_ROOT                : /content/gdrive_day23/MyDrive/FinGPT_Project
DAY23_RESULTS_ROOT          : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day23

[RECOVERED ARTIFACT STATUS]
PASS   — project_root
PASS   — day23_results_root
PASS   — stage_c_dir
MISS   — stage_c_capped_dir
MISS   — stage_c_capped_checkpoint_dir
PASS   — stage_d_dir
PASS   — stage_d_selected_adapter
PASS   — stage_e_dir
PASS   — stage_e_selected_adapter
PASS   — final_freeze_complete
PASS   — final_histval_predictions
PASS   — final_histval_complete

[STAGE C CAPPED-BALANCED]
Checkpoint directory        : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day23/stage_c_sampling_completion/runs/capped_balanced/model/checkpoints
Existing checkpoints        : []
Completion marker detected  : None

------------------------------------------------------------------------------------------------------------------------
DAY 23 RESTART RECOVERY STATUS: PASS — CORE PAT

In [14]:
# ==================================================================================================
# DAY 23 — CELL 4
# STAGE C CAPPED-BALANCED CANDIDATE
# BNB INTEGRITY + FRESH MODEL + WEIGHTED TRAINER INFRASTRUCTURE
# ==================================================================================================

import gc
import json
import random
from pathlib import Path

import numpy as np
import torch
import bitsandbytes as bnb

from torch.utils.data import (
    DataLoader,
    WeightedRandomSampler,
)

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


# --------------------------------------------------------------------------------------------------
# 1. Candidate identity + paths
# --------------------------------------------------------------------------------------------------

STAGE_C_CURRENT_CONDITION = "capped_balanced"

STAGE_C_CAPPED_DIR = (
    DAY23_STAGE_C_DIR
    / "capped_balanced"
)

STAGE_C_CAPPED_ARTIFACT_DIR = (
    STAGE_C_CAPPED_DIR
    / "artifacts"
)

STAGE_C_CAPPED_MODEL_DIR = (
    STAGE_C_CAPPED_DIR
    / "model"
)

STAGE_C_CAPPED_CHECKPOINT_DIR = (
    STAGE_C_CAPPED_MODEL_DIR
    / "checkpoints"
)


for directory in [
    STAGE_C_CAPPED_ARTIFACT_DIR,
    STAGE_C_CAPPED_MODEL_DIR,
    STAGE_C_CAPPED_CHECKPOINT_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# --------------------------------------------------------------------------------------------------
# 2. Candidate directory must be clean
# --------------------------------------------------------------------------------------------------

existing_capped_checkpoints = sorted(
    [
        path
        for path
        in STAGE_C_CAPPED_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if path.is_dir()
    ]
)


# --------------------------------------------------------------------------------------------------
# Restart-safe checkpoint handling
# --------------------------------------------------------------------------------------------------

existing_capped_checkpoints = sorted(
    STAGE_C_CAPPED_CHECKPOINT_DIR.glob(
        "checkpoint-*"
    )
)

capped_training_already_complete = (
    STAGE_C_CAPPED_TRAINING_COMPLETE_PATH.exists()
)


if capped_training_already_complete:

    print(
        "RESTART-SAFE RESUME: existing completed capped-balanced training detected."
    )

    print(
        "Training will NOT be rerun."
    )

    print(
        f"Completion marker           : "
        f"{STAGE_C_CAPPED_TRAINING_COMPLETE_PATH}"
    )

    print(
        f"Existing checkpoints        : "
        f"{[p.name for p in existing_capped_checkpoints]}"
    )

else:

    if len(
        existing_capped_checkpoints
    ) > 0:

        raise RuntimeError(
            "DAY 23 STOP: partial capped-balanced checkpoints exist "
            "without a formal completion marker. "
            "Do not silently resume or overwrite."
        )

# --------------------------------------------------------------------------------------------------
# 3. BitsAndBytes / CUDA integrity
#
# Day 22 required bitsandbytes 0.50.2 for CUDA 13 / Tesla T4.
# --------------------------------------------------------------------------------------------------

bnb_version = getattr(
    bnb,
    "__version__",
    "UNKNOWN",
)


bnb_backend_name = None
bnb_native_loaded = False


try:

    bnb_backend_name = (
        type(
            bnb.cextension.lib
        ).__name__
    )

    bnb_native_loaded = (
        bnb.cextension.lib
        is not None
    )

except Exception:

    bnb_backend_name = "UNAVAILABLE"
    bnb_native_loaded = False


bnb_checks = {

    "cuda_available":
        torch.cuda.is_available(),

    "cuda_runtime_13":
        (
            torch.version.cuda
            is not None
            and
            str(
                torch.version.cuda
            ).startswith(
                "13."
            )
        ),

    "bitsandbytes_0_50_2":
        str(
            bnb_version
        )
        ==
        "0.50.2",

    "bnb_native_backend_loaded":
        bnb_native_loaded,
}


assert all(
    bnb_checks.values()
), (
    "DAY 23 STOP: BitsAndBytes/CUDA integrity check failed. "
    "Do not initialize 4-bit model."
)


# --------------------------------------------------------------------------------------------------
# 4. Clean GPU before fresh model initialization
# --------------------------------------------------------------------------------------------------

gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()


gpu_before_model_gb = (
    torch.cuda.memory_allocated()
    /
    (1024 ** 3)
)


# --------------------------------------------------------------------------------------------------
# 5. Frozen seed
# --------------------------------------------------------------------------------------------------

STAGE_C_SEED = 42


random.seed(
    STAGE_C_SEED
)

np.random.seed(
    STAGE_C_SEED
)

torch.manual_seed(
    STAGE_C_SEED
)

torch.cuda.manual_seed_all(
    STAGE_C_SEED
)

set_seed(
    STAGE_C_SEED
)


# --------------------------------------------------------------------------------------------------
# 6. Fresh 4-bit base model
# --------------------------------------------------------------------------------------------------

stage_c_capped_bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_use_double_quant=True,

    bnb_4bit_compute_dtype=torch.float16,
)


stage_c_capped_base_model = (
    AutoModelForCausalLM.from_pretrained(

        BASE_MODEL_NAME,

        quantization_config=
            stage_c_capped_bnb_config,

        device_map=
            "auto",

        trust_remote_code=
            True,
    )
)


stage_c_capped_base_model.config.use_cache = False


stage_c_capped_base_model = (
    prepare_model_for_kbit_training(

        stage_c_capped_base_model,

        use_gradient_checkpointing=
            True,
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Fresh LoRA
#
# Exactly identical to Stage B and inverse-sqrt candidate.
# --------------------------------------------------------------------------------------------------

set_seed(
    STAGE_C_SEED
)


stage_c_capped_lora_config = LoraConfig(

    r=8,

    lora_alpha=32,

    lora_dropout=0.1,

    target_modules=[
        "q_proj",
        "v_proj",
    ],

    bias="none",

    task_type="CAUSAL_LM",

    inference_mode=False,
)


stage_c_capped_model = (
    get_peft_model(
        stage_c_capped_base_model,
        stage_c_capped_lora_config,
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Reuse exact frozen capped-balanced sample weights
# --------------------------------------------------------------------------------------------------

stage_c_capped_sampler_weights = (
    capped_balanced_sample_weights
    .clone()
    .detach()
    .double()
)


assert len(
    stage_c_capped_sampler_weights
) == 10831


# --------------------------------------------------------------------------------------------------
# 9. Weighted Trainer
#
# Exactly 4000 sampled observations:
#
#     500 optimizer steps
#       × 8 gradient accumulation
#       × batch size 1
#
# The sampler is reset from seed 42 when training dataloader is created.
# --------------------------------------------------------------------------------------------------

class StageCWeightedSamplerTrainer(
    Trainer
):

    def __init__(
        self,
        *args,
        sampler_weights=None,
        sampler_num_samples=None,
        sampler_seed=42,
        **kwargs,
    ):

        super().__init__(
            *args,
            **kwargs,
        )

        self.stage_c_sampler_weights = (
            sampler_weights
        )

        self.stage_c_sampler_num_samples = int(
            sampler_num_samples
        )

        self.stage_c_sampler_seed = int(
            sampler_seed
        )


    def get_train_dataloader(
        self,
    ):

        if self.train_dataset is None:

            raise ValueError(
                "Trainer requires train_dataset."
            )


        generator = torch.Generator()

        generator.manual_seed(
            self.stage_c_sampler_seed
        )


        sampler = WeightedRandomSampler(

            weights=
                self.stage_c_sampler_weights,

            num_samples=
                self.stage_c_sampler_num_samples,

            replacement=
                True,

            generator=
                generator,
        )


        dataloader = DataLoader(

            self.train_dataset,

            batch_size=
                self.args.per_device_train_batch_size,

            sampler=
                sampler,

            collate_fn=
                self.data_collator,

            drop_last=
                self.args.dataloader_drop_last,

            num_workers=
                self.args.dataloader_num_workers,

            pin_memory=
                self.args.dataloader_pin_memory,
        )


        return self.accelerator.prepare(
            dataloader
        )


# --------------------------------------------------------------------------------------------------
# 10. Final-only evaluation / checkpoint callback
# --------------------------------------------------------------------------------------------------

class FinalStepEvaluationSaveCallback(
    TrainerCallback
):

    def __init__(
        self,
        final_step=500,
    ):

        self.final_step = int(
            final_step
        )

        self.eval_triggered = False
        self.save_triggered = False


    def on_step_end(
        self,
        args,
        state,
        control,
        **kwargs,
    ):

        control.should_evaluate = False
        control.should_save = False


        if int(
            state.global_step
        ) == self.final_step:

            control.should_evaluate = True
            control.should_save = True

            self.eval_triggered = True
            self.save_triggered = True


        return control


stage_c_capped_callback = (
    FinalStepEvaluationSaveCallback(
        final_step=500
    )
)


# --------------------------------------------------------------------------------------------------
# 11. Frozen TrainingArguments
# --------------------------------------------------------------------------------------------------

stage_c_capped_training_args = TrainingArguments(

    output_dir=
        str(
            STAGE_C_CAPPED_CHECKPOINT_DIR
        ),

    max_steps=500,

    per_device_train_batch_size=1,

    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=1e-4,

    lr_scheduler_type="linear",

    warmup_steps=10,

    weight_decay=0.0,

    logging_strategy="steps",

    logging_steps=10,

    # Final callback controls evaluation/save.
    eval_strategy="no",

    save_strategy="no",

    load_best_model_at_end=False,

    fp16=True,

    bf16=False,

    gradient_checkpointing=True,

    seed=42,

    data_seed=42,

    report_to=[],

    remove_unused_columns=True,

    label_names=[
        "labels"
    ],

    disable_tqdm=False,
)


# --------------------------------------------------------------------------------------------------
# 12. Build capped-balanced Trainer
# --------------------------------------------------------------------------------------------------

stage_c_capped_trainer = (
    StageCWeightedSamplerTrainer(

        model=
            stage_c_capped_model,

        args=
            stage_c_capped_training_args,

        train_dataset=
            hf_opt_train_model,

        eval_dataset=
            hf_opt_dev_model,

        data_collator=
            data_collator,

        processing_class=
            tokenizer,

        callbacks=[
            stage_c_capped_callback
        ],

        sampler_weights=
            stage_c_capped_sampler_weights,

        sampler_num_samples=
            STAGE_C_TOTAL_DRAWS,

        sampler_seed=
            STAGE_C_SAMPLER_SEED,
    )
)


# --------------------------------------------------------------------------------------------------
# 13. Trainable parameter audit
# --------------------------------------------------------------------------------------------------

stage_c_capped_total_params = 0
stage_c_capped_trainable_params = 0
stage_c_capped_trainable_names = []


for name, param in stage_c_capped_model.named_parameters():

    stage_c_capped_total_params += (
        param.numel()
    )

    if param.requires_grad:

        stage_c_capped_trainable_params += (
            param.numel()
        )

        stage_c_capped_trainable_names.append(
            name
        )


stage_c_capped_unexpected_trainable = [

    name

    for name
    in stage_c_capped_trainable_names

    if "lora_" not in name
]


# --------------------------------------------------------------------------------------------------
# 14. Independent sampler dry-run
#
# Must exactly reproduce Cell 3's 4000-index sequence.
# --------------------------------------------------------------------------------------------------

dry_generator = torch.Generator()

dry_generator.manual_seed(
    STAGE_C_SAMPLER_SEED
)


dry_sampler = WeightedRandomSampler(

    weights=
        stage_c_capped_sampler_weights,

    num_samples=
        STAGE_C_TOTAL_DRAWS,

    replacement=
        True,

    generator=
        dry_generator,
)


dry_indices = list(
    dry_sampler
)


dry_sampler_exact_match = (
    dry_indices
    ==
    capped_balanced_indices
)


# --------------------------------------------------------------------------------------------------
# 15. Formal infrastructure checks
# --------------------------------------------------------------------------------------------------

stage_c_capped_checks = {

    "bnb_integrity_pass":
        all(
            bnb_checks.values()
        ),

    "checkpoint_dir_clean":
        len(
            existing_capped_checkpoints
        )
        == 0,

    "fresh_model_loaded":
        stage_c_capped_model
        is not None,

    "train_n_10831":
        len(
            hf_opt_train_model
        )
        == 10831,

    "dev_n_2708":
        len(
            hf_opt_dev_model
        )
        == 2708,

    "sampler_weights_n_10831":
        len(
            stage_c_capped_sampler_weights
        )
        == 10831,

    "sampler_draws_4000":
        STAGE_C_TOTAL_DRAWS
        == 4000,

    "sampler_seed_42":
        STAGE_C_SAMPLER_SEED
        == 42,

    "dry_sampler_exact_match_cell3":
        dry_sampler_exact_match,

    "max_steps_500":
        stage_c_capped_training_args.max_steps
        == 500,

    "gradient_accumulation_8":
        (
            stage_c_capped_training_args
            .gradient_accumulation_steps
            ==
            8
        ),

    "learning_rate_1e4":
        abs(
            stage_c_capped_training_args.learning_rate
            -
            1e-4
        )
        <
        1e-12,

    "warmup_10":
        stage_c_capped_training_args.warmup_steps
        == 10,

    "seed_42":
        stage_c_capped_training_args.seed
        == 42,

    "lora_r8":
        (
            stage_c_capped_model
            .peft_config[
                "default"
            ]
            .r
            ==
            8
        ),

    "lora_alpha32":
        (
            stage_c_capped_model
            .peft_config[
                "default"
            ]
            .lora_alpha
            ==
            32
        ),

    "trainable_params_3407872":
        stage_c_capped_trainable_params
        ==
        3_407_872,

    "only_lora_trainable":
        len(
            stage_c_capped_unexpected_trainable
        )
        == 0,

    "trainer_global_step_zero":
        int(
            stage_c_capped_trainer
            .state
            .global_step
        )
        == 0,

    "final_eval_not_triggered":
        stage_c_capped_callback.eval_triggered
        is False,

    "final_save_not_triggered":
        stage_c_capped_callback.save_triggered
        is False,

    "historical_validation_locked":
        day22_resume_manifest[
            "historical_validation_locked"
        ]
        is True,

    "future_locked":
        day22_resume_manifest[
            "future_f1_f6_locked"
        ]
        is True,
}


all_stage_c_capped_checks_pass = bool(
    all(
        stage_c_capped_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 16. Save infrastructure manifest
# --------------------------------------------------------------------------------------------------

STAGE_C_CAPPED_INFRA_PATH = (
    STAGE_C_CAPPED_ARTIFACT_DIR
    / "capped_balanced_training_infrastructure.json"
)


stage_c_capped_infra_payload = {

    "day":
        23,

    "stage":
        "C",

    "condition":
        "capped_balanced",

    "base_model":
        BASE_MODEL_NAME,

    "quantization": {

        "load_in_4bit":
            True,

        "quant_type":
            "nf4",

        "double_quant":
            True,

        "compute_dtype":
            "float16",

        "bitsandbytes_version":
            bnb_version,

        "bnb_backend":
            bnb_backend_name,
    },

    "lora": {

        "r":
            8,

        "alpha":
            32,

        "dropout":
            0.1,

        "target_modules":
            [
                "q_proj",
                "v_proj",
            ],
    },

    "training": {

        "max_steps":
            500,

        "learning_rate":
            1e-4,

        "scheduler":
            "linear",

        "warmup_steps":
            10,

        "effective_batch":
            8,

        "seed":
            42,
    },

    "sampling": {

        "type":
            "WeightedRandomSampler",

        "replacement":
            True,

        "seed":
            42,

        "num_samples":
            4000,

        "draw_sequence_sha256":
            capped_draw_sequence_sha256,

        "exact_match_cell3":
            dry_sampler_exact_match,
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in stage_c_capped_checks.items()
    },

    "all_checks_pass":
        all_stage_c_capped_checks_pass,
}


with open(
    STAGE_C_CAPPED_INFRA_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_c_capped_infra_payload,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 17. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE C CAPPED-BALANCED TRAINER INFRASTRUCTURE")
print("=" * 120)


print("\n[BITSANDBYTES]")
print(
    f"Version                     : "
    f"{bnb_version}"
)
print(
    f"Backend                     : "
    f"{bnb_backend_name}"
)

for name, passed in bnb_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[CANDIDATE]")
print(
    "Condition                   : capped_balanced"
)
print(
    "Initialization              : fresh base + fresh LoRA"
)
print(
    "Stage B adapter             : NOT LOADED"
)
print(
    "Inverse-sqrt adapter        : NOT LOADED"
)


print("\n[SAMPLER]")
print(
    "Type                        : WeightedRandomSampler"
)
print(
    "Replacement                 : True"
)
print(
    f"Seed                        : "
    f"{STAGE_C_SAMPLER_SEED}"
)
print(
    f"Total draws                 : "
    f"{STAGE_C_TOTAL_DRAWS:,}"
)
print(
    f"Exact Cell 3 match          : "
    f"{dry_sampler_exact_match}"
)
print(
    f"Draw SHA256                 : "
    f"{capped_draw_sequence_sha256}"
)


print("\n[TRAINING PROTOCOL]")
print(
    "Max steps                   : 500"
)
print(
    "Gradient accumulation       : 8"
)
print(
    "Effective batch             : 8"
)
print(
    "Learning rate               : 0.0001"
)
print(
    "Scheduler                   : linear"
)
print(
    "Warmup                      : 10"
)
print(
    "Formal Dev evaluation       : step 500 only"
)


print("\n[PARAMETERS]")
print(
    f"Trainable parameters        : "
    f"{stage_c_capped_trainable_params:,}"
)
print(
    f"Total parameters            : "
    f"{stage_c_capped_total_params:,}"
)
print(
    f"Unexpected trainable        : "
    f"{len(stage_c_capped_unexpected_trainable)}"
)


print("\n[PRE-TRAINING STATE]")
print(
    f"Global step                 : "
    f"{stage_c_capped_trainer.state.global_step}"
)
print(
    f"Final eval triggered        : "
    f"{stage_c_capped_callback.eval_triggered}"
)
print(
    f"Final save triggered        : "
    f"{stage_c_capped_callback.save_triggered}"
)


print("\n[GPU]")
print(
    f"Allocated before model      : "
    f"{gpu_before_model_gb:.2f} GB"
)
print(
    f"Allocated after model       : "
    f"{torch.cuda.memory_allocated() / (1024 ** 3):.2f} GB"
)


print("\n[INFRASTRUCTURE CHECKS]")

for name, passed in stage_c_capped_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACT]")
print(
    STAGE_C_CAPPED_INFRA_PATH
)


print("\n" + "-" * 120)

if all_stage_c_capped_checks_pass:

    print(
        "DAY 23 CELL 4 STATUS: PASS — "
        "CAPPED-BALANCED WEIGHTED TRAINER READY; "
        "TRAINING NOT YET STARTED"
    )

else:

    print(
        "DAY 23 CELL 4 STATUS: FAIL — "
        "DO NOT START CAPPED-BALANCED TRAINING"
    )

print("=" * 120)

NameError: name 'STAGE_C_CAPPED_TRAINING_COMPLETE_PATH' is not defined

In [11]:
# ==================================================================================================
# DAY 23 — CELL 5
# STAGE C CAPPED-BALANCED CANDIDATE
# CONTROLLED 500-STEP WEIGHTED-SAMPLING TRAINING
# ==================================================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch


# --------------------------------------------------------------------------------------------------
# 1. Formal artifact paths
# --------------------------------------------------------------------------------------------------

STAGE_C_CAPPED_TRAINING_LOG_PATH = (
    STAGE_C_CAPPED_ARTIFACT_DIR
    / "capped_balanced_training_log.csv"
)

STAGE_C_CAPPED_TRAINING_SUMMARY_PATH = (
    STAGE_C_CAPPED_ARTIFACT_DIR
    / "capped_balanced_training_summary.json"
)

STAGE_C_CAPPED_TRAINING_COMPLETE_PATH = (
    STAGE_C_CAPPED_ARTIFACT_DIR
    / "capped_balanced_training_complete.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Pre-training safety gate
# --------------------------------------------------------------------------------------------------

assert int(
    stage_c_capped_trainer.state.global_step
) == 0, (
    "DAY 23 STOP: capped-balanced trainer has already advanced beyond step 0."
)


assert (
    stage_c_capped_callback.eval_triggered
    is False
), (
    "DAY 23 STOP: final evaluation callback was already triggered."
)


assert (
    stage_c_capped_callback.save_triggered
    is False
), (
    "DAY 23 STOP: final save callback was already triggered."
)


existing_checkpoints_before_training = sorted(
    [
        p
        for p
        in STAGE_C_CAPPED_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


assert len(
    existing_checkpoints_before_training
) == 0, (
    "DAY 23 STOP: capped-balanced checkpoint directory is not clean."
)


assert not STAGE_C_CAPPED_TRAINING_COMPLETE_PATH.exists(), (
    "DAY 23 STOP: capped-balanced completion marker already exists."
)


# --------------------------------------------------------------------------------------------------
# 3. Reset deterministic seeds immediately before training
# --------------------------------------------------------------------------------------------------

random.seed(
    STAGE_C_SEED
)

np.random.seed(
    STAGE_C_SEED
)

torch.manual_seed(
    STAGE_C_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        STAGE_C_SEED
    )

    torch.cuda.reset_peak_memory_stats()


set_seed(
    STAGE_C_SEED
)


# --------------------------------------------------------------------------------------------------
# 4. Train
# --------------------------------------------------------------------------------------------------

training_start_time = time.time()


stage_c_capped_train_result = (
    stage_c_capped_trainer.train()
)


training_end_time = time.time()


training_runtime_seconds = (
    training_end_time
    -
    training_start_time
)

training_runtime_minutes = (
    training_runtime_seconds
    /
    60.0
)


# --------------------------------------------------------------------------------------------------
# 5. Collect Trainer log history
# --------------------------------------------------------------------------------------------------

stage_c_capped_log_history = (
    stage_c_capped_trainer.state.log_history
)


stage_c_capped_log_df = pd.DataFrame(
    stage_c_capped_log_history
)


stage_c_capped_log_df.to_csv(
    STAGE_C_CAPPED_TRAINING_LOG_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 6. Final state
# --------------------------------------------------------------------------------------------------

final_global_step = int(
    stage_c_capped_trainer.state.global_step
)


final_training_loss = float(
    stage_c_capped_train_result.training_loss
)


# --------------------------------------------------------------------------------------------------
# 7. Extract final Dev evaluation
# --------------------------------------------------------------------------------------------------

final_eval_rows = (
    stage_c_capped_log_df.loc[
        stage_c_capped_log_df[
            "eval_loss"
        ].notna()
    ]
    .copy()
)


if len(
    final_eval_rows
) > 0:

    final_eval_row = (
        final_eval_rows
        .sort_values(
            "step"
        )
        .iloc[-1]
    )

    final_eval_step = int(
        final_eval_row[
            "step"
        ]
    )

    final_eval_loss = float(
        final_eval_row[
            "eval_loss"
        ]
    )

    final_eval_runtime = float(
        final_eval_row.get(
            "eval_runtime",
            np.nan,
        )
    )

else:

    final_eval_step = None
    final_eval_loss = None
    final_eval_runtime = None


# --------------------------------------------------------------------------------------------------
# 8. Learning-rate trajectory
# --------------------------------------------------------------------------------------------------

lr_rows = (
    stage_c_capped_log_df.loc[
        stage_c_capped_log_df[
            "learning_rate"
        ].notna()
    ]
    .copy()
)


def get_logged_lr_at_step(
    step,
):

    rows = (
        lr_rows.loc[
            lr_rows[
                "step"
            ]
            ==
            step
        ]
    )

    if len(
        rows
    ) == 0:

        return None

    return float(
        rows.iloc[-1][
            "learning_rate"
        ]
    )


lr_step_100 = get_logged_lr_at_step(
    100
)

lr_step_200 = get_logged_lr_at_step(
    200
)

lr_step_300 = get_logged_lr_at_step(
    300
)

lr_step_400 = get_logged_lr_at_step(
    400
)

lr_step_500 = get_logged_lr_at_step(
    500
)


# --------------------------------------------------------------------------------------------------
# 9. Live scheduler end-state
# --------------------------------------------------------------------------------------------------

final_scheduler_epoch = None
final_scheduler_lr = None


if (
    stage_c_capped_trainer.lr_scheduler
    is not None
):

    final_scheduler_epoch = int(
        stage_c_capped_trainer.lr_scheduler.last_epoch
    )

    final_scheduler_lr = [
        float(
            value
        )
        for value
        in stage_c_capped_trainer.lr_scheduler.get_last_lr()
    ]


# --------------------------------------------------------------------------------------------------
# 10. Check final checkpoint
# --------------------------------------------------------------------------------------------------

capped_final_checkpoint_dir = (
    STAGE_C_CAPPED_CHECKPOINT_DIR
    / "checkpoint-500"
)


checkpoint_500_checks = {

    "checkpoint_exists":
        capped_final_checkpoint_dir.exists(),

    "adapter_config_exists":
        (
            capped_final_checkpoint_dir
            / "adapter_config.json"
        ).exists(),

    "adapter_weights_exist":
        (
            (
                capped_final_checkpoint_dir
                / "adapter_model.safetensors"
            ).exists()
            or
            (
                capped_final_checkpoint_dir
                / "adapter_model.bin"
            ).exists()
        ),

    "trainer_state_exists":
        (
            capped_final_checkpoint_dir
            / "trainer_state.json"
        ).exists(),

    "optimizer_state_exists":
        (
            capped_final_checkpoint_dir
            / "optimizer.pt"
        ).exists(),

    "scheduler_state_exists":
        (
            capped_final_checkpoint_dir
            / "scheduler.pt"
        ).exists(),
}


# --------------------------------------------------------------------------------------------------
# 11. Detect unintended intermediate checkpoints
# --------------------------------------------------------------------------------------------------

all_capped_checkpoints = sorted(
    [
        p.name
        for p
        in STAGE_C_CAPPED_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


# --------------------------------------------------------------------------------------------------
# 12. GPU state
# --------------------------------------------------------------------------------------------------

peak_gpu_gb = None


if torch.cuda.is_available():

    peak_gpu_gb = (
        torch.cuda.max_memory_allocated()
        /
        (1024 ** 3)
    )


# --------------------------------------------------------------------------------------------------
# 13. Formal post-training checks
# --------------------------------------------------------------------------------------------------

stage_c_capped_training_checks = {

    "training_completed_500_steps":
        final_global_step
        == 500,

    "training_loss_finite":
        np.isfinite(
            final_training_loss
        ),

    "training_log_nonempty":
        len(
            stage_c_capped_log_df
        )
        > 0,

    "final_eval_triggered":
        stage_c_capped_callback.eval_triggered
        is True,

    "final_save_triggered":
        stage_c_capped_callback.save_triggered
        is True,

    "exactly_one_eval":
        len(
            final_eval_rows
        )
        == 1,

    "eval_at_step_500":
        final_eval_step
        == 500,

    "eval_loss_finite":
        (
            final_eval_loss
            is not None
            and
            np.isfinite(
                final_eval_loss
            )
        ),

    "only_checkpoint_500":
        all_capped_checkpoints
        ==
        [
            "checkpoint-500"
        ],

    "checkpoint_500_complete":
        all(
            checkpoint_500_checks.values()
        ),

    "lr_step_100_observed":
        lr_step_100
        is not None,

    "lr_step_200_observed":
        lr_step_200
        is not None,

    "lr_step_300_observed":
        lr_step_300
        is not None,

    "lr_step_400_observed":
        lr_step_400
        is not None,

    "lr_step_500_observed":
        lr_step_500
        is not None,

    "lr_positive_before_end":
        (
            lr_step_100 is not None
            and
            lr_step_200 is not None
            and
            lr_step_300 is not None
            and
            lr_step_400 is not None
            and
            lr_step_100 > 0
            and
            lr_step_200 > 0
            and
            lr_step_300 > 0
            and
            lr_step_400 > 0
        ),

    "scheduler_reaches_end_at_500":
        final_scheduler_epoch
        == 500,

    "historical_validation_not_used":
        True,

    "future_not_used":
        True,
}


all_capped_training_checks_pass = bool(
    all(
        stage_c_capped_training_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 14. Save summary
# --------------------------------------------------------------------------------------------------

stage_c_capped_training_summary = {

    "day":
        23,

    "stage":
        "C",

    "condition":
        "capped_balanced",

    "sampling": {

        "sampler":
            "WeightedRandomSampler",

        "replacement":
            True,

        "seed":
            STAGE_C_SAMPLER_SEED,

        "draws":
            STAGE_C_TOTAL_DRAWS,

        "draw_sequence_sha256":
            capped_draw_sequence_sha256,

        "class_weights":
            capped_balanced_class_weights,
    },

    "training": {

        "final_global_step":
            final_global_step,

        "training_loss":
            final_training_loss,

        "runtime_seconds":
            float(
                training_runtime_seconds
            ),

        "runtime_minutes":
            float(
                training_runtime_minutes
            ),

        "peak_gpu_gb":
            (
                float(
                    peak_gpu_gb
                )
                if peak_gpu_gb
                is not None
                else None
            ),
    },

    "evaluation": {

        "step":
            final_eval_step,

        "eval_loss":
            final_eval_loss,

        "eval_runtime":
            final_eval_runtime,
    },

    "learning_rate": {

        "step_100":
            lr_step_100,

        "step_200":
            lr_step_200,

        "step_300":
            lr_step_300,

        "step_400":
            lr_step_400,

        "step_500":
            lr_step_500,

        "final_scheduler_epoch":
            final_scheduler_epoch,

        "final_scheduler_lr":
            final_scheduler_lr,
    },

    "checkpoint": {

        "directory":
            str(
                capped_final_checkpoint_dir
            ),

        "all_checkpoint_dirs":
            all_capped_checkpoints,

        "checks": {

            key:
                bool(
                    value
                )

            for key, value
            in checkpoint_500_checks.items()
        },
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in stage_c_capped_training_checks.items()
    },

    "all_checks_pass":
        all_capped_training_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_C_CAPPED_TRAINING_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_c_capped_training_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 15. Completion marker
# --------------------------------------------------------------------------------------------------

if all_capped_training_checks_pass:

    completion_payload = {

        "status":
            "PASS",

        "condition":
            "capped_balanced",

        "global_step":
            final_global_step,

        "checkpoint":
            str(
                capped_final_checkpoint_dir
            ),

        "created_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


    with open(
        STAGE_C_CAPPED_TRAINING_COMPLETE_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            completion_payload,
            f,
            indent=2,
            ensure_ascii=False,
        )


# --------------------------------------------------------------------------------------------------
# 16. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE C CAPPED-BALANCED 500-STEP TRAINING")
print("=" * 120)


print("\n[TRAINING]")
print(
    f"Final global step           : "
    f"{final_global_step}"
)
print(
    f"Final training loss         : "
    f"{final_training_loss:.6f}"
)
print(
    f"Runtime                     : "
    f"{training_runtime_minutes:.2f} minutes"
)

if peak_gpu_gb is not None:

    print(
        f"Peak GPU allocation         : "
        f"{peak_gpu_gb:.2f} GB"
    )


print("\n[FINAL DEV EVALUATION]")
print(
    f"Evaluation step             : "
    f"{final_eval_step}"
)
print(
    f"Evaluation loss             : "
    f"{final_eval_loss:.6f}"
)
print(
    f"Evaluation runtime          : "
    f"{final_eval_runtime:.2f} sec"
)


print("\n[LEARNING-RATE TRAJECTORY]")
print(
    f"Step 100                    : "
    f"{lr_step_100}"
)
print(
    f"Step 200                    : "
    f"{lr_step_200}"
)
print(
    f"Step 300                    : "
    f"{lr_step_300}"
)
print(
    f"Step 400                    : "
    f"{lr_step_400}"
)
print(
    f"Step 500                    : "
    f"{lr_step_500}"
)
print(
    f"Final scheduler epoch       : "
    f"{final_scheduler_epoch}"
)
print(
    f"Final scheduler LR          : "
    f"{final_scheduler_lr}"
)


print("\n[CALLBACK]")
print(
    f"Final eval triggered        : "
    f"{stage_c_capped_callback.eval_triggered}"
)
print(
    f"Final save triggered        : "
    f"{stage_c_capped_callback.save_triggered}"
)


print("\n[CHECKPOINT]")
print(
    f"Checkpoint directory        : "
    f"{capped_final_checkpoint_dir}"
)
print(
    f"All checkpoints             : "
    f"{all_capped_checkpoints}"
)

for name, passed in checkpoint_500_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[POST-TRAINING CHECKS]")

for name, passed in stage_c_capped_training_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Training log                : "
    f"{STAGE_C_CAPPED_TRAINING_LOG_PATH}"
)
print(
    f"Training summary            : "
    f"{STAGE_C_CAPPED_TRAINING_SUMMARY_PATH}"
)
print(
    f"Completion marker           : "
    f"{STAGE_C_CAPPED_TRAINING_COMPLETE_PATH}"
)


print("\n" + "-" * 120)

if all_capped_training_checks_pass:

    print(
        "DAY 23 CELL 5 STATUS: PASS — "
        "CAPPED-BALANCED 500-STEP TRAINING COMPLETE; "
        "CLASSIFICATION EVALUATION PENDING"
    )

else:

    print(
        "DAY 23 CELL 5 STATUS: FAIL — "
        "DO NOT EVALUATE OR PROCEED"
    )

print("=" * 120)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


DAY 23 — STAGE C CAPPED-BALANCED 500-STEP TRAINING

[TRAINING]
Final global step           : 500
Final training loss         : 0.464265
Runtime                     : 57.64 minutes
Peak GPU allocation         : 8.80 GB

[FINAL DEV EVALUATION]
Evaluation step             : 500
Evaluation loss             : 0.228869
Evaluation runtime          : 500.76 sec

[LEARNING-RATE TRAJECTORY]
Step 100                    : 8.183673469387756e-05
Step 200                    : 6.142857142857143e-05
Step 300                    : 4.102040816326531e-05
Step 400                    : 2.0612244897959186e-05
Step 500                    : 2.0408163265306124e-07
Final scheduler epoch       : 500
Final scheduler LR          : [0.0, 0.0]

[CALLBACK]
Final eval triggered        : True
Final save triggered        : True

[CHECKPOINT]
Checkpoint directory        : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day23/stage_c_sampling_completion/capped_balanced/model/checkpoints/checkpoint-500
All checkpoints  

In [ ]:
# ==================================================================================================
# DAY 23 — CELL 6
# STAGE C CAPPED-BALANCED
# OPTIMIZATION DEV CLASSIFICATION INFERENCE + INCUMBENT COMPARISON
# ==================================================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

from tqdm.auto import tqdm
from transformers.utils import logging as hf_logging


# --------------------------------------------------------------------------------------------------
# 1. Formal output paths
# --------------------------------------------------------------------------------------------------

STAGE_C_CAPPED_PREDICTIONS_PATH = (
    STAGE_C_CAPPED_ARTIFACT_DIR
    / "capped_balanced_optimization_dev_predictions.csv"
)

STAGE_C_CAPPED_METRICS_PATH = (
    STAGE_C_CAPPED_ARTIFACT_DIR
    / "capped_balanced_optimization_dev_metrics.json"
)

STAGE_C_CAPPED_CLASS_METRICS_PATH = (
    STAGE_C_CAPPED_ARTIFACT_DIR
    / "capped_balanced_optimization_dev_class_metrics.csv"
)

STAGE_C_CAPPED_CONFUSION_PATH = (
    STAGE_C_CAPPED_ARTIFACT_DIR
    / "capped_balanced_optimization_dev_confusion_matrix.csv"
)


# --------------------------------------------------------------------------------------------------
# 2. Safety gate
# --------------------------------------------------------------------------------------------------

assert (
    STAGE_C_CAPPED_TRAINING_COMPLETE_PATH.exists()
), (
    "DAY 23 STOP: capped-balanced training completion marker missing."
)


assert (
    int(stage_c_capped_trainer.state.global_step)
    == 500
), (
    "DAY 23 STOP: capped-balanced trainer is not at step 500."
)


assert len(
    optimization_dev
) == 2708


# --------------------------------------------------------------------------------------------------
# 3. Frozen inference protocol
# --------------------------------------------------------------------------------------------------

INFERENCE_BATCH_SIZE = 8

VALID_LABELS = [
    "negative",
    "neutral",
    "positive",
]


# Save original tokenizer state so training semantics remain recoverable.
original_padding_side = (
    tokenizer.padding_side
)

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token


stage_c_capped_model.eval()

stage_c_capped_model.config.use_cache = True


# --------------------------------------------------------------------------------------------------
# 4. Inference
#
# Exact Day 22 semantics:
# - identical prompt
# - left padding
# - greedy generation
# - max_new_tokens = 4
# - generated portion only
# - lowercase + strip
# - exact class validation
# --------------------------------------------------------------------------------------------------

all_predictions = []
all_raw_generations = []


inference_start = time.time()


# Suppress the harmless max_length/max_new_tokens logging noise observed on Day 22.
previous_hf_verbosity = (
    hf_logging.get_verbosity()
)

hf_logging.set_verbosity_error()


try:

    for batch_start in tqdm(
        range(
            0,
            len(optimization_dev),
            INFERENCE_BATCH_SIZE,
        ),
        desc="Capped-balanced Dev inference",
    ):

        batch_end = min(
            batch_start
            +
            INFERENCE_BATCH_SIZE,

            len(
                optimization_dev
            ),
        )


        batch_df = (
            optimization_dev.iloc[
                batch_start:batch_end
            ]
        )


        batch_prompts = [
            build_inference_prompt(
                text
            )
            for text
            in batch_df[
                "text"
            ].tolist()
        ]


        encoded = tokenizer(
            batch_prompts,

            return_tensors=
                "pt",

            padding=
                True,

            truncation=
                True,

            max_length=
                MAX_SEQ_LENGTH,

            add_special_tokens=
                False,
        )


        encoded = {
            key:
                value.to(
                    stage_c_capped_model.device
                )

            for key, value
            in encoded.items()
        }


        input_length = (
            encoded[
                "input_ids"
            ].shape[1]
        )


        with torch.inference_mode():

            generated = (
                stage_c_capped_model.generate(

                    **encoded,

                    do_sample=
                        False,

                    max_new_tokens=
                        4,

                    pad_token_id=
                        tokenizer.pad_token_id,

                    eos_token_id=
                        tokenizer.eos_token_id,

                    use_cache=
                        True,
                )
            )


        generated_only = (
            generated[
                :,
                input_length:
            ]
        )


        decoded = tokenizer.batch_decode(

            generated_only,

            skip_special_tokens=
                True,

            clean_up_tokenization_spaces=
                False,
        )


        for raw_text in decoded:

            normalized = (
                raw_text
                .strip()
                .lower()
            )


            all_raw_generations.append(
                raw_text
            )


            if normalized in VALID_LABELS:

                all_predictions.append(
                    normalized
                )

            else:

                all_predictions.append(
                    "__invalid__"
                )


finally:

    hf_logging.set_verbosity(
        previous_hf_verbosity
    )

    tokenizer.padding_side = (
        original_padding_side
    )


inference_runtime_seconds = (
    time.time()
    -
    inference_start
)


# --------------------------------------------------------------------------------------------------
# 5. Prediction population
# --------------------------------------------------------------------------------------------------

y_true = (
    optimization_dev[
        "sentiment"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .tolist()
)


y_pred = (
    all_predictions
)


assert len(
    y_true
) == 2708


assert len(
    y_pred
) == 2708


invalid_prediction_count = int(
    sum(
        prediction
        ==
        "__invalid__"

        for prediction
        in y_pred
    )
)


invalid_prediction_rate = (
    invalid_prediction_count
    /
    len(
        y_pred
    )
)


# --------------------------------------------------------------------------------------------------
# 6. Classification metrics
# --------------------------------------------------------------------------------------------------

accuracy = float(
    accuracy_score(
        y_true,
        y_pred,
    )
)


macro_f1 = float(
    f1_score(
        y_true,
        y_pred,
        labels=
            VALID_LABELS,
        average=
            "macro",
        zero_division=
            0,
    )
)


weighted_f1 = float(
    f1_score(
        y_true,
        y_pred,
        labels=
            VALID_LABELS,
        average=
            "weighted",
        zero_division=
            0,
    )
)


(
    class_precision,
    class_recall,
    class_f1,
    class_support,
) = precision_recall_fscore_support(

    y_true,
    y_pred,

    labels=
        VALID_LABELS,

    zero_division=
        0,
)


class_metrics_df = pd.DataFrame(
    {
        "label":
            VALID_LABELS,

        "precision":
            class_precision,

        "recall":
            class_recall,

        "f1":
            class_f1,

        "support":
            class_support.astype(
                int
            ),
    }
)


negative_recall = float(
    class_metrics_df.loc[
        class_metrics_df[
            "label"
        ]
        ==
        "negative",

        "recall",
    ].iloc[0]
)


neutral_recall = float(
    class_metrics_df.loc[
        class_metrics_df[
            "label"
        ]
        ==
        "neutral",

        "recall",
    ].iloc[0]
)


positive_recall = float(
    class_metrics_df.loc[
        class_metrics_df[
            "label"
        ]
        ==
        "positive",

        "recall",
    ].iloc[0]
)


minority_recall_floor = float(
    min(
        negative_recall,
        positive_recall,
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Confusion matrix
# --------------------------------------------------------------------------------------------------

cm = confusion_matrix(

    y_true,
    y_pred,

    labels=
        VALID_LABELS,
)


confusion_df = pd.DataFrame(

    cm,

    index=[
        f"true_{label}"
        for label
        in VALID_LABELS
    ],

    columns=[
        f"pred_{label}"
        for label
        in VALID_LABELS
    ],
)


# --------------------------------------------------------------------------------------------------
# 8. Prediction-level artifact
# --------------------------------------------------------------------------------------------------

prediction_df = pd.DataFrame(
    {
        "observation_id":
            optimization_dev[
                "observation_id"
            ].tolist(),

        "date":
            optimization_dev[
                "date"
            ].astype(str).tolist(),

        "true_label":
            y_true,

        "predicted_label":
            y_pred,

        "raw_generation":
            all_raw_generations,
    }
)


prediction_df.to_csv(
    STAGE_C_CAPPED_PREDICTIONS_PATH,
    index=False,
)


class_metrics_df.to_csv(
    STAGE_C_CAPPED_CLASS_METRICS_PATH,
    index=False,
)


confusion_df.to_csv(
    STAGE_C_CAPPED_CONFUSION_PATH
)


# --------------------------------------------------------------------------------------------------
# 9. Compare against current incumbent:
#    Stage B-500 natural sampling
# --------------------------------------------------------------------------------------------------

accuracy_delta_vs_incumbent = (
    accuracy
    -
    INCUMBENT_ACCURACY
)


macro_f1_delta_vs_incumbent = (
    macro_f1
    -
    INCUMBENT_MACRO_F1
)


negative_recall_delta_vs_incumbent = (
    negative_recall
    -
    INCUMBENT_NEG_RECALL
)


neutral_recall_delta_vs_incumbent = (
    neutral_recall
    -
    INCUMBENT_NEU_RECALL
)


positive_recall_delta_vs_incumbent = (
    positive_recall
    -
    INCUMBENT_POS_RECALL
)


incumbent_minority_recall_floor = min(
    INCUMBENT_NEG_RECALL,
    INCUMBENT_POS_RECALL,
)


minority_floor_delta_vs_incumbent = (
    minority_recall_floor
    -
    incumbent_minority_recall_floor
)


beats_incumbent_accuracy = bool(
    accuracy
    >
    INCUMBENT_ACCURACY
)


# --------------------------------------------------------------------------------------------------
# 10. Formal evaluation checks
# --------------------------------------------------------------------------------------------------

cell6_checks = {

    "dev_population_2708":
        len(
            y_true
        )
        == 2708,

    "prediction_count_2708":
        len(
            y_pred
        )
        == 2708,

    "zero_invalid_predictions":
        invalid_prediction_count
        == 0,

    "accuracy_finite":
        np.isfinite(
            accuracy
        ),

    "macro_f1_finite":
        np.isfinite(
            macro_f1
        ),

    "weighted_f1_finite":
        np.isfinite(
            weighted_f1
        ),

    "negative_recall_finite":
        np.isfinite(
            negative_recall
        ),

    "neutral_recall_finite":
        np.isfinite(
            neutral_recall
        ),

    "positive_recall_finite":
        np.isfinite(
            positive_recall
        ),

    "confusion_total_2708":
        int(
            cm.sum()
        )
        == 2708,

    "prediction_artifact_saved":
        STAGE_C_CAPPED_PREDICTIONS_PATH.exists(),

    "class_metrics_saved":
        STAGE_C_CAPPED_CLASS_METRICS_PATH.exists(),

    "confusion_matrix_saved":
        STAGE_C_CAPPED_CONFUSION_PATH.exists(),

    "historical_validation_not_used":
        True,

    "future_not_used":
        True,
}


all_cell6_checks_pass = bool(
    all(
        cell6_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 11. Save formal metrics
# --------------------------------------------------------------------------------------------------

stage_c_capped_metrics = {

    "day":
        23,

    "stage":
        "C",

    "condition":
        "capped_balanced",

    "evaluation_population":
        "optimization_dev",

    "n":
        len(
            y_true
        ),

    "metrics": {

        "accuracy":
            accuracy,

        "macro_f1":
            macro_f1,

        "weighted_f1":
            weighted_f1,

        "negative_recall":
            negative_recall,

        "neutral_recall":
            neutral_recall,

        "positive_recall":
            positive_recall,

        "minority_recall_floor":
            minority_recall_floor,

        "invalid_predictions":
            invalid_prediction_count,

        "invalid_prediction_rate":
            invalid_prediction_rate,
    },

    "comparison_vs_incumbent": {

        "incumbent":
            INCUMBENT_MODEL_NAME,

        "incumbent_accuracy":
            INCUMBENT_ACCURACY,

        "accuracy_delta":
            float(
                accuracy_delta_vs_incumbent
            ),

        "macro_f1_delta":
            float(
                macro_f1_delta_vs_incumbent
            ),

        "negative_recall_delta":
            float(
                negative_recall_delta_vs_incumbent
            ),

        "neutral_recall_delta":
            float(
                neutral_recall_delta_vs_incumbent
            ),

        "positive_recall_delta":
            float(
                positive_recall_delta_vs_incumbent
            ),

        "minority_recall_floor_delta":
            float(
                minority_floor_delta_vs_incumbent
            ),

        "beats_incumbent_accuracy":
            beats_incumbent_accuracy,
    },

    "inference": {

        "batch_size":
            INFERENCE_BATCH_SIZE,

        "do_sample":
            False,

        "max_new_tokens":
            4,

        "padding_side":
            "left",

        "runtime_seconds":
            float(
                inference_runtime_seconds
            ),
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in cell6_checks.items()
    },

    "all_checks_pass":
        all_cell6_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_C_CAPPED_METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_c_capped_metrics,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 12. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE C CAPPED-BALANCED OPTIMIZATION DEV CLASSIFICATION")
print("=" * 120)


print("\n[EVALUATION POPULATION]")
print(
    f"Split                       : Optimization Dev"
)
print(
    f"N                           : {len(y_true):,}"
)
print(
    f"Inference runtime           : "
    f"{inference_runtime_seconds / 60:.2f} minutes"
)
print(
    f"Invalid predictions         : "
    f"{invalid_prediction_count}"
)


print("\n[OVERALL METRICS]")
print(
    f"Accuracy                    : "
    f"{accuracy:.6f}"
)
print(
    f"Macro-F1                    : "
    f"{macro_f1:.6f}"
)
print(
    f"Weighted-F1                 : "
    f"{weighted_f1:.6f}"
)


print("\n[CLASS RECALL]")
print(
    f"Negative recall             : "
    f"{negative_recall:.6f}"
)
print(
    f"Neutral recall              : "
    f"{neutral_recall:.6f}"
)
print(
    f"Positive recall             : "
    f"{positive_recall:.6f}"
)
print(
    f"Minority recall floor       : "
    f"{minority_recall_floor:.6f}"
)


print("\n[CLASS METRICS]")
display(
    class_metrics_df
)


print("\n[CONFUSION MATRIX]")
display(
    confusion_df
)


print("\n[COMPARISON VS CURRENT INCUMBENT]")
print(
    f"Incumbent                   : "
    f"{INCUMBENT_MODEL_NAME}"
)
print(
    f"Incumbent Accuracy          : "
    f"{INCUMBENT_ACCURACY:.6f}"
)
print(
    f"Capped-balanced Accuracy    : "
    f"{accuracy:.6f}"
)
print(
    f"Accuracy delta              : "
    f"{accuracy_delta_vs_incumbent:+.6f} "
    f"({accuracy_delta_vs_incumbent * 100:+.2f} pp)"
)
print(
    f"Macro-F1 delta              : "
    f"{macro_f1_delta_vs_incumbent:+.6f}"
)
print(
    f"Negative recall delta       : "
    f"{negative_recall_delta_vs_incumbent:+.6f}"
)
print(
    f"Neutral recall delta        : "
    f"{neutral_recall_delta_vs_incumbent:+.6f}"
)
print(
    f"Positive recall delta       : "
    f"{positive_recall_delta_vs_incumbent:+.6f}"
)
print(
    f"Minority floor delta        : "
    f"{minority_floor_delta_vs_incumbent:+.6f}"
)
print(
    f"Beats incumbent Accuracy    : "
    f"{beats_incumbent_accuracy}"
)


print("\n[FORMAL CHECKS]")

for name, passed in cell6_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Predictions                 : "
    f"{STAGE_C_CAPPED_PREDICTIONS_PATH}"
)
print(
    f"Metrics                     : "
    f"{STAGE_C_CAPPED_METRICS_PATH}"
)
print(
    f"Class metrics               : "
    f"{STAGE_C_CAPPED_CLASS_METRICS_PATH}"
)
print(
    f"Confusion matrix            : "
    f"{STAGE_C_CAPPED_CONFUSION_PATH}"
)


print("\n" + "-" * 120)

if all_cell6_checks_pass:

    print(
        "DAY 23 CELL 6 STATUS: PASS — "
        "CAPPED-BALANCED CLASSIFICATION EVALUATION COMPLETE; "
        "STAGE C THREE-CONDITION SELECTION READY"
    )

else:

    print(
        "DAY 23 CELL 6 STATUS: FAIL — "
        "DO NOT PERFORM STAGE C SELECTION"
    )

print("=" * 120)

In [11]:
# ==================================================================================================
# DAY 23 — CELL 7
# STAGE C THREE-CONDITION COMPARISON + FORMAL WINNER FREEZE
#
# SCHEMA-SAFE VERSION:
# - overall metrics: read ONLY from approved metric containers
# - class recalls: read from class-metrics CSV
# - comparison/delta fields are NEVER used as raw performance metrics
# ==================================================================================================

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Source artifacts
# --------------------------------------------------------------------------------------------------

# Natural / Stage B-500
NATURAL_METRICS_PATH = (
    DAY22_RESULTS_ROOT
    / "runs"
    / "stage_b_horizon_audit"
    / "artifacts"
    / "stage_b_500_optimization_dev_metrics.json"
)

NATURAL_CLASS_METRICS_PATH = (
    DAY22_RESULTS_ROOT
    / "runs"
    / "stage_b_horizon_audit"
    / "artifacts"
    / "stage_b_500_optimization_dev_class_metrics.csv"
)


# Inverse-sqrt
INVERSE_METRICS_PATH = (
    STAGE_C_DIR
    / "inverse_sqrt"
    / "artifacts"
    / "inverse_sqrt_optimization_dev_metrics.json"
)

INVERSE_CLASS_METRICS_PATH = (
    STAGE_C_DIR
    / "inverse_sqrt"
    / "artifacts"
    / "inverse_sqrt_optimization_dev_class_metrics.csv"
)


# Capped-balanced
CAPPED_METRICS_PATH = (
    STAGE_C_CAPPED_METRICS_PATH
)

CAPPED_CLASS_METRICS_PATH = (
    STAGE_C_CAPPED_CLASS_METRICS_PATH
)


required_stage_c_files = {
    "natural_metrics": NATURAL_METRICS_PATH,
    "natural_class_metrics": NATURAL_CLASS_METRICS_PATH,
    "inverse_metrics": INVERSE_METRICS_PATH,
    "inverse_class_metrics": INVERSE_CLASS_METRICS_PATH,
    "capped_metrics": CAPPED_METRICS_PATH,
    "capped_class_metrics": CAPPED_CLASS_METRICS_PATH,
}


for name, path in required_stage_c_files.items():

    assert path.exists(), (
        f"DAY 23 STOP: missing {name}:\n{path}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Load JSONs
# --------------------------------------------------------------------------------------------------

with open(
    NATURAL_METRICS_PATH,
    "r",
    encoding="utf-8",
) as f:
    natural_payload = load_artifact_json(f)


with open(
    INVERSE_METRICS_PATH,
    "r",
    encoding="utf-8",
) as f:
    inverse_payload = load_artifact_json(f)


with open(
    CAPPED_METRICS_PATH,
    "r",
    encoding="utf-8",
) as f:
    capped_payload = load_artifact_json(f)


# --------------------------------------------------------------------------------------------------
# 3. Schema-safe overall metric extractor
#
# IMPORTANT:
# We do NOT recursively search through comparison/delta sections.
#
# Approved locations:
#   payload["metrics"][metric]
#   payload["overall_metrics"][metric]
#   payload["classification_metrics"][metric]
#   payload[metric]
#
# If none exist, inspect only immediate non-comparison dictionaries.
# --------------------------------------------------------------------------------------------------

APPROVED_METRIC_CONTAINERS = [
    "metrics",
    "overall_metrics",
    "classification_metrics",
    "evaluation_metrics",
]


FORBIDDEN_CONTAINER_TOKENS = [
    "comparison",
    "delta",
    "incumbent",
    "vs_",
    "checks",
    "inference",
    "artifact",
]


def extract_overall_metric(
    payload,
    metric_name,
    condition_name,
):

    # --------------------------------------------------------------
    # A. Preferred formal containers
    # --------------------------------------------------------------

    for container_name in APPROVED_METRIC_CONTAINERS:

        container = payload.get(
            container_name
        )

        if (
            isinstance(container, dict)
            and
            metric_name in container
        ):

            value = container[
                metric_name
            ]

            if isinstance(
                value,
                (
                    int,
                    float,
                    np.integer,
                    np.floating,
                ),
            ):

                return float(value)


    # --------------------------------------------------------------
    # B. Direct top-level metric
    # --------------------------------------------------------------

    if metric_name in payload:

        value = payload[
            metric_name
        ]

        if isinstance(
            value,
            (
                int,
                float,
                np.integer,
                np.floating,
            ),
        ):

            return float(value)


    # --------------------------------------------------------------
    # C. Conservative inspection of immediate child dictionaries
    #    Explicitly exclude comparison/delta structures.
    # --------------------------------------------------------------

    candidates = []


    for container_name, container in payload.items():

        if not isinstance(
            container,
            dict,
        ):
            continue


        container_lower = (
            str(container_name)
            .strip()
            .lower()
        )


        if any(
            forbidden_token in container_lower
            for forbidden_token
            in FORBIDDEN_CONTAINER_TOKENS
        ):
            continue


        if metric_name not in container:
            continue


        value = container[
            metric_name
        ]


        if isinstance(
            value,
            (
                int,
                float,
                np.integer,
                np.floating,
            ),
        ):

            candidates.append(
                (
                    container_name,
                    float(value),
                )
            )


    if len(
        candidates
    ) == 1:

        return candidates[0][1]


    if len(
        candidates
    ) == 0:

        raise KeyError(
            f"{condition_name}: overall metric '{metric_name}' "
            f"not found in an approved metrics location.\n"
            f"Top-level keys: {list(payload.keys())}"
        )


    raise ValueError(
        f"{condition_name}: ambiguous overall metric "
        f"'{metric_name}': {candidates}"
    )


# --------------------------------------------------------------------------------------------------
# 4. Load and normalize class-metric CSVs
# --------------------------------------------------------------------------------------------------

VALID_LABELS = [
    "negative",
    "neutral",
    "positive",
]


def load_class_metrics(
    path,
    condition_name,
):

    df = pd.read_csv(
        path
    )


    required_columns = {
        "label",
        "precision",
        "recall",
        "f1",
        "support",
    }


    assert required_columns.issubset(
        df.columns
    ), (
        f"{condition_name}: invalid class-metrics schema.\n"
        f"Columns: {df.columns.tolist()}"
    )


    df[
        "label"
    ] = (
        df[
            "label"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
    )


    assert set(
        df[
            "label"
        ]
    ) == set(
        VALID_LABELS
    ), (
        f"{condition_name}: unexpected labels."
    )


    return df


natural_class_df = load_class_metrics(
    NATURAL_CLASS_METRICS_PATH,
    "natural",
)

inverse_class_df = load_class_metrics(
    INVERSE_CLASS_METRICS_PATH,
    "inverse_sqrt",
)

capped_class_df = load_class_metrics(
    CAPPED_CLASS_METRICS_PATH,
    "capped_balanced",
)


def get_recall(
    df,
    label,
):

    row = df.loc[
        df[
            "label"
        ]
        ==
        label
    ]


    assert len(
        row
    ) == 1


    return float(
        row.iloc[0][
            "recall"
        ]
    )


# --------------------------------------------------------------------------------------------------
# 5. Define conditions
# --------------------------------------------------------------------------------------------------

conditions = {

    "natural": {
        "sampling": "Natural",
        "payload": natural_payload,
        "class_df": natural_class_df,
    },

    "inverse_sqrt": {
        "sampling": "Inverse-sqrt",
        "payload": inverse_payload,
        "class_df": inverse_class_df,
    },

    "capped_balanced": {
        "sampling": "Capped-balanced",
        "payload": capped_payload,
        "class_df": capped_class_df,
    },
}


# --------------------------------------------------------------------------------------------------
# 6. Build unified comparison table
# --------------------------------------------------------------------------------------------------

stage_c_rows = []


for condition, info in conditions.items():

    payload = info[
        "payload"
    ]

    class_df = info[
        "class_df"
    ]


    accuracy_value = extract_overall_metric(
        payload,
        "accuracy",
        condition,
    )


    macro_f1_value = extract_overall_metric(
        payload,
        "macro_f1",
        condition,
    )


    weighted_f1_value = extract_overall_metric(
        payload,
        "weighted_f1",
        condition,
    )


    negative_recall_value = get_recall(
        class_df,
        "negative",
    )


    neutral_recall_value = get_recall(
        class_df,
        "neutral",
    )


    positive_recall_value = get_recall(
        class_df,
        "positive",
    )


    minority_floor_value = min(
        negative_recall_value,
        positive_recall_value,
    )


    stage_c_rows.append(
        {
            "condition":
                condition,

            "sampling":
                info[
                    "sampling"
                ],

            "accuracy":
                accuracy_value,

            "macro_f1":
                macro_f1_value,

            "weighted_f1":
                weighted_f1_value,

            "negative_recall":
                negative_recall_value,

            "neutral_recall":
                neutral_recall_value,

            "positive_recall":
                positive_recall_value,

            "minority_recall_floor":
                minority_floor_value,
        }
    )


stage_c_comparison_df = pd.DataFrame(
    stage_c_rows
)


# --------------------------------------------------------------------------------------------------
# 7. Sanity-print extracted raw metrics BEFORE selection
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE C METRIC EXTRACTION AUDIT")
print("=" * 120)

display(
    stage_c_comparison_df
)


# --------------------------------------------------------------------------------------------------
# 8. Hard numerical identity checks against already observed results
#
# These prevent accidental schema extraction errors.
# --------------------------------------------------------------------------------------------------

expected_metrics = {

    "natural": {
        "accuracy": 0.840842,
        "macro_f1": 0.808201,
    },

    "inverse_sqrt": {
        "accuracy": 0.822009,
        "macro_f1": 0.796423,
    },

    "capped_balanced": {
        "accuracy": 0.810561,
        "macro_f1": 0.789809,
    },
}


identity_checks = {}


for condition, expected in expected_metrics.items():

    row = (
        stage_c_comparison_df.loc[
            stage_c_comparison_df[
                "condition"
            ]
            ==
            condition
        ]
        .iloc[0]
    )


    identity_checks[
        f"{condition}_accuracy_match"
    ] = (
        abs(
            float(
                row[
                    "accuracy"
                ]
            )
            -
            expected[
                "accuracy"
            ]
        )
        <
        1e-6
    )


    identity_checks[
        f"{condition}_macro_f1_match"
    ] = (
        abs(
            float(
                row[
                    "macro_f1"
                ]
            )
            -
            expected[
                "macro_f1"
            ]
        )
        <
        1e-6
    )


assert all(
    identity_checks.values()
), (
    "DAY 23 STOP: extracted Stage C metrics do not match "
    "the already observed formal results."
)


# --------------------------------------------------------------------------------------------------
# 9. Ranking
#
# Primary selection criterion = Accuracy
# Secondary = Macro-F1
# Tertiary diagnostic = minority recall floor
# --------------------------------------------------------------------------------------------------

stage_c_comparison_df[
    "accuracy_rank"
] = (
    stage_c_comparison_df[
        "accuracy"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)


stage_c_comparison_df[
    "macro_f1_rank"
] = (
    stage_c_comparison_df[
        "macro_f1"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)


stage_c_comparison_df[
    "minority_floor_rank"
] = (
    stage_c_comparison_df[
        "minority_recall_floor"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)


stage_c_comparison_df = (
    stage_c_comparison_df
    .sort_values(
        by=[
            "accuracy",
            "macro_f1",
            "minority_recall_floor",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 10. Explicit rows
# --------------------------------------------------------------------------------------------------

natural_row = (
    stage_c_comparison_df.loc[
        stage_c_comparison_df[
            "condition"
        ]
        ==
        "natural"
    ]
    .iloc[0]
)


inverse_row = (
    stage_c_comparison_df.loc[
        stage_c_comparison_df[
            "condition"
        ]
        ==
        "inverse_sqrt"
    ]
    .iloc[0]
)


capped_row = (
    stage_c_comparison_df.loc[
        stage_c_comparison_df[
            "condition"
        ]
        ==
        "capped_balanced"
    ]
    .iloc[0]
)


# --------------------------------------------------------------------------------------------------
# 11. Delta vs natural
# --------------------------------------------------------------------------------------------------

for metric in [
    "accuracy",
    "macro_f1",
    "negative_recall",
    "neutral_recall",
    "positive_recall",
    "minority_recall_floor",
]:

    stage_c_comparison_df[
        f"{metric}_delta_vs_natural"
    ] = (
        stage_c_comparison_df[
            metric
        ]
        -
        float(
            natural_row[
                metric
            ]
        )
    )


# --------------------------------------------------------------------------------------------------
# 12. Freeze winner
# --------------------------------------------------------------------------------------------------

winner_row = (
    stage_c_comparison_df.iloc[
        0
    ]
)


STAGE_C_WINNER = str(
    winner_row[
        "condition"
    ]
)

STAGE_C_WINNER_SAMPLING = str(
    winner_row[
        "sampling"
    ]
)

STAGE_C_WINNER_ACCURACY = float(
    winner_row[
        "accuracy"
    ]
)

STAGE_C_WINNER_MACRO_F1 = float(
    winner_row[
        "macro_f1"
    ]
)

STAGE_C_WINNER_NEG_RECALL = float(
    winner_row[
        "negative_recall"
    ]
)

STAGE_C_WINNER_NEU_RECALL = float(
    winner_row[
        "neutral_recall"
    ]
)

STAGE_C_WINNER_POS_RECALL = float(
    winner_row[
        "positive_recall"
    ]
)

STAGE_C_WINNER_MINORITY_FLOOR = float(
    winner_row[
        "minority_recall_floor"
    ]
)


# --------------------------------------------------------------------------------------------------
# 13. Interpretation
# --------------------------------------------------------------------------------------------------

natural_highest_accuracy = bool(
    natural_row[
        "accuracy"
    ]
    ==
    stage_c_comparison_df[
        "accuracy"
    ].max()
)


natural_highest_macro_f1 = bool(
    natural_row[
        "macro_f1"
    ]
    ==
    stage_c_comparison_df[
        "macro_f1"
    ].max()
)


inverse_improves_minority_floor = bool(
    inverse_row[
        "minority_recall_floor"
    ]
    >
    natural_row[
        "minority_recall_floor"
    ]
)


capped_improves_minority_floor = bool(
    capped_row[
        "minority_recall_floor"
    ]
    >
    natural_row[
        "minority_recall_floor"
    ]
)


inverse_reduces_accuracy = bool(
    inverse_row[
        "accuracy"
    ]
    <
    natural_row[
        "accuracy"
    ]
)


capped_reduces_accuracy = bool(
    capped_row[
        "accuracy"
    ]
    <
    natural_row[
        "accuracy"
    ]
)


inverse_reduces_neutral_recall = bool(
    inverse_row[
        "neutral_recall"
    ]
    <
    natural_row[
        "neutral_recall"
    ]
)


capped_reduces_neutral_recall = bool(
    capped_row[
        "neutral_recall"
    ]
    <
    natural_row[
        "neutral_recall"
    ]
)


# --------------------------------------------------------------------------------------------------
# 14. Winner adapter
# --------------------------------------------------------------------------------------------------

STAGE_C_WINNER_ADAPTER_PATH = (
    DAY22_RESULTS_ROOT
    / "runs"
    / "stage_b_horizon_audit"
    / "model"
    / "selected_adapter"
)


# --------------------------------------------------------------------------------------------------
# 15. Formal checks
# --------------------------------------------------------------------------------------------------

stage_c_selection_checks = {

    **identity_checks,

    "three_conditions_present":
        set(
            stage_c_comparison_df[
                "condition"
            ]
        )
        ==
        {
            "natural",
            "inverse_sqrt",
            "capped_balanced",
        },

    "winner_is_natural":
        STAGE_C_WINNER
        ==
        "natural",

    "natural_highest_accuracy":
        natural_highest_accuracy,

    "natural_highest_macro_f1":
        natural_highest_macro_f1,

    "inverse_improves_minority_floor":
        inverse_improves_minority_floor,

    "capped_improves_minority_floor":
        capped_improves_minority_floor,

    "inverse_reduces_accuracy":
        inverse_reduces_accuracy,

    "capped_reduces_accuracy":
        capped_reduces_accuracy,

    "inverse_reduces_neutral_recall":
        inverse_reduces_neutral_recall,

    "capped_reduces_neutral_recall":
        capped_reduces_neutral_recall,

    "winner_adapter_exists":
        STAGE_C_WINNER_ADAPTER_PATH.exists(),

    "no_posthoc_numeric_guardrail":
        True,

    "historical_validation_not_used":
        True,

    "future_not_used":
        True,
}


all_stage_c_selection_checks_pass = bool(
    all(
        stage_c_selection_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 16. Save artifacts
# --------------------------------------------------------------------------------------------------

STAGE_C_SELECTION_DIR = (
    DAY23_RESULTS_ROOT
    / "stage_c_selection"
)

STAGE_C_SELECTION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


STAGE_C_COMPARISON_PATH = (
    STAGE_C_SELECTION_DIR
    / "stage_c_three_condition_comparison.csv"
)

STAGE_C_SELECTION_SUMMARY_PATH = (
    STAGE_C_SELECTION_DIR
    / "stage_c_selection_summary.json"
)


stage_c_comparison_df.to_csv(
    STAGE_C_COMPARISON_PATH,
    index=False,
)


stage_c_selection_summary = {

    "day":
        23,

    "stage":
        "C",

    "objective":
        "historical optimization with primary target Accuracy",

    "evaluation_population":
        "optimization_dev",

    "selection_rule": {
        "primary":
            "accuracy",

        "secondary":
            "macro_f1",

        "diagnostic":
            "minority_recall_floor",

        "posthoc_numeric_guardrail_added":
            False,
    },

    "winner": {
        "condition":
            STAGE_C_WINNER,

        "sampling":
            STAGE_C_WINNER_SAMPLING,

        "accuracy":
            STAGE_C_WINNER_ACCURACY,

        "macro_f1":
            STAGE_C_WINNER_MACRO_F1,

        "negative_recall":
            STAGE_C_WINNER_NEG_RECALL,

        "neutral_recall":
            STAGE_C_WINNER_NEU_RECALL,

        "positive_recall":
            STAGE_C_WINNER_POS_RECALL,

        "minority_recall_floor":
            STAGE_C_WINNER_MINORITY_FLOOR,

        "adapter_path":
            str(
                STAGE_C_WINNER_ADAPTER_PATH
            ),
    },

    "interpretation": {
        "natural_highest_accuracy":
            natural_highest_accuracy,

        "natural_highest_macro_f1":
            natural_highest_macro_f1,

        "inverse_sqrt_improves_minority_floor":
            inverse_improves_minority_floor,

        "inverse_sqrt_reduces_accuracy":
            inverse_reduces_accuracy,

        "capped_balanced_improves_minority_floor":
            capped_improves_minority_floor,

        "capped_balanced_reduces_accuracy":
            capped_reduces_accuracy,

        "reweighting_shifts_recall_from_neutral_toward_minority_classes":
            bool(
                inverse_reduces_neutral_recall
                and
                capped_reduces_neutral_recall
                and
                inverse_improves_minority_floor
                and
                capped_improves_minority_floor
            ),
    },

    "checks": {
        key:
            bool(value)

        for key, value
        in stage_c_selection_checks.items()
    },

    "all_checks_pass":
        all_stage_c_selection_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_C_SELECTION_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_c_selection_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 17. Report
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("DAY 23 — STAGE C THREE-CONDITION FORMAL SELECTION")
print("=" * 120)


print("\n[THREE-CONDITION COMPARISON]")

display(
    stage_c_comparison_df[
        [
            "sampling",
            "accuracy",
            "macro_f1",
            "weighted_f1",
            "negative_recall",
            "neutral_recall",
            "positive_recall",
            "minority_recall_floor",
            "accuracy_rank",
            "macro_f1_rank",
        ]
    ]
)


print("\n[DELTA VS NATURAL]")

display(
    stage_c_comparison_df[
        [
            "sampling",
            "accuracy_delta_vs_natural",
            "macro_f1_delta_vs_natural",
            "negative_recall_delta_vs_natural",
            "neutral_recall_delta_vs_natural",
            "positive_recall_delta_vs_natural",
            "minority_recall_floor_delta_vs_natural",
        ]
    ]
)


print("\n[STAGE C WINNER]")
print(
    f"Condition                   : "
    f"{STAGE_C_WINNER}"
)
print(
    f"Sampling                    : "
    f"{STAGE_C_WINNER_SAMPLING}"
)
print(
    f"Accuracy                    : "
    f"{STAGE_C_WINNER_ACCURACY:.6f}"
)
print(
    f"Macro-F1                    : "
    f"{STAGE_C_WINNER_MACRO_F1:.6f}"
)
print(
    f"Negative recall             : "
    f"{STAGE_C_WINNER_NEG_RECALL:.6f}"
)
print(
    f"Neutral recall              : "
    f"{STAGE_C_WINNER_NEU_RECALL:.6f}"
)
print(
    f"Positive recall             : "
    f"{STAGE_C_WINNER_POS_RECALL:.6f}"
)
print(
    f"Selected adapter            : "
    f"{STAGE_C_WINNER_ADAPTER_PATH}"
)


print("\n[SCIENTIFIC INTERPRETATION]")
print(
    f"Natural highest Accuracy             : "
    f"{natural_highest_accuracy}"
)
print(
    f"Natural highest Macro-F1             : "
    f"{natural_highest_macro_f1}"
)
print(
    f"Inverse-sqrt improves minority floor : "
    f"{inverse_improves_minority_floor}"
)
print(
    f"Inverse-sqrt reduces Accuracy        : "
    f"{inverse_reduces_accuracy}"
)
print(
    f"Capped improves minority floor       : "
    f"{capped_improves_minority_floor}"
)
print(
    f"Capped reduces Accuracy              : "
    f"{capped_reduces_accuracy}"
)


print("\n[FORMAL CHECKS]")

for name, passed in stage_c_selection_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Comparison                  : "
    f"{STAGE_C_COMPARISON_PATH}"
)
print(
    f"Selection summary           : "
    f"{STAGE_C_SELECTION_SUMMARY_PATH}"
)


print("\n" + "-" * 120)

if all_stage_c_selection_checks_pass:

    print(
        "DAY 23 CELL 7 STATUS: PASS — "
        "STAGE C COMPLETE; NATURAL SAMPLING FROZEN AS WINNER; "
        "READY FOR STAGE D LEARNING-RATE AUDIT"
    )

else:

    print(
        "DAY 23 CELL 7 STATUS: FAIL — "
        "DO NOT START STAGE D"
    )

print("=" * 120)

NameError: name 'STAGE_C_CAPPED_METRICS_PATH' is not defined

In [16]:
# ==================================================================================================
# DAY 23 — CELL 8
# STAGE D — LEARNING-RATE AUDIT PROTOCOL FREEZE
# ==================================================================================================

import json
from pathlib import Path
from datetime import datetime, timezone


# --------------------------------------------------------------------------------------------------
# 1. Stage D identity
# --------------------------------------------------------------------------------------------------

STAGE_D_NAME = "learning_rate_audit"

STAGE_D_DIR = (
    DAY23_RESULTS_ROOT
    / "stage_d_learning_rate_audit"
)

STAGE_D_ARTIFACT_DIR = (
    STAGE_D_DIR
    / "artifacts"
)

STAGE_D_RUNS_DIR = (
    STAGE_D_DIR
    / "runs"
)


for directory in [
    STAGE_D_ARTIFACT_DIR,
    STAGE_D_RUNS_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# --------------------------------------------------------------------------------------------------
# 2. Preconditions from Stage C
# --------------------------------------------------------------------------------------------------

assert (
    STAGE_C_WINNER
    ==
    "natural"
), (
    "DAY 23 STOP: Stage C winner is not natural sampling."
)


assert (
    abs(
        STAGE_C_WINNER_ACCURACY
        -
        0.840842
    )
    <
    1e-6
), (
    "DAY 23 STOP: Stage C winner Accuracy mismatch."
)


assert (
    abs(
        STAGE_C_WINNER_MACRO_F1
        -
        0.808201
    )
    <
    1e-6
), (
    "DAY 23 STOP: Stage C winner Macro-F1 mismatch."
)


assert (
    STAGE_C_WINNER_ADAPTER_PATH.exists()
), (
    "DAY 23 STOP: Stage C winner adapter missing."
)


# --------------------------------------------------------------------------------------------------
# 3. Frozen Stage D candidate learning rates
# --------------------------------------------------------------------------------------------------

STAGE_D_LEARNING_RATES = [
    5e-5,
    1e-4,
    2e-4,
]


STAGE_D_EXISTING_REFERENCE_LR = 1e-4


STAGE_D_NEW_TRAINING_LRS = [
    5e-5,
    2e-4,
]


# --------------------------------------------------------------------------------------------------
# 4. Frozen shared training configuration
# --------------------------------------------------------------------------------------------------

STAGE_D_BASE_MODEL = (
    "meta-llama/Meta-Llama-3-8B"
)

STAGE_D_SAMPLING = (
    "natural"
)

STAGE_D_MAX_STEPS = 500

STAGE_D_TRAIN_BATCH_SIZE = 1

STAGE_D_GRADIENT_ACCUMULATION = 8

STAGE_D_EFFECTIVE_BATCH = (
    STAGE_D_TRAIN_BATCH_SIZE
    *
    STAGE_D_GRADIENT_ACCUMULATION
)

STAGE_D_SCHEDULER = (
    "linear"
)

STAGE_D_WARMUP_STEPS = 10

STAGE_D_SEED = 42

STAGE_D_MAX_SEQ_LENGTH = 512


# Frozen LoRA
STAGE_D_LORA_R = 8

STAGE_D_LORA_ALPHA = 32

STAGE_D_LORA_DROPOUT = 0.1

STAGE_D_LORA_TARGET_MODULES = [
    "q_proj",
    "v_proj",
]


# Frozen quantization
STAGE_D_LOAD_IN_4BIT = True

STAGE_D_QUANT_TYPE = "nf4"

STAGE_D_DOUBLE_QUANT = True

STAGE_D_COMPUTE_DTYPE = "float16"


# --------------------------------------------------------------------------------------------------
# 5. Existing 1e-4 reference
#
# This is the frozen Stage B-500 natural-sampling run.
# It MUST be reused, not retrained.
# --------------------------------------------------------------------------------------------------

STAGE_D_REFERENCE_ADAPTER_PATH = (
    DAY22_RESULTS_ROOT
    / "runs"
    / "stage_b_horizon_audit"
    / "model"
    / "selected_adapter"
)


STAGE_D_REFERENCE_METRICS_PATH = (
    DAY22_RESULTS_ROOT
    / "runs"
    / "stage_b_horizon_audit"
    / "artifacts"
    / "stage_b_500_optimization_dev_metrics.json"
)


STAGE_D_REFERENCE_CLASS_METRICS_PATH = (
    DAY22_RESULTS_ROOT
    / "runs"
    / "stage_b_horizon_audit"
    / "artifacts"
    / "stage_b_500_optimization_dev_class_metrics.csv"
)


assert STAGE_D_REFERENCE_ADAPTER_PATH.exists()
assert STAGE_D_REFERENCE_METRICS_PATH.exists()
assert STAGE_D_REFERENCE_CLASS_METRICS_PATH.exists()


# --------------------------------------------------------------------------------------------------
# 6. New candidate output locations
# --------------------------------------------------------------------------------------------------

STAGE_D_LR5E5_DIR = (
    STAGE_D_RUNS_DIR
    / "lr_5e-5"
)

STAGE_D_LR2E4_DIR = (
    STAGE_D_RUNS_DIR
    / "lr_2e-4"
)


for directory in [
    STAGE_D_LR5E5_DIR,
    STAGE_D_LR2E4_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# --------------------------------------------------------------------------------------------------
# 7. Selection rule
#
# Frozen Day 22 / Day 23 optimization objective:
#
#   1. Primary   : Accuracy
#   2. Secondary : Macro-F1
#   3. Diagnostic: minority recall floor
#
# No post-hoc numerical guardrail will be introduced.
# --------------------------------------------------------------------------------------------------

STAGE_D_SELECTION_RULE = {

    "primary":
        "accuracy",

    "secondary":
        "macro_f1",

    "diagnostic":
        "minority_recall_floor",

    "tie_break_if_needed":
        [
            "higher_macro_f1",
            "higher_minority_recall_floor",
            "lower_eval_loss",
            "simpler_existing_configuration_if_effectively_tied",
        ],

    "posthoc_numeric_guardrail_allowed":
        False,
}


# --------------------------------------------------------------------------------------------------
# 8. Firewall
# --------------------------------------------------------------------------------------------------

STAGE_D_EVALUATION_SPLIT = (
    "optimization_dev"
)

STAGE_D_HISTORICAL_VALIDATION_ALLOWED = False

STAGE_D_FUTURE_ALLOWED = False


# --------------------------------------------------------------------------------------------------
# 9. Candidate manifest
# --------------------------------------------------------------------------------------------------

stage_d_candidate_manifest = {

    "lr_5e-5": {

        "learning_rate":
            5e-5,

        "status":
            "PENDING_FRESH_TRAINING",

        "reuse_existing_run":
            False,

        "run_dir":
            str(
                STAGE_D_LR5E5_DIR
            ),
    },

    "lr_1e-4": {

        "learning_rate":
            1e-4,

        "status":
            "EXISTING_REFERENCE",

        "reuse_existing_run":
            True,

        "source":
            "Day22 Stage B-500 natural sampling",

        "adapter_path":
            str(
                STAGE_D_REFERENCE_ADAPTER_PATH
            ),

        "optimization_dev_accuracy":
            0.840842,

        "optimization_dev_macro_f1":
            0.808201,
    },

    "lr_2e-4": {

        "learning_rate":
            2e-4,

        "status":
            "PENDING_FRESH_TRAINING",

        "reuse_existing_run":
            False,

        "run_dir":
            str(
                STAGE_D_LR2E4_DIR
            ),
    },
}


# --------------------------------------------------------------------------------------------------
# 10. Formal protocol checks
# --------------------------------------------------------------------------------------------------

stage_d_protocol_checks = {

    "stage_c_complete":
        all_stage_c_selection_checks_pass,

    "stage_c_winner_natural":
        STAGE_C_WINNER
        ==
        "natural",

    "three_learning_rates":
        STAGE_D_LEARNING_RATES
        ==
        [
            5e-5,
            1e-4,
            2e-4,
        ],

    "reference_lr_1e4":
        STAGE_D_EXISTING_REFERENCE_LR
        ==
        1e-4,

    "only_two_new_training_runs":
        STAGE_D_NEW_TRAINING_LRS
        ==
        [
            5e-5,
            2e-4,
        ],

    "natural_sampling_fixed":
        STAGE_D_SAMPLING
        ==
        "natural",

    "max_steps_500":
        STAGE_D_MAX_STEPS
        ==
        500,

    "effective_batch_8":
        STAGE_D_EFFECTIVE_BATCH
        ==
        8,

    "scheduler_linear":
        STAGE_D_SCHEDULER
        ==
        "linear",

    "warmup_10":
        STAGE_D_WARMUP_STEPS
        ==
        10,

    "seed_42":
        STAGE_D_SEED
        ==
        42,

    "lora_r8":
        STAGE_D_LORA_R
        ==
        8,

    "lora_alpha32":
        STAGE_D_LORA_ALPHA
        ==
        32,

    "lora_dropout_0_1":
        abs(
            STAGE_D_LORA_DROPOUT
            -
            0.1
        )
        <
        1e-12,

    "reference_adapter_exists":
        STAGE_D_REFERENCE_ADAPTER_PATH.exists(),

    "reference_metrics_exist":
        STAGE_D_REFERENCE_METRICS_PATH.exists(),

    "primary_metric_accuracy":
        STAGE_D_SELECTION_RULE[
            "primary"
        ]
        ==
        "accuracy",

    "no_posthoc_guardrail":
        STAGE_D_SELECTION_RULE[
            "posthoc_numeric_guardrail_allowed"
        ]
        is False,

    "optimization_dev_only":
        STAGE_D_EVALUATION_SPLIT
        ==
        "optimization_dev",

    "historical_validation_locked":
        STAGE_D_HISTORICAL_VALIDATION_ALLOWED
        is False,

    "future_locked":
        STAGE_D_FUTURE_ALLOWED
        is False,
}


all_stage_d_protocol_checks_pass = bool(
    all(
        stage_d_protocol_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 11. Save frozen Stage D protocol
# --------------------------------------------------------------------------------------------------

STAGE_D_PROTOCOL_PATH = (
    STAGE_D_ARTIFACT_DIR
    / "stage_d_learning_rate_audit_protocol.json"
)


stage_d_protocol_payload = {

    "day":
        23,

    "stage":
        "D",

    "name":
        STAGE_D_NAME,

    "freeze_status":
        "STAGE_D_LEARNING_RATE_PROTOCOL_FROZEN",

    "objective":
        "Optimize historical model performance with primary target Accuracy.",

    "evaluation_population":
        STAGE_D_EVALUATION_SPLIT,

    "fixed_configuration": {

        "base_model":
            STAGE_D_BASE_MODEL,

        "sampling":
            STAGE_D_SAMPLING,

        "max_steps":
            STAGE_D_MAX_STEPS,

        "train_batch_size":
            STAGE_D_TRAIN_BATCH_SIZE,

        "gradient_accumulation_steps":
            STAGE_D_GRADIENT_ACCUMULATION,

        "effective_batch":
            STAGE_D_EFFECTIVE_BATCH,

        "scheduler":
            STAGE_D_SCHEDULER,

        "warmup_steps":
            STAGE_D_WARMUP_STEPS,

        "seed":
            STAGE_D_SEED,

        "max_seq_length":
            STAGE_D_MAX_SEQ_LENGTH,

        "lora": {
            "r":
                STAGE_D_LORA_R,

            "alpha":
                STAGE_D_LORA_ALPHA,

            "dropout":
                STAGE_D_LORA_DROPOUT,

            "target_modules":
                STAGE_D_LORA_TARGET_MODULES,
        },

        "quantization": {
            "load_in_4bit":
                STAGE_D_LOAD_IN_4BIT,

            "quant_type":
                STAGE_D_QUANT_TYPE,

            "double_quant":
                STAGE_D_DOUBLE_QUANT,

            "compute_dtype":
                STAGE_D_COMPUTE_DTYPE,
        },
    },

    "candidate_learning_rates":
        STAGE_D_LEARNING_RATES,

    "candidate_manifest":
        stage_d_candidate_manifest,

    "selection_rule":
        STAGE_D_SELECTION_RULE,

    "firewall": {
        "historical_validation_allowed":
            False,

        "future_f1_f6_allowed":
            False,
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in stage_d_protocol_checks.items()
    },

    "all_checks_pass":
        all_stage_d_protocol_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_D_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_d_protocol_payload,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 12. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE D LEARNING-RATE AUDIT PROTOCOL FREEZE")
print("=" * 120)


print("\n[STAGE C INPUT]")
print(
    f"Frozen sampling winner      : "
    f"{STAGE_C_WINNER_SAMPLING}"
)
print(
    f"Reference Accuracy          : "
    f"{STAGE_C_WINNER_ACCURACY:.6f}"
)
print(
    f"Reference Macro-F1          : "
    f"{STAGE_C_WINNER_MACRO_F1:.6f}"
)


print("\n[FIXED CONFIGURATION]")
print(
    f"Base model                  : "
    f"{STAGE_D_BASE_MODEL}"
)
print(
    f"Sampling                    : "
    f"{STAGE_D_SAMPLING}"
)
print(
    f"Training horizon            : "
    f"{STAGE_D_MAX_STEPS}"
)
print(
    f"Effective batch             : "
    f"{STAGE_D_EFFECTIVE_BATCH}"
)
print(
    f"Scheduler                   : "
    f"{STAGE_D_SCHEDULER}"
)
print(
    f"Warmup                      : "
    f"{STAGE_D_WARMUP_STEPS}"
)
print(
    f"LoRA                        : "
    f"r={STAGE_D_LORA_R}, "
    f"alpha={STAGE_D_LORA_ALPHA}, "
    f"dropout={STAGE_D_LORA_DROPOUT}"
)
print(
    f"Seed                        : "
    f"{STAGE_D_SEED}"
)


print("\n[LEARNING-RATE CANDIDATES]")
print(
    "5e-5                        : NEW FRESH TRAINING REQUIRED"
)
print(
    "1e-4                        : REUSE STAGE B-500 — NO RETRAIN"
)
print(
    "2e-4                        : NEW FRESH TRAINING REQUIRED"
)


print("\n[SELECTION]")
print(
    "Primary                     : Accuracy"
)
print(
    "Secondary                   : Macro-F1"
)
print(
    "Diagnostic                  : Minority recall floor"
)
print(
    "Post-hoc numerical guardrail: NOT ALLOWED"
)


print("\n[FIREWALL]")
print(
    "Optimization Dev            : ALLOWED"
)
print(
    "Historical Validation       : LOCKED"
)
print(
    "Future F1-F6                : LOCKED"
)


print("\n[FORMAL CHECKS]")

for name, passed in stage_d_protocol_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACT]")
print(
    STAGE_D_PROTOCOL_PATH
)


print("\n" + "-" * 120)

if all_stage_d_protocol_checks_pass:

    print(
        "DAY 23 CELL 8 STATUS: PASS — "
        "STAGE D LEARNING-RATE PROTOCOL FROZEN; "
        "READY FOR LR=5E-5 FRESH MODEL INITIALIZATION"
    )

else:

    print(
        "DAY 23 CELL 8 STATUS: FAIL — "
        "DO NOT INITIALIZE STAGE D TRAINING"
    )

print("=" * 120)

DAY 23 — STAGE D LEARNING-RATE AUDIT PROTOCOL FREEZE

[STAGE C INPUT]
Frozen sampling winner      : Natural
Reference Accuracy          : 0.840842
Reference Macro-F1          : 0.808201

[FIXED CONFIGURATION]
Base model                  : meta-llama/Meta-Llama-3-8B
Sampling                    : natural
Training horizon            : 500
Effective batch             : 8
Scheduler                   : linear
Warmup                      : 10
LoRA                        : r=8, alpha=32, dropout=0.1
Seed                        : 42

[LEARNING-RATE CANDIDATES]
5e-5                        : NEW FRESH TRAINING REQUIRED
1e-4                        : REUSE STAGE B-500 — NO RETRAIN
2e-4                        : NEW FRESH TRAINING REQUIRED

[SELECTION]
Primary                     : Accuracy
Secondary                   : Macro-F1
Diagnostic                  : Minority recall floor
Post-hoc numerical guardrail: NOT ALLOWED

[FIREWALL]
Optimization Dev            : ALLOWED
Historical Validation       : 

In [17]:
# ==================================================================================================
# DAY 23 — CELL 9
# STAGE D — LR = 5e-5
# FRESH MODEL + NATURAL-SAMPLING TRAINER INFRASTRUCTURE
# ==================================================================================================

import gc
import json
import random
from pathlib import Path

import numpy as np
import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


# --------------------------------------------------------------------------------------------------
# 1. Candidate identity + paths
# --------------------------------------------------------------------------------------------------

STAGE_D_CURRENT_LR = 5e-5

STAGE_D_LR5E5_ARTIFACT_DIR = (
    STAGE_D_LR5E5_DIR
    / "artifacts"
)

STAGE_D_LR5E5_MODEL_DIR = (
    STAGE_D_LR5E5_DIR
    / "model"
)

STAGE_D_LR5E5_CHECKPOINT_DIR = (
    STAGE_D_LR5E5_MODEL_DIR
    / "checkpoints"
)


for directory in [
    STAGE_D_LR5E5_ARTIFACT_DIR,
    STAGE_D_LR5E5_MODEL_DIR,
    STAGE_D_LR5E5_CHECKPOINT_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# --------------------------------------------------------------------------------------------------
# 2. Candidate directory must be clean
# --------------------------------------------------------------------------------------------------

existing_lr5e5_checkpoints = sorted(
    [
        p
        for p
        in STAGE_D_LR5E5_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


assert len(
    existing_lr5e5_checkpoints
) == 0, (
    "DAY 23 STOP: LR=5e-5 checkpoint directory is not clean."
)


# --------------------------------------------------------------------------------------------------
# 3. Release capped-balanced model/trainer GPU state
# --------------------------------------------------------------------------------------------------

objects_to_delete = [
    "stage_c_capped_trainer",
    "stage_c_capped_model",
    "stage_c_capped_base_model",
]


for obj_name in objects_to_delete:

    if obj_name in globals():

        del globals()[
            obj_name
        ]


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()


gpu_after_cleanup_gb = (
    torch.cuda.memory_allocated()
    /
    (1024 ** 3)
)


# --------------------------------------------------------------------------------------------------
# 4. Deterministic seed
# --------------------------------------------------------------------------------------------------

random.seed(
    STAGE_D_SEED
)

np.random.seed(
    STAGE_D_SEED
)

torch.manual_seed(
    STAGE_D_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        STAGE_D_SEED
    )


set_seed(
    STAGE_D_SEED
)


# --------------------------------------------------------------------------------------------------
# 5. Fresh 4-bit base model
# --------------------------------------------------------------------------------------------------

stage_d_lr5e5_bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_use_double_quant=True,

    bnb_4bit_compute_dtype=torch.float16,
)


stage_d_lr5e5_base_model = (
    AutoModelForCausalLM.from_pretrained(

        STAGE_D_BASE_MODEL,

        quantization_config=
            stage_d_lr5e5_bnb_config,

        device_map=
            "auto",

        trust_remote_code=
            True,
    )
)


stage_d_lr5e5_base_model.config.use_cache = False


stage_d_lr5e5_base_model = (
    prepare_model_for_kbit_training(

        stage_d_lr5e5_base_model,

        use_gradient_checkpointing=True,
    )
)


# --------------------------------------------------------------------------------------------------
# 6. Fresh LoRA
# --------------------------------------------------------------------------------------------------

set_seed(
    STAGE_D_SEED
)


stage_d_lr5e5_lora_config = LoraConfig(

    r=
        STAGE_D_LORA_R,

    lora_alpha=
        STAGE_D_LORA_ALPHA,

    lora_dropout=
        STAGE_D_LORA_DROPOUT,

    target_modules=
        STAGE_D_LORA_TARGET_MODULES,

    bias=
        "none",

    task_type=
        "CAUSAL_LM",

    inference_mode=
        False,
)


stage_d_lr5e5_model = (
    get_peft_model(
        stage_d_lr5e5_base_model,
        stage_d_lr5e5_lora_config,
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Final-only evaluation/save callback
# --------------------------------------------------------------------------------------------------

class StageDFinalStepCallback(
    TrainerCallback
):

    def __init__(
        self,
        final_step=500,
    ):

        self.final_step = int(
            final_step
        )

        self.eval_triggered = False
        self.save_triggered = False


    def on_step_end(
        self,
        args,
        state,
        control,
        **kwargs,
    ):

        control.should_evaluate = False
        control.should_save = False


        if int(
            state.global_step
        ) == self.final_step:

            control.should_evaluate = True
            control.should_save = True

            self.eval_triggered = True
            self.save_triggered = True


        return control


stage_d_lr5e5_callback = (
    StageDFinalStepCallback(
        final_step=
            STAGE_D_MAX_STEPS
    )
)


# --------------------------------------------------------------------------------------------------
# 8. TrainingArguments
#
# Natural sampling:
# no WeightedRandomSampler override.
# Standard Trainer shuffle behavior is retained.
# --------------------------------------------------------------------------------------------------

stage_d_lr5e5_training_args = TrainingArguments(

    output_dir=
        str(
            STAGE_D_LR5E5_CHECKPOINT_DIR
        ),

    max_steps=
        STAGE_D_MAX_STEPS,

    per_device_train_batch_size=
        STAGE_D_TRAIN_BATCH_SIZE,

    per_device_eval_batch_size=
        1,

    gradient_accumulation_steps=
        STAGE_D_GRADIENT_ACCUMULATION,

    learning_rate=
        STAGE_D_CURRENT_LR,

    lr_scheduler_type=
        STAGE_D_SCHEDULER,

    warmup_steps=
        STAGE_D_WARMUP_STEPS,

    weight_decay=
        0.0,

    logging_strategy=
        "steps",

    logging_steps=
        10,

    eval_strategy=
        "no",

    save_strategy=
        "no",

    load_best_model_at_end=
        False,

    fp16=
        True,

    bf16=
        False,

    gradient_checkpointing=
        True,

    seed=
        STAGE_D_SEED,

    data_seed=
        STAGE_D_SEED,

    report_to=
        [],

    remove_unused_columns=
        True,

    label_names=[
        "labels"
    ],

    disable_tqdm=
        False,
)


# --------------------------------------------------------------------------------------------------
# 9. Build standard natural-sampling Trainer
# --------------------------------------------------------------------------------------------------

stage_d_lr5e5_trainer = Trainer(

    model=
        stage_d_lr5e5_model,

    args=
        stage_d_lr5e5_training_args,

    train_dataset=
        hf_opt_train_model,

    eval_dataset=
        hf_opt_dev_model,

    data_collator=
        data_collator,

    processing_class=
        tokenizer,

    callbacks=[
        stage_d_lr5e5_callback
    ],
)


# --------------------------------------------------------------------------------------------------
# 10. Trainable parameter audit
# --------------------------------------------------------------------------------------------------

stage_d_lr5e5_total_params = 0
stage_d_lr5e5_trainable_params = 0
stage_d_lr5e5_trainable_names = []


for name, param in stage_d_lr5e5_model.named_parameters():

    stage_d_lr5e5_total_params += (
        param.numel()
    )

    if param.requires_grad:

        stage_d_lr5e5_trainable_params += (
            param.numel()
        )

        stage_d_lr5e5_trainable_names.append(
            name
        )


stage_d_lr5e5_unexpected_trainable = [

    name

    for name
    in stage_d_lr5e5_trainable_names

    if "lora_" not in name
]


# --------------------------------------------------------------------------------------------------
# 11. Natural-sampling integrity
# --------------------------------------------------------------------------------------------------

natural_sampling_checks = {

    "trainer_is_standard_trainer":
        type(
            stage_d_lr5e5_trainer
        )
        is Trainer,

    "no_weighted_sampler":
        True,

    "train_dataset_identity":
        stage_d_lr5e5_trainer.train_dataset
        is hf_opt_train_model,

    "train_population_10831":
        len(
            stage_d_lr5e5_trainer.train_dataset
        )
        == 10831,
}


# --------------------------------------------------------------------------------------------------
# 12. Formal infrastructure checks
# --------------------------------------------------------------------------------------------------

stage_d_lr5e5_checks = {

    "checkpoint_dir_clean":
        len(
            existing_lr5e5_checkpoints
        )
        == 0,

    "gpu_cleanup_completed":
        gpu_after_cleanup_gb
        <
        3.0,

    "fresh_model_loaded":
        stage_d_lr5e5_model
        is not None,

    "natural_sampling":
        all(
            natural_sampling_checks.values()
        ),

    "train_n_10831":
        len(
            hf_opt_train_model
        )
        == 10831,

    "dev_n_2708":
        len(
            hf_opt_dev_model
        )
        == 2708,

    "max_steps_500":
        stage_d_lr5e5_training_args.max_steps
        == 500,

    "lr_5e5":
        abs(
            stage_d_lr5e5_training_args.learning_rate
            -
            5e-5
        )
        <
        1e-12,

    "gradient_accumulation_8":
        (
            stage_d_lr5e5_training_args
            .gradient_accumulation_steps
            ==
            8
        ),

    "scheduler_linear":
        str(
            stage_d_lr5e5_training_args.lr_scheduler_type
        )
        .lower()
        .endswith(
            "linear"
        ),

    "warmup_10":
        stage_d_lr5e5_training_args.warmup_steps
        == 10,

    "seed_42":
        stage_d_lr5e5_training_args.seed
        == 42,

    "lora_r8":
        (
            stage_d_lr5e5_model
            .peft_config[
                "default"
            ]
            .r
            ==
            8
        ),

    "lora_alpha32":
        (
            stage_d_lr5e5_model
            .peft_config[
                "default"
            ]
            .lora_alpha
            ==
            32
        ),

    "trainable_params_3407872":
        stage_d_lr5e5_trainable_params
        ==
        3_407_872,

    "only_lora_trainable":
        len(
            stage_d_lr5e5_unexpected_trainable
        )
        == 0,

    "trainer_global_step_zero":
        int(
            stage_d_lr5e5_trainer
            .state
            .global_step
        )
        == 0,

    "eval_not_triggered":
        stage_d_lr5e5_callback.eval_triggered
        is False,

    "save_not_triggered":
        stage_d_lr5e5_callback.save_triggered
        is False,

    "historical_validation_locked":
        STAGE_D_HISTORICAL_VALIDATION_ALLOWED
        is False,

    "future_locked":
        STAGE_D_FUTURE_ALLOWED
        is False,
}


all_stage_d_lr5e5_checks_pass = bool(
    all(
        stage_d_lr5e5_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 13. Save infrastructure manifest
# --------------------------------------------------------------------------------------------------

STAGE_D_LR5E5_INFRA_PATH = (
    STAGE_D_LR5E5_ARTIFACT_DIR
    / "lr_5e-5_training_infrastructure.json"
)


stage_d_lr5e5_infra_payload = {

    "day":
        23,

    "stage":
        "D",

    "condition":
        "lr_5e-5",

    "learning_rate":
        STAGE_D_CURRENT_LR,

    "sampling":
        "natural",

    "initialization":
        "fresh_base_plus_fresh_lora",

    "base_model":
        STAGE_D_BASE_MODEL,

    "training": {

        "max_steps":
            STAGE_D_MAX_STEPS,

        "learning_rate":
            STAGE_D_CURRENT_LR,

        "scheduler":
            STAGE_D_SCHEDULER,

        "warmup_steps":
            STAGE_D_WARMUP_STEPS,

        "effective_batch":
            STAGE_D_EFFECTIVE_BATCH,

        "seed":
            STAGE_D_SEED,
    },

    "lora": {

        "r":
            STAGE_D_LORA_R,

        "alpha":
            STAGE_D_LORA_ALPHA,

        "dropout":
            STAGE_D_LORA_DROPOUT,

        "target_modules":
            STAGE_D_LORA_TARGET_MODULES,
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in stage_d_lr5e5_checks.items()
    },

    "all_checks_pass":
        all_stage_d_lr5e5_checks_pass,
}


with open(
    STAGE_D_LR5E5_INFRA_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_d_lr5e5_infra_payload,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 14. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE D LR=5E-5 TRAINER INFRASTRUCTURE")
print("=" * 120)


print("\n[GPU CLEANUP]")
print(
    f"Allocated after cleanup     : "
    f"{gpu_after_cleanup_gb:.2f} GB"
)


print("\n[CANDIDATE]")
print(
    "Learning rate               : 5e-5"
)
print(
    "Sampling                    : Natural"
)
print(
    "Initialization              : Fresh base + fresh LoRA"
)
print(
    "Previous adapters loaded    : False"
)


print("\n[TRAINING PROTOCOL]")
print(
    "Max steps                   : 500"
)
print(
    "Effective batch             : 8"
)
print(
    "Scheduler                   : linear"
)
print(
    "Warmup                      : 10"
)
print(
    "Seed                        : 42"
)
print(
    "Formal Dev evaluation       : step 500 only"
)


print("\n[PARAMETERS]")
print(
    f"Trainable parameters        : "
    f"{stage_d_lr5e5_trainable_params:,}"
)
print(
    f"Total parameters            : "
    f"{stage_d_lr5e5_total_params:,}"
)
print(
    f"Unexpected trainable        : "
    f"{len(stage_d_lr5e5_unexpected_trainable)}"
)


print("\n[PRE-TRAINING STATE]")
print(
    f"Global step                 : "
    f"{stage_d_lr5e5_trainer.state.global_step}"
)
print(
    f"Final eval triggered        : "
    f"{stage_d_lr5e5_callback.eval_triggered}"
)
print(
    f"Final save triggered        : "
    f"{stage_d_lr5e5_callback.save_triggered}"
)


print("\n[GPU AFTER MODEL LOAD]")
print(
    f"Allocated                   : "
    f"{torch.cuda.memory_allocated() / (1024 ** 3):.2f} GB"
)


print("\n[FORMAL CHECKS]")

for name, passed in stage_d_lr5e5_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACT]")
print(
    STAGE_D_LR5E5_INFRA_PATH
)


print("\n" + "-" * 120)

if all_stage_d_lr5e5_checks_pass:

    print(
        "DAY 23 CELL 9 STATUS: PASS — "
        "LR=5E-5 NATURAL-SAMPLING TRAINER READY; "
        "TRAINING NOT YET STARTED"
    )

else:

    print(
        "DAY 23 CELL 9 STATUS: FAIL — "
        "DO NOT START LR=5E-5 TRAINING"
    )

print("=" * 120)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

DAY 23 — STAGE D LR=5E-5 TRAINER INFRASTRUCTURE

[GPU CLEANUP]
Allocated after cleanup     : 1.97 GB

[CANDIDATE]
Learning rate               : 5e-5
Sampling                    : Natural
Initialization              : Fresh base + fresh LoRA
Previous adapters loaded    : False

[TRAINING PROTOCOL]
Max steps                   : 500
Effective batch             : 8
Scheduler                   : linear
Warmup                      : 10
Seed                        : 42
Formal Dev evaluation       : step 500 only

[PARAMETERS]
Trainable parameters        : 3,407,872
Total parameters            : 4,544,008,192
Unexpected trainable        : 0

[PRE-TRAINING STATE]
Global step                 : 0
Final eval triggered        : False
Final save triggered        : False

[GPU AFTER MODEL LOAD]
Allocated                   : 7.30 GB

[FORMAL CHECKS]
PASS   — checkpoint_dir_clean
PASS   — gpu_cleanup_completed
PASS   — fresh_model_loaded
PASS   — natural_sampling
PASS   — train_n_10831
PASS   — dev_n_2

In [18]:
# ==================================================================================================
# DAY 23 — CELL 10
# STAGE D — LR = 5e-5
# CONTROLLED 500-STEP NATURAL-SAMPLING TRAINING
# ==================================================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch


# --------------------------------------------------------------------------------------------------
# 1. Formal artifact paths
# --------------------------------------------------------------------------------------------------

STAGE_D_LR5E5_TRAINING_LOG_PATH = (
    STAGE_D_LR5E5_ARTIFACT_DIR
    / "lr_5e-5_training_log.csv"
)

STAGE_D_LR5E5_TRAINING_SUMMARY_PATH = (
    STAGE_D_LR5E5_ARTIFACT_DIR
    / "lr_5e-5_training_summary.json"
)

STAGE_D_LR5E5_TRAINING_COMPLETE_PATH = (
    STAGE_D_LR5E5_ARTIFACT_DIR
    / "lr_5e-5_training_complete.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Pre-training safety gate
# --------------------------------------------------------------------------------------------------

assert int(
    stage_d_lr5e5_trainer.state.global_step
) == 0, (
    "DAY 23 STOP: LR=5e-5 trainer already advanced beyond step 0."
)


assert (
    stage_d_lr5e5_callback.eval_triggered
    is False
), (
    "DAY 23 STOP: LR=5e-5 evaluation callback already triggered."
)


assert (
    stage_d_lr5e5_callback.save_triggered
    is False
), (
    "DAY 23 STOP: LR=5e-5 save callback already triggered."
)


existing_checkpoints_before_training = sorted(
    [
        p
        for p
        in STAGE_D_LR5E5_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


assert len(
    existing_checkpoints_before_training
) == 0, (
    "DAY 23 STOP: LR=5e-5 checkpoint directory is not clean."
)


assert not STAGE_D_LR5E5_TRAINING_COMPLETE_PATH.exists(), (
    "DAY 23 STOP: LR=5e-5 completion marker already exists."
)


# --------------------------------------------------------------------------------------------------
# 3. Reset deterministic seeds immediately before training
# --------------------------------------------------------------------------------------------------

random.seed(
    STAGE_D_SEED
)

np.random.seed(
    STAGE_D_SEED
)

torch.manual_seed(
    STAGE_D_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        STAGE_D_SEED
    )

    torch.cuda.reset_peak_memory_stats()


set_seed(
    STAGE_D_SEED
)


# --------------------------------------------------------------------------------------------------
# 4. Train
# --------------------------------------------------------------------------------------------------

training_start_time = time.time()


stage_d_lr5e5_train_result = (
    stage_d_lr5e5_trainer.train()
)


training_end_time = time.time()


training_runtime_seconds = (
    training_end_time
    -
    training_start_time
)

training_runtime_minutes = (
    training_runtime_seconds
    /
    60.0
)


# --------------------------------------------------------------------------------------------------
# 5. Collect Trainer history
# --------------------------------------------------------------------------------------------------

stage_d_lr5e5_log_history = (
    stage_d_lr5e5_trainer.state.log_history
)


stage_d_lr5e5_log_df = pd.DataFrame(
    stage_d_lr5e5_log_history
)


stage_d_lr5e5_log_df.to_csv(
    STAGE_D_LR5E5_TRAINING_LOG_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 6. Final state
# --------------------------------------------------------------------------------------------------

final_global_step = int(
    stage_d_lr5e5_trainer.state.global_step
)


final_training_loss = float(
    stage_d_lr5e5_train_result.training_loss
)


# --------------------------------------------------------------------------------------------------
# 7. Extract final Dev evaluation
# --------------------------------------------------------------------------------------------------

final_eval_rows = (
    stage_d_lr5e5_log_df.loc[
        stage_d_lr5e5_log_df[
            "eval_loss"
        ].notna()
    ]
    .copy()
)


if len(
    final_eval_rows
) > 0:

    final_eval_row = (
        final_eval_rows
        .sort_values(
            "step"
        )
        .iloc[-1]
    )

    final_eval_step = int(
        final_eval_row[
            "step"
        ]
    )

    final_eval_loss = float(
        final_eval_row[
            "eval_loss"
        ]
    )

    final_eval_runtime = float(
        final_eval_row.get(
            "eval_runtime",
            np.nan,
        )
    )

else:

    final_eval_step = None
    final_eval_loss = None
    final_eval_runtime = None


# --------------------------------------------------------------------------------------------------
# 8. Learning-rate trajectory
# --------------------------------------------------------------------------------------------------

lr_rows = (
    stage_d_lr5e5_log_df.loc[
        stage_d_lr5e5_log_df[
            "learning_rate"
        ].notna()
    ]
    .copy()
)


def get_lr5e5_logged_lr(
    step,
):

    rows = lr_rows.loc[
        lr_rows[
            "step"
        ]
        ==
        step
    ]


    if len(
        rows
    ) == 0:

        return None


    return float(
        rows.iloc[-1][
            "learning_rate"
        ]
    )


lr_step_100 = get_lr5e5_logged_lr(
    100
)

lr_step_200 = get_lr5e5_logged_lr(
    200
)

lr_step_300 = get_lr5e5_logged_lr(
    300
)

lr_step_400 = get_lr5e5_logged_lr(
    400
)

lr_step_500 = get_lr5e5_logged_lr(
    500
)


# --------------------------------------------------------------------------------------------------
# 9. Scheduler end-state
# --------------------------------------------------------------------------------------------------

final_scheduler_epoch = None
final_scheduler_lr = None


if (
    stage_d_lr5e5_trainer.lr_scheduler
    is not None
):

    final_scheduler_epoch = int(
        stage_d_lr5e5_trainer
        .lr_scheduler
        .last_epoch
    )

    final_scheduler_lr = [
        float(
            value
        )
        for value
        in stage_d_lr5e5_trainer
        .lr_scheduler
        .get_last_lr()
    ]


# --------------------------------------------------------------------------------------------------
# 10. Check final checkpoint
# --------------------------------------------------------------------------------------------------

stage_d_lr5e5_final_checkpoint = (
    STAGE_D_LR5E5_CHECKPOINT_DIR
    / "checkpoint-500"
)


checkpoint_500_checks = {

    "checkpoint_exists":
        stage_d_lr5e5_final_checkpoint.exists(),

    "adapter_config_exists":
        (
            stage_d_lr5e5_final_checkpoint
            / "adapter_config.json"
        ).exists(),

    "adapter_weights_exist":
        (
            (
                stage_d_lr5e5_final_checkpoint
                / "adapter_model.safetensors"
            ).exists()
            or
            (
                stage_d_lr5e5_final_checkpoint
                / "adapter_model.bin"
            ).exists()
        ),

    "trainer_state_exists":
        (
            stage_d_lr5e5_final_checkpoint
            / "trainer_state.json"
        ).exists(),

    "optimizer_state_exists":
        (
            stage_d_lr5e5_final_checkpoint
            / "optimizer.pt"
        ).exists(),

    "scheduler_state_exists":
        (
            stage_d_lr5e5_final_checkpoint
            / "scheduler.pt"
        ).exists(),
}


# --------------------------------------------------------------------------------------------------
# 11. Detect unintended intermediate checkpoints
# --------------------------------------------------------------------------------------------------

all_lr5e5_checkpoints = sorted(
    [
        p.name
        for p
        in STAGE_D_LR5E5_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


# --------------------------------------------------------------------------------------------------
# 12. GPU state
# --------------------------------------------------------------------------------------------------

peak_gpu_gb = None


if torch.cuda.is_available():

    peak_gpu_gb = (
        torch.cuda.max_memory_allocated()
        /
        (1024 ** 3)
    )


# --------------------------------------------------------------------------------------------------
# 13. Formal post-training checks
# --------------------------------------------------------------------------------------------------

stage_d_lr5e5_training_checks = {

    "training_completed_500_steps":
        final_global_step
        == 500,

    "training_loss_finite":
        np.isfinite(
            final_training_loss
        ),

    "training_log_nonempty":
        len(
            stage_d_lr5e5_log_df
        )
        > 0,

    "final_eval_triggered":
        stage_d_lr5e5_callback.eval_triggered
        is True,

    "final_save_triggered":
        stage_d_lr5e5_callback.save_triggered
        is True,

    "exactly_one_eval":
        len(
            final_eval_rows
        )
        == 1,

    "eval_at_step_500":
        final_eval_step
        == 500,

    "eval_loss_finite":
        (
            final_eval_loss
            is not None
            and
            np.isfinite(
                final_eval_loss
            )
        ),

    "only_checkpoint_500":
        all_lr5e5_checkpoints
        ==
        [
            "checkpoint-500"
        ],

    "checkpoint_500_complete":
        all(
            checkpoint_500_checks.values()
        ),

    "lr_100_observed":
        lr_step_100
        is not None,

    "lr_200_observed":
        lr_step_200
        is not None,

    "lr_300_observed":
        lr_step_300
        is not None,

    "lr_400_observed":
        lr_step_400
        is not None,

    "lr_500_observed":
        lr_step_500
        is not None,

    "lr_positive_before_end":
        (
            lr_step_100 is not None
            and
            lr_step_200 is not None
            and
            lr_step_300 is not None
            and
            lr_step_400 is not None
            and
            lr_step_100 > 0
            and
            lr_step_200 > 0
            and
            lr_step_300 > 0
            and
            lr_step_400 > 0
        ),

    "scheduler_reaches_end_at_500":
        final_scheduler_epoch
        == 500,

    "historical_validation_not_used":
        True,

    "future_not_used":
        True,
}


all_lr5e5_training_checks_pass = bool(
    all(
        stage_d_lr5e5_training_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 14. Save summary
# --------------------------------------------------------------------------------------------------

stage_d_lr5e5_training_summary = {

    "day":
        23,

    "stage":
        "D",

    "condition":
        "lr_5e-5",

    "sampling":
        "natural",

    "learning_rate":
        5e-5,

    "training": {

        "final_global_step":
            final_global_step,

        "training_loss":
            final_training_loss,

        "runtime_seconds":
            float(
                training_runtime_seconds
            ),

        "runtime_minutes":
            float(
                training_runtime_minutes
            ),

        "peak_gpu_gb":
            (
                float(
                    peak_gpu_gb
                )
                if peak_gpu_gb
                is not None
                else None
            ),
    },

    "evaluation": {

        "split":
            "optimization_dev",

        "step":
            final_eval_step,

        "eval_loss":
            final_eval_loss,

        "eval_runtime":
            final_eval_runtime,
    },

    "learning_rate_trajectory": {

        "step_100":
            lr_step_100,

        "step_200":
            lr_step_200,

        "step_300":
            lr_step_300,

        "step_400":
            lr_step_400,

        "step_500":
            lr_step_500,

        "final_scheduler_epoch":
            final_scheduler_epoch,

        "final_scheduler_lr":
            final_scheduler_lr,
    },

    "checkpoint": {

        "directory":
            str(
                stage_d_lr5e5_final_checkpoint
            ),

        "all_checkpoint_dirs":
            all_lr5e5_checkpoints,

        "checks": {

            key:
                bool(
                    value
                )

            for key, value
            in checkpoint_500_checks.items()
        },
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in stage_d_lr5e5_training_checks.items()
    },

    "all_checks_pass":
        all_lr5e5_training_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_D_LR5E5_TRAINING_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_d_lr5e5_training_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 15. Completion marker
# --------------------------------------------------------------------------------------------------

if all_lr5e5_training_checks_pass:

    completion_payload = {

        "status":
            "PASS",

        "condition":
            "lr_5e-5",

        "sampling":
            "natural",

        "learning_rate":
            5e-5,

        "global_step":
            final_global_step,

        "checkpoint":
            str(
                stage_d_lr5e5_final_checkpoint
            ),

        "created_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


    with open(
        STAGE_D_LR5E5_TRAINING_COMPLETE_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            completion_payload,
            f,
            indent=2,
            ensure_ascii=False,
        )


# --------------------------------------------------------------------------------------------------
# 16. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE D LR=5E-5 500-STEP TRAINING")
print("=" * 120)


print("\n[TRAINING]")
print(
    f"Final global step           : "
    f"{final_global_step}"
)
print(
    f"Final training loss         : "
    f"{final_training_loss:.6f}"
)
print(
    f"Runtime                     : "
    f"{training_runtime_minutes:.2f} minutes"
)

if peak_gpu_gb is not None:

    print(
        f"Peak GPU allocation         : "
        f"{peak_gpu_gb:.2f} GB"
    )


print("\n[FINAL DEV EVALUATION]")
print(
    f"Evaluation step             : "
    f"{final_eval_step}"
)
print(
    f"Evaluation loss             : "
    f"{final_eval_loss:.6f}"
)
print(
    f"Evaluation runtime          : "
    f"{final_eval_runtime:.2f} sec"
)


print("\n[LEARNING-RATE TRAJECTORY]")
print(
    f"Step 100                    : "
    f"{lr_step_100}"
)
print(
    f"Step 200                    : "
    f"{lr_step_200}"
)
print(
    f"Step 300                    : "
    f"{lr_step_300}"
)
print(
    f"Step 400                    : "
    f"{lr_step_400}"
)
print(
    f"Step 500                    : "
    f"{lr_step_500}"
)
print(
    f"Final scheduler epoch       : "
    f"{final_scheduler_epoch}"
)
print(
    f"Final scheduler LR          : "
    f"{final_scheduler_lr}"
)


print("\n[CALLBACK]")
print(
    f"Final eval triggered        : "
    f"{stage_d_lr5e5_callback.eval_triggered}"
)
print(
    f"Final save triggered        : "
    f"{stage_d_lr5e5_callback.save_triggered}"
)


print("\n[CHECKPOINT]")
print(
    f"Checkpoint directory        : "
    f"{stage_d_lr5e5_final_checkpoint}"
)
print(
    f"All checkpoints             : "
    f"{all_lr5e5_checkpoints}"
)

for name, passed in checkpoint_500_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[POST-TRAINING CHECKS]")

for name, passed in stage_d_lr5e5_training_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Training log                : "
    f"{STAGE_D_LR5E5_TRAINING_LOG_PATH}"
)
print(
    f"Training summary            : "
    f"{STAGE_D_LR5E5_TRAINING_SUMMARY_PATH}"
)
print(
    f"Completion marker           : "
    f"{STAGE_D_LR5E5_TRAINING_COMPLETE_PATH}"
)


print("\n" + "-" * 120)

if all_lr5e5_training_checks_pass:

    print(
        "DAY 23 CELL 10 STATUS: PASS — "
        "LR=5E-5 500-STEP TRAINING COMPLETE; "
        "CLASSIFICATION EVALUATION PENDING"
    )

else:

    print(
        "DAY 23 CELL 10 STATUS: FAIL — "
        "DO NOT EVALUATE OR PROCEED"
    )

print("=" * 120)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


DAY 23 — STAGE D LR=5E-5 500-STEP TRAINING

[TRAINING]
Final global step           : 500
Final training loss         : 0.555189
Runtime                     : 57.59 minutes
Peak GPU allocation         : 8.81 GB

[FINAL DEV EVALUATION]
Evaluation step             : 500
Evaluation loss             : 0.240016
Evaluation runtime          : 499.87 sec

[LEARNING-RATE TRAJECTORY]
Step 100                    : 4.091836734693878e-05
Step 200                    : 3.071428571428572e-05
Step 300                    : 2.0510204081632654e-05
Step 400                    : 1.0306122448979593e-05
Step 500                    : 1.0204081632653062e-07
Final scheduler epoch       : 500
Final scheduler LR          : [0.0, 0.0]

[CALLBACK]
Final eval triggered        : True
Final save triggered        : True

[CHECKPOINT]
Checkpoint directory        : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day23/stage_d_learning_rate_audit/runs/lr_5e-5/model/checkpoints/checkpoint-500
All checkpoints            

In [19]:
# ==================================================================================================
# DAY 23 — CELL 11
# STAGE D — LR = 5e-5
# OPTIMIZATION DEV CLASSIFICATION INFERENCE + COMPARISON VS LR=1e-4 INCUMBENT
# ==================================================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

from tqdm.auto import tqdm
from transformers.utils import logging as hf_logging


# --------------------------------------------------------------------------------------------------
# 1. Formal output paths
# --------------------------------------------------------------------------------------------------

STAGE_D_LR5E5_PREDICTIONS_PATH = (
    STAGE_D_LR5E5_ARTIFACT_DIR
    / "lr_5e-5_optimization_dev_predictions.csv"
)

STAGE_D_LR5E5_METRICS_PATH = (
    STAGE_D_LR5E5_ARTIFACT_DIR
    / "lr_5e-5_optimization_dev_metrics.json"
)

STAGE_D_LR5E5_CLASS_METRICS_PATH = (
    STAGE_D_LR5E5_ARTIFACT_DIR
    / "lr_5e-5_optimization_dev_class_metrics.csv"
)

STAGE_D_LR5E5_CONFUSION_PATH = (
    STAGE_D_LR5E5_ARTIFACT_DIR
    / "lr_5e-5_optimization_dev_confusion_matrix.csv"
)


# --------------------------------------------------------------------------------------------------
# 2. Safety gate
# --------------------------------------------------------------------------------------------------

assert STAGE_D_LR5E5_TRAINING_COMPLETE_PATH.exists(), (
    "DAY 23 STOP: LR=5e-5 training completion marker missing."
)


assert int(
    stage_d_lr5e5_trainer.state.global_step
) == 500, (
    "DAY 23 STOP: LR=5e-5 trainer is not at step 500."
)


assert len(
    optimization_dev
) == 2708


# --------------------------------------------------------------------------------------------------
# 3. Frozen inference protocol
# --------------------------------------------------------------------------------------------------

INFERENCE_BATCH_SIZE = 8

VALID_LABELS = [
    "negative",
    "neutral",
    "positive",
]


original_padding_side = (
    tokenizer.padding_side
)


tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token


stage_d_lr5e5_model.eval()

stage_d_lr5e5_model.config.use_cache = True


# --------------------------------------------------------------------------------------------------
# 4. Inference
# --------------------------------------------------------------------------------------------------

all_predictions = []
all_raw_generations = []


inference_start = time.time()


previous_hf_verbosity = (
    hf_logging.get_verbosity()
)

hf_logging.set_verbosity_error()


try:

    for batch_start in tqdm(
        range(
            0,
            len(optimization_dev),
            INFERENCE_BATCH_SIZE,
        ),
        desc="LR=5e-5 Dev inference",
    ):

        batch_end = min(
            batch_start
            +
            INFERENCE_BATCH_SIZE,

            len(
                optimization_dev
            ),
        )


        batch_df = (
            optimization_dev.iloc[
                batch_start:batch_end
            ]
        )


        batch_prompts = [
            build_inference_prompt(
                text
            )
            for text
            in batch_df[
                "text"
            ].tolist()
        ]


        encoded = tokenizer(
            batch_prompts,

            return_tensors=
                "pt",

            padding=
                True,

            truncation=
                True,

            max_length=
                MAX_SEQ_LENGTH,

            add_special_tokens=
                False,
        )


        encoded = {
            key:
                value.to(
                    stage_d_lr5e5_model.device
                )

            for key, value
            in encoded.items()
        }


        input_length = (
            encoded[
                "input_ids"
            ].shape[1]
        )


        with torch.inference_mode():

            generated = (
                stage_d_lr5e5_model.generate(

                    **encoded,

                    do_sample=
                        False,

                    max_new_tokens=
                        4,

                    pad_token_id=
                        tokenizer.pad_token_id,

                    eos_token_id=
                        tokenizer.eos_token_id,

                    use_cache=
                        True,
                )
            )


        generated_only = (
            generated[
                :,
                input_length:
            ]
        )


        decoded = tokenizer.batch_decode(

            generated_only,

            skip_special_tokens=
                True,

            clean_up_tokenization_spaces=
                False,
        )


        for raw_text in decoded:

            normalized = (
                raw_text
                .strip()
                .lower()
            )


            all_raw_generations.append(
                raw_text
            )


            if normalized in VALID_LABELS:

                all_predictions.append(
                    normalized
                )

            else:

                all_predictions.append(
                    "__invalid__"
                )


finally:

    hf_logging.set_verbosity(
        previous_hf_verbosity
    )

    tokenizer.padding_side = (
        original_padding_side
    )


inference_runtime_seconds = (
    time.time()
    -
    inference_start
)


# --------------------------------------------------------------------------------------------------
# 5. True / predicted labels
# --------------------------------------------------------------------------------------------------

y_true = (
    optimization_dev[
        "sentiment"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .tolist()
)


y_pred = (
    all_predictions
)


assert len(
    y_true
) == 2708


assert len(
    y_pred
) == 2708


invalid_prediction_count = int(
    sum(
        pred
        ==
        "__invalid__"

        for pred
        in y_pred
    )
)


invalid_prediction_rate = (
    invalid_prediction_count
    /
    len(
        y_pred
    )
)


# --------------------------------------------------------------------------------------------------
# 6. Overall metrics
# --------------------------------------------------------------------------------------------------

accuracy = float(
    accuracy_score(
        y_true,
        y_pred,
    )
)


macro_f1 = float(
    f1_score(
        y_true,
        y_pred,

        labels=
            VALID_LABELS,

        average=
            "macro",

        zero_division=
            0,
    )
)


weighted_f1 = float(
    f1_score(
        y_true,
        y_pred,

        labels=
            VALID_LABELS,

        average=
            "weighted",

        zero_division=
            0,
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Class-level metrics
# --------------------------------------------------------------------------------------------------

(
    class_precision,
    class_recall,
    class_f1,
    class_support,
) = precision_recall_fscore_support(

    y_true,
    y_pred,

    labels=
        VALID_LABELS,

    zero_division=
        0,
)


class_metrics_df = pd.DataFrame(
    {
        "label":
            VALID_LABELS,

        "precision":
            class_precision,

        "recall":
            class_recall,

        "f1":
            class_f1,

        "support":
            class_support.astype(
                int
            ),
    }
)


negative_recall = float(
    class_metrics_df.loc[
        class_metrics_df[
            "label"
        ]
        ==
        "negative",

        "recall",
    ].iloc[0]
)


neutral_recall = float(
    class_metrics_df.loc[
        class_metrics_df[
            "label"
        ]
        ==
        "neutral",

        "recall",
    ].iloc[0]
)


positive_recall = float(
    class_metrics_df.loc[
        class_metrics_df[
            "label"
        ]
        ==
        "positive",

        "recall",
    ].iloc[0]
)


minority_recall_floor = float(
    min(
        negative_recall,
        positive_recall,
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Confusion matrix
# --------------------------------------------------------------------------------------------------

cm = confusion_matrix(

    y_true,
    y_pred,

    labels=
        VALID_LABELS,
)


confusion_df = pd.DataFrame(

    cm,

    index=[
        f"true_{label}"
        for label
        in VALID_LABELS
    ],

    columns=[
        f"pred_{label}"
        for label
        in VALID_LABELS
    ],
)


# --------------------------------------------------------------------------------------------------
# 9. Prediction artifact
# --------------------------------------------------------------------------------------------------

prediction_df = pd.DataFrame(
    {
        "observation_id":
            optimization_dev[
                "observation_id"
            ].tolist(),

        "date":
            optimization_dev[
                "date"
            ].astype(str).tolist(),

        "true_label":
            y_true,

        "predicted_label":
            y_pred,

        "raw_generation":
            all_raw_generations,
    }
)


prediction_df.to_csv(
    STAGE_D_LR5E5_PREDICTIONS_PATH,
    index=False,
)


class_metrics_df.to_csv(
    STAGE_D_LR5E5_CLASS_METRICS_PATH,
    index=False,
)


confusion_df.to_csv(
    STAGE_D_LR5E5_CONFUSION_PATH
)


# --------------------------------------------------------------------------------------------------
# 10. Compare with LR=1e-4 Stage B-500 incumbent
# --------------------------------------------------------------------------------------------------

LR1E4_ACCURACY = (
    STAGE_C_WINNER_ACCURACY
)

LR1E4_MACRO_F1 = (
    STAGE_C_WINNER_MACRO_F1
)

LR1E4_NEG_RECALL = (
    STAGE_C_WINNER_NEG_RECALL
)

LR1E4_NEU_RECALL = (
    STAGE_C_WINNER_NEU_RECALL
)

LR1E4_POS_RECALL = (
    STAGE_C_WINNER_POS_RECALL
)

LR1E4_MINORITY_FLOOR = min(
    LR1E4_NEG_RECALL,
    LR1E4_POS_RECALL,
)


accuracy_delta_vs_1e4 = (
    accuracy
    -
    LR1E4_ACCURACY
)


macro_f1_delta_vs_1e4 = (
    macro_f1
    -
    LR1E4_MACRO_F1
)


negative_recall_delta_vs_1e4 = (
    negative_recall
    -
    LR1E4_NEG_RECALL
)


neutral_recall_delta_vs_1e4 = (
    neutral_recall
    -
    LR1E4_NEU_RECALL
)


positive_recall_delta_vs_1e4 = (
    positive_recall
    -
    LR1E4_POS_RECALL
)


minority_floor_delta_vs_1e4 = (
    minority_recall_floor
    -
    LR1E4_MINORITY_FLOOR
)


beats_lr1e4_accuracy = bool(
    accuracy
    >
    LR1E4_ACCURACY
)


# --------------------------------------------------------------------------------------------------
# 11. Formal checks
# --------------------------------------------------------------------------------------------------

cell11_checks = {

    "dev_population_2708":
        len(
            y_true
        )
        == 2708,

    "prediction_count_2708":
        len(
            y_pred
        )
        == 2708,

    "zero_invalid_predictions":
        invalid_prediction_count
        == 0,

    "accuracy_finite":
        np.isfinite(
            accuracy
        ),

    "macro_f1_finite":
        np.isfinite(
            macro_f1
        ),

    "weighted_f1_finite":
        np.isfinite(
            weighted_f1
        ),

    "negative_recall_finite":
        np.isfinite(
            negative_recall
        ),

    "neutral_recall_finite":
        np.isfinite(
            neutral_recall
        ),

    "positive_recall_finite":
        np.isfinite(
            positive_recall
        ),

    "confusion_total_2708":
        int(
            cm.sum()
        )
        == 2708,

    "prediction_artifact_saved":
        STAGE_D_LR5E5_PREDICTIONS_PATH.exists(),

    "class_metrics_saved":
        STAGE_D_LR5E5_CLASS_METRICS_PATH.exists(),

    "confusion_matrix_saved":
        STAGE_D_LR5E5_CONFUSION_PATH.exists(),

    "historical_validation_not_used":
        True,

    "future_not_used":
        True,
}


all_cell11_checks_pass = bool(
    all(
        cell11_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. Save metrics JSON
# --------------------------------------------------------------------------------------------------

stage_d_lr5e5_metrics = {

    "day":
        23,

    "stage":
        "D",

    "condition":
        "lr_5e-5",

    "learning_rate":
        5e-5,

    "sampling":
        "natural",

    "evaluation_population":
        "optimization_dev",

    "n":
        len(
            y_true
        ),

    "metrics": {

        "accuracy":
            accuracy,

        "macro_f1":
            macro_f1,

        "weighted_f1":
            weighted_f1,

        "negative_recall":
            negative_recall,

        "neutral_recall":
            neutral_recall,

        "positive_recall":
            positive_recall,

        "minority_recall_floor":
            minority_recall_floor,

        "invalid_predictions":
            invalid_prediction_count,

        "invalid_prediction_rate":
            invalid_prediction_rate,
    },

    "comparison_vs_lr_1e-4": {

        "reference":
            "Day22 Stage B-500 natural sampling",

        "reference_accuracy":
            LR1E4_ACCURACY,

        "accuracy_delta":
            float(
                accuracy_delta_vs_1e4
            ),

        "macro_f1_delta":
            float(
                macro_f1_delta_vs_1e4
            ),

        "negative_recall_delta":
            float(
                negative_recall_delta_vs_1e4
            ),

        "neutral_recall_delta":
            float(
                neutral_recall_delta_vs_1e4
            ),

        "positive_recall_delta":
            float(
                positive_recall_delta_vs_1e4
            ),

        "minority_recall_floor_delta":
            float(
                minority_floor_delta_vs_1e4
            ),

        "beats_reference_accuracy":
            beats_lr1e4_accuracy,
    },

    "inference": {

        "batch_size":
            INFERENCE_BATCH_SIZE,

        "do_sample":
            False,

        "max_new_tokens":
            4,

        "padding_side":
            "left",

        "runtime_seconds":
            float(
                inference_runtime_seconds
            ),
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in cell11_checks.items()
    },

    "all_checks_pass":
        all_cell11_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_D_LR5E5_METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_d_lr5e5_metrics,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 13. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE D LR=5E-5 OPTIMIZATION DEV CLASSIFICATION")
print("=" * 120)


print("\n[EVALUATION POPULATION]")
print(
    "Split                       : Optimization Dev"
)
print(
    f"N                           : "
    f"{len(y_true):,}"
)
print(
    f"Inference runtime           : "
    f"{inference_runtime_seconds / 60:.2f} minutes"
)
print(
    f"Invalid predictions         : "
    f"{invalid_prediction_count}"
)


print("\n[OVERALL METRICS]")
print(
    f"Accuracy                    : "
    f"{accuracy:.6f}"
)
print(
    f"Macro-F1                    : "
    f"{macro_f1:.6f}"
)
print(
    f"Weighted-F1                 : "
    f"{weighted_f1:.6f}"
)


print("\n[CLASS RECALL]")
print(
    f"Negative recall             : "
    f"{negative_recall:.6f}"
)
print(
    f"Neutral recall              : "
    f"{neutral_recall:.6f}"
)
print(
    f"Positive recall             : "
    f"{positive_recall:.6f}"
)
print(
    f"Minority recall floor       : "
    f"{minority_recall_floor:.6f}"
)


print("\n[CLASS METRICS]")
display(
    class_metrics_df
)


print("\n[CONFUSION MATRIX]")
display(
    confusion_df
)


print("\n[COMPARISON VS LR=1E-4 INCUMBENT]")
print(
    f"LR=1e-4 Accuracy            : "
    f"{LR1E4_ACCURACY:.6f}"
)
print(
    f"LR=5e-5 Accuracy            : "
    f"{accuracy:.6f}"
)
print(
    f"Accuracy delta              : "
    f"{accuracy_delta_vs_1e4:+.6f} "
    f"({accuracy_delta_vs_1e4 * 100:+.2f} pp)"
)
print(
    f"Macro-F1 delta              : "
    f"{macro_f1_delta_vs_1e4:+.6f}"
)
print(
    f"Negative recall delta       : "
    f"{negative_recall_delta_vs_1e4:+.6f}"
)
print(
    f"Neutral recall delta        : "
    f"{neutral_recall_delta_vs_1e4:+.6f}"
)
print(
    f"Positive recall delta       : "
    f"{positive_recall_delta_vs_1e4:+.6f}"
)
print(
    f"Minority floor delta        : "
    f"{minority_floor_delta_vs_1e4:+.6f}"
)
print(
    f"Beats LR=1e-4 Accuracy      : "
    f"{beats_lr1e4_accuracy}"
)


print("\n[FORMAL CHECKS]")

for name, passed in cell11_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Predictions                 : "
    f"{STAGE_D_LR5E5_PREDICTIONS_PATH}"
)
print(
    f"Metrics                     : "
    f"{STAGE_D_LR5E5_METRICS_PATH}"
)
print(
    f"Class metrics               : "
    f"{STAGE_D_LR5E5_CLASS_METRICS_PATH}"
)
print(
    f"Confusion matrix            : "
    f"{STAGE_D_LR5E5_CONFUSION_PATH}"
)


print("\n" + "-" * 120)

if all_cell11_checks_pass:

    print(
        "DAY 23 CELL 11 STATUS: PASS — "
        "LR=5E-5 CLASSIFICATION EVALUATION COMPLETE; "
        "LR=2E-4 CANDIDATE STILL PENDING"
    )

else:

    print(
        "DAY 23 CELL 11 STATUS: FAIL — "
        "DO NOT PROCEED TO LR=2E-4"
    )

print("=" * 120)

LR=5e-5 Dev inference:   0%|          | 0/339 [00:00<?, ?it/s]

DAY 23 — STAGE D LR=5E-5 OPTIMIZATION DEV CLASSIFICATION

[EVALUATION POPULATION]
Split                       : Optimization Dev
N                           : 2,708
Inference runtime           : 6.09 minutes
Invalid predictions         : 0

[OVERALL METRICS]
Accuracy                    : 0.818316
Macro-F1                    : 0.778698
Weighted-F1                 : 0.816447

[CLASS RECALL]
Negative recall             : 0.671937
Neutral recall              : 0.885560
Positive recall             : 0.746717
Minority recall floor       : 0.671937

[CLASS METRICS]


,label,precision,recall,f1,support
0,negative,0.783410,0.671937,0.723404,506
1,neutral,0.849914,0.885560,0.867371,1669
2,positive,0.743925,0.746717,0.745318,533



[CONFUSION MATRIX]


,pred_negative,pred_neutral,pred_positive
true_negative,340,144,22
true_neutral,76,1478,115
true_positive,18,117,398



[COMPARISON VS LR=1E-4 INCUMBENT]
LR=1e-4 Accuracy            : 0.840842
LR=5e-5 Accuracy            : 0.818316
Accuracy delta              : -0.022526 (-2.25 pp)
Macro-F1 delta              : -0.029503
Negative recall delta       : -0.049407
Neutral recall delta        : -0.004194
Positive recall delta       : -0.054409
Minority floor delta        : -0.049407
Beats LR=1e-4 Accuracy      : False

[FORMAL CHECKS]
PASS   — dev_population_2708
PASS   — prediction_count_2708
PASS   — zero_invalid_predictions
PASS   — accuracy_finite
PASS   — macro_f1_finite
PASS   — weighted_f1_finite
PASS   — negative_recall_finite
PASS   — neutral_recall_finite
PASS   — positive_recall_finite
PASS   — confusion_total_2708
PASS   — prediction_artifact_saved
PASS   — class_metrics_saved
PASS   — confusion_matrix_saved
PASS   — historical_validation_not_used
PASS   — future_not_used

[ARTIFACTS]
Predictions                 : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day23/stage_d_learning_rate_a

In [20]:
# ==================================================================================================
# DAY 23 — CELL 12
# STAGE D — LR = 2e-4
# FRESH MODEL + NATURAL-SAMPLING TRAINER INFRASTRUCTURE
# ==================================================================================================

import gc
import json
import random
from pathlib import Path

import numpy as np
import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


# --------------------------------------------------------------------------------------------------
# 1. Candidate identity + paths
# --------------------------------------------------------------------------------------------------

STAGE_D_CURRENT_LR = 2e-4

STAGE_D_LR2E4_ARTIFACT_DIR = (
    STAGE_D_LR2E4_DIR
    / "artifacts"
)

STAGE_D_LR2E4_MODEL_DIR = (
    STAGE_D_LR2E4_DIR
    / "model"
)

STAGE_D_LR2E4_CHECKPOINT_DIR = (
    STAGE_D_LR2E4_MODEL_DIR
    / "checkpoints"
)


for directory in [
    STAGE_D_LR2E4_ARTIFACT_DIR,
    STAGE_D_LR2E4_MODEL_DIR,
    STAGE_D_LR2E4_CHECKPOINT_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# --------------------------------------------------------------------------------------------------
# 2. Candidate directory must be clean
# --------------------------------------------------------------------------------------------------

existing_lr2e4_checkpoints = sorted(
    [
        p
        for p
        in STAGE_D_LR2E4_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


assert len(
    existing_lr2e4_checkpoints
) == 0, (
    "DAY 23 STOP: LR=2e-4 checkpoint directory is not clean."
)


# --------------------------------------------------------------------------------------------------
# 3. Release LR=5e-5 model/trainer GPU state
# --------------------------------------------------------------------------------------------------

objects_to_delete = [
    "stage_d_lr5e5_trainer",
    "stage_d_lr5e5_model",
    "stage_d_lr5e5_base_model",
]


for obj_name in objects_to_delete:

    if obj_name in globals():

        del globals()[
            obj_name
        ]


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()


gpu_after_cleanup_gb = (
    torch.cuda.memory_allocated()
    /
    (1024 ** 3)
)


# --------------------------------------------------------------------------------------------------
# 4. Deterministic seed
# --------------------------------------------------------------------------------------------------

random.seed(
    STAGE_D_SEED
)

np.random.seed(
    STAGE_D_SEED
)

torch.manual_seed(
    STAGE_D_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        STAGE_D_SEED
    )


set_seed(
    STAGE_D_SEED
)


# --------------------------------------------------------------------------------------------------
# 5. Fresh 4-bit base model
# --------------------------------------------------------------------------------------------------

stage_d_lr2e4_bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_use_double_quant=True,

    bnb_4bit_compute_dtype=torch.float16,
)


stage_d_lr2e4_base_model = (
    AutoModelForCausalLM.from_pretrained(

        STAGE_D_BASE_MODEL,

        quantization_config=
            stage_d_lr2e4_bnb_config,

        device_map=
            "auto",

        trust_remote_code=
            True,
    )
)


stage_d_lr2e4_base_model.config.use_cache = False


stage_d_lr2e4_base_model = (
    prepare_model_for_kbit_training(

        stage_d_lr2e4_base_model,

        use_gradient_checkpointing=True,
    )
)


# --------------------------------------------------------------------------------------------------
# 6. Fresh LoRA
# --------------------------------------------------------------------------------------------------

set_seed(
    STAGE_D_SEED
)


stage_d_lr2e4_lora_config = LoraConfig(

    r=
        STAGE_D_LORA_R,

    lora_alpha=
        STAGE_D_LORA_ALPHA,

    lora_dropout=
        STAGE_D_LORA_DROPOUT,

    target_modules=
        STAGE_D_LORA_TARGET_MODULES,

    bias=
        "none",

    task_type=
        "CAUSAL_LM",

    inference_mode=
        False,
)


stage_d_lr2e4_model = (
    get_peft_model(
        stage_d_lr2e4_base_model,
        stage_d_lr2e4_lora_config,
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Final-only evaluation/save callback
# --------------------------------------------------------------------------------------------------

class StageDLR2E4FinalStepCallback(
    TrainerCallback
):

    def __init__(
        self,
        final_step=500,
    ):

        self.final_step = int(
            final_step
        )

        self.eval_triggered = False
        self.save_triggered = False


    def on_step_end(
        self,
        args,
        state,
        control,
        **kwargs,
    ):

        control.should_evaluate = False
        control.should_save = False


        if int(
            state.global_step
        ) == self.final_step:

            control.should_evaluate = True
            control.should_save = True

            self.eval_triggered = True
            self.save_triggered = True


        return control


stage_d_lr2e4_callback = (
    StageDLR2E4FinalStepCallback(
        final_step=
            STAGE_D_MAX_STEPS
    )
)


# --------------------------------------------------------------------------------------------------
# 8. TrainingArguments
#
# Natural sampling:
# standard Trainer behavior, no weighted sampler.
# --------------------------------------------------------------------------------------------------

stage_d_lr2e4_training_args = TrainingArguments(

    output_dir=
        str(
            STAGE_D_LR2E4_CHECKPOINT_DIR
        ),

    max_steps=
        STAGE_D_MAX_STEPS,

    per_device_train_batch_size=
        STAGE_D_TRAIN_BATCH_SIZE,

    per_device_eval_batch_size=
        1,

    gradient_accumulation_steps=
        STAGE_D_GRADIENT_ACCUMULATION,

    learning_rate=
        STAGE_D_CURRENT_LR,

    lr_scheduler_type=
        STAGE_D_SCHEDULER,

    warmup_steps=
        STAGE_D_WARMUP_STEPS,

    weight_decay=
        0.0,

    logging_strategy=
        "steps",

    logging_steps=
        10,

    eval_strategy=
        "no",

    save_strategy=
        "no",

    load_best_model_at_end=
        False,

    fp16=
        True,

    bf16=
        False,

    gradient_checkpointing=
        True,

    seed=
        STAGE_D_SEED,

    data_seed=
        STAGE_D_SEED,

    report_to=
        [],

    remove_unused_columns=
        True,

    label_names=[
        "labels"
    ],

    disable_tqdm=
        False,
)


# --------------------------------------------------------------------------------------------------
# 9. Standard natural-sampling Trainer
# --------------------------------------------------------------------------------------------------

stage_d_lr2e4_trainer = Trainer(

    model=
        stage_d_lr2e4_model,

    args=
        stage_d_lr2e4_training_args,

    train_dataset=
        hf_opt_train_model,

    eval_dataset=
        hf_opt_dev_model,

    data_collator=
        data_collator,

    processing_class=
        tokenizer,

    callbacks=[
        stage_d_lr2e4_callback
    ],
)


# --------------------------------------------------------------------------------------------------
# 10. Trainable parameter audit
# --------------------------------------------------------------------------------------------------

stage_d_lr2e4_total_params = 0
stage_d_lr2e4_trainable_params = 0
stage_d_lr2e4_trainable_names = []


for name, param in stage_d_lr2e4_model.named_parameters():

    stage_d_lr2e4_total_params += (
        param.numel()
    )

    if param.requires_grad:

        stage_d_lr2e4_trainable_params += (
            param.numel()
        )

        stage_d_lr2e4_trainable_names.append(
            name
        )


stage_d_lr2e4_unexpected_trainable = [

    name

    for name
    in stage_d_lr2e4_trainable_names

    if "lora_" not in name
]


# --------------------------------------------------------------------------------------------------
# 11. Natural-sampling integrity
# --------------------------------------------------------------------------------------------------

natural_sampling_checks = {

    "trainer_is_standard_trainer":
        type(
            stage_d_lr2e4_trainer
        )
        is Trainer,

    "no_weighted_sampler":
        True,

    "train_dataset_identity":
        stage_d_lr2e4_trainer.train_dataset
        is hf_opt_train_model,

    "train_population_10831":
        len(
            stage_d_lr2e4_trainer.train_dataset
        )
        == 10831,
}


# --------------------------------------------------------------------------------------------------
# 12. Formal infrastructure checks
# --------------------------------------------------------------------------------------------------

stage_d_lr2e4_checks = {

    "checkpoint_dir_clean":
        len(
            existing_lr2e4_checkpoints
        )
        == 0,

    "gpu_cleanup_completed":
        gpu_after_cleanup_gb
        <
        3.0,

    "fresh_model_loaded":
        stage_d_lr2e4_model
        is not None,

    "natural_sampling":
        all(
            natural_sampling_checks.values()
        ),

    "train_n_10831":
        len(
            hf_opt_train_model
        )
        == 10831,

    "dev_n_2708":
        len(
            hf_opt_dev_model
        )
        == 2708,

    "max_steps_500":
        stage_d_lr2e4_training_args.max_steps
        == 500,

    "lr_2e4":
        abs(
            stage_d_lr2e4_training_args.learning_rate
            -
            2e-4
        )
        <
        1e-12,

    "gradient_accumulation_8":
        (
            stage_d_lr2e4_training_args
            .gradient_accumulation_steps
            ==
            8
        ),

    "scheduler_linear":
        str(
            stage_d_lr2e4_training_args.lr_scheduler_type
        )
        .lower()
        .endswith(
            "linear"
        ),

    "warmup_10":
        stage_d_lr2e4_training_args.warmup_steps
        == 10,

    "seed_42":
        stage_d_lr2e4_training_args.seed
        == 42,

    "lora_r8":
        (
            stage_d_lr2e4_model
            .peft_config[
                "default"
            ]
            .r
            ==
            8
        ),

    "lora_alpha32":
        (
            stage_d_lr2e4_model
            .peft_config[
                "default"
            ]
            .lora_alpha
            ==
            32
        ),

    "trainable_params_3407872":
        stage_d_lr2e4_trainable_params
        ==
        3_407_872,

    "only_lora_trainable":
        len(
            stage_d_lr2e4_unexpected_trainable
        )
        == 0,

    "trainer_global_step_zero":
        int(
            stage_d_lr2e4_trainer
            .state
            .global_step
        )
        == 0,

    "eval_not_triggered":
        stage_d_lr2e4_callback.eval_triggered
        is False,

    "save_not_triggered":
        stage_d_lr2e4_callback.save_triggered
        is False,

    "historical_validation_locked":
        STAGE_D_HISTORICAL_VALIDATION_ALLOWED
        is False,

    "future_locked":
        STAGE_D_FUTURE_ALLOWED
        is False,
}


all_stage_d_lr2e4_checks_pass = bool(
    all(
        stage_d_lr2e4_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 13. Save infrastructure manifest
# --------------------------------------------------------------------------------------------------

STAGE_D_LR2E4_INFRA_PATH = (
    STAGE_D_LR2E4_ARTIFACT_DIR
    / "lr_2e-4_training_infrastructure.json"
)


stage_d_lr2e4_infra_payload = {

    "day":
        23,

    "stage":
        "D",

    "condition":
        "lr_2e-4",

    "learning_rate":
        STAGE_D_CURRENT_LR,

    "sampling":
        "natural",

    "initialization":
        "fresh_base_plus_fresh_lora",

    "base_model":
        STAGE_D_BASE_MODEL,

    "training": {

        "max_steps":
            STAGE_D_MAX_STEPS,

        "learning_rate":
            STAGE_D_CURRENT_LR,

        "scheduler":
            STAGE_D_SCHEDULER,

        "warmup_steps":
            STAGE_D_WARMUP_STEPS,

        "effective_batch":
            STAGE_D_EFFECTIVE_BATCH,

        "seed":
            STAGE_D_SEED,
    },

    "lora": {

        "r":
            STAGE_D_LORA_R,

        "alpha":
            STAGE_D_LORA_ALPHA,

        "dropout":
            STAGE_D_LORA_DROPOUT,

        "target_modules":
            STAGE_D_LORA_TARGET_MODULES,
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in stage_d_lr2e4_checks.items()
    },

    "all_checks_pass":
        all_stage_d_lr2e4_checks_pass,
}


with open(
    STAGE_D_LR2E4_INFRA_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_d_lr2e4_infra_payload,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 14. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE D LR=2E-4 TRAINER INFRASTRUCTURE")
print("=" * 120)


print("\n[GPU CLEANUP]")
print(
    f"Allocated after cleanup     : "
    f"{gpu_after_cleanup_gb:.2f} GB"
)


print("\n[CANDIDATE]")
print(
    "Learning rate               : 2e-4"
)
print(
    "Sampling                    : Natural"
)
print(
    "Initialization              : Fresh base + fresh LoRA"
)
print(
    "Previous adapters loaded    : False"
)


print("\n[TRAINING PROTOCOL]")
print(
    "Max steps                   : 500"
)
print(
    "Effective batch             : 8"
)
print(
    "Scheduler                   : linear"
)
print(
    "Warmup                      : 10"
)
print(
    "Seed                        : 42"
)
print(
    "Formal Dev evaluation       : step 500 only"
)


print("\n[PARAMETERS]")
print(
    f"Trainable parameters        : "
    f"{stage_d_lr2e4_trainable_params:,}"
)
print(
    f"Total parameters            : "
    f"{stage_d_lr2e4_total_params:,}"
)
print(
    f"Unexpected trainable        : "
    f"{len(stage_d_lr2e4_unexpected_trainable)}"
)


print("\n[PRE-TRAINING STATE]")
print(
    f"Global step                 : "
    f"{stage_d_lr2e4_trainer.state.global_step}"
)
print(
    f"Final eval triggered        : "
    f"{stage_d_lr2e4_callback.eval_triggered}"
)
print(
    f"Final save triggered        : "
    f"{stage_d_lr2e4_callback.save_triggered}"
)


print("\n[GPU AFTER MODEL LOAD]")
print(
    f"Allocated                   : "
    f"{torch.cuda.memory_allocated() / (1024 ** 3):.2f} GB"
)


print("\n[FORMAL CHECKS]")

for name, passed in stage_d_lr2e4_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACT]")
print(
    STAGE_D_LR2E4_INFRA_PATH
)


print("\n" + "-" * 120)

if all_stage_d_lr2e4_checks_pass:

    print(
        "DAY 23 CELL 12 STATUS: PASS — "
        "LR=2E-4 NATURAL-SAMPLING TRAINER READY; "
        "TRAINING NOT YET STARTED"
    )

else:

    print(
        "DAY 23 CELL 12 STATUS: FAIL — "
        "DO NOT START LR=2E-4 TRAINING"
    )

print("=" * 120)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

DAY 23 — STAGE D LR=2E-4 TRAINER INFRASTRUCTURE

[GPU CLEANUP]
Allocated after cleanup     : 1.97 GB

[CANDIDATE]
Learning rate               : 2e-4
Sampling                    : Natural
Initialization              : Fresh base + fresh LoRA
Previous adapters loaded    : False

[TRAINING PROTOCOL]
Max steps                   : 500
Effective batch             : 8
Scheduler                   : linear
Warmup                      : 10
Seed                        : 42
Formal Dev evaluation       : step 500 only

[PARAMETERS]
Trainable parameters        : 3,407,872
Total parameters            : 4,544,008,192
Unexpected trainable        : 0

[PRE-TRAINING STATE]
Global step                 : 0
Final eval triggered        : False
Final save triggered        : False

[GPU AFTER MODEL LOAD]
Allocated                   : 7.30 GB

[FORMAL CHECKS]
PASS   — checkpoint_dir_clean
PASS   — gpu_cleanup_completed
PASS   — fresh_model_loaded
PASS   — natural_sampling
PASS   — train_n_10831
PASS   — dev_n_2

In [21]:
# ==================================================================================================
# DAY 23 — CELL 13
# STAGE D — LR = 2e-4
# CONTROLLED 500-STEP NATURAL-SAMPLING TRAINING
# ==================================================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch


# --------------------------------------------------------------------------------------------------
# 1. Formal artifact paths
# --------------------------------------------------------------------------------------------------

STAGE_D_LR2E4_TRAINING_LOG_PATH = (
    STAGE_D_LR2E4_ARTIFACT_DIR
    / "lr_2e-4_training_log.csv"
)

STAGE_D_LR2E4_TRAINING_SUMMARY_PATH = (
    STAGE_D_LR2E4_ARTIFACT_DIR
    / "lr_2e-4_training_summary.json"
)

STAGE_D_LR2E4_TRAINING_COMPLETE_PATH = (
    STAGE_D_LR2E4_ARTIFACT_DIR
    / "lr_2e-4_training_complete.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Pre-training safety gate
# --------------------------------------------------------------------------------------------------

assert int(
    stage_d_lr2e4_trainer.state.global_step
) == 0, (
    "DAY 23 STOP: LR=2e-4 trainer already advanced beyond step 0."
)


assert (
    stage_d_lr2e4_callback.eval_triggered
    is False
), (
    "DAY 23 STOP: LR=2e-4 evaluation callback already triggered."
)


assert (
    stage_d_lr2e4_callback.save_triggered
    is False
), (
    "DAY 23 STOP: LR=2e-4 save callback already triggered."
)


existing_checkpoints_before_training = sorted(
    [
        p
        for p
        in STAGE_D_LR2E4_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


assert len(
    existing_checkpoints_before_training
) == 0, (
    "DAY 23 STOP: LR=2e-4 checkpoint directory is not clean."
)


assert not STAGE_D_LR2E4_TRAINING_COMPLETE_PATH.exists(), (
    "DAY 23 STOP: LR=2e-4 completion marker already exists."
)


# --------------------------------------------------------------------------------------------------
# 3. Reset deterministic seeds
# --------------------------------------------------------------------------------------------------

random.seed(
    STAGE_D_SEED
)

np.random.seed(
    STAGE_D_SEED
)

torch.manual_seed(
    STAGE_D_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        STAGE_D_SEED
    )

    torch.cuda.reset_peak_memory_stats()


set_seed(
    STAGE_D_SEED
)


# --------------------------------------------------------------------------------------------------
# 4. Train
# --------------------------------------------------------------------------------------------------

training_start_time = time.time()


stage_d_lr2e4_train_result = (
    stage_d_lr2e4_trainer.train()
)


training_end_time = time.time()


training_runtime_seconds = (
    training_end_time
    -
    training_start_time
)

training_runtime_minutes = (
    training_runtime_seconds
    /
    60.0
)


# --------------------------------------------------------------------------------------------------
# 5. Collect log history
# --------------------------------------------------------------------------------------------------

stage_d_lr2e4_log_history = (
    stage_d_lr2e4_trainer.state.log_history
)


stage_d_lr2e4_log_df = pd.DataFrame(
    stage_d_lr2e4_log_history
)


stage_d_lr2e4_log_df.to_csv(
    STAGE_D_LR2E4_TRAINING_LOG_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 6. Final state
# --------------------------------------------------------------------------------------------------

final_global_step = int(
    stage_d_lr2e4_trainer.state.global_step
)


final_training_loss = float(
    stage_d_lr2e4_train_result.training_loss
)


# --------------------------------------------------------------------------------------------------
# 7. Extract final Dev evaluation
# --------------------------------------------------------------------------------------------------

final_eval_rows = (
    stage_d_lr2e4_log_df.loc[
        stage_d_lr2e4_log_df[
            "eval_loss"
        ].notna()
    ]
    .copy()
)


if len(
    final_eval_rows
) > 0:

    final_eval_row = (
        final_eval_rows
        .sort_values(
            "step"
        )
        .iloc[-1]
    )

    final_eval_step = int(
        final_eval_row[
            "step"
        ]
    )

    final_eval_loss = float(
        final_eval_row[
            "eval_loss"
        ]
    )

    final_eval_runtime = float(
        final_eval_row.get(
            "eval_runtime",
            np.nan,
        )
    )

else:

    final_eval_step = None
    final_eval_loss = None
    final_eval_runtime = None


# --------------------------------------------------------------------------------------------------
# 8. Learning-rate trajectory
# --------------------------------------------------------------------------------------------------

lr_rows = (
    stage_d_lr2e4_log_df.loc[
        stage_d_lr2e4_log_df[
            "learning_rate"
        ].notna()
    ]
    .copy()
)


def get_lr2e4_logged_lr(
    step,
):

    rows = lr_rows.loc[
        lr_rows[
            "step"
        ]
        ==
        step
    ]


    if len(
        rows
    ) == 0:

        return None


    return float(
        rows.iloc[-1][
            "learning_rate"
        ]
    )


lr_step_100 = get_lr2e4_logged_lr(
    100
)

lr_step_200 = get_lr2e4_logged_lr(
    200
)

lr_step_300 = get_lr2e4_logged_lr(
    300
)

lr_step_400 = get_lr2e4_logged_lr(
    400
)

lr_step_500 = get_lr2e4_logged_lr(
    500
)


# --------------------------------------------------------------------------------------------------
# 9. Scheduler end-state
# --------------------------------------------------------------------------------------------------

final_scheduler_epoch = None
final_scheduler_lr = None


if (
    stage_d_lr2e4_trainer.lr_scheduler
    is not None
):

    final_scheduler_epoch = int(
        stage_d_lr2e4_trainer
        .lr_scheduler
        .last_epoch
    )

    final_scheduler_lr = [
        float(
            value
        )
        for value
        in stage_d_lr2e4_trainer
        .lr_scheduler
        .get_last_lr()
    ]


# --------------------------------------------------------------------------------------------------
# 10. Check final checkpoint
# --------------------------------------------------------------------------------------------------

stage_d_lr2e4_final_checkpoint = (
    STAGE_D_LR2E4_CHECKPOINT_DIR
    / "checkpoint-500"
)


checkpoint_500_checks = {

    "checkpoint_exists":
        stage_d_lr2e4_final_checkpoint.exists(),

    "adapter_config_exists":
        (
            stage_d_lr2e4_final_checkpoint
            / "adapter_config.json"
        ).exists(),

    "adapter_weights_exist":
        (
            (
                stage_d_lr2e4_final_checkpoint
                / "adapter_model.safetensors"
            ).exists()
            or
            (
                stage_d_lr2e4_final_checkpoint
                / "adapter_model.bin"
            ).exists()
        ),

    "trainer_state_exists":
        (
            stage_d_lr2e4_final_checkpoint
            / "trainer_state.json"
        ).exists(),

    "optimizer_state_exists":
        (
            stage_d_lr2e4_final_checkpoint
            / "optimizer.pt"
        ).exists(),

    "scheduler_state_exists":
        (
            stage_d_lr2e4_final_checkpoint
            / "scheduler.pt"
        ).exists(),
}


# --------------------------------------------------------------------------------------------------
# 11. Detect unintended intermediate checkpoints
# --------------------------------------------------------------------------------------------------

all_lr2e4_checkpoints = sorted(
    [
        p.name
        for p
        in STAGE_D_LR2E4_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


# --------------------------------------------------------------------------------------------------
# 12. GPU state
# --------------------------------------------------------------------------------------------------

peak_gpu_gb = None


if torch.cuda.is_available():

    peak_gpu_gb = (
        torch.cuda.max_memory_allocated()
        /
        (1024 ** 3)
    )


# --------------------------------------------------------------------------------------------------
# 13. Formal post-training checks
# --------------------------------------------------------------------------------------------------

stage_d_lr2e4_training_checks = {

    "training_completed_500_steps":
        final_global_step
        == 500,

    "training_loss_finite":
        np.isfinite(
            final_training_loss
        ),

    "training_log_nonempty":
        len(
            stage_d_lr2e4_log_df
        )
        > 0,

    "final_eval_triggered":
        stage_d_lr2e4_callback.eval_triggered
        is True,

    "final_save_triggered":
        stage_d_lr2e4_callback.save_triggered
        is True,

    "exactly_one_eval":
        len(
            final_eval_rows
        )
        == 1,

    "eval_at_step_500":
        final_eval_step
        == 500,

    "eval_loss_finite":
        (
            final_eval_loss
            is not None
            and
            np.isfinite(
                final_eval_loss
            )
        ),

    "only_checkpoint_500":
        all_lr2e4_checkpoints
        ==
        [
            "checkpoint-500"
        ],

    "checkpoint_500_complete":
        all(
            checkpoint_500_checks.values()
        ),

    "lr_100_observed":
        lr_step_100
        is not None,

    "lr_200_observed":
        lr_step_200
        is not None,

    "lr_300_observed":
        lr_step_300
        is not None,

    "lr_400_observed":
        lr_step_400
        is not None,

    "lr_500_observed":
        lr_step_500
        is not None,

    "lr_positive_before_end":
        (
            lr_step_100 is not None
            and
            lr_step_200 is not None
            and
            lr_step_300 is not None
            and
            lr_step_400 is not None
            and
            lr_step_100 > 0
            and
            lr_step_200 > 0
            and
            lr_step_300 > 0
            and
            lr_step_400 > 0
        ),

    "scheduler_reaches_end_at_500":
        final_scheduler_epoch
        == 500,

    "historical_validation_not_used":
        True,

    "future_not_used":
        True,
}


all_lr2e4_training_checks_pass = bool(
    all(
        stage_d_lr2e4_training_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 14. Save summary
# --------------------------------------------------------------------------------------------------

stage_d_lr2e4_training_summary = {

    "day":
        23,

    "stage":
        "D",

    "condition":
        "lr_2e-4",

    "sampling":
        "natural",

    "learning_rate":
        2e-4,

    "training": {

        "final_global_step":
            final_global_step,

        "training_loss":
            final_training_loss,

        "runtime_seconds":
            float(
                training_runtime_seconds
            ),

        "runtime_minutes":
            float(
                training_runtime_minutes
            ),

        "peak_gpu_gb":
            (
                float(
                    peak_gpu_gb
                )
                if peak_gpu_gb
                is not None
                else None
            ),
    },

    "evaluation": {

        "split":
            "optimization_dev",

        "step":
            final_eval_step,

        "eval_loss":
            final_eval_loss,

        "eval_runtime":
            final_eval_runtime,
    },

    "learning_rate_trajectory": {

        "step_100":
            lr_step_100,

        "step_200":
            lr_step_200,

        "step_300":
            lr_step_300,

        "step_400":
            lr_step_400,

        "step_500":
            lr_step_500,

        "final_scheduler_epoch":
            final_scheduler_epoch,

        "final_scheduler_lr":
            final_scheduler_lr,
    },

    "checkpoint": {

        "directory":
            str(
                stage_d_lr2e4_final_checkpoint
            ),

        "all_checkpoint_dirs":
            all_lr2e4_checkpoints,

        "checks": {

            key:
                bool(
                    value
                )

            for key, value
            in checkpoint_500_checks.items()
        },
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in stage_d_lr2e4_training_checks.items()
    },

    "all_checks_pass":
        all_lr2e4_training_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_D_LR2E4_TRAINING_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_d_lr2e4_training_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 15. Completion marker
# --------------------------------------------------------------------------------------------------

if all_lr2e4_training_checks_pass:

    completion_payload = {

        "status":
            "PASS",

        "condition":
            "lr_2e-4",

        "sampling":
            "natural",

        "learning_rate":
            2e-4,

        "global_step":
            final_global_step,

        "checkpoint":
            str(
                stage_d_lr2e4_final_checkpoint
            ),

        "created_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


    with open(
        STAGE_D_LR2E4_TRAINING_COMPLETE_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            completion_payload,
            f,
            indent=2,
            ensure_ascii=False,
        )


# --------------------------------------------------------------------------------------------------
# 16. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE D LR=2E-4 500-STEP TRAINING")
print("=" * 120)


print("\n[TRAINING]")
print(
    f"Final global step           : "
    f"{final_global_step}"
)
print(
    f"Final training loss         : "
    f"{final_training_loss:.6f}"
)
print(
    f"Runtime                     : "
    f"{training_runtime_minutes:.2f} minutes"
)

if peak_gpu_gb is not None:

    print(
        f"Peak GPU allocation         : "
        f"{peak_gpu_gb:.2f} GB"
    )


print("\n[FINAL DEV EVALUATION]")
print(
    f"Evaluation step             : "
    f"{final_eval_step}"
)
print(
    f"Evaluation loss             : "
    f"{final_eval_loss:.6f}"
)
print(
    f"Evaluation runtime          : "
    f"{final_eval_runtime:.2f} sec"
)


print("\n[LEARNING-RATE TRAJECTORY]")
print(
    f"Step 100                    : "
    f"{lr_step_100}"
)
print(
    f"Step 200                    : "
    f"{lr_step_200}"
)
print(
    f"Step 300                    : "
    f"{lr_step_300}"
)
print(
    f"Step 400                    : "
    f"{lr_step_400}"
)
print(
    f"Step 500                    : "
    f"{lr_step_500}"
)
print(
    f"Final scheduler epoch       : "
    f"{final_scheduler_epoch}"
)
print(
    f"Final scheduler LR          : "
    f"{final_scheduler_lr}"
)


print("\n[CALLBACK]")
print(
    f"Final eval triggered        : "
    f"{stage_d_lr2e4_callback.eval_triggered}"
)
print(
    f"Final save triggered        : "
    f"{stage_d_lr2e4_callback.save_triggered}"
)


print("\n[CHECKPOINT]")
print(
    f"Checkpoint directory        : "
    f"{stage_d_lr2e4_final_checkpoint}"
)
print(
    f"All checkpoints             : "
    f"{all_lr2e4_checkpoints}"
)

for name, passed in checkpoint_500_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[POST-TRAINING CHECKS]")

for name, passed in stage_d_lr2e4_training_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Training log                : "
    f"{STAGE_D_LR2E4_TRAINING_LOG_PATH}"
)
print(
    f"Training summary            : "
    f"{STAGE_D_LR2E4_TRAINING_SUMMARY_PATH}"
)
print(
    f"Completion marker           : "
    f"{STAGE_D_LR2E4_TRAINING_COMPLETE_PATH}"
)


print("\n" + "-" * 120)

if all_lr2e4_training_checks_pass:

    print(
        "DAY 23 CELL 13 STATUS: PASS — "
        "LR=2E-4 500-STEP TRAINING COMPLETE; "
        "CLASSIFICATION EVALUATION PENDING"
    )

else:

    print(
        "DAY 23 CELL 13 STATUS: FAIL — "
        "DO NOT EVALUATE OR PROCEED"
    )

print("=" * 120)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


DAY 23 — STAGE D LR=2E-4 500-STEP TRAINING

[TRAINING]
Final global step           : 500
Final training loss         : 0.402645
Runtime                     : 57.33 minutes
Peak GPU allocation         : 8.81 GB

[FINAL DEV EVALUATION]
Evaluation step             : 500
Evaluation loss             : 0.212594
Evaluation runtime          : 499.97 sec

[LEARNING-RATE TRAJECTORY]
Step 100                    : 0.00016367346938775512
Step 200                    : 0.00012285714285714287
Step 300                    : 8.204081632653062e-05
Step 400                    : 4.122448979591837e-05
Step 500                    : 4.081632653061225e-07
Final scheduler epoch       : 500
Final scheduler LR          : [0.0, 0.0]

[CALLBACK]
Final eval triggered        : True
Final save triggered        : True

[CHECKPOINT]
Checkpoint directory        : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day23/stage_d_learning_rate_audit/runs/lr_2e-4/model/checkpoints/checkpoint-500
All checkpoints             

In [22]:
# ==================================================================================================
# DAY 23 — CELL 14
# STAGE D — LR = 2e-4
# OPTIMIZATION DEV CLASSIFICATION INFERENCE + COMPARISON VS LR=1e-4 INCUMBENT
# ==================================================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

from tqdm.auto import tqdm
from transformers.utils import logging as hf_logging


# --------------------------------------------------------------------------------------------------
# 1. Formal output paths
# --------------------------------------------------------------------------------------------------

STAGE_D_LR2E4_PREDICTIONS_PATH = (
    STAGE_D_LR2E4_ARTIFACT_DIR
    / "lr_2e-4_optimization_dev_predictions.csv"
)

STAGE_D_LR2E4_METRICS_PATH = (
    STAGE_D_LR2E4_ARTIFACT_DIR
    / "lr_2e-4_optimization_dev_metrics.json"
)

STAGE_D_LR2E4_CLASS_METRICS_PATH = (
    STAGE_D_LR2E4_ARTIFACT_DIR
    / "lr_2e-4_optimization_dev_class_metrics.csv"
)

STAGE_D_LR2E4_CONFUSION_PATH = (
    STAGE_D_LR2E4_ARTIFACT_DIR
    / "lr_2e-4_optimization_dev_confusion_matrix.csv"
)


# --------------------------------------------------------------------------------------------------
# 2. Safety gate
# --------------------------------------------------------------------------------------------------

assert STAGE_D_LR2E4_TRAINING_COMPLETE_PATH.exists(), (
    "DAY 23 STOP: LR=2e-4 training completion marker missing."
)

assert int(
    stage_d_lr2e4_trainer.state.global_step
) == 500, (
    "DAY 23 STOP: LR=2e-4 trainer is not at step 500."
)

assert len(
    optimization_dev
) == 2708


# --------------------------------------------------------------------------------------------------
# 3. Frozen inference protocol
# --------------------------------------------------------------------------------------------------

INFERENCE_BATCH_SIZE = 8

VALID_LABELS = [
    "negative",
    "neutral",
    "positive",
]


original_padding_side = (
    tokenizer.padding_side
)

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token


stage_d_lr2e4_model.eval()
stage_d_lr2e4_model.config.use_cache = True


# --------------------------------------------------------------------------------------------------
# 4. Inference
# --------------------------------------------------------------------------------------------------

all_predictions = []
all_raw_generations = []


inference_start = time.time()


previous_hf_verbosity = (
    hf_logging.get_verbosity()
)

hf_logging.set_verbosity_error()


try:

    for batch_start in tqdm(
        range(
            0,
            len(optimization_dev),
            INFERENCE_BATCH_SIZE,
        ),
        desc="LR=2e-4 Dev inference",
    ):

        batch_end = min(
            batch_start
            + INFERENCE_BATCH_SIZE,

            len(
                optimization_dev
            ),
        )


        batch_df = (
            optimization_dev.iloc[
                batch_start:batch_end
            ]
        )


        batch_prompts = [
            build_inference_prompt(
                text
            )
            for text
            in batch_df[
                "text"
            ].tolist()
        ]


        encoded = tokenizer(
            batch_prompts,

            return_tensors=
                "pt",

            padding=
                True,

            truncation=
                True,

            max_length=
                MAX_SEQ_LENGTH,

            add_special_tokens=
                False,
        )


        encoded = {
            key:
                value.to(
                    stage_d_lr2e4_model.device
                )

            for key, value
            in encoded.items()
        }


        input_length = (
            encoded[
                "input_ids"
            ].shape[1]
        )


        with torch.inference_mode():

            generated = (
                stage_d_lr2e4_model.generate(

                    **encoded,

                    do_sample=
                        False,

                    max_new_tokens=
                        4,

                    pad_token_id=
                        tokenizer.pad_token_id,

                    eos_token_id=
                        tokenizer.eos_token_id,

                    use_cache=
                        True,
                )
            )


        generated_only = (
            generated[
                :,
                input_length:
            ]
        )


        decoded = tokenizer.batch_decode(

            generated_only,

            skip_special_tokens=
                True,

            clean_up_tokenization_spaces=
                False,
        )


        for raw_text in decoded:

            normalized = (
                raw_text
                .strip()
                .lower()
            )


            all_raw_generations.append(
                raw_text
            )


            if normalized in VALID_LABELS:

                all_predictions.append(
                    normalized
                )

            else:

                all_predictions.append(
                    "__invalid__"
                )


finally:

    hf_logging.set_verbosity(
        previous_hf_verbosity
    )

    tokenizer.padding_side = (
        original_padding_side
    )


inference_runtime_seconds = (
    time.time()
    -
    inference_start
)


# --------------------------------------------------------------------------------------------------
# 5. True / predicted labels
# --------------------------------------------------------------------------------------------------

y_true = (
    optimization_dev[
        "sentiment"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .tolist()
)


y_pred = (
    all_predictions
)


assert len(
    y_true
) == 2708

assert len(
    y_pred
) == 2708


invalid_prediction_count = int(
    sum(
        pred
        ==
        "__invalid__"

        for pred
        in y_pred
    )
)


invalid_prediction_rate = (
    invalid_prediction_count
    /
    len(
        y_pred
    )
)


# --------------------------------------------------------------------------------------------------
# 6. Overall metrics
# --------------------------------------------------------------------------------------------------

accuracy = float(
    accuracy_score(
        y_true,
        y_pred,
    )
)


macro_f1 = float(
    f1_score(
        y_true,
        y_pred,

        labels=
            VALID_LABELS,

        average=
            "macro",

        zero_division=
            0,
    )
)


weighted_f1 = float(
    f1_score(
        y_true,
        y_pred,

        labels=
            VALID_LABELS,

        average=
            "weighted",

        zero_division=
            0,
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Class-level metrics
# --------------------------------------------------------------------------------------------------

(
    class_precision,
    class_recall,
    class_f1,
    class_support,
) = precision_recall_fscore_support(

    y_true,
    y_pred,

    labels=
        VALID_LABELS,

    zero_division=
        0,
)


class_metrics_df = pd.DataFrame(
    {
        "label":
            VALID_LABELS,

        "precision":
            class_precision,

        "recall":
            class_recall,

        "f1":
            class_f1,

        "support":
            class_support.astype(
                int
            ),
    }
)


negative_recall = float(
    class_metrics_df.loc[
        class_metrics_df[
            "label"
        ]
        ==
        "negative",

        "recall",
    ].iloc[0]
)


neutral_recall = float(
    class_metrics_df.loc[
        class_metrics_df[
            "label"
        ]
        ==
        "neutral",

        "recall",
    ].iloc[0]
)


positive_recall = float(
    class_metrics_df.loc[
        class_metrics_df[
            "label"
        ]
        ==
        "positive",

        "recall",
    ].iloc[0]
)


minority_recall_floor = float(
    min(
        negative_recall,
        positive_recall,
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Confusion matrix
# --------------------------------------------------------------------------------------------------

cm = confusion_matrix(

    y_true,
    y_pred,

    labels=
        VALID_LABELS,
)


confusion_df = pd.DataFrame(

    cm,

    index=[
        f"true_{label}"
        for label
        in VALID_LABELS
    ],

    columns=[
        f"pred_{label}"
        for label
        in VALID_LABELS
    ],
)


# --------------------------------------------------------------------------------------------------
# 9. Prediction artifact
# --------------------------------------------------------------------------------------------------

prediction_df = pd.DataFrame(
    {
        "observation_id":
            optimization_dev[
                "observation_id"
            ].tolist(),

        "date":
            optimization_dev[
                "date"
            ].astype(str).tolist(),

        "true_label":
            y_true,

        "predicted_label":
            y_pred,

        "raw_generation":
            all_raw_generations,
    }
)


prediction_df.to_csv(
    STAGE_D_LR2E4_PREDICTIONS_PATH,
    index=False,
)


class_metrics_df.to_csv(
    STAGE_D_LR2E4_CLASS_METRICS_PATH,
    index=False,
)


confusion_df.to_csv(
    STAGE_D_LR2E4_CONFUSION_PATH
)


# --------------------------------------------------------------------------------------------------
# 10. Compare against LR=1e-4 incumbent
# --------------------------------------------------------------------------------------------------

LR1E4_ACCURACY = (
    STAGE_C_WINNER_ACCURACY
)

LR1E4_MACRO_F1 = (
    STAGE_C_WINNER_MACRO_F1
)

LR1E4_NEG_RECALL = (
    STAGE_C_WINNER_NEG_RECALL
)

LR1E4_NEU_RECALL = (
    STAGE_C_WINNER_NEU_RECALL
)

LR1E4_POS_RECALL = (
    STAGE_C_WINNER_POS_RECALL
)

LR1E4_MINORITY_FLOOR = min(
    LR1E4_NEG_RECALL,
    LR1E4_POS_RECALL,
)


accuracy_delta_vs_1e4 = (
    accuracy
    -
    LR1E4_ACCURACY
)


macro_f1_delta_vs_1e4 = (
    macro_f1
    -
    LR1E4_MACRO_F1
)


negative_recall_delta_vs_1e4 = (
    negative_recall
    -
    LR1E4_NEG_RECALL
)


neutral_recall_delta_vs_1e4 = (
    neutral_recall
    -
    LR1E4_NEU_RECALL
)


positive_recall_delta_vs_1e4 = (
    positive_recall
    -
    LR1E4_POS_RECALL
)


minority_floor_delta_vs_1e4 = (
    minority_recall_floor
    -
    LR1E4_MINORITY_FLOOR
)


beats_lr1e4_accuracy = bool(
    accuracy
    >
    LR1E4_ACCURACY
)


# --------------------------------------------------------------------------------------------------
# 11. Formal checks
# --------------------------------------------------------------------------------------------------

cell14_checks = {

    "dev_population_2708":
        len(
            y_true
        )
        == 2708,

    "prediction_count_2708":
        len(
            y_pred
        )
        == 2708,

    "zero_invalid_predictions":
        invalid_prediction_count
        == 0,

    "accuracy_finite":
        np.isfinite(
            accuracy
        ),

    "macro_f1_finite":
        np.isfinite(
            macro_f1
        ),

    "weighted_f1_finite":
        np.isfinite(
            weighted_f1
        ),

    "negative_recall_finite":
        np.isfinite(
            negative_recall
        ),

    "neutral_recall_finite":
        np.isfinite(
            neutral_recall
        ),

    "positive_recall_finite":
        np.isfinite(
            positive_recall
        ),

    "confusion_total_2708":
        int(
            cm.sum()
        )
        == 2708,

    "prediction_artifact_saved":
        STAGE_D_LR2E4_PREDICTIONS_PATH.exists(),

    "class_metrics_saved":
        STAGE_D_LR2E4_CLASS_METRICS_PATH.exists(),

    "confusion_matrix_saved":
        STAGE_D_LR2E4_CONFUSION_PATH.exists(),

    "historical_validation_not_used":
        True,

    "future_not_used":
        True,
}


all_cell14_checks_pass = bool(
    all(
        cell14_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. Save metrics JSON
# --------------------------------------------------------------------------------------------------

stage_d_lr2e4_metrics = {

    "day":
        23,

    "stage":
        "D",

    "condition":
        "lr_2e-4",

    "learning_rate":
        2e-4,

    "sampling":
        "natural",

    "evaluation_population":
        "optimization_dev",

    "n":
        len(
            y_true
        ),

    "metrics": {

        "accuracy":
            accuracy,

        "macro_f1":
            macro_f1,

        "weighted_f1":
            weighted_f1,

        "negative_recall":
            negative_recall,

        "neutral_recall":
            neutral_recall,

        "positive_recall":
            positive_recall,

        "minority_recall_floor":
            minority_recall_floor,

        "invalid_predictions":
            invalid_prediction_count,

        "invalid_prediction_rate":
            invalid_prediction_rate,
    },

    "comparison_vs_lr_1e-4": {

        "reference":
            "Day22 Stage B-500 natural sampling",

        "reference_accuracy":
            LR1E4_ACCURACY,

        "accuracy_delta":
            float(
                accuracy_delta_vs_1e4
            ),

        "macro_f1_delta":
            float(
                macro_f1_delta_vs_1e4
            ),

        "negative_recall_delta":
            float(
                negative_recall_delta_vs_1e4
            ),

        "neutral_recall_delta":
            float(
                neutral_recall_delta_vs_1e4
            ),

        "positive_recall_delta":
            float(
                positive_recall_delta_vs_1e4
            ),

        "minority_recall_floor_delta":
            float(
                minority_floor_delta_vs_1e4
            ),

        "beats_reference_accuracy":
            beats_lr1e4_accuracy,
    },

    "inference": {

        "batch_size":
            INFERENCE_BATCH_SIZE,

        "do_sample":
            False,

        "max_new_tokens":
            4,

        "padding_side":
            "left",

        "runtime_seconds":
            float(
                inference_runtime_seconds
            ),
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in cell14_checks.items()
    },

    "all_checks_pass":
        all_cell14_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_D_LR2E4_METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_d_lr2e4_metrics,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 13. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE D LR=2E-4 OPTIMIZATION DEV CLASSIFICATION")
print("=" * 120)


print("\n[EVALUATION POPULATION]")
print(
    "Split                       : Optimization Dev"
)
print(
    f"N                           : "
    f"{len(y_true):,}"
)
print(
    f"Inference runtime           : "
    f"{inference_runtime_seconds / 60:.2f} minutes"
)
print(
    f"Invalid predictions         : "
    f"{invalid_prediction_count}"
)


print("\n[OVERALL METRICS]")
print(
    f"Accuracy                    : "
    f"{accuracy:.6f}"
)
print(
    f"Macro-F1                    : "
    f"{macro_f1:.6f}"
)
print(
    f"Weighted-F1                 : "
    f"{weighted_f1:.6f}"
)


print("\n[CLASS RECALL]")
print(
    f"Negative recall             : "
    f"{negative_recall:.6f}"
)
print(
    f"Neutral recall              : "
    f"{neutral_recall:.6f}"
)
print(
    f"Positive recall             : "
    f"{positive_recall:.6f}"
)
print(
    f"Minority recall floor       : "
    f"{minority_recall_floor:.6f}"
)


print("\n[CLASS METRICS]")
display(
    class_metrics_df
)


print("\n[CONFUSION MATRIX]")
display(
    confusion_df
)


print("\n[COMPARISON VS LR=1E-4 INCUMBENT]")
print(
    f"LR=1e-4 Accuracy            : "
    f"{LR1E4_ACCURACY:.6f}"
)
print(
    f"LR=2e-4 Accuracy            : "
    f"{accuracy:.6f}"
)
print(
    f"Accuracy delta              : "
    f"{accuracy_delta_vs_1e4:+.6f} "
    f"({accuracy_delta_vs_1e4 * 100:+.2f} pp)"
)
print(
    f"Macro-F1 delta              : "
    f"{macro_f1_delta_vs_1e4:+.6f}"
)
print(
    f"Negative recall delta       : "
    f"{negative_recall_delta_vs_1e4:+.6f}"
)
print(
    f"Neutral recall delta        : "
    f"{neutral_recall_delta_vs_1e4:+.6f}"
)
print(
    f"Positive recall delta       : "
    f"{positive_recall_delta_vs_1e4:+.6f}"
)
print(
    f"Minority floor delta        : "
    f"{minority_floor_delta_vs_1e4:+.6f}"
)
print(
    f"Beats LR=1e-4 Accuracy      : "
    f"{beats_lr1e4_accuracy}"
)


print("\n[FORMAL CHECKS]")

for name, passed in cell14_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Predictions                 : "
    f"{STAGE_D_LR2E4_PREDICTIONS_PATH}"
)
print(
    f"Metrics                     : "
    f"{STAGE_D_LR2E4_METRICS_PATH}"
)
print(
    f"Class metrics               : "
    f"{STAGE_D_LR2E4_CLASS_METRICS_PATH}"
)
print(
    f"Confusion matrix            : "
    f"{STAGE_D_LR2E4_CONFUSION_PATH}"
)


print("\n" + "-" * 120)

if all_cell14_checks_pass:

    print(
        "DAY 23 CELL 14 STATUS: PASS — "
        "LR=2E-4 CLASSIFICATION EVALUATION COMPLETE; "
        "STAGE D THREE-RATE SELECTION READY"
    )

else:

    print(
        "DAY 23 CELL 14 STATUS: FAIL — "
        "DO NOT PERFORM STAGE D SELECTION"
    )

print("=" * 120)

LR=2e-4 Dev inference:   0%|          | 0/339 [00:00<?, ?it/s]

DAY 23 — STAGE D LR=2E-4 OPTIMIZATION DEV CLASSIFICATION

[EVALUATION POPULATION]
Split                       : Optimization Dev
N                           : 2,708
Inference runtime           : 6.08 minutes
Invalid predictions         : 0

[OVERALL METRICS]
Accuracy                    : 0.843427
Macro-F1                    : 0.813516
Weighted-F1                 : 0.843665

[CLASS RECALL]
Negative recall             : 0.750988
Neutral recall              : 0.880168
Positive recall             : 0.816135
Minority recall floor       : 0.750988

[CLASS METRICS]


,label,precision,recall,f1,support
0,negative,0.794979,0.750988,0.772358,506
1,neutral,0.888687,0.880168,0.884407,1669
2,positive,0.753899,0.816135,0.783784,533



[CONFUSION MATRIX]


,pred_negative,pred_neutral,pred_positive
true_negative,380,101,25
true_neutral,83,1469,117
true_positive,15,83,435



[COMPARISON VS LR=1E-4 INCUMBENT]
LR=1e-4 Accuracy            : 0.840842
LR=2e-4 Accuracy            : 0.843427
Accuracy delta              : +0.002585 (+0.26 pp)
Macro-F1 delta              : +0.005315
Negative recall delta       : +0.029644
Neutral recall delta        : -0.009587
Positive recall delta       : +0.015009
Minority floor delta        : +0.029644
Beats LR=1e-4 Accuracy      : True

[FORMAL CHECKS]
PASS   — dev_population_2708
PASS   — prediction_count_2708
PASS   — zero_invalid_predictions
PASS   — accuracy_finite
PASS   — macro_f1_finite
PASS   — weighted_f1_finite
PASS   — negative_recall_finite
PASS   — neutral_recall_finite
PASS   — positive_recall_finite
PASS   — confusion_total_2708
PASS   — prediction_artifact_saved
PASS   — class_metrics_saved
PASS   — confusion_matrix_saved
PASS   — historical_validation_not_used
PASS   — future_not_used

[ARTIFACTS]
Predictions                 : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day23/stage_d_learning_rate_au

In [23]:
# ==================================================================================================
# DAY 23 — CELL 15
# STAGE D THREE-RATE COMPARISON + FORMAL WINNER FREEZE
# ==================================================================================================

import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Required formal artifacts
# --------------------------------------------------------------------------------------------------

required_stage_d_artifacts = {

    "lr5e5_metrics":
        STAGE_D_LR5E5_METRICS_PATH,

    "lr5e5_training_summary":
        STAGE_D_LR5E5_TRAINING_SUMMARY_PATH,

    "lr2e4_metrics":
        STAGE_D_LR2E4_METRICS_PATH,

    "lr2e4_training_summary":
        STAGE_D_LR2E4_TRAINING_SUMMARY_PATH,

    "lr2e4_checkpoint":
        stage_d_lr2e4_final_checkpoint,

    "lr1e4_reference_adapter":
        STAGE_D_REFERENCE_ADAPTER_PATH,
}


for name, path in required_stage_d_artifacts.items():

    assert Path(path).exists(), (
        f"DAY 23 STOP: missing Stage D artifact:\n"
        f"{name}: {path}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Load fresh-run artifacts
# --------------------------------------------------------------------------------------------------

with open(
    STAGE_D_LR5E5_METRICS_PATH,
    "r",
    encoding="utf-8",
) as f:

    lr5e5_metrics_payload = load_artifact_json(f)


with open(
    STAGE_D_LR5E5_TRAINING_SUMMARY_PATH,
    "r",
    encoding="utf-8",
) as f:

    lr5e5_training_payload = load_artifact_json(f)


with open(
    STAGE_D_LR2E4_METRICS_PATH,
    "r",
    encoding="utf-8",
) as f:

    lr2e4_metrics_payload = load_artifact_json(f)


with open(
    STAGE_D_LR2E4_TRAINING_SUMMARY_PATH,
    "r",
    encoding="utf-8",
) as f:

    lr2e4_training_payload = load_artifact_json(f)


# --------------------------------------------------------------------------------------------------
# 3. Extract LR=5e-5
# --------------------------------------------------------------------------------------------------

lr5e5_metrics = (
    lr5e5_metrics_payload[
        "metrics"
    ]
)


LR5E5_ACCURACY = float(
    lr5e5_metrics[
        "accuracy"
    ]
)

LR5E5_MACRO_F1 = float(
    lr5e5_metrics[
        "macro_f1"
    ]
)

LR5E5_WEIGHTED_F1 = float(
    lr5e5_metrics[
        "weighted_f1"
    ]
)

LR5E5_NEG_RECALL = float(
    lr5e5_metrics[
        "negative_recall"
    ]
)

LR5E5_NEU_RECALL = float(
    lr5e5_metrics[
        "neutral_recall"
    ]
)

LR5E5_POS_RECALL = float(
    lr5e5_metrics[
        "positive_recall"
    ]
)

LR5E5_MINORITY_FLOOR = float(
    lr5e5_metrics[
        "minority_recall_floor"
    ]
)

LR5E5_EVAL_LOSS = float(
    lr5e5_training_payload[
        "evaluation"
    ][
        "eval_loss"
    ]
)


# --------------------------------------------------------------------------------------------------
# 4. Extract frozen LR=1e-4 reference
#
# Metrics come from Stage C frozen winner.
# Eval loss comes from the frozen Stage B-500 horizon audit.
# --------------------------------------------------------------------------------------------------

LR1E4_ACCURACY = float(
    STAGE_C_WINNER_ACCURACY
)

LR1E4_MACRO_F1 = float(
    STAGE_C_WINNER_MACRO_F1
)

LR1E4_NEG_RECALL = float(
    STAGE_C_WINNER_NEG_RECALL
)

LR1E4_NEU_RECALL = float(
    STAGE_C_WINNER_NEU_RECALL
)

LR1E4_POS_RECALL = float(
    STAGE_C_WINNER_POS_RECALL
)

LR1E4_MINORITY_FLOOR = float(
    min(
        LR1E4_NEG_RECALL,
        LR1E4_POS_RECALL,
    )
)

LR1E4_WEIGHTED_F1 = 0.840209

LR1E4_EVAL_LOSS = 0.215775


# --------------------------------------------------------------------------------------------------
# 5. Extract LR=2e-4
# --------------------------------------------------------------------------------------------------

lr2e4_metrics = (
    lr2e4_metrics_payload[
        "metrics"
    ]
)


LR2E4_ACCURACY = float(
    lr2e4_metrics[
        "accuracy"
    ]
)

LR2E4_MACRO_F1 = float(
    lr2e4_metrics[
        "macro_f1"
    ]
)

LR2E4_WEIGHTED_F1 = float(
    lr2e4_metrics[
        "weighted_f1"
    ]
)

LR2E4_NEG_RECALL = float(
    lr2e4_metrics[
        "negative_recall"
    ]
)

LR2E4_NEU_RECALL = float(
    lr2e4_metrics[
        "neutral_recall"
    ]
)

LR2E4_POS_RECALL = float(
    lr2e4_metrics[
        "positive_recall"
    ]
)

LR2E4_MINORITY_FLOOR = float(
    lr2e4_metrics[
        "minority_recall_floor"
    ]
)

LR2E4_EVAL_LOSS = float(
    lr2e4_training_payload[
        "evaluation"
    ][
        "eval_loss"
    ]
)


# --------------------------------------------------------------------------------------------------
# 6. Construct three-rate comparison table
# --------------------------------------------------------------------------------------------------

stage_d_comparison_df = pd.DataFrame(
    [
        {
            "condition":
                "lr_5e-5",

            "learning_rate":
                5e-5,

            "accuracy":
                LR5E5_ACCURACY,

            "macro_f1":
                LR5E5_MACRO_F1,

            "weighted_f1":
                LR5E5_WEIGHTED_F1,

            "negative_recall":
                LR5E5_NEG_RECALL,

            "neutral_recall":
                LR5E5_NEU_RECALL,

            "positive_recall":
                LR5E5_POS_RECALL,

            "minority_recall_floor":
                LR5E5_MINORITY_FLOOR,

            "eval_loss":
                LR5E5_EVAL_LOSS,

            "model_source":
                str(
                    stage_d_lr5e5_final_checkpoint
                ),
        },

        {
            "condition":
                "lr_1e-4",

            "learning_rate":
                1e-4,

            "accuracy":
                LR1E4_ACCURACY,

            "macro_f1":
                LR1E4_MACRO_F1,

            "weighted_f1":
                LR1E4_WEIGHTED_F1,

            "negative_recall":
                LR1E4_NEG_RECALL,

            "neutral_recall":
                LR1E4_NEU_RECALL,

            "positive_recall":
                LR1E4_POS_RECALL,

            "minority_recall_floor":
                LR1E4_MINORITY_FLOOR,

            "eval_loss":
                LR1E4_EVAL_LOSS,

            "model_source":
                str(
                    STAGE_D_REFERENCE_ADAPTER_PATH
                ),
        },

        {
            "condition":
                "lr_2e-4",

            "learning_rate":
                2e-4,

            "accuracy":
                LR2E4_ACCURACY,

            "macro_f1":
                LR2E4_MACRO_F1,

            "weighted_f1":
                LR2E4_WEIGHTED_F1,

            "negative_recall":
                LR2E4_NEG_RECALL,

            "neutral_recall":
                LR2E4_NEU_RECALL,

            "positive_recall":
                LR2E4_POS_RECALL,

            "minority_recall_floor":
                LR2E4_MINORITY_FLOOR,

            "eval_loss":
                LR2E4_EVAL_LOSS,

            "model_source":
                str(
                    stage_d_lr2e4_final_checkpoint
                ),
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 7. Hard identity checks
#
# Protects against accidental artifact/schema mismatch.
# --------------------------------------------------------------------------------------------------

identity_checks = {

    "lr5e5_accuracy_match":
        abs(
            LR5E5_ACCURACY
            -
            0.818316
        )
        <
        1e-6,

    "lr5e5_macro_f1_match":
        abs(
            LR5E5_MACRO_F1
            -
            0.778698
        )
        <
        1e-6,

    "lr1e4_accuracy_match":
        abs(
            LR1E4_ACCURACY
            -
            0.840842
        )
        <
        1e-6,

    "lr1e4_macro_f1_match":
        abs(
            LR1E4_MACRO_F1
            -
            0.808201
        )
        <
        1e-6,

    "lr2e4_accuracy_match":
        abs(
            LR2E4_ACCURACY
            -
            0.843427
        )
        <
        1e-6,

    "lr2e4_macro_f1_match":
        abs(
            LR2E4_MACRO_F1
            -
            0.813516
        )
        <
        1e-6,

    "lr5e5_eval_loss_match":
        abs(
            LR5E5_EVAL_LOSS
            -
            0.240016
        )
        <
        1e-6,

    "lr1e4_eval_loss_match":
        abs(
            LR1E4_EVAL_LOSS
            -
            0.215775
        )
        <
        1e-6,

    "lr2e4_eval_loss_match":
        abs(
            LR2E4_EVAL_LOSS
            -
            0.212594
        )
        <
        1e-6,
}


assert all(
    identity_checks.values()
), (
    "DAY 23 STOP: Stage D artifact values do not match "
    "the already observed formal results."
)


# --------------------------------------------------------------------------------------------------
# 8. Ranking
#
# Frozen protocol:
#   Primary   = Accuracy
#   Secondary = Macro-F1
#   Diagnostic = minority recall floor
#   eval_loss only as later tie-break / supporting diagnostic
# --------------------------------------------------------------------------------------------------

stage_d_comparison_df[
    "accuracy_rank"
] = (
    stage_d_comparison_df[
        "accuracy"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)


stage_d_comparison_df[
    "macro_f1_rank"
] = (
    stage_d_comparison_df[
        "macro_f1"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)


stage_d_comparison_df[
    "minority_floor_rank"
] = (
    stage_d_comparison_df[
        "minority_recall_floor"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)


stage_d_comparison_df[
    "eval_loss_rank"
] = (
    stage_d_comparison_df[
        "eval_loss"
    ]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)


stage_d_comparison_df = (
    stage_d_comparison_df
    .sort_values(
        by=[
            "accuracy",
            "macro_f1",
            "minority_recall_floor",
            "eval_loss",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 9. Explicit rows
# --------------------------------------------------------------------------------------------------

lr5e5_row = (
    stage_d_comparison_df.loc[
        stage_d_comparison_df[
            "condition"
        ]
        ==
        "lr_5e-5"
    ]
    .iloc[0]
)


lr1e4_row = (
    stage_d_comparison_df.loc[
        stage_d_comparison_df[
            "condition"
        ]
        ==
        "lr_1e-4"
    ]
    .iloc[0]
)


lr2e4_row = (
    stage_d_comparison_df.loc[
        stage_d_comparison_df[
            "condition"
        ]
        ==
        "lr_2e-4"
    ]
    .iloc[0]
)


# --------------------------------------------------------------------------------------------------
# 10. Deltas versus previous incumbent LR=1e-4
# --------------------------------------------------------------------------------------------------

for metric in [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "negative_recall",
    "neutral_recall",
    "positive_recall",
    "minority_recall_floor",
    "eval_loss",
]:

    stage_d_comparison_df[
        f"{metric}_delta_vs_lr1e4"
    ] = (
        stage_d_comparison_df[
            metric
        ]
        -
        float(
            lr1e4_row[
                metric
            ]
        )
    )


# --------------------------------------------------------------------------------------------------
# 11. Freeze winner
# --------------------------------------------------------------------------------------------------

stage_d_winner_row = (
    stage_d_comparison_df.iloc[
        0
    ]
)


STAGE_D_WINNER = str(
    stage_d_winner_row[
        "condition"
    ]
)

STAGE_D_WINNER_LR = float(
    stage_d_winner_row[
        "learning_rate"
    ]
)

STAGE_D_WINNER_ACCURACY = float(
    stage_d_winner_row[
        "accuracy"
    ]
)

STAGE_D_WINNER_MACRO_F1 = float(
    stage_d_winner_row[
        "macro_f1"
    ]
)

STAGE_D_WINNER_WEIGHTED_F1 = float(
    stage_d_winner_row[
        "weighted_f1"
    ]
)

STAGE_D_WINNER_NEG_RECALL = float(
    stage_d_winner_row[
        "negative_recall"
    ]
)

STAGE_D_WINNER_NEU_RECALL = float(
    stage_d_winner_row[
        "neutral_recall"
    ]
)

STAGE_D_WINNER_POS_RECALL = float(
    stage_d_winner_row[
        "positive_recall"
    ]
)

STAGE_D_WINNER_MINORITY_FLOOR = float(
    stage_d_winner_row[
        "minority_recall_floor"
    ]
)

STAGE_D_WINNER_EVAL_LOSS = float(
    stage_d_winner_row[
        "eval_loss"
    ]
)


# --------------------------------------------------------------------------------------------------
# 12. Selected model source
# --------------------------------------------------------------------------------------------------

STAGE_D_WINNER_SOURCE_CHECKPOINT = (
    stage_d_lr2e4_final_checkpoint
)


assert (
    STAGE_D_WINNER
    ==
    "lr_2e-4"
)


assert (
    STAGE_D_WINNER_SOURCE_CHECKPOINT.exists()
)


# --------------------------------------------------------------------------------------------------
# 13. Create clean selected-adapter directory
#
# Copy only adapter/inference-relevant files.
# Do NOT copy optimizer.pt or scheduler.pt.
# --------------------------------------------------------------------------------------------------

STAGE_D_SELECTED_ADAPTER_DIR = (
    STAGE_D_DIR
    / "selected_adapter"
)


if STAGE_D_SELECTED_ADAPTER_DIR.exists():

    shutil.rmtree(
        STAGE_D_SELECTED_ADAPTER_DIR
    )


STAGE_D_SELECTED_ADAPTER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


adapter_files_to_copy = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "adapter_model.bin",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
]


copied_adapter_files = []


for filename in adapter_files_to_copy:

    source_file = (
        STAGE_D_WINNER_SOURCE_CHECKPOINT
        / filename
    )


    if source_file.exists():

        destination_file = (
            STAGE_D_SELECTED_ADAPTER_DIR
            / filename
        )


        shutil.copy2(
            source_file,
            destination_file,
        )


        copied_adapter_files.append(
            filename
        )


# --------------------------------------------------------------------------------------------------
# 14. Selected-adapter integrity
# --------------------------------------------------------------------------------------------------

selected_adapter_has_config = (
    STAGE_D_SELECTED_ADAPTER_DIR
    / "adapter_config.json"
).exists()


selected_adapter_has_weights = (
    (
        STAGE_D_SELECTED_ADAPTER_DIR
        / "adapter_model.safetensors"
    ).exists()
    or
    (
        STAGE_D_SELECTED_ADAPTER_DIR
        / "adapter_model.bin"
    ).exists()
)


# --------------------------------------------------------------------------------------------------
# 15. Scientific interpretation
# --------------------------------------------------------------------------------------------------

lr2e4_beats_lr1e4_accuracy = bool(
    LR2E4_ACCURACY
    >
    LR1E4_ACCURACY
)


lr2e4_beats_lr1e4_macro_f1 = bool(
    LR2E4_MACRO_F1
    >
    LR1E4_MACRO_F1
)


lr2e4_improves_minority_floor = bool(
    LR2E4_MINORITY_FLOOR
    >
    LR1E4_MINORITY_FLOOR
)


lr2e4_lower_eval_loss = bool(
    LR2E4_EVAL_LOSS
    <
    LR1E4_EVAL_LOSS
)


lr2e4_neutral_recall_tradeoff = bool(
    LR2E4_NEU_RECALL
    <
    LR1E4_NEU_RECALL
)


# --------------------------------------------------------------------------------------------------
# 16. Formal checks
# --------------------------------------------------------------------------------------------------

stage_d_selection_checks = {

    **identity_checks,

    "three_learning_rates_present":
        set(
            stage_d_comparison_df[
                "condition"
            ]
        )
        ==
        {
            "lr_5e-5",
            "lr_1e-4",
            "lr_2e-4",
        },

    "winner_is_lr2e4":
        STAGE_D_WINNER
        ==
        "lr_2e-4",

    "lr2e4_highest_accuracy":
        LR2E4_ACCURACY
        ==
        stage_d_comparison_df[
            "accuracy"
        ].max(),

    "lr2e4_highest_macro_f1":
        LR2E4_MACRO_F1
        ==
        stage_d_comparison_df[
            "macro_f1"
        ].max(),

    "lr2e4_highest_minority_floor":
        LR2E4_MINORITY_FLOOR
        ==
        stage_d_comparison_df[
            "minority_recall_floor"
        ].max(),

    "lr2e4_lowest_eval_loss":
        LR2E4_EVAL_LOSS
        ==
        stage_d_comparison_df[
            "eval_loss"
        ].min(),

    "lr2e4_beats_lr1e4_accuracy":
        lr2e4_beats_lr1e4_accuracy,

    "lr2e4_beats_lr1e4_macro_f1":
        lr2e4_beats_lr1e4_macro_f1,

    "lr2e4_improves_minority_floor":
        lr2e4_improves_minority_floor,

    "lr2e4_lower_eval_loss":
        lr2e4_lower_eval_loss,

    "neutral_recall_tradeoff_recorded":
        lr2e4_neutral_recall_tradeoff,

    "selected_adapter_config_exists":
        selected_adapter_has_config,

    "selected_adapter_weights_exist":
        selected_adapter_has_weights,

    "no_posthoc_numeric_guardrail":
        True,

    "historical_validation_not_used":
        True,

    "future_not_used":
        True,
}


all_stage_d_selection_checks_pass = bool(
    all(
        stage_d_selection_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 17. Save formal Stage D selection artifacts
# --------------------------------------------------------------------------------------------------

STAGE_D_COMPARISON_PATH = (
    STAGE_D_ARTIFACT_DIR
    / "stage_d_three_rate_comparison.csv"
)

STAGE_D_SELECTION_SUMMARY_PATH = (
    STAGE_D_ARTIFACT_DIR
    / "stage_d_selection_summary.json"
)


stage_d_comparison_df.to_csv(
    STAGE_D_COMPARISON_PATH,
    index=False,
)


stage_d_selection_summary = {

    "day":
        23,

    "stage":
        "D",

    "objective":
        "historical optimization with primary target Accuracy",

    "evaluation_population":
        "optimization_dev",

    "fixed_configuration": {

        "sampling":
            "natural",

        "max_steps":
            500,

        "seed":
            42,

        "lora_r":
            8,

        "lora_alpha":
            32,

        "lora_dropout":
            0.1,
    },

    "candidate_learning_rates": [
        5e-5,
        1e-4,
        2e-4,
    ],

    "selection_rule": {

        "primary":
            "accuracy",

        "secondary":
            "macro_f1",

        "diagnostic":
            "minority_recall_floor",

        "later_tie_break":
            "eval_loss",

        "posthoc_numeric_guardrail_added":
            False,
    },

    "winner": {

        "condition":
            STAGE_D_WINNER,

        "learning_rate":
            STAGE_D_WINNER_LR,

        "accuracy":
            STAGE_D_WINNER_ACCURACY,

        "macro_f1":
            STAGE_D_WINNER_MACRO_F1,

        "weighted_f1":
            STAGE_D_WINNER_WEIGHTED_F1,

        "negative_recall":
            STAGE_D_WINNER_NEG_RECALL,

        "neutral_recall":
            STAGE_D_WINNER_NEU_RECALL,

        "positive_recall":
            STAGE_D_WINNER_POS_RECALL,

        "minority_recall_floor":
            STAGE_D_WINNER_MINORITY_FLOOR,

        "eval_loss":
            STAGE_D_WINNER_EVAL_LOSS,

        "source_checkpoint":
            str(
                STAGE_D_WINNER_SOURCE_CHECKPOINT
            ),

        "selected_adapter":
            str(
                STAGE_D_SELECTED_ADAPTER_DIR
            ),
    },

    "comparison_vs_previous_incumbent_lr1e4": {

        "accuracy_delta":
            float(
                LR2E4_ACCURACY
                -
                LR1E4_ACCURACY
            ),

        "macro_f1_delta":
            float(
                LR2E4_MACRO_F1
                -
                LR1E4_MACRO_F1
            ),

        "negative_recall_delta":
            float(
                LR2E4_NEG_RECALL
                -
                LR1E4_NEG_RECALL
            ),

        "neutral_recall_delta":
            float(
                LR2E4_NEU_RECALL
                -
                LR1E4_NEU_RECALL
            ),

        "positive_recall_delta":
            float(
                LR2E4_POS_RECALL
                -
                LR1E4_POS_RECALL
            ),

        "minority_floor_delta":
            float(
                LR2E4_MINORITY_FLOOR
                -
                LR1E4_MINORITY_FLOOR
            ),

        "eval_loss_delta":
            float(
                LR2E4_EVAL_LOSS
                -
                LR1E4_EVAL_LOSS
            ),
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in stage_d_selection_checks.items()
    },

    "all_checks_pass":
        all_stage_d_selection_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_D_SELECTION_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_d_selection_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 18. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE D THREE-RATE FORMAL SELECTION")
print("=" * 120)


print("\n[THREE-RATE COMPARISON]")

display(
    stage_d_comparison_df[
        [
            "learning_rate",
            "accuracy",
            "macro_f1",
            "weighted_f1",
            "negative_recall",
            "neutral_recall",
            "positive_recall",
            "minority_recall_floor",
            "eval_loss",
            "accuracy_rank",
            "macro_f1_rank",
            "minority_floor_rank",
            "eval_loss_rank",
        ]
    ]
)


print("\n[DELTA VS LR=1E-4]")

display(
    stage_d_comparison_df[
        [
            "learning_rate",
            "accuracy_delta_vs_lr1e4",
            "macro_f1_delta_vs_lr1e4",
            "negative_recall_delta_vs_lr1e4",
            "neutral_recall_delta_vs_lr1e4",
            "positive_recall_delta_vs_lr1e4",
            "minority_recall_floor_delta_vs_lr1e4",
            "eval_loss_delta_vs_lr1e4",
        ]
    ]
)


print("\n[STAGE D WINNER]")
print(
    f"Condition                   : "
    f"{STAGE_D_WINNER}"
)
print(
    f"Learning rate               : "
    f"{STAGE_D_WINNER_LR}"
)
print(
    f"Accuracy                    : "
    f"{STAGE_D_WINNER_ACCURACY:.6f}"
)
print(
    f"Macro-F1                    : "
    f"{STAGE_D_WINNER_MACRO_F1:.6f}"
)
print(
    f"Weighted-F1                 : "
    f"{STAGE_D_WINNER_WEIGHTED_F1:.6f}"
)
print(
    f"Negative recall             : "
    f"{STAGE_D_WINNER_NEG_RECALL:.6f}"
)
print(
    f"Neutral recall              : "
    f"{STAGE_D_WINNER_NEU_RECALL:.6f}"
)
print(
    f"Positive recall             : "
    f"{STAGE_D_WINNER_POS_RECALL:.6f}"
)
print(
    f"Minority recall floor       : "
    f"{STAGE_D_WINNER_MINORITY_FLOOR:.6f}"
)
print(
    f"Eval loss                   : "
    f"{STAGE_D_WINNER_EVAL_LOSS:.6f}"
)


print("\n[VS PREVIOUS LR=1E-4 INCUMBENT]")
print(
    f"Accuracy delta              : "
    f"{LR2E4_ACCURACY - LR1E4_ACCURACY:+.6f} "
    f"({(LR2E4_ACCURACY - LR1E4_ACCURACY) * 100:+.2f} pp)"
)
print(
    f"Macro-F1 delta              : "
    f"{LR2E4_MACRO_F1 - LR1E4_MACRO_F1:+.6f}"
)
print(
    f"Negative recall delta       : "
    f"{LR2E4_NEG_RECALL - LR1E4_NEG_RECALL:+.6f}"
)
print(
    f"Neutral recall delta        : "
    f"{LR2E4_NEU_RECALL - LR1E4_NEU_RECALL:+.6f}"
)
print(
    f"Positive recall delta       : "
    f"{LR2E4_POS_RECALL - LR1E4_POS_RECALL:+.6f}"
)
print(
    f"Minority floor delta        : "
    f"{LR2E4_MINORITY_FLOOR - LR1E4_MINORITY_FLOOR:+.6f}"
)
print(
    f"Eval-loss delta             : "
    f"{LR2E4_EVAL_LOSS - LR1E4_EVAL_LOSS:+.6f}"
)


print("\n[SELECTED ADAPTER]")
print(
    f"Source checkpoint           : "
    f"{STAGE_D_WINNER_SOURCE_CHECKPOINT}"
)
print(
    f"Frozen adapter              : "
    f"{STAGE_D_SELECTED_ADAPTER_DIR}"
)
print(
    f"Copied files                : "
    f"{copied_adapter_files}"
)


print("\n[FORMAL CHECKS]")

for name, passed in stage_d_selection_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Comparison                  : "
    f"{STAGE_D_COMPARISON_PATH}"
)
print(
    f"Selection summary           : "
    f"{STAGE_D_SELECTION_SUMMARY_PATH}"
)


print("\n" + "-" * 120)

if all_stage_d_selection_checks_pass:

    print(
        "DAY 23 CELL 15 STATUS: PASS — "
        "STAGE D COMPLETE; LR=2E-4 FROZEN AS WINNER; "
        "READY FOR STAGE E LORA CAPACITY AUDIT"
    )

else:

    print(
        "DAY 23 CELL 15 STATUS: FAIL — "
        "DO NOT START STAGE E"
    )

print("=" * 120)

DAY 23 — STAGE D THREE-RATE FORMAL SELECTION

[THREE-RATE COMPARISON]


,learning_rate,accuracy,macro_f1,weighted_f1,negative_recall,neutral_recall,positive_recall,minority_recall_floor,eval_loss,accuracy_rank,macro_f1_rank,minority_floor_rank,eval_loss_rank
0,0.00020,0.843427,0.813516,0.843665,0.750988,0.880168,0.816135,0.750988,0.212594,1,1,1,1
1,0.00010,0.840842,0.808201,0.840209,0.721344,0.889754,0.801126,0.721344,0.215775,2,2,2,2
2,0.00005,0.818316,0.778698,0.816447,0.671937,0.885560,0.746717,0.671937,0.240016,3,3,3,3



[DELTA VS LR=1E-4]


,learning_rate,accuracy_delta_vs_lr1e4,macro_f1_delta_vs_lr1e4,negative_recall_delta_vs_lr1e4,neutral_recall_delta_vs_lr1e4,positive_recall_delta_vs_lr1e4,minority_recall_floor_delta_vs_lr1e4,eval_loss_delta_vs_lr1e4
0,0.00020,0.002585,0.005315,0.029644,-0.009587,0.015009,0.029644,-0.003181
1,0.00010,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.00005,-0.022526,-0.029503,-0.049407,-0.004194,-0.054409,-0.049407,0.024241



[STAGE D WINNER]
Condition                   : lr_2e-4
Learning rate               : 0.0002
Accuracy                    : 0.843427
Macro-F1                    : 0.813516
Weighted-F1                 : 0.843665
Negative recall             : 0.750988
Neutral recall              : 0.880168
Positive recall             : 0.816135
Minority recall floor       : 0.750988
Eval loss                   : 0.212594

[VS PREVIOUS LR=1E-4 INCUMBENT]
Accuracy delta              : +0.002585 (+0.26 pp)
Macro-F1 delta              : +0.005315
Negative recall delta       : +0.029644
Neutral recall delta        : -0.009587
Positive recall delta       : +0.015009
Minority floor delta        : +0.029644
Eval-loss delta             : -0.003181

[SELECTED ADAPTER]
Source checkpoint           : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day23/stage_d_learning_rate_audit/runs/lr_2e-4/model/checkpoints/checkpoint-500
Frozen adapter              : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day23

In [24]:
# ==================================================================================================
# DAY 23 — CELL 16
# STAGE E — LORA CAPACITY AUDIT PROTOCOL FREEZE
# ==================================================================================================

import json
from datetime import datetime, timezone
from pathlib import Path


# --------------------------------------------------------------------------------------------------
# 1. Stage E directories
# --------------------------------------------------------------------------------------------------

STAGE_E_NAME = "lora_capacity_audit"

STAGE_E_DIR = (
    DAY23_RESULTS_ROOT
    / "stage_e_lora_capacity_audit"
)

STAGE_E_ARTIFACT_DIR = (
    STAGE_E_DIR
    / "artifacts"
)

STAGE_E_RUNS_DIR = (
    STAGE_E_DIR
    / "runs"
)


for directory in [
    STAGE_E_ARTIFACT_DIR,
    STAGE_E_RUNS_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# --------------------------------------------------------------------------------------------------
# 2. Preconditions from Stage D
# --------------------------------------------------------------------------------------------------

assert (
    STAGE_D_WINNER
    ==
    "lr_2e-4"
), (
    "DAY 23 STOP: Stage D winner is not LR=2e-4."
)


assert (
    abs(
        STAGE_D_WINNER_LR
        -
        2e-4
    )
    <
    1e-12
)


assert (
    abs(
        STAGE_D_WINNER_ACCURACY
        -
        0.843427
    )
    <
    1e-6
)


assert (
    abs(
        STAGE_D_WINNER_MACRO_F1
        -
        0.813516
    )
    <
    1e-6
)


assert (
    STAGE_D_SELECTED_ADAPTER_DIR.exists()
)


# --------------------------------------------------------------------------------------------------
# 3. Frozen Stage E shared configuration
# --------------------------------------------------------------------------------------------------

STAGE_E_BASE_MODEL = (
    "meta-llama/Meta-Llama-3-8B"
)

STAGE_E_SAMPLING = "natural"

STAGE_E_LEARNING_RATE = 2e-4

STAGE_E_MAX_STEPS = 500

STAGE_E_TRAIN_BATCH_SIZE = 1

STAGE_E_GRADIENT_ACCUMULATION = 8

STAGE_E_EFFECTIVE_BATCH = (
    STAGE_E_TRAIN_BATCH_SIZE
    *
    STAGE_E_GRADIENT_ACCUMULATION
)

STAGE_E_SCHEDULER = "linear"

STAGE_E_WARMUP_STEPS = 10

STAGE_E_SEED = 42

STAGE_E_MAX_SEQ_LENGTH = 512


# --------------------------------------------------------------------------------------------------
# 4. LoRA candidates
# --------------------------------------------------------------------------------------------------

STAGE_E_REFERENCE_CONFIG = {
    "condition":
        "r8_a32",

    "r":
        8,

    "alpha":
        32,

    "dropout":
        0.1,

    "status":
        "EXISTING_STAGE_D_WINNER",
}


STAGE_E_NEW_CONFIG = {
    "condition":
        "r16_a64",

    "r":
        16,

    "alpha":
        64,

    "dropout":
        0.1,

    "status":
        "PENDING_FRESH_TRAINING",
}


# --------------------------------------------------------------------------------------------------
# 5. Existing reference artifact
# --------------------------------------------------------------------------------------------------

STAGE_E_REFERENCE_ADAPTER_PATH = (
    STAGE_D_SELECTED_ADAPTER_DIR
)


assert STAGE_E_REFERENCE_ADAPTER_PATH.exists()


# --------------------------------------------------------------------------------------------------
# 6. New r16/a64 run directories
# --------------------------------------------------------------------------------------------------

STAGE_E_R16A64_DIR = (
    STAGE_E_RUNS_DIR
    / "r16_a64"
)

STAGE_E_R16A64_ARTIFACT_DIR = (
    STAGE_E_R16A64_DIR
    / "artifacts"
)

STAGE_E_R16A64_MODEL_DIR = (
    STAGE_E_R16A64_DIR
    / "model"
)

STAGE_E_R16A64_CHECKPOINT_DIR = (
    STAGE_E_R16A64_MODEL_DIR
    / "checkpoints"
)


for directory in [
    STAGE_E_R16A64_DIR,
    STAGE_E_R16A64_ARTIFACT_DIR,
    STAGE_E_R16A64_MODEL_DIR,
    STAGE_E_R16A64_CHECKPOINT_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# --------------------------------------------------------------------------------------------------
# 7. Candidate manifest
# --------------------------------------------------------------------------------------------------

stage_e_candidate_manifest = {

    "r8_a32": {

        "r":
            8,

        "alpha":
            32,

        "dropout":
            0.1,

        "status":
            "EXISTING_REFERENCE",

        "reuse_existing_run":
            True,

        "source":
            "Stage D winner",

        "learning_rate":
            2e-4,

        "accuracy":
            STAGE_D_WINNER_ACCURACY,

        "macro_f1":
            STAGE_D_WINNER_MACRO_F1,

        "minority_recall_floor":
            STAGE_D_WINNER_MINORITY_FLOOR,

        "eval_loss":
            STAGE_D_WINNER_EVAL_LOSS,

        "adapter_path":
            str(
                STAGE_E_REFERENCE_ADAPTER_PATH
            ),
    },

    "r16_a64": {

        "r":
            16,

        "alpha":
            64,

        "dropout":
            0.1,

        "status":
            "PENDING_FRESH_TRAINING",

        "reuse_existing_run":
            False,

        "learning_rate":
            2e-4,

        "run_dir":
            str(
                STAGE_E_R16A64_DIR
            ),
    },
}


# --------------------------------------------------------------------------------------------------
# 8. Selection rule
# --------------------------------------------------------------------------------------------------

STAGE_E_SELECTION_RULE = {

    "primary":
        "accuracy",

    "secondary":
        "macro_f1",

    "diagnostic":
        "minority_recall_floor",

    "later_tie_break":
        "eval_loss",

    "complexity_preference_if_effectively_tied":
        "prefer_r8_a32",

    "posthoc_numeric_guardrail_allowed":
        False,
}


# --------------------------------------------------------------------------------------------------
# 9. Firewall
# --------------------------------------------------------------------------------------------------

STAGE_E_EVALUATION_SPLIT = (
    "optimization_dev"
)

STAGE_E_HISTORICAL_VALIDATION_ALLOWED = False

STAGE_E_FUTURE_ALLOWED = False


# --------------------------------------------------------------------------------------------------
# 10. Formal checks
# --------------------------------------------------------------------------------------------------

stage_e_protocol_checks = {

    "stage_d_complete":
        all_stage_d_selection_checks_pass,

    "stage_d_winner_lr2e4":
        STAGE_D_WINNER
        ==
        "lr_2e-4",

    "learning_rate_fixed_2e4":
        abs(
            STAGE_E_LEARNING_RATE
            -
            2e-4
        )
        <
        1e-12,

    "natural_sampling_fixed":
        STAGE_E_SAMPLING
        ==
        "natural",

    "max_steps_500":
        STAGE_E_MAX_STEPS
        ==
        500,

    "effective_batch_8":
        STAGE_E_EFFECTIVE_BATCH
        ==
        8,

    "scheduler_linear":
        STAGE_E_SCHEDULER
        ==
        "linear",

    "warmup_10":
        STAGE_E_WARMUP_STEPS
        ==
        10,

    "seed_42":
        STAGE_E_SEED
        ==
        42,

    "reference_r8_a32":
        (
            STAGE_E_REFERENCE_CONFIG[
                "r"
            ]
            ==
            8
            and
            STAGE_E_REFERENCE_CONFIG[
                "alpha"
            ]
            ==
            32
        ),

    "new_candidate_r16_a64":
        (
            STAGE_E_NEW_CONFIG[
                "r"
            ]
            ==
            16
            and
            STAGE_E_NEW_CONFIG[
                "alpha"
            ]
            ==
            64
        ),

    "same_dropout":
        (
            STAGE_E_REFERENCE_CONFIG[
                "dropout"
            ]
            ==
            STAGE_E_NEW_CONFIG[
                "dropout"
            ]
            ==
            0.1
        ),

    "reference_adapter_exists":
        STAGE_E_REFERENCE_ADAPTER_PATH.exists(),

    "only_one_new_training_run":
        STAGE_E_NEW_CONFIG[
            "status"
        ]
        ==
        "PENDING_FRESH_TRAINING",

    "primary_metric_accuracy":
        STAGE_E_SELECTION_RULE[
            "primary"
        ]
        ==
        "accuracy",

    "no_posthoc_guardrail":
        STAGE_E_SELECTION_RULE[
            "posthoc_numeric_guardrail_allowed"
        ]
        is False,

    "optimization_dev_only":
        STAGE_E_EVALUATION_SPLIT
        ==
        "optimization_dev",

    "historical_validation_locked":
        STAGE_E_HISTORICAL_VALIDATION_ALLOWED
        is False,

    "future_locked":
        STAGE_E_FUTURE_ALLOWED
        is False,
}


all_stage_e_protocol_checks_pass = bool(
    all(
        stage_e_protocol_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 11. Save frozen protocol
# --------------------------------------------------------------------------------------------------

STAGE_E_PROTOCOL_PATH = (
    STAGE_E_ARTIFACT_DIR
    / "stage_e_lora_capacity_audit_protocol.json"
)


stage_e_protocol_payload = {

    "day":
        23,

    "stage":
        "E",

    "name":
        STAGE_E_NAME,

    "freeze_status":
        "STAGE_E_LORA_CAPACITY_PROTOCOL_FROZEN",

    "objective":
        "Test whether increasing LoRA capacity improves historical Optimization Dev performance.",

    "evaluation_population":
        STAGE_E_EVALUATION_SPLIT,

    "fixed_configuration": {

        "base_model":
            STAGE_E_BASE_MODEL,

        "sampling":
            STAGE_E_SAMPLING,

        "learning_rate":
            STAGE_E_LEARNING_RATE,

        "max_steps":
            STAGE_E_MAX_STEPS,

        "effective_batch":
            STAGE_E_EFFECTIVE_BATCH,

        "scheduler":
            STAGE_E_SCHEDULER,

        "warmup_steps":
            STAGE_E_WARMUP_STEPS,

        "seed":
            STAGE_E_SEED,

        "max_seq_length":
            STAGE_E_MAX_SEQ_LENGTH,

        "dropout":
            0.1,

        "target_modules": [
            "q_proj",
            "v_proj",
        ],
    },

    "candidate_manifest":
        stage_e_candidate_manifest,

    "selection_rule":
        STAGE_E_SELECTION_RULE,

    "firewall": {

        "historical_validation_allowed":
            False,

        "future_f1_f6_allowed":
            False,
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in stage_e_protocol_checks.items()
    },

    "all_checks_pass":
        all_stage_e_protocol_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_E_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_e_protocol_payload,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 12. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE E LORA CAPACITY AUDIT PROTOCOL FREEZE")
print("=" * 120)


print("\n[CURRENT INCUMBENT]")
print(
    "Sampling                    : Natural"
)
print(
    "Learning rate               : 2e-4"
)
print(
    "LoRA                        : r=8, alpha=32"
)
print(
    f"Accuracy                    : "
    f"{STAGE_D_WINNER_ACCURACY:.6f}"
)
print(
    f"Macro-F1                    : "
    f"{STAGE_D_WINNER_MACRO_F1:.6f}"
)
print(
    f"Minority recall floor       : "
    f"{STAGE_D_WINNER_MINORITY_FLOOR:.6f}"
)
print(
    f"Eval loss                   : "
    f"{STAGE_D_WINNER_EVAL_LOSS:.6f}"
)


print("\n[LORA CAPACITY CANDIDATES]")
print(
    "r=8, alpha=32               : EXISTING REFERENCE — NO RETRAIN"
)
print(
    "r=16, alpha=64              : NEW FRESH TRAINING REQUIRED"
)


print("\n[FIXED CONFIGURATION]")
print(
    "Sampling                    : Natural"
)
print(
    "Learning rate               : 2e-4"
)
print(
    "Training horizon            : 500"
)
print(
    "Effective batch             : 8"
)
print(
    "Scheduler                   : linear"
)
print(
    "Warmup                      : 10"
)
print(
    "Dropout                     : 0.1"
)
print(
    "Target modules              : q_proj, v_proj"
)
print(
    "Seed                        : 42"
)


print("\n[SELECTION]")
print(
    "Primary                     : Accuracy"
)
print(
    "Secondary                   : Macro-F1"
)
print(
    "Diagnostic                  : Minority recall floor"
)
print(
    "Later tie-break             : Eval loss"
)
print(
    "Complexity preference       : Prefer r8/a32 if effectively tied"
)
print(
    "Post-hoc numerical guardrail: NOT ALLOWED"
)


print("\n[FIREWALL]")
print(
    "Optimization Dev            : ALLOWED"
)
print(
    "Historical Validation       : LOCKED"
)
print(
    "Future F1-F6                : LOCKED"
)


print("\n[FORMAL CHECKS]")

for name, passed in stage_e_protocol_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACT]")
print(
    STAGE_E_PROTOCOL_PATH
)


print("\n" + "-" * 120)

if all_stage_e_protocol_checks_pass:

    print(
        "DAY 23 CELL 16 STATUS: PASS — "
        "STAGE E LORA CAPACITY PROTOCOL FROZEN; "
        "READY FOR R16/A64 FRESH MODEL INITIALIZATION"
    )

else:

    print(
        "DAY 23 CELL 16 STATUS: FAIL — "
        "DO NOT INITIALIZE STAGE E TRAINING"
    )

print("=" * 120)

DAY 23 — STAGE E LORA CAPACITY AUDIT PROTOCOL FREEZE

[CURRENT INCUMBENT]
Sampling                    : Natural
Learning rate               : 2e-4
LoRA                        : r=8, alpha=32
Accuracy                    : 0.843427
Macro-F1                    : 0.813516
Minority recall floor       : 0.750988
Eval loss                   : 0.212594

[LORA CAPACITY CANDIDATES]
r=8, alpha=32               : EXISTING REFERENCE — NO RETRAIN
r=16, alpha=64              : NEW FRESH TRAINING REQUIRED

[FIXED CONFIGURATION]
Sampling                    : Natural
Learning rate               : 2e-4
Training horizon            : 500
Effective batch             : 8
Scheduler                   : linear
Warmup                      : 10
Dropout                     : 0.1
Target modules              : q_proj, v_proj
Seed                        : 42

[SELECTION]
Primary                     : Accuracy
Secondary                   : Macro-F1
Diagnostic                  : Minority recall floor
Later tie-break   

In [25]:
# ==================================================================================================
# DAY 23 — CELL 17
# STAGE E — LORA r=16, alpha=64
# FRESH MODEL + NATURAL-SAMPLING TRAINER INFRASTRUCTURE
# ==================================================================================================

import gc
import json
import random
from pathlib import Path

import numpy as np
import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


# --------------------------------------------------------------------------------------------------
# 1. Candidate identity
# --------------------------------------------------------------------------------------------------

STAGE_E_CURRENT_R = 16
STAGE_E_CURRENT_ALPHA = 64
STAGE_E_CURRENT_DROPOUT = 0.1


# --------------------------------------------------------------------------------------------------
# 2. Candidate directory must be clean
# --------------------------------------------------------------------------------------------------

existing_r16a64_checkpoints = sorted(
    [
        p
        for p
        in STAGE_E_R16A64_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


assert len(
    existing_r16a64_checkpoints
) == 0, (
    "DAY 23 STOP: Stage E r16/a64 checkpoint directory is not clean."
)


# --------------------------------------------------------------------------------------------------
# 3. Release Stage D LR=2e-4 model/trainer from GPU
# --------------------------------------------------------------------------------------------------

objects_to_delete = [
    "stage_d_lr2e4_trainer",
    "stage_d_lr2e4_model",
    "stage_d_lr2e4_base_model",
]


for obj_name in objects_to_delete:

    if obj_name in globals():

        del globals()[
            obj_name
        ]


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()


gpu_after_cleanup_gb = (
    torch.cuda.memory_allocated()
    /
    (1024 ** 3)
)


# --------------------------------------------------------------------------------------------------
# 4. Deterministic seed
# --------------------------------------------------------------------------------------------------

random.seed(
    STAGE_E_SEED
)

np.random.seed(
    STAGE_E_SEED
)

torch.manual_seed(
    STAGE_E_SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        STAGE_E_SEED
    )


set_seed(
    STAGE_E_SEED
)


# --------------------------------------------------------------------------------------------------
# 5. Fresh 4-bit base model
# --------------------------------------------------------------------------------------------------

stage_e_r16a64_bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_use_double_quant=True,

    bnb_4bit_compute_dtype=torch.float16,
)


stage_e_r16a64_base_model = (
    AutoModelForCausalLM.from_pretrained(

        STAGE_E_BASE_MODEL,

        quantization_config=
            stage_e_r16a64_bnb_config,

        device_map=
            "auto",

        trust_remote_code=
            True,
    )
)


stage_e_r16a64_base_model.config.use_cache = False


stage_e_r16a64_base_model = (
    prepare_model_for_kbit_training(

        stage_e_r16a64_base_model,

        use_gradient_checkpointing=True,
    )
)


# --------------------------------------------------------------------------------------------------
# 6. Fresh LoRA r16 / alpha64
# --------------------------------------------------------------------------------------------------

set_seed(
    STAGE_E_SEED
)


stage_e_r16a64_lora_config = LoraConfig(

    r=
        STAGE_E_CURRENT_R,

    lora_alpha=
        STAGE_E_CURRENT_ALPHA,

    lora_dropout=
        STAGE_E_CURRENT_DROPOUT,

    target_modules=[
        "q_proj",
        "v_proj",
    ],

    bias=
        "none",

    task_type=
        "CAUSAL_LM",

    inference_mode=
        False,
)


stage_e_r16a64_model = (
    get_peft_model(
        stage_e_r16a64_base_model,
        stage_e_r16a64_lora_config,
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Final-step-only evaluation / save callback
# --------------------------------------------------------------------------------------------------

class StageER16A64FinalStepCallback(
    TrainerCallback
):

    def __init__(
        self,
        final_step=500,
    ):

        self.final_step = int(
            final_step
        )

        self.eval_triggered = False
        self.save_triggered = False


    def on_step_end(
        self,
        args,
        state,
        control,
        **kwargs,
    ):

        control.should_evaluate = False
        control.should_save = False


        if int(
            state.global_step
        ) == self.final_step:

            control.should_evaluate = True
            control.should_save = True

            self.eval_triggered = True
            self.save_triggered = True


        return control


stage_e_r16a64_callback = (
    StageER16A64FinalStepCallback(
        final_step=
            STAGE_E_MAX_STEPS
    )
)


# --------------------------------------------------------------------------------------------------
# 8. TrainingArguments
#
# Everything fixed from Stage D winner except LoRA capacity.
# --------------------------------------------------------------------------------------------------

stage_e_r16a64_training_args = TrainingArguments(

    output_dir=
        str(
            STAGE_E_R16A64_CHECKPOINT_DIR
        ),

    max_steps=
        STAGE_E_MAX_STEPS,

    per_device_train_batch_size=
        STAGE_E_TRAIN_BATCH_SIZE,

    per_device_eval_batch_size=
        1,

    gradient_accumulation_steps=
        STAGE_E_GRADIENT_ACCUMULATION,

    learning_rate=
        STAGE_E_LEARNING_RATE,

    lr_scheduler_type=
        STAGE_E_SCHEDULER,

    warmup_steps=
        STAGE_E_WARMUP_STEPS,

    weight_decay=
        0.0,

    logging_strategy=
        "steps",

    logging_steps=
        10,

    eval_strategy=
        "no",

    save_strategy=
        "no",

    load_best_model_at_end=
        False,

    fp16=
        True,

    bf16=
        False,

    gradient_checkpointing=
        True,

    seed=
        STAGE_E_SEED,

    data_seed=
        STAGE_E_SEED,

    report_to=
        [],

    remove_unused_columns=
        True,

    label_names=[
        "labels"
    ],

    disable_tqdm=
        False,
)


# --------------------------------------------------------------------------------------------------
# 9. Standard Trainer = natural sampling
# --------------------------------------------------------------------------------------------------

stage_e_r16a64_trainer = Trainer(

    model=
        stage_e_r16a64_model,

    args=
        stage_e_r16a64_training_args,

    train_dataset=
        hf_opt_train_model,

    eval_dataset=
        hf_opt_dev_model,

    data_collator=
        data_collator,

    processing_class=
        tokenizer,

    callbacks=[
        stage_e_r16a64_callback
    ],
)


# --------------------------------------------------------------------------------------------------
# 10. Trainable parameter audit
# --------------------------------------------------------------------------------------------------

stage_e_r16a64_total_params = 0
stage_e_r16a64_trainable_params = 0
stage_e_r16a64_trainable_names = []


for name, param in stage_e_r16a64_model.named_parameters():

    stage_e_r16a64_total_params += (
        param.numel()
    )

    if param.requires_grad:

        stage_e_r16a64_trainable_params += (
            param.numel()
        )

        stage_e_r16a64_trainable_names.append(
            name
        )


stage_e_r16a64_unexpected_trainable = [

    name

    for name
    in stage_e_r16a64_trainable_names

    if "lora_" not in name
]


# --------------------------------------------------------------------------------------------------
# 11. Expected trainable parameter count
#
# r16 doubles trainable LoRA parameters relative to r8:
# 3,407,872 -> 6,815,744
# --------------------------------------------------------------------------------------------------

EXPECTED_R16A64_TRAINABLE_PARAMS = (
    6_815_744
)


# --------------------------------------------------------------------------------------------------
# 12. Natural-sampling integrity
# --------------------------------------------------------------------------------------------------

natural_sampling_checks = {

    "trainer_is_standard_trainer":
        type(
            stage_e_r16a64_trainer
        )
        is Trainer,

    "no_weighted_sampler":
        True,

    "train_dataset_identity":
        stage_e_r16a64_trainer.train_dataset
        is hf_opt_train_model,

    "train_population_10831":
        len(
            stage_e_r16a64_trainer.train_dataset
        )
        == 10831,
}


# --------------------------------------------------------------------------------------------------
# 13. Formal infrastructure checks
# --------------------------------------------------------------------------------------------------

stage_e_r16a64_checks = {

    "checkpoint_dir_clean":
        len(
            existing_r16a64_checkpoints
        )
        == 0,

    "gpu_cleanup_completed":
        gpu_after_cleanup_gb
        <
        3.0,

    "fresh_model_loaded":
        stage_e_r16a64_model
        is not None,

    "natural_sampling":
        all(
            natural_sampling_checks.values()
        ),

    "train_n_10831":
        len(
            hf_opt_train_model
        )
        == 10831,

    "dev_n_2708":
        len(
            hf_opt_dev_model
        )
        == 2708,

    "max_steps_500":
        stage_e_r16a64_training_args.max_steps
        ==
        500,

    "lr_2e4":
        abs(
            stage_e_r16a64_training_args.learning_rate
            -
            2e-4
        )
        <
        1e-12,

    "gradient_accumulation_8":
        (
            stage_e_r16a64_training_args
            .gradient_accumulation_steps
            ==
            8
        ),

    "scheduler_linear":
        str(
            stage_e_r16a64_training_args.lr_scheduler_type
        )
        .lower()
        .endswith(
            "linear"
        ),

    "warmup_10":
        stage_e_r16a64_training_args.warmup_steps
        ==
        10,

    "seed_42":
        stage_e_r16a64_training_args.seed
        ==
        42,

    "lora_r16":
        (
            stage_e_r16a64_model
            .peft_config[
                "default"
            ]
            .r
            ==
            16
        ),

    "lora_alpha64":
        (
            stage_e_r16a64_model
            .peft_config[
                "default"
            ]
            .lora_alpha
            ==
            64
        ),

    "lora_dropout_0_1":
        abs(
            stage_e_r16a64_model
            .peft_config[
                "default"
            ]
            .lora_dropout
            -
            0.1
        )
        <
        1e-12,

    "trainable_params_6815744":
        stage_e_r16a64_trainable_params
        ==
        EXPECTED_R16A64_TRAINABLE_PARAMS,

    "only_lora_trainable":
        len(
            stage_e_r16a64_unexpected_trainable
        )
        ==
        0,

    "trainer_global_step_zero":
        int(
            stage_e_r16a64_trainer
            .state
            .global_step
        )
        ==
        0,

    "eval_not_triggered":
        stage_e_r16a64_callback.eval_triggered
        is False,

    "save_not_triggered":
        stage_e_r16a64_callback.save_triggered
        is False,

    "historical_validation_locked":
        STAGE_E_HISTORICAL_VALIDATION_ALLOWED
        is False,

    "future_locked":
        STAGE_E_FUTURE_ALLOWED
        is False,
}


all_stage_e_r16a64_checks_pass = bool(
    all(
        stage_e_r16a64_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 14. Save infrastructure manifest
# --------------------------------------------------------------------------------------------------

STAGE_E_R16A64_INFRA_PATH = (
    STAGE_E_R16A64_ARTIFACT_DIR
    / "r16_a64_training_infrastructure.json"
)


stage_e_r16a64_infra_payload = {

    "day":
        23,

    "stage":
        "E",

    "condition":
        "r16_a64",

    "sampling":
        "natural",

    "learning_rate":
        2e-4,

    "initialization":
        "fresh_base_plus_fresh_lora",

    "base_model":
        STAGE_E_BASE_MODEL,

    "training": {

        "max_steps":
            STAGE_E_MAX_STEPS,

        "learning_rate":
            STAGE_E_LEARNING_RATE,

        "scheduler":
            STAGE_E_SCHEDULER,

        "warmup_steps":
            STAGE_E_WARMUP_STEPS,

        "effective_batch":
            STAGE_E_EFFECTIVE_BATCH,

        "seed":
            STAGE_E_SEED,
    },

    "lora": {

        "r":
            STAGE_E_CURRENT_R,

        "alpha":
            STAGE_E_CURRENT_ALPHA,

        "dropout":
            STAGE_E_CURRENT_DROPOUT,

        "target_modules": [
            "q_proj",
            "v_proj",
        ],
    },

    "parameter_audit": {

        "trainable_params":
            int(
                stage_e_r16a64_trainable_params
            ),

        "total_params":
            int(
                stage_e_r16a64_total_params
            ),

        "unexpected_trainable_count":
            len(
                stage_e_r16a64_unexpected_trainable
            ),
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in stage_e_r16a64_checks.items()
    },

    "all_checks_pass":
        all_stage_e_r16a64_checks_pass,
}


with open(
    STAGE_E_R16A64_INFRA_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_e_r16a64_infra_payload,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 15. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE E R16/A64 TRAINER INFRASTRUCTURE")
print("=" * 120)


print("\n[GPU CLEANUP]")
print(
    f"Allocated after cleanup     : "
    f"{gpu_after_cleanup_gb:.2f} GB"
)


print("\n[CANDIDATE]")
print(
    "LoRA                        : r=16, alpha=64"
)
print(
    "Learning rate               : 2e-4"
)
print(
    "Sampling                    : Natural"
)
print(
    "Initialization              : Fresh base + fresh LoRA"
)
print(
    "Previous adapters loaded    : False"
)


print("\n[TRAINING PROTOCOL]")
print(
    "Max steps                   : 500"
)
print(
    "Effective batch             : 8"
)
print(
    "Scheduler                   : linear"
)
print(
    "Warmup                      : 10"
)
print(
    "Seed                        : 42"
)
print(
    "Formal Dev evaluation       : step 500 only"
)


print("\n[PARAMETERS]")
print(
    f"Trainable parameters        : "
    f"{stage_e_r16a64_trainable_params:,}"
)
print(
    f"Expected trainable          : "
    f"{EXPECTED_R16A64_TRAINABLE_PARAMS:,}"
)
print(
    f"Total parameters            : "
    f"{stage_e_r16a64_total_params:,}"
)
print(
    f"Unexpected trainable        : "
    f"{len(stage_e_r16a64_unexpected_trainable)}"
)


print("\n[PRE-TRAINING STATE]")
print(
    f"Global step                 : "
    f"{stage_e_r16a64_trainer.state.global_step}"
)
print(
    f"Final eval triggered        : "
    f"{stage_e_r16a64_callback.eval_triggered}"
)
print(
    f"Final save triggered        : "
    f"{stage_e_r16a64_callback.save_triggered}"
)


print("\n[GPU AFTER MODEL LOAD]")
print(
    f"Allocated                   : "
    f"{torch.cuda.memory_allocated() / (1024 ** 3):.2f} GB"
)


print("\n[FORMAL CHECKS]")

for name, passed in stage_e_r16a64_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACT]")
print(
    STAGE_E_R16A64_INFRA_PATH
)


print("\n" + "-" * 120)

if all_stage_e_r16a64_checks_pass:

    print(
        "DAY 23 CELL 17 STATUS: PASS — "
        "R16/A64 NATURAL-SAMPLING TRAINER READY; "
        "TRAINING NOT YET STARTED"
    )

else:

    print(
        "DAY 23 CELL 17 STATUS: FAIL — "
        "DO NOT START R16/A64 TRAINING"
    )

print("=" * 120)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

DAY 23 — STAGE E R16/A64 TRAINER INFRASTRUCTURE

[GPU CLEANUP]
Allocated after cleanup     : 1.97 GB

[CANDIDATE]
LoRA                        : r=16, alpha=64
Learning rate               : 2e-4
Sampling                    : Natural
Initialization              : Fresh base + fresh LoRA
Previous adapters loaded    : False

[TRAINING PROTOCOL]
Max steps                   : 500
Effective batch             : 8
Scheduler                   : linear
Warmup                      : 10
Seed                        : 42
Formal Dev evaluation       : step 500 only

[PARAMETERS]
Trainable parameters        : 6,815,744
Expected trainable          : 6,815,744
Total parameters            : 4,547,416,064
Unexpected trainable        : 0

[PRE-TRAINING STATE]
Global step                 : 0
Final eval triggered        : False
Final save triggered        : False

[GPU AFTER MODEL LOAD]
Allocated                   : 7.32 GB

[FORMAL CHECKS]
PASS   — checkpoint_dir_clean
PASS   — gpu_cleanup_completed
PASS   —

In [26]:
# ==================================================================================================
# DAY 23 — CELL 18
# STAGE E — LORA r=16, alpha=64
# CONTROLLED 500-STEP NATURAL-SAMPLING TRAINING
# ==================================================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch


# --------------------------------------------------------------------------------------------------
# 1. Formal artifact paths
# --------------------------------------------------------------------------------------------------

STAGE_E_R16A64_TRAINING_LOG_PATH = (
    STAGE_E_R16A64_ARTIFACT_DIR
    / "r16_a64_training_log.csv"
)

STAGE_E_R16A64_TRAINING_SUMMARY_PATH = (
    STAGE_E_R16A64_ARTIFACT_DIR
    / "r16_a64_training_summary.json"
)

STAGE_E_R16A64_TRAINING_COMPLETE_PATH = (
    STAGE_E_R16A64_ARTIFACT_DIR
    / "r16_a64_training_complete.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Pre-training safety gate
# --------------------------------------------------------------------------------------------------

assert int(
    stage_e_r16a64_trainer.state.global_step
) == 0, (
    "DAY 23 STOP: r16/a64 trainer already advanced beyond step 0."
)


assert (
    stage_e_r16a64_callback.eval_triggered
    is False
), (
    "DAY 23 STOP: r16/a64 evaluation callback already triggered."
)


assert (
    stage_e_r16a64_callback.save_triggered
    is False
), (
    "DAY 23 STOP: r16/a64 save callback already triggered."
)


existing_checkpoints_before_training = sorted(
    [
        p
        for p
        in STAGE_E_R16A64_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


assert len(
    existing_checkpoints_before_training
) == 0, (
    "DAY 23 STOP: r16/a64 checkpoint directory is not clean."
)


assert not STAGE_E_R16A64_TRAINING_COMPLETE_PATH.exists(), (
    "DAY 23 STOP: r16/a64 completion marker already exists."
)


# --------------------------------------------------------------------------------------------------
# 3. Reset deterministic seeds immediately before training
# --------------------------------------------------------------------------------------------------

random.seed(
    STAGE_E_SEED
)

np.random.seed(
    STAGE_E_SEED
)

torch.manual_seed(
    STAGE_E_SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        STAGE_E_SEED
    )

    torch.cuda.reset_peak_memory_stats()


set_seed(
    STAGE_E_SEED
)


# --------------------------------------------------------------------------------------------------
# 4. Train
# --------------------------------------------------------------------------------------------------

training_start_time = time.time()


stage_e_r16a64_train_result = (
    stage_e_r16a64_trainer.train()
)


training_end_time = time.time()


training_runtime_seconds = (
    training_end_time
    -
    training_start_time
)

training_runtime_minutes = (
    training_runtime_seconds
    /
    60.0
)


# --------------------------------------------------------------------------------------------------
# 5. Collect Trainer history
# --------------------------------------------------------------------------------------------------

stage_e_r16a64_log_history = (
    stage_e_r16a64_trainer.state.log_history
)


stage_e_r16a64_log_df = pd.DataFrame(
    stage_e_r16a64_log_history
)


stage_e_r16a64_log_df.to_csv(
    STAGE_E_R16A64_TRAINING_LOG_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 6. Final state
# --------------------------------------------------------------------------------------------------

final_global_step = int(
    stage_e_r16a64_trainer.state.global_step
)


final_training_loss = float(
    stage_e_r16a64_train_result.training_loss
)


# --------------------------------------------------------------------------------------------------
# 7. Extract final Dev evaluation
# --------------------------------------------------------------------------------------------------

final_eval_rows = (
    stage_e_r16a64_log_df.loc[
        stage_e_r16a64_log_df[
            "eval_loss"
        ].notna()
    ]
    .copy()
)


if len(
    final_eval_rows
) > 0:

    final_eval_row = (
        final_eval_rows
        .sort_values(
            "step"
        )
        .iloc[-1]
    )

    final_eval_step = int(
        final_eval_row[
            "step"
        ]
    )

    final_eval_loss = float(
        final_eval_row[
            "eval_loss"
        ]
    )

    final_eval_runtime = float(
        final_eval_row.get(
            "eval_runtime",
            np.nan,
        )
    )

else:

    final_eval_step = None
    final_eval_loss = None
    final_eval_runtime = None


# --------------------------------------------------------------------------------------------------
# 8. Learning-rate trajectory
# --------------------------------------------------------------------------------------------------

lr_rows = (
    stage_e_r16a64_log_df.loc[
        stage_e_r16a64_log_df[
            "learning_rate"
        ].notna()
    ]
    .copy()
)


def get_r16a64_logged_lr(
    step,
):

    rows = lr_rows.loc[
        lr_rows[
            "step"
        ]
        ==
        step
    ]


    if len(
        rows
    ) == 0:

        return None


    return float(
        rows.iloc[-1][
            "learning_rate"
        ]
    )


lr_step_100 = get_r16a64_logged_lr(
    100
)

lr_step_200 = get_r16a64_logged_lr(
    200
)

lr_step_300 = get_r16a64_logged_lr(
    300
)

lr_step_400 = get_r16a64_logged_lr(
    400
)

lr_step_500 = get_r16a64_logged_lr(
    500
)


# --------------------------------------------------------------------------------------------------
# 9. Scheduler end-state
# --------------------------------------------------------------------------------------------------

final_scheduler_epoch = None
final_scheduler_lr = None


if (
    stage_e_r16a64_trainer.lr_scheduler
    is not None
):

    final_scheduler_epoch = int(
        stage_e_r16a64_trainer
        .lr_scheduler
        .last_epoch
    )

    final_scheduler_lr = [
        float(
            value
        )
        for value
        in stage_e_r16a64_trainer
        .lr_scheduler
        .get_last_lr()
    ]


# --------------------------------------------------------------------------------------------------
# 10. Final checkpoint audit
# --------------------------------------------------------------------------------------------------

stage_e_r16a64_final_checkpoint = (
    STAGE_E_R16A64_CHECKPOINT_DIR
    / "checkpoint-500"
)


checkpoint_500_checks = {

    "checkpoint_exists":
        stage_e_r16a64_final_checkpoint.exists(),

    "adapter_config_exists":
        (
            stage_e_r16a64_final_checkpoint
            / "adapter_config.json"
        ).exists(),

    "adapter_weights_exist":
        (
            (
                stage_e_r16a64_final_checkpoint
                / "adapter_model.safetensors"
            ).exists()
            or
            (
                stage_e_r16a64_final_checkpoint
                / "adapter_model.bin"
            ).exists()
        ),

    "trainer_state_exists":
        (
            stage_e_r16a64_final_checkpoint
            / "trainer_state.json"
        ).exists(),

    "optimizer_state_exists":
        (
            stage_e_r16a64_final_checkpoint
            / "optimizer.pt"
        ).exists(),

    "scheduler_state_exists":
        (
            stage_e_r16a64_final_checkpoint
            / "scheduler.pt"
        ).exists(),
}


# --------------------------------------------------------------------------------------------------
# 11. Detect unintended intermediate checkpoints
# --------------------------------------------------------------------------------------------------

all_r16a64_checkpoints = sorted(
    [
        p.name
        for p
        in STAGE_E_R16A64_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


# --------------------------------------------------------------------------------------------------
# 12. GPU state
# --------------------------------------------------------------------------------------------------

peak_gpu_gb = None


if torch.cuda.is_available():

    peak_gpu_gb = (
        torch.cuda.max_memory_allocated()
        /
        (1024 ** 3)
    )


# --------------------------------------------------------------------------------------------------
# 13. Formal post-training checks
# --------------------------------------------------------------------------------------------------

stage_e_r16a64_training_checks = {

    "training_completed_500_steps":
        final_global_step
        == 500,

    "training_loss_finite":
        np.isfinite(
            final_training_loss
        ),

    "training_log_nonempty":
        len(
            stage_e_r16a64_log_df
        )
        > 0,

    "final_eval_triggered":
        stage_e_r16a64_callback.eval_triggered
        is True,

    "final_save_triggered":
        stage_e_r16a64_callback.save_triggered
        is True,

    "exactly_one_eval":
        len(
            final_eval_rows
        )
        == 1,

    "eval_at_step_500":
        final_eval_step
        == 500,

    "eval_loss_finite":
        (
            final_eval_loss
            is not None
            and
            np.isfinite(
                final_eval_loss
            )
        ),

    "only_checkpoint_500":
        all_r16a64_checkpoints
        ==
        [
            "checkpoint-500"
        ],

    "checkpoint_500_complete":
        all(
            checkpoint_500_checks.values()
        ),

    "lr_100_observed":
        lr_step_100
        is not None,

    "lr_200_observed":
        lr_step_200
        is not None,

    "lr_300_observed":
        lr_step_300
        is not None,

    "lr_400_observed":
        lr_step_400
        is not None,

    "lr_500_observed":
        lr_step_500
        is not None,

    "lr_positive_before_end":
        (
            lr_step_100 is not None
            and
            lr_step_200 is not None
            and
            lr_step_300 is not None
            and
            lr_step_400 is not None
            and
            lr_step_100 > 0
            and
            lr_step_200 > 0
            and
            lr_step_300 > 0
            and
            lr_step_400 > 0
        ),

    "scheduler_reaches_end_at_500":
        final_scheduler_epoch
        == 500,

    "historical_validation_not_used":
        True,

    "future_not_used":
        True,
}


all_r16a64_training_checks_pass = bool(
    all(
        stage_e_r16a64_training_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 14. Save training summary
# --------------------------------------------------------------------------------------------------

stage_e_r16a64_training_summary = {

    "day":
        23,

    "stage":
        "E",

    "condition":
        "r16_a64",

    "sampling":
        "natural",

    "learning_rate":
        2e-4,

    "lora": {
        "r":
            16,

        "alpha":
            64,

        "dropout":
            0.1,
    },

    "training": {

        "final_global_step":
            final_global_step,

        "training_loss":
            final_training_loss,

        "runtime_seconds":
            float(
                training_runtime_seconds
            ),

        "runtime_minutes":
            float(
                training_runtime_minutes
            ),

        "peak_gpu_gb":
            (
                float(
                    peak_gpu_gb
                )
                if peak_gpu_gb
                is not None
                else None
            ),
    },

    "evaluation": {

        "split":
            "optimization_dev",

        "step":
            final_eval_step,

        "eval_loss":
            final_eval_loss,

        "eval_runtime":
            final_eval_runtime,
    },

    "learning_rate_trajectory": {

        "step_100":
            lr_step_100,

        "step_200":
            lr_step_200,

        "step_300":
            lr_step_300,

        "step_400":
            lr_step_400,

        "step_500":
            lr_step_500,

        "final_scheduler_epoch":
            final_scheduler_epoch,

        "final_scheduler_lr":
            final_scheduler_lr,
    },

    "checkpoint": {

        "directory":
            str(
                stage_e_r16a64_final_checkpoint
            ),

        "all_checkpoint_dirs":
            all_r16a64_checkpoints,

        "checks": {

            key:
                bool(
                    value
                )

            for key, value
            in checkpoint_500_checks.items()
        },
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in stage_e_r16a64_training_checks.items()
    },

    "all_checks_pass":
        all_r16a64_training_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_E_R16A64_TRAINING_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_e_r16a64_training_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 15. Completion marker
# --------------------------------------------------------------------------------------------------

if all_r16a64_training_checks_pass:

    completion_payload = {

        "status":
            "PASS",

        "condition":
            "r16_a64",

        "sampling":
            "natural",

        "learning_rate":
            2e-4,

        "lora_r":
            16,

        "lora_alpha":
            64,

        "global_step":
            final_global_step,

        "checkpoint":
            str(
                stage_e_r16a64_final_checkpoint
            ),

        "created_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


    with open(
        STAGE_E_R16A64_TRAINING_COMPLETE_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            completion_payload,
            f,
            indent=2,
            ensure_ascii=False,
        )


# --------------------------------------------------------------------------------------------------
# 16. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE E R16/A64 500-STEP TRAINING")
print("=" * 120)


print("\n[TRAINING]")
print(
    f"Final global step           : "
    f"{final_global_step}"
)
print(
    f"Final training loss         : "
    f"{final_training_loss:.6f}"
)
print(
    f"Runtime                     : "
    f"{training_runtime_minutes:.2f} minutes"
)

if peak_gpu_gb is not None:

    print(
        f"Peak GPU allocation         : "
        f"{peak_gpu_gb:.2f} GB"
    )


print("\n[FINAL DEV EVALUATION]")
print(
    f"Evaluation step             : "
    f"{final_eval_step}"
)
print(
    f"Evaluation loss             : "
    f"{final_eval_loss:.6f}"
)
print(
    f"Evaluation runtime          : "
    f"{final_eval_runtime:.2f} sec"
)


print("\n[LEARNING-RATE TRAJECTORY]")
print(
    f"Step 100                    : "
    f"{lr_step_100}"
)
print(
    f"Step 200                    : "
    f"{lr_step_200}"
)
print(
    f"Step 300                    : "
    f"{lr_step_300}"
)
print(
    f"Step 400                    : "
    f"{lr_step_400}"
)
print(
    f"Step 500                    : "
    f"{lr_step_500}"
)
print(
    f"Final scheduler epoch       : "
    f"{final_scheduler_epoch}"
)
print(
    f"Final scheduler LR          : "
    f"{final_scheduler_lr}"
)


print("\n[CALLBACK]")
print(
    f"Final eval triggered        : "
    f"{stage_e_r16a64_callback.eval_triggered}"
)
print(
    f"Final save triggered        : "
    f"{stage_e_r16a64_callback.save_triggered}"
)


print("\n[CHECKPOINT]")
print(
    f"Checkpoint directory        : "
    f"{stage_e_r16a64_final_checkpoint}"
)
print(
    f"All checkpoints             : "
    f"{all_r16a64_checkpoints}"
)

for name, passed in checkpoint_500_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[POST-TRAINING CHECKS]")

for name, passed in stage_e_r16a64_training_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Training log                : "
    f"{STAGE_E_R16A64_TRAINING_LOG_PATH}"
)
print(
    f"Training summary            : "
    f"{STAGE_E_R16A64_TRAINING_SUMMARY_PATH}"
)
print(
    f"Completion marker           : "
    f"{STAGE_E_R16A64_TRAINING_COMPLETE_PATH}"
)


print("\n" + "-" * 120)

if all_r16a64_training_checks_pass:

    print(
        "DAY 23 CELL 18 STATUS: PASS — "
        "R16/A64 500-STEP TRAINING COMPLETE; "
        "CLASSIFICATION EVALUATION PENDING"
    )

else:

    print(
        "DAY 23 CELL 18 STATUS: FAIL — "
        "DO NOT EVALUATE OR PROCEED"
    )

print("=" * 120)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


DAY 23 — STAGE E R16/A64 500-STEP TRAINING

[TRAINING]
Final global step           : 500
Final training loss         : 0.399962
Runtime                     : 57.44 minutes
Peak GPU allocation         : 8.86 GB

[FINAL DEV EVALUATION]
Evaluation step             : 500
Evaluation loss             : 0.196898
Evaluation runtime          : 501.93 sec

[LEARNING-RATE TRAJECTORY]
Step 100                    : 0.00016367346938775512
Step 200                    : 0.00012285714285714287
Step 300                    : 8.204081632653062e-05
Step 400                    : 4.122448979591837e-05
Step 500                    : 4.081632653061225e-07
Final scheduler epoch       : 500
Final scheduler LR          : [0.0, 0.0]

[CALLBACK]
Final eval triggered        : True
Final save triggered        : True

[CHECKPOINT]
Checkpoint directory        : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day23/stage_e_lora_capacity_audit/runs/r16_a64/model/checkpoints/checkpoint-500
All checkpoints             

In [27]:
# ==================================================================================================
# DAY 23 — CELL 19
# STAGE E — r16/a64
# OPTIMIZATION DEV CLASSIFICATION INFERENCE + COMPARISON VS r8/a32 INCUMBENT
# ==================================================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

from tqdm.auto import tqdm
from transformers.utils import logging as hf_logging


# --------------------------------------------------------------------------------------------------
# 1. Formal output paths
# --------------------------------------------------------------------------------------------------

STAGE_E_R16A64_PREDICTIONS_PATH = (
    STAGE_E_R16A64_ARTIFACT_DIR
    / "r16_a64_optimization_dev_predictions.csv"
)

STAGE_E_R16A64_METRICS_PATH = (
    STAGE_E_R16A64_ARTIFACT_DIR
    / "r16_a64_optimization_dev_metrics.json"
)

STAGE_E_R16A64_CLASS_METRICS_PATH = (
    STAGE_E_R16A64_ARTIFACT_DIR
    / "r16_a64_optimization_dev_class_metrics.csv"
)

STAGE_E_R16A64_CONFUSION_PATH = (
    STAGE_E_R16A64_ARTIFACT_DIR
    / "r16_a64_optimization_dev_confusion_matrix.csv"
)


# --------------------------------------------------------------------------------------------------
# 2. Safety gate
# --------------------------------------------------------------------------------------------------

assert STAGE_E_R16A64_TRAINING_COMPLETE_PATH.exists(), (
    "DAY 23 STOP: r16/a64 training completion marker missing."
)

assert int(
    stage_e_r16a64_trainer.state.global_step
) == 500, (
    "DAY 23 STOP: r16/a64 trainer is not at step 500."
)

assert len(
    optimization_dev
) == 2708


# --------------------------------------------------------------------------------------------------
# 3. Frozen inference protocol
# --------------------------------------------------------------------------------------------------

INFERENCE_BATCH_SIZE = 8

VALID_LABELS = [
    "negative",
    "neutral",
    "positive",
]


original_padding_side = tokenizer.padding_side

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token


stage_e_r16a64_model.eval()
stage_e_r16a64_model.config.use_cache = True


# --------------------------------------------------------------------------------------------------
# 4. Inference
# --------------------------------------------------------------------------------------------------

all_predictions = []
all_raw_generations = []


inference_start = time.time()


previous_hf_verbosity = hf_logging.get_verbosity()
hf_logging.set_verbosity_error()


try:

    for batch_start in tqdm(
        range(
            0,
            len(optimization_dev),
            INFERENCE_BATCH_SIZE,
        ),
        desc="r16/a64 Dev inference",
    ):

        batch_end = min(
            batch_start + INFERENCE_BATCH_SIZE,
            len(optimization_dev),
        )


        batch_df = optimization_dev.iloc[
            batch_start:batch_end
        ]


        batch_prompts = [
            build_inference_prompt(
                text
            )
            for text
            in batch_df[
                "text"
            ].tolist()
        ]


        encoded = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            add_special_tokens=False,
        )


        encoded = {
            key:
                value.to(
                    stage_e_r16a64_model.device
                )
            for key, value
            in encoded.items()
        }


        input_length = encoded[
            "input_ids"
        ].shape[1]


        with torch.inference_mode():

            generated = stage_e_r16a64_model.generate(
                **encoded,
                do_sample=False,
                max_new_tokens=4,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )


        generated_only = generated[
            :,
            input_length:
        ]


        decoded = tokenizer.batch_decode(
            generated_only,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )


        for raw_text in decoded:

            normalized = (
                raw_text
                .strip()
                .lower()
            )


            all_raw_generations.append(
                raw_text
            )


            if normalized in VALID_LABELS:

                all_predictions.append(
                    normalized
                )

            else:

                all_predictions.append(
                    "__invalid__"
                )


finally:

    hf_logging.set_verbosity(
        previous_hf_verbosity
    )

    tokenizer.padding_side = (
        original_padding_side
    )


inference_runtime_seconds = (
    time.time()
    -
    inference_start
)


# --------------------------------------------------------------------------------------------------
# 5. Labels
# --------------------------------------------------------------------------------------------------

y_true = (
    optimization_dev[
        "sentiment"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .tolist()
)


y_pred = all_predictions


assert len(y_true) == 2708
assert len(y_pred) == 2708


invalid_prediction_count = int(
    sum(
        pred == "__invalid__"
        for pred in y_pred
    )
)


invalid_prediction_rate = (
    invalid_prediction_count
    /
    len(y_pred)
)


# --------------------------------------------------------------------------------------------------
# 6. Overall metrics
# --------------------------------------------------------------------------------------------------

accuracy = float(
    accuracy_score(
        y_true,
        y_pred,
    )
)


macro_f1 = float(
    f1_score(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        average="macro",
        zero_division=0,
    )
)


weighted_f1 = float(
    f1_score(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        average="weighted",
        zero_division=0,
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Class metrics
# --------------------------------------------------------------------------------------------------

(
    class_precision,
    class_recall,
    class_f1,
    class_support,
) = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=VALID_LABELS,
    zero_division=0,
)


class_metrics_df = pd.DataFrame(
    {
        "label": VALID_LABELS,
        "precision": class_precision,
        "recall": class_recall,
        "f1": class_f1,
        "support": class_support.astype(int),
    }
)


negative_recall = float(
    class_metrics_df.loc[
        class_metrics_df["label"] == "negative",
        "recall",
    ].iloc[0]
)


neutral_recall = float(
    class_metrics_df.loc[
        class_metrics_df["label"] == "neutral",
        "recall",
    ].iloc[0]
)


positive_recall = float(
    class_metrics_df.loc[
        class_metrics_df["label"] == "positive",
        "recall",
    ].iloc[0]
)


minority_recall_floor = float(
    min(
        negative_recall,
        positive_recall,
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Confusion matrix
# --------------------------------------------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=VALID_LABELS,
)


confusion_df = pd.DataFrame(
    cm,
    index=[
        f"true_{label}"
        for label
        in VALID_LABELS
    ],
    columns=[
        f"pred_{label}"
        for label
        in VALID_LABELS
    ],
)


# --------------------------------------------------------------------------------------------------
# 9. Prediction artifact
# --------------------------------------------------------------------------------------------------

prediction_df = pd.DataFrame(
    {
        "observation_id":
            optimization_dev[
                "observation_id"
            ].tolist(),

        "date":
            optimization_dev[
                "date"
            ].astype(str).tolist(),

        "true_label":
            y_true,

        "predicted_label":
            y_pred,

        "raw_generation":
            all_raw_generations,
    }
)


prediction_df.to_csv(
    STAGE_E_R16A64_PREDICTIONS_PATH,
    index=False,
)


class_metrics_df.to_csv(
    STAGE_E_R16A64_CLASS_METRICS_PATH,
    index=False,
)


confusion_df.to_csv(
    STAGE_E_R16A64_CONFUSION_PATH
)


# --------------------------------------------------------------------------------------------------
# 10. Compare vs current r8/a32 incumbent
# --------------------------------------------------------------------------------------------------

R8A32_ACCURACY = (
    STAGE_D_WINNER_ACCURACY
)

R8A32_MACRO_F1 = (
    STAGE_D_WINNER_MACRO_F1
)

R8A32_WEIGHTED_F1 = (
    STAGE_D_WINNER_WEIGHTED_F1
)

R8A32_NEG_RECALL = (
    STAGE_D_WINNER_NEG_RECALL
)

R8A32_NEU_RECALL = (
    STAGE_D_WINNER_NEU_RECALL
)

R8A32_POS_RECALL = (
    STAGE_D_WINNER_POS_RECALL
)

R8A32_MINORITY_FLOOR = (
    STAGE_D_WINNER_MINORITY_FLOOR
)

R8A32_EVAL_LOSS = (
    STAGE_D_WINNER_EVAL_LOSS
)


accuracy_delta_vs_r8 = (
    accuracy
    -
    R8A32_ACCURACY
)


macro_f1_delta_vs_r8 = (
    macro_f1
    -
    R8A32_MACRO_F1
)


weighted_f1_delta_vs_r8 = (
    weighted_f1
    -
    R8A32_WEIGHTED_F1
)


negative_recall_delta_vs_r8 = (
    negative_recall
    -
    R8A32_NEG_RECALL
)


neutral_recall_delta_vs_r8 = (
    neutral_recall
    -
    R8A32_NEU_RECALL
)


positive_recall_delta_vs_r8 = (
    positive_recall
    -
    R8A32_POS_RECALL
)


minority_floor_delta_vs_r8 = (
    minority_recall_floor
    -
    R8A32_MINORITY_FLOOR
)


eval_loss_delta_vs_r8 = (
    final_eval_loss
    -
    R8A32_EVAL_LOSS
)


beats_r8_accuracy = bool(
    accuracy
    >
    R8A32_ACCURACY
)


# --------------------------------------------------------------------------------------------------
# 11. Formal checks
# --------------------------------------------------------------------------------------------------

cell19_checks = {

    "dev_population_2708":
        len(y_true)
        == 2708,

    "prediction_count_2708":
        len(y_pred)
        == 2708,

    "zero_invalid_predictions":
        invalid_prediction_count
        == 0,

    "accuracy_finite":
        np.isfinite(accuracy),

    "macro_f1_finite":
        np.isfinite(macro_f1),

    "weighted_f1_finite":
        np.isfinite(weighted_f1),

    "negative_recall_finite":
        np.isfinite(negative_recall),

    "neutral_recall_finite":
        np.isfinite(neutral_recall),

    "positive_recall_finite":
        np.isfinite(positive_recall),

    "confusion_total_2708":
        int(cm.sum())
        == 2708,

    "prediction_artifact_saved":
        STAGE_E_R16A64_PREDICTIONS_PATH.exists(),

    "class_metrics_saved":
        STAGE_E_R16A64_CLASS_METRICS_PATH.exists(),

    "confusion_matrix_saved":
        STAGE_E_R16A64_CONFUSION_PATH.exists(),

    "historical_validation_not_used":
        True,

    "future_not_used":
        True,
}


all_cell19_checks_pass = bool(
    all(
        cell19_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. Save metrics JSON
# --------------------------------------------------------------------------------------------------

stage_e_r16a64_metrics = {

    "day":
        23,

    "stage":
        "E",

    "condition":
        "r16_a64",

    "sampling":
        "natural",

    "learning_rate":
        2e-4,

    "lora": {
        "r": 16,
        "alpha": 64,
        "dropout": 0.1,
    },

    "evaluation_population":
        "optimization_dev",

    "n":
        len(y_true),

    "metrics": {

        "accuracy":
            accuracy,

        "macro_f1":
            macro_f1,

        "weighted_f1":
            weighted_f1,

        "negative_recall":
            negative_recall,

        "neutral_recall":
            neutral_recall,

        "positive_recall":
            positive_recall,

        "minority_recall_floor":
            minority_recall_floor,

        "eval_loss":
            float(
                final_eval_loss
            ),

        "invalid_predictions":
            invalid_prediction_count,

        "invalid_prediction_rate":
            invalid_prediction_rate,
    },

    "comparison_vs_r8_a32": {

        "reference":
            "Stage D winner r8/a32",

        "accuracy_delta":
            float(
                accuracy_delta_vs_r8
            ),

        "macro_f1_delta":
            float(
                macro_f1_delta_vs_r8
            ),

        "weighted_f1_delta":
            float(
                weighted_f1_delta_vs_r8
            ),

        "negative_recall_delta":
            float(
                negative_recall_delta_vs_r8
            ),

        "neutral_recall_delta":
            float(
                neutral_recall_delta_vs_r8
            ),

        "positive_recall_delta":
            float(
                positive_recall_delta_vs_r8
            ),

        "minority_recall_floor_delta":
            float(
                minority_floor_delta_vs_r8
            ),

        "eval_loss_delta":
            float(
                eval_loss_delta_vs_r8
            ),

        "beats_reference_accuracy":
            beats_r8_accuracy,
    },

    "inference": {

        "batch_size":
            INFERENCE_BATCH_SIZE,

        "do_sample":
            False,

        "max_new_tokens":
            4,

        "padding_side":
            "left",

        "runtime_seconds":
            float(
                inference_runtime_seconds
            ),
    },

    "checks": {

        key:
            bool(value)

        for key, value
        in cell19_checks.items()
    },

    "all_checks_pass":
        all_cell19_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_E_R16A64_METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_e_r16a64_metrics,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 13. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE E R16/A64 OPTIMIZATION DEV CLASSIFICATION")
print("=" * 120)


print("\n[EVALUATION POPULATION]")
print(
    "Split                       : Optimization Dev"
)
print(
    f"N                           : "
    f"{len(y_true):,}"
)
print(
    f"Inference runtime           : "
    f"{inference_runtime_seconds / 60:.2f} minutes"
)
print(
    f"Invalid predictions         : "
    f"{invalid_prediction_count}"
)


print("\n[OVERALL METRICS]")
print(
    f"Accuracy                    : "
    f"{accuracy:.6f}"
)
print(
    f"Macro-F1                    : "
    f"{macro_f1:.6f}"
)
print(
    f"Weighted-F1                 : "
    f"{weighted_f1:.6f}"
)


print("\n[CLASS RECALL]")
print(
    f"Negative recall             : "
    f"{negative_recall:.6f}"
)
print(
    f"Neutral recall              : "
    f"{neutral_recall:.6f}"
)
print(
    f"Positive recall             : "
    f"{positive_recall:.6f}"
)
print(
    f"Minority recall floor       : "
    f"{minority_recall_floor:.6f}"
)


print("\n[CLASS METRICS]")
display(
    class_metrics_df
)


print("\n[CONFUSION MATRIX]")
display(
    confusion_df
)


print("\n[COMPARISON VS R8/A32 INCUMBENT]")
print(
    f"r8/a32 Accuracy             : "
    f"{R8A32_ACCURACY:.6f}"
)
print(
    f"r16/a64 Accuracy            : "
    f"{accuracy:.6f}"
)
print(
    f"Accuracy delta              : "
    f"{accuracy_delta_vs_r8:+.6f} "
    f"({accuracy_delta_vs_r8 * 100:+.2f} pp)"
)
print(
    f"Macro-F1 delta              : "
    f"{macro_f1_delta_vs_r8:+.6f}"
)
print(
    f"Weighted-F1 delta           : "
    f"{weighted_f1_delta_vs_r8:+.6f}"
)
print(
    f"Negative recall delta       : "
    f"{negative_recall_delta_vs_r8:+.6f}"
)
print(
    f"Neutral recall delta        : "
    f"{neutral_recall_delta_vs_r8:+.6f}"
)
print(
    f"Positive recall delta       : "
    f"{positive_recall_delta_vs_r8:+.6f}"
)
print(
    f"Minority floor delta        : "
    f"{minority_floor_delta_vs_r8:+.6f}"
)
print(
    f"Eval-loss delta             : "
    f"{eval_loss_delta_vs_r8:+.6f}"
)
print(
    f"Beats r8/a32 Accuracy       : "
    f"{beats_r8_accuracy}"
)


print("\n[FORMAL CHECKS]")

for name, passed in cell19_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Predictions                 : "
    f"{STAGE_E_R16A64_PREDICTIONS_PATH}"
)
print(
    f"Metrics                     : "
    f"{STAGE_E_R16A64_METRICS_PATH}"
)
print(
    f"Class metrics               : "
    f"{STAGE_E_R16A64_CLASS_METRICS_PATH}"
)
print(
    f"Confusion matrix            : "
    f"{STAGE_E_R16A64_CONFUSION_PATH}"
)


print("\n" + "-" * 120)

if all_cell19_checks_pass:

    print(
        "DAY 23 CELL 19 STATUS: PASS — "
        "R16/A64 CLASSIFICATION EVALUATION COMPLETE; "
        "STAGE E TWO-CONDITION SELECTION READY"
    )

else:

    print(
        "DAY 23 CELL 19 STATUS: FAIL — "
        "DO NOT PERFORM STAGE E SELECTION"
    )

print("=" * 120)

r16/a64 Dev inference:   0%|          | 0/339 [00:00<?, ?it/s]

DAY 23 — STAGE E R16/A64 OPTIMIZATION DEV CLASSIFICATION

[EVALUATION POPULATION]
Split                       : Optimization Dev
N                           : 2,708
Inference runtime           : 6.16 minutes
Invalid predictions         : 0

[OVERALL METRICS]
Accuracy                    : 0.852659
Macro-F1                    : 0.820731
Weighted-F1                 : 0.851846

[CLASS RECALL]
Negative recall             : 0.735178
Neutral recall              : 0.903535
Positive recall             : 0.804878
Minority recall floor       : 0.735178

[CLASS METRICS]


,label,precision,recall,f1,support
0,negative,0.826667,0.735178,0.778243,506
1,neutral,0.884457,0.903535,0.893894,1669
2,positive,0.775769,0.804878,0.790055,533



[CONFUSION MATRIX]


,pred_negative,pred_neutral,pred_positive
true_negative,372,110,24
true_neutral,61,1508,100
true_positive,17,87,429



[COMPARISON VS R8/A32 INCUMBENT]
r8/a32 Accuracy             : 0.843427
r16/a64 Accuracy            : 0.852659
Accuracy delta              : +0.009232 (+0.92 pp)
Macro-F1 delta              : +0.007215
Weighted-F1 delta           : +0.008181
Negative recall delta       : -0.015810
Neutral recall delta        : +0.023367
Positive recall delta       : -0.011257
Minority floor delta        : -0.015810
Eval-loss delta             : -0.015697
Beats r8/a32 Accuracy       : True

[FORMAL CHECKS]
PASS   — dev_population_2708
PASS   — prediction_count_2708
PASS   — zero_invalid_predictions
PASS   — accuracy_finite
PASS   — macro_f1_finite
PASS   — weighted_f1_finite
PASS   — negative_recall_finite
PASS   — neutral_recall_finite
PASS   — positive_recall_finite
PASS   — confusion_total_2708
PASS   — prediction_artifact_saved
PASS   — class_metrics_saved
PASS   — confusion_matrix_saved
PASS   — historical_validation_not_used
PASS   — future_not_used

[ARTIFACTS]
Predictions                 : /con

In [28]:
# ==================================================================================================
# DAY 23 — CELL 20
# STAGE E TWO-CONDITION COMPARISON + FORMAL WINNER FREEZE
# ==================================================================================================

import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Required formal artifacts
# --------------------------------------------------------------------------------------------------

required_stage_e_artifacts = {

    "r16a64_metrics":
        STAGE_E_R16A64_METRICS_PATH,

    "r16a64_training_summary":
        STAGE_E_R16A64_TRAINING_SUMMARY_PATH,

    "r16a64_checkpoint":
        stage_e_r16a64_final_checkpoint,

    "r8a32_reference_adapter":
        STAGE_E_REFERENCE_ADAPTER_PATH,
}


for name, path in required_stage_e_artifacts.items():

    assert Path(path).exists(), (
        f"DAY 23 STOP: missing Stage E artifact:\n"
        f"{name}: {path}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Load r16/a64 formal artifacts
# --------------------------------------------------------------------------------------------------

with open(
    STAGE_E_R16A64_METRICS_PATH,
    "r",
    encoding="utf-8",
) as f:

    r16a64_metrics_payload = load_artifact_json(f)


with open(
    STAGE_E_R16A64_TRAINING_SUMMARY_PATH,
    "r",
    encoding="utf-8",
) as f:

    r16a64_training_payload = load_artifact_json(f)


# --------------------------------------------------------------------------------------------------
# 3. Recover r8/a32 incumbent from frozen Stage D winner
# --------------------------------------------------------------------------------------------------

R8A32_ACCURACY = float(
    STAGE_D_WINNER_ACCURACY
)

R8A32_MACRO_F1 = float(
    STAGE_D_WINNER_MACRO_F1
)

R8A32_WEIGHTED_F1 = float(
    STAGE_D_WINNER_WEIGHTED_F1
)

R8A32_NEG_RECALL = float(
    STAGE_D_WINNER_NEG_RECALL
)

R8A32_NEU_RECALL = float(
    STAGE_D_WINNER_NEU_RECALL
)

R8A32_POS_RECALL = float(
    STAGE_D_WINNER_POS_RECALL
)

R8A32_MINORITY_FLOOR = float(
    STAGE_D_WINNER_MINORITY_FLOOR
)

R8A32_EVAL_LOSS = float(
    STAGE_D_WINNER_EVAL_LOSS
)


# --------------------------------------------------------------------------------------------------
# 4. Recover r16/a64 metrics
# --------------------------------------------------------------------------------------------------

r16_metrics = (
    r16a64_metrics_payload[
        "metrics"
    ]
)


R16A64_ACCURACY = float(
    r16_metrics[
        "accuracy"
    ]
)

R16A64_MACRO_F1 = float(
    r16_metrics[
        "macro_f1"
    ]
)

R16A64_WEIGHTED_F1 = float(
    r16_metrics[
        "weighted_f1"
    ]
)

R16A64_NEG_RECALL = float(
    r16_metrics[
        "negative_recall"
    ]
)

R16A64_NEU_RECALL = float(
    r16_metrics[
        "neutral_recall"
    ]
)

R16A64_POS_RECALL = float(
    r16_metrics[
        "positive_recall"
    ]
)

R16A64_MINORITY_FLOOR = float(
    r16_metrics[
        "minority_recall_floor"
    ]
)

R16A64_EVAL_LOSS = float(
    r16a64_training_payload[
        "evaluation"
    ][
        "eval_loss"
    ]
)


# --------------------------------------------------------------------------------------------------
# 5. Construct two-condition comparison table
# --------------------------------------------------------------------------------------------------

stage_e_comparison_df = pd.DataFrame(
    [
        {
            "condition":
                "r8_a32",

            "lora_r":
                8,

            "lora_alpha":
                32,

            "trainable_params":
                3_407_872,

            "accuracy":
                R8A32_ACCURACY,

            "macro_f1":
                R8A32_MACRO_F1,

            "weighted_f1":
                R8A32_WEIGHTED_F1,

            "negative_recall":
                R8A32_NEG_RECALL,

            "neutral_recall":
                R8A32_NEU_RECALL,

            "positive_recall":
                R8A32_POS_RECALL,

            "minority_recall_floor":
                R8A32_MINORITY_FLOOR,

            "eval_loss":
                R8A32_EVAL_LOSS,

            "model_source":
                str(
                    STAGE_E_REFERENCE_ADAPTER_PATH
                ),
        },

        {
            "condition":
                "r16_a64",

            "lora_r":
                16,

            "lora_alpha":
                64,

            "trainable_params":
                6_815_744,

            "accuracy":
                R16A64_ACCURACY,

            "macro_f1":
                R16A64_MACRO_F1,

            "weighted_f1":
                R16A64_WEIGHTED_F1,

            "negative_recall":
                R16A64_NEG_RECALL,

            "neutral_recall":
                R16A64_NEU_RECALL,

            "positive_recall":
                R16A64_POS_RECALL,

            "minority_recall_floor":
                R16A64_MINORITY_FLOOR,

            "eval_loss":
                R16A64_EVAL_LOSS,

            "model_source":
                str(
                    stage_e_r16a64_final_checkpoint
                ),
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 6. Hard identity checks
# --------------------------------------------------------------------------------------------------

identity_checks = {

    "r8a32_accuracy_match":
        abs(
            R8A32_ACCURACY
            -
            0.843427
        )
        <
        1e-6,

    "r8a32_macro_f1_match":
        abs(
            R8A32_MACRO_F1
            -
            0.813516
        )
        <
        1e-6,

    "r8a32_eval_loss_match":
        abs(
            R8A32_EVAL_LOSS
            -
            0.212594
        )
        <
        1e-6,

    "r16a64_accuracy_match":
        abs(
            R16A64_ACCURACY
            -
            0.852659
        )
        <
        1e-6,

    "r16a64_macro_f1_match":
        abs(
            R16A64_MACRO_F1
            -
            0.820731
        )
        <
        1e-6,

    "r16a64_eval_loss_match":
        abs(
            R16A64_EVAL_LOSS
            -
            0.196898
        )
        <
        1e-6,
}


assert all(
    identity_checks.values()
), (
    "DAY 23 STOP: Stage E artifact values do not match "
    "the already observed formal results."
)


# --------------------------------------------------------------------------------------------------
# 7. Ranking
#
# Frozen Stage E rule:
#   Primary   = Accuracy
#   Secondary = Macro-F1
#   Diagnostic = Minority recall floor
#   Later tie-break = Eval loss
#
# Complexity preference for r8/a32 applies only if candidates are effectively tied.
# No post-hoc numerical tie threshold is introduced here.
# --------------------------------------------------------------------------------------------------

stage_e_comparison_df[
    "accuracy_rank"
] = (
    stage_e_comparison_df[
        "accuracy"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)


stage_e_comparison_df[
    "macro_f1_rank"
] = (
    stage_e_comparison_df[
        "macro_f1"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)


stage_e_comparison_df[
    "minority_floor_rank"
] = (
    stage_e_comparison_df[
        "minority_recall_floor"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)


stage_e_comparison_df[
    "eval_loss_rank"
] = (
    stage_e_comparison_df[
        "eval_loss"
    ]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)


stage_e_comparison_df = (
    stage_e_comparison_df
    .sort_values(
        by=[
            "accuracy",
            "macro_f1",
            "minority_recall_floor",
            "eval_loss",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Explicit rows
# --------------------------------------------------------------------------------------------------

r8_row = (
    stage_e_comparison_df.loc[
        stage_e_comparison_df[
            "condition"
        ]
        ==
        "r8_a32"
    ]
    .iloc[0]
)


r16_row = (
    stage_e_comparison_df.loc[
        stage_e_comparison_df[
            "condition"
        ]
        ==
        "r16_a64"
    ]
    .iloc[0]
)


# --------------------------------------------------------------------------------------------------
# 9. Deltas versus r8/a32 incumbent
# --------------------------------------------------------------------------------------------------

for metric in [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "negative_recall",
    "neutral_recall",
    "positive_recall",
    "minority_recall_floor",
    "eval_loss",
]:

    stage_e_comparison_df[
        f"{metric}_delta_vs_r8a32"
    ] = (
        stage_e_comparison_df[
            metric
        ]
        -
        float(
            r8_row[
                metric
            ]
        )
    )


# --------------------------------------------------------------------------------------------------
# 10. Freeze winner
# --------------------------------------------------------------------------------------------------

stage_e_winner_row = (
    stage_e_comparison_df.iloc[
        0
    ]
)


STAGE_E_WINNER = str(
    stage_e_winner_row[
        "condition"
    ]
)

STAGE_E_WINNER_R = int(
    stage_e_winner_row[
        "lora_r"
    ]
)

STAGE_E_WINNER_ALPHA = int(
    stage_e_winner_row[
        "lora_alpha"
    ]
)

STAGE_E_WINNER_ACCURACY = float(
    stage_e_winner_row[
        "accuracy"
    ]
)

STAGE_E_WINNER_MACRO_F1 = float(
    stage_e_winner_row[
        "macro_f1"
    ]
)

STAGE_E_WINNER_WEIGHTED_F1 = float(
    stage_e_winner_row[
        "weighted_f1"
    ]
)

STAGE_E_WINNER_NEG_RECALL = float(
    stage_e_winner_row[
        "negative_recall"
    ]
)

STAGE_E_WINNER_NEU_RECALL = float(
    stage_e_winner_row[
        "neutral_recall"
    ]
)

STAGE_E_WINNER_POS_RECALL = float(
    stage_e_winner_row[
        "positive_recall"
    ]
)

STAGE_E_WINNER_MINORITY_FLOOR = float(
    stage_e_winner_row[
        "minority_recall_floor"
    ]
)

STAGE_E_WINNER_EVAL_LOSS = float(
    stage_e_winner_row[
        "eval_loss"
    ]
)


# --------------------------------------------------------------------------------------------------
# 11. Scientific comparison flags
# --------------------------------------------------------------------------------------------------

r16_beats_accuracy = bool(
    R16A64_ACCURACY
    >
    R8A32_ACCURACY
)


r16_beats_macro_f1 = bool(
    R16A64_MACRO_F1
    >
    R8A32_MACRO_F1
)


r16_beats_weighted_f1 = bool(
    R16A64_WEIGHTED_F1
    >
    R8A32_WEIGHTED_F1
)


r16_lower_eval_loss = bool(
    R16A64_EVAL_LOSS
    <
    R8A32_EVAL_LOSS
)


r16_minority_floor_tradeoff = bool(
    R16A64_MINORITY_FLOOR
    <
    R8A32_MINORITY_FLOOR
)


r16_negative_recall_tradeoff = bool(
    R16A64_NEG_RECALL
    <
    R8A32_NEG_RECALL
)


r16_positive_recall_tradeoff = bool(
    R16A64_POS_RECALL
    <
    R8A32_POS_RECALL
)


r16_neutral_recall_improves = bool(
    R16A64_NEU_RECALL
    >
    R8A32_NEU_RECALL
)


# --------------------------------------------------------------------------------------------------
# 12. Winner source checkpoint
# --------------------------------------------------------------------------------------------------

STAGE_E_WINNER_SOURCE_CHECKPOINT = (
    stage_e_r16a64_final_checkpoint
)


assert STAGE_E_WINNER == "r16_a64"

assert STAGE_E_WINNER_SOURCE_CHECKPOINT.exists()


# --------------------------------------------------------------------------------------------------
# 13. Create clean Stage E selected adapter
#
# Copy only inference-relevant adapter/tokenizer files.
# --------------------------------------------------------------------------------------------------

STAGE_E_SELECTED_ADAPTER_DIR = (
    STAGE_E_DIR
    / "selected_adapter"
)


if STAGE_E_SELECTED_ADAPTER_DIR.exists():

    shutil.rmtree(
        STAGE_E_SELECTED_ADAPTER_DIR
    )


STAGE_E_SELECTED_ADAPTER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


adapter_files_to_copy = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "adapter_model.bin",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
]


copied_adapter_files = []


for filename in adapter_files_to_copy:

    source_file = (
        STAGE_E_WINNER_SOURCE_CHECKPOINT
        / filename
    )


    if source_file.exists():

        destination_file = (
            STAGE_E_SELECTED_ADAPTER_DIR
            / filename
        )


        shutil.copy2(
            source_file,
            destination_file,
        )


        copied_adapter_files.append(
            filename
        )


selected_adapter_has_config = (
    STAGE_E_SELECTED_ADAPTER_DIR
    / "adapter_config.json"
).exists()


selected_adapter_has_weights = (
    (
        STAGE_E_SELECTED_ADAPTER_DIR
        / "adapter_model.safetensors"
    ).exists()
    or
    (
        STAGE_E_SELECTED_ADAPTER_DIR
        / "adapter_model.bin"
    ).exists()
)


# --------------------------------------------------------------------------------------------------
# 14. Formal checks
# --------------------------------------------------------------------------------------------------

stage_e_selection_checks = {

    **identity_checks,

    "two_lora_conditions_present":
        set(
            stage_e_comparison_df[
                "condition"
            ]
        )
        ==
        {
            "r8_a32",
            "r16_a64",
        },

    "winner_is_r16_a64":
        STAGE_E_WINNER
        ==
        "r16_a64",

    "r16_highest_accuracy":
        R16A64_ACCURACY
        ==
        stage_e_comparison_df[
            "accuracy"
        ].max(),

    "r16_highest_macro_f1":
        R16A64_MACRO_F1
        ==
        stage_e_comparison_df[
            "macro_f1"
        ].max(),

    "r16_highest_weighted_f1":
        R16A64_WEIGHTED_F1
        ==
        stage_e_comparison_df[
            "weighted_f1"
        ].max(),

    "r16_lowest_eval_loss":
        R16A64_EVAL_LOSS
        ==
        stage_e_comparison_df[
            "eval_loss"
        ].min(),

    "r16_beats_r8_accuracy":
        r16_beats_accuracy,

    "r16_beats_r8_macro_f1":
        r16_beats_macro_f1,

    "r16_lower_eval_loss":
        r16_lower_eval_loss,

    "minority_floor_tradeoff_recorded":
        r16_minority_floor_tradeoff,

    "negative_recall_tradeoff_recorded":
        r16_negative_recall_tradeoff,

    "positive_recall_tradeoff_recorded":
        r16_positive_recall_tradeoff,

    "neutral_recall_improvement_recorded":
        r16_neutral_recall_improves,

    "selected_adapter_config_exists":
        selected_adapter_has_config,

    "selected_adapter_weights_exist":
        selected_adapter_has_weights,

    "no_posthoc_numeric_tie_threshold":
        True,

    "historical_validation_not_used":
        True,

    "future_not_used":
        True,
}


all_stage_e_selection_checks_pass = bool(
    all(
        stage_e_selection_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 15. Save comparison + selection artifacts
# --------------------------------------------------------------------------------------------------

STAGE_E_COMPARISON_PATH = (
    STAGE_E_ARTIFACT_DIR
    / "stage_e_two_condition_comparison.csv"
)

STAGE_E_SELECTION_SUMMARY_PATH = (
    STAGE_E_ARTIFACT_DIR
    / "stage_e_selection_summary.json"
)


stage_e_comparison_df.to_csv(
    STAGE_E_COMPARISON_PATH,
    index=False,
)


stage_e_selection_summary = {

    "day":
        23,

    "stage":
        "E",

    "objective":
        "LoRA capacity audit under frozen Stage D winner settings.",

    "evaluation_population":
        "optimization_dev",

    "fixed_configuration": {

        "sampling":
            "natural",

        "learning_rate":
            2e-4,

        "max_steps":
            500,

        "seed":
            42,

        "dropout":
            0.1,

        "target_modules": [
            "q_proj",
            "v_proj",
        ],
    },

    "candidates": [
        {
            "condition":
                "r8_a32",

            "r":
                8,

            "alpha":
                32,
        },

        {
            "condition":
                "r16_a64",

            "r":
                16,

            "alpha":
                64,
        },
    ],

    "selection_rule": {

        "primary":
            "accuracy",

        "secondary":
            "macro_f1",

        "diagnostic":
            "minority_recall_floor",

        "later_tie_break":
            "eval_loss",

        "complexity_preference_if_effectively_tied":
            "prefer_r8_a32",

        "posthoc_numeric_tie_threshold_added":
            False,
    },

    "winner": {

        "condition":
            STAGE_E_WINNER,

        "lora_r":
            STAGE_E_WINNER_R,

        "lora_alpha":
            STAGE_E_WINNER_ALPHA,

        "accuracy":
            STAGE_E_WINNER_ACCURACY,

        "macro_f1":
            STAGE_E_WINNER_MACRO_F1,

        "weighted_f1":
            STAGE_E_WINNER_WEIGHTED_F1,

        "negative_recall":
            STAGE_E_WINNER_NEG_RECALL,

        "neutral_recall":
            STAGE_E_WINNER_NEU_RECALL,

        "positive_recall":
            STAGE_E_WINNER_POS_RECALL,

        "minority_recall_floor":
            STAGE_E_WINNER_MINORITY_FLOOR,

        "eval_loss":
            STAGE_E_WINNER_EVAL_LOSS,

        "source_checkpoint":
            str(
                STAGE_E_WINNER_SOURCE_CHECKPOINT
            ),

        "selected_adapter":
            str(
                STAGE_E_SELECTED_ADAPTER_DIR
            ),
    },

    "comparison_vs_r8_a32": {

        "accuracy_delta":
            float(
                R16A64_ACCURACY
                -
                R8A32_ACCURACY
            ),

        "macro_f1_delta":
            float(
                R16A64_MACRO_F1
                -
                R8A32_MACRO_F1
            ),

        "weighted_f1_delta":
            float(
                R16A64_WEIGHTED_F1
                -
                R8A32_WEIGHTED_F1
            ),

        "negative_recall_delta":
            float(
                R16A64_NEG_RECALL
                -
                R8A32_NEG_RECALL
            ),

        "neutral_recall_delta":
            float(
                R16A64_NEU_RECALL
                -
                R8A32_NEU_RECALL
            ),

        "positive_recall_delta":
            float(
                R16A64_POS_RECALL
                -
                R8A32_POS_RECALL
            ),

        "minority_floor_delta":
            float(
                R16A64_MINORITY_FLOOR
                -
                R8A32_MINORITY_FLOOR
            ),

        "eval_loss_delta":
            float(
                R16A64_EVAL_LOSS
                -
                R8A32_EVAL_LOSS
            ),
    },

    "interpretation": {

        "accuracy_improved":
            r16_beats_accuracy,

        "macro_f1_improved":
            r16_beats_macro_f1,

        "weighted_f1_improved":
            r16_beats_weighted_f1,

        "eval_loss_improved":
            r16_lower_eval_loss,

        "minority_recall_floor_tradeoff":
            r16_minority_floor_tradeoff,

        "negative_recall_tradeoff":
            r16_negative_recall_tradeoff,

        "positive_recall_tradeoff":
            r16_positive_recall_tradeoff,

        "neutral_recall_improved":
            r16_neutral_recall_improves,
    },

    "checks": {

        key:
            bool(value)

        for key, value
        in stage_e_selection_checks.items()
    },

    "all_checks_pass":
        all_stage_e_selection_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_E_SELECTION_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_e_selection_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 16. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — STAGE E TWO-CONDITION FORMAL SELECTION")
print("=" * 120)


print("\n[TWO-CONDITION COMPARISON]")

display(
    stage_e_comparison_df[
        [
            "condition",
            "lora_r",
            "lora_alpha",
            "trainable_params",
            "accuracy",
            "macro_f1",
            "weighted_f1",
            "negative_recall",
            "neutral_recall",
            "positive_recall",
            "minority_recall_floor",
            "eval_loss",
            "accuracy_rank",
            "macro_f1_rank",
            "minority_floor_rank",
            "eval_loss_rank",
        ]
    ]
)


print("\n[DELTA VS R8/A32]")

display(
    stage_e_comparison_df[
        [
            "condition",
            "accuracy_delta_vs_r8a32",
            "macro_f1_delta_vs_r8a32",
            "weighted_f1_delta_vs_r8a32",
            "negative_recall_delta_vs_r8a32",
            "neutral_recall_delta_vs_r8a32",
            "positive_recall_delta_vs_r8a32",
            "minority_recall_floor_delta_vs_r8a32",
            "eval_loss_delta_vs_r8a32",
        ]
    ]
)


print("\n[STAGE E WINNER]")
print(
    f"Condition                   : "
    f"{STAGE_E_WINNER}"
)
print(
    f"LoRA                        : "
    f"r={STAGE_E_WINNER_R}, "
    f"alpha={STAGE_E_WINNER_ALPHA}"
)
print(
    f"Accuracy                    : "
    f"{STAGE_E_WINNER_ACCURACY:.6f}"
)
print(
    f"Macro-F1                    : "
    f"{STAGE_E_WINNER_MACRO_F1:.6f}"
)
print(
    f"Weighted-F1                 : "
    f"{STAGE_E_WINNER_WEIGHTED_F1:.6f}"
)
print(
    f"Negative recall             : "
    f"{STAGE_E_WINNER_NEG_RECALL:.6f}"
)
print(
    f"Neutral recall              : "
    f"{STAGE_E_WINNER_NEU_RECALL:.6f}"
)
print(
    f"Positive recall             : "
    f"{STAGE_E_WINNER_POS_RECALL:.6f}"
)
print(
    f"Minority recall floor       : "
    f"{STAGE_E_WINNER_MINORITY_FLOOR:.6f}"
)
print(
    f"Eval loss                   : "
    f"{STAGE_E_WINNER_EVAL_LOSS:.6f}"
)


print("\n[VS R8/A32 INCUMBENT]")
print(
    f"Accuracy delta              : "
    f"{R16A64_ACCURACY - R8A32_ACCURACY:+.6f} "
    f"({(R16A64_ACCURACY - R8A32_ACCURACY) * 100:+.2f} pp)"
)
print(
    f"Macro-F1 delta              : "
    f"{R16A64_MACRO_F1 - R8A32_MACRO_F1:+.6f}"
)
print(
    f"Weighted-F1 delta           : "
    f"{R16A64_WEIGHTED_F1 - R8A32_WEIGHTED_F1:+.6f}"
)
print(
    f"Negative recall delta       : "
    f"{R16A64_NEG_RECALL - R8A32_NEG_RECALL:+.6f}"
)
print(
    f"Neutral recall delta        : "
    f"{R16A64_NEU_RECALL - R8A32_NEU_RECALL:+.6f}"
)
print(
    f"Positive recall delta       : "
    f"{R16A64_POS_RECALL - R8A32_POS_RECALL:+.6f}"
)
print(
    f"Minority floor delta        : "
    f"{R16A64_MINORITY_FLOOR - R8A32_MINORITY_FLOOR:+.6f}"
)
print(
    f"Eval-loss delta             : "
    f"{R16A64_EVAL_LOSS - R8A32_EVAL_LOSS:+.6f}"
)


print("\n[SELECTED ADAPTER]")
print(
    f"Source checkpoint           : "
    f"{STAGE_E_WINNER_SOURCE_CHECKPOINT}"
)
print(
    f"Frozen adapter              : "
    f"{STAGE_E_SELECTED_ADAPTER_DIR}"
)
print(
    f"Copied files                : "
    f"{copied_adapter_files}"
)


print("\n[FORMAL CHECKS]")

for name, passed in stage_e_selection_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Comparison                  : "
    f"{STAGE_E_COMPARISON_PATH}"
)
print(
    f"Selection summary           : "
    f"{STAGE_E_SELECTION_SUMMARY_PATH}"
)


print("\n" + "-" * 120)

if all_stage_e_selection_checks_pass:

    print(
        "DAY 23 CELL 20 STATUS: PASS — "
        "STAGE E COMPLETE; R16/A64 FROZEN AS WINNER; "
        "HISTORICAL TRAINING-SIDE OPTIMIZATION COMPLETE"
    )

else:

    print(
        "DAY 23 CELL 20 STATUS: FAIL — "
        "DO NOT PROCEED"
    )

print("=" * 120)

DAY 23 — STAGE E TWO-CONDITION FORMAL SELECTION

[TWO-CONDITION COMPARISON]


,condition,lora_r,lora_alpha,trainable_params,accuracy,macro_f1,weighted_f1,negative_recall,neutral_recall,positive_recall,minority_recall_floor,eval_loss,accuracy_rank,macro_f1_rank,minority_floor_rank,eval_loss_rank
0,r16_a64,16,64,6815744,0.852659,0.820731,0.851846,0.735178,0.903535,0.804878,0.735178,0.196898,1,1,2,1
1,r8_a32,8,32,3407872,0.843427,0.813516,0.843665,0.750988,0.880168,0.816135,0.750988,0.212594,2,2,1,2



[DELTA VS R8/A32]


,condition,accuracy_delta_vs_r8a32,macro_f1_delta_vs_r8a32,weighted_f1_delta_vs_r8a32,negative_recall_delta_vs_r8a32,neutral_recall_delta_vs_r8a32,positive_recall_delta_vs_r8a32,minority_recall_floor_delta_vs_r8a32,eval_loss_delta_vs_r8a32
0,r16_a64,0.009232,0.007215,0.008181,-0.01581,0.023367,-0.011257,-0.01581,-0.015697
1,r8_a32,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000



[STAGE E WINNER]
Condition                   : r16_a64
LoRA                        : r=16, alpha=64
Accuracy                    : 0.852659
Macro-F1                    : 0.820731
Weighted-F1                 : 0.851846
Negative recall             : 0.735178
Neutral recall              : 0.903535
Positive recall             : 0.804878
Minority recall floor       : 0.735178
Eval loss                   : 0.196898

[VS R8/A32 INCUMBENT]
Accuracy delta              : +0.009232 (+0.92 pp)
Macro-F1 delta              : +0.007215
Weighted-F1 delta           : +0.008181
Negative recall delta       : -0.015810
Neutral recall delta        : +0.023367
Positive recall delta       : -0.011257
Minority floor delta        : -0.015810
Eval-loss delta             : -0.015697

[SELECTED ADAPTER]
Source checkpoint           : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day23/stage_e_lora_capacity_audit/runs/r16_a64/model/checkpoints/checkpoint-500
Frozen adapter              : /content/gdrive_day2

In [30]:
# ==================================================================================================
# DAY 23 — CELL 21A
# HISTORICAL VALIDATION POPULATION RECONSTRUCTION
#
# IMPORTANT:
# - reconstruct population only
# - NO model inference
# - NO performance metrics
# - NO tuning
# ==================================================================================================

import hashlib
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Reconstruct Historical Validation directly from frozen source population
#
# Frozen date range:
# 2025-11-01 through 2025-12-31
# --------------------------------------------------------------------------------------------------

source_dates_utc = pd.to_datetime(
    source_df[
        "date"
    ],
    utc=True,
)


historical_validation_mask = (
    (
        source_dates_utc
        >=
        pd.Timestamp(
            "2025-11-01 00:00:00",
            tz="UTC",
        )
    )
    &
    (
        source_dates_utc
        <
        pd.Timestamp(
            "2026-01-01 00:00:00",
            tz="UTC",
        )
    )
)


historical_validation = (
    source_df.loc[
        historical_validation_mask
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 2. Reconstruct observation_id if needed
# --------------------------------------------------------------------------------------------------

if (
    "observation_id"
    not in historical_validation.columns
):

    historical_validation[
        "observation_id"
    ] = (
        historical_validation[
            "channel_username"
        ].astype(str)
        +
        "::"
        +
        historical_validation[
            "message_id"
        ].astype(str)
    )


# --------------------------------------------------------------------------------------------------
# 3. Frozen identity checks
# --------------------------------------------------------------------------------------------------

historical_validation_dates = pd.to_datetime(
    historical_validation[
        "date"
    ],
    utc=True,
)


historical_validation_label_counts = (
    historical_validation[
        "sentiment"
    ]
    .value_counts()
    .to_dict()
)


historical_validation_start = (
    historical_validation_dates.min()
)


historical_validation_end = (
    historical_validation_dates.max()
)


historical_validation_ordered_ids = (
    historical_validation[
        "observation_id"
    ]
    .astype(str)
    .tolist()
)


historical_validation_id_sha256 = (
    hashlib.sha256(
        "\n".join(
            historical_validation_ordered_ids
        ).encode(
            "utf-8"
        )
    )
    .hexdigest()
)


EXPECTED_HISTVAL_ID_SHA256 = (
    "09ccb61d3ae6cc4e384c7fe5364043a5f7806ca31ee72830324d149f573748bf"
)


histval_reconstruction_checks = {

    "n_5859":
        len(
            historical_validation
        )
        ==
        5859,

    "id_unique":
        historical_validation[
            "observation_id"
        ]
        .nunique()
        ==
        5859,

    "negative_1132":
        int(
            historical_validation_label_counts.get(
                "negative",
                0,
            )
        )
        ==
        1132,

    "neutral_3687":
        int(
            historical_validation_label_counts.get(
                "neutral",
                0,
            )
        )
        ==
        3687,

    "positive_1040":
        int(
            historical_validation_label_counts.get(
                "positive",
                0,
            )
        )
        ==
        1040,

    "date_start_match":
        historical_validation_start
        ==
        pd.Timestamp(
            "2025-11-01 01:50:30",
            tz="UTC",
        ),

    "date_end_match":
        historical_validation_end
        ==
        pd.Timestamp(
            "2025-12-31 21:05:31",
            tz="UTC",
        ),

    "ordered_id_hash_match":
        historical_validation_id_sha256
        ==
        EXPECTED_HISTVAL_ID_SHA256,

    "no_model_inference_performed":
        True,

    "no_performance_metric_computed":
        True,

    "future_not_accessed":
        True,
}


all_histval_reconstruction_checks_pass = bool(
    all(
        histval_reconstruction_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 4. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — HISTORICAL VALIDATION POPULATION RECONSTRUCTION")
print("=" * 120)


print("\n[POPULATION]")
print(
    f"N                           : "
    f"{len(historical_validation):,}"
)
print(
    f"Date start                  : "
    f"{historical_validation_start}"
)
print(
    f"Date end                    : "
    f"{historical_validation_end}"
)
print(
    f"Label counts                : "
    f"{historical_validation_label_counts}"
)
print(
    f"Ordered ID SHA256           : "
    f"{historical_validation_id_sha256}"
)


print("\n[FIREWALL]")
print(
    "Model inference             : NOT PERFORMED"
)
print(
    "Performance metrics         : NOT COMPUTED"
)
print(
    "Historical Validation use   : POPULATION IDENTITY ONLY"
)
print(
    "Future F1-F6                : NOT ACCESSED"
)


print("\n[FORMAL CHECKS]")

for name, passed in histval_reconstruction_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n" + "-" * 120)

if all_histval_reconstruction_checks_pass:

    print(
        "DAY 23 CELL 21A STATUS: PASS — "
        "HISTORICAL VALIDATION POPULATION RECONSTRUCTED EXACTLY; "
        "PERFORMANCE REMAINS UNSEEN"
    )

else:

    print(
        "DAY 23 CELL 21A STATUS: FAIL — "
        "DO NOT PROCEED TO FINAL CONFIGURATION FREEZE"
    )

print("=" * 120)

DAY 23 — HISTORICAL VALIDATION POPULATION RECONSTRUCTION

[POPULATION]
N                           : 5,859
Date start                  : 2025-11-01 01:50:30+00:00
Date end                    : 2025-12-31 21:05:31+00:00
Label counts                : {'neutral': 3687, 'negative': 1132, 'positive': 1040}
Ordered ID SHA256           : 09ccb61d3ae6cc4e384c7fe5364043a5f7806ca31ee72830324d149f573748bf

[FIREWALL]
Model inference             : NOT PERFORMED
Performance metrics         : NOT COMPUTED
Historical Validation use   : POPULATION IDENTITY ONLY
Future F1-F6                : NOT ACCESSED

[FORMAL CHECKS]
PASS   — n_5859
PASS   — id_unique
PASS   — negative_1132
PASS   — neutral_3687
PASS   — positive_1040
PASS   — date_start_match
PASS   — date_end_match
PASS   — ordered_id_hash_match
PASS   — no_model_inference_performed
PASS   — no_performance_metric_computed
PASS   — future_not_accessed

------------------------------------------------------------------------------------------------

In [31]:
# ==================================================================================================
# DAY 23 — CELL 21
# FINAL HISTORICAL MODEL CONFIGURATION FREEZE
#
# Purpose:
#   Freeze the complete training + inference configuration BEFORE the first formal use
#   of Historical Validation.
#
# Historical Validation is NOT evaluated in this cell.
# ==================================================================================================

import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Final configuration directory
# --------------------------------------------------------------------------------------------------

FINAL_FREEZE_DIR = (
    DAY23_RESULTS_ROOT
    / "final_configuration_freeze"
)

FINAL_FREEZE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


FINAL_CONFIG_PATH = (
    FINAL_FREEZE_DIR
    / "final_historical_model_configuration.json"
)

FINAL_FIREWALL_PATH = (
    FINAL_FREEZE_DIR
    / "historical_validation_firewall.json"
)

FINAL_FREEZE_COMPLETE_PATH = (
    FINAL_FREEZE_DIR
    / "final_configuration_freeze_complete.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Final selected adapter
# --------------------------------------------------------------------------------------------------

FINAL_SELECTED_ADAPTER_PATH = (
    STAGE_E_SELECTED_ADAPTER_DIR
)


assert FINAL_SELECTED_ADAPTER_PATH.exists(), (
    "DAY 23 STOP: Stage E selected adapter is missing."
)


FINAL_ADAPTER_CONFIG_PATH = (
    FINAL_SELECTED_ADAPTER_PATH
    / "adapter_config.json"
)


assert FINAL_ADAPTER_CONFIG_PATH.exists(), (
    "DAY 23 STOP: final adapter_config.json is missing."
)


FINAL_ADAPTER_WEIGHTS_PATH = None


for candidate_name in [
    "adapter_model.safetensors",
    "adapter_model.bin",
]:

    candidate_path = (
        FINAL_SELECTED_ADAPTER_PATH
        / candidate_name
    )

    if candidate_path.exists():

        FINAL_ADAPTER_WEIGHTS_PATH = (
            candidate_path
        )

        break


assert FINAL_ADAPTER_WEIGHTS_PATH is not None, (
    "DAY 23 STOP: final adapter weights are missing."
)


# --------------------------------------------------------------------------------------------------
# 3. Hash helper
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    hasher = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:

                break

            hasher.update(
                chunk
            )


    return hasher.hexdigest()


FINAL_ADAPTER_CONFIG_SHA256 = (
    sha256_file(
        FINAL_ADAPTER_CONFIG_PATH
    )
)


FINAL_ADAPTER_WEIGHTS_SHA256 = (
    sha256_file(
        FINAL_ADAPTER_WEIGHTS_PATH
    )
)


# --------------------------------------------------------------------------------------------------
# 4. Reconstruct exact final configuration
# --------------------------------------------------------------------------------------------------

FINAL_BASE_MODEL = (
    "meta-llama/Meta-Llama-3-8B"
)

FINAL_SAMPLING = (
    "natural"
)

FINAL_MAX_STEPS = 500

FINAL_LEARNING_RATE = 2e-4

FINAL_SCHEDULER = (
    "linear"
)

FINAL_WARMUP_STEPS = 10

FINAL_TRAIN_BATCH_SIZE = 1

FINAL_GRADIENT_ACCUMULATION = 8

FINAL_EFFECTIVE_BATCH = 8

FINAL_SEED = 42

FINAL_MAX_SEQ_LENGTH = 512


FINAL_LORA_R = 16

FINAL_LORA_ALPHA = 64

FINAL_LORA_DROPOUT = 0.1

FINAL_LORA_TARGET_MODULES = [
    "q_proj",
    "v_proj",
]


FINAL_QUANTIZATION = {

    "load_in_4bit":
        True,

    "quant_type":
        "nf4",

    "double_quant":
        True,

    "compute_dtype":
        "float16",
}


# --------------------------------------------------------------------------------------------------
# 5. Freeze exact prompt / tokenizer / inference semantics
# --------------------------------------------------------------------------------------------------

FINAL_INSTRUCTION = (
    FROZEN_INSTRUCTION
)

FINAL_PROMPT_TEMPLATE = (
    FROZEN_PROMPT_TEMPLATE
)


EXPECTED_FINAL_INSTRUCTION = (
    "Classify the sentiment of the following financial news as negative, neutral, or positive."
)


EXPECTED_FINAL_PROMPT_TEMPLATE = (
    r"Instruction:{instruction}\nInput:{input}\nAnswer: "
)


FINAL_VALID_LABELS = [
    "negative",
    "neutral",
    "positive",
]


FINAL_INFERENCE_PROTOCOL = {

    "prompt_template":
        FINAL_PROMPT_TEMPLATE,

    "instruction":
        FINAL_INSTRUCTION,

    "padding_side":
        "left",

    "pad_token":
        "eos",

    "pad_token_id":
        int(
            tokenizer.eos_token_id
        ),

    "eos_token_id":
        int(
            tokenizer.eos_token_id
        ),

    "add_special_tokens":
        False,

    "max_input_length":
        512,

    "do_sample":
        False,

    "max_new_tokens":
        4,

    "decode_generated_tokens_only":
        True,

    "skip_special_tokens":
        True,

    "clean_up_tokenization_spaces":
        False,

    "normalization":
        "strip_then_lowercase",

    "valid_labels":
        FINAL_VALID_LABELS,

    "invalid_prediction_policy":
        "__invalid__",
}


# --------------------------------------------------------------------------------------------------
# 6. Frozen optimization-stage performance
#
# IMPORTANT:
# These are NOT Historical Validation metrics.
# They are Optimization Dev selection results only.
# --------------------------------------------------------------------------------------------------

FINAL_OPT_DEV_METRICS = {

    "n":
        2708,

    "accuracy":
        float(
            STAGE_E_WINNER_ACCURACY
        ),

    "macro_f1":
        float(
            STAGE_E_WINNER_MACRO_F1
        ),

    "weighted_f1":
        float(
            STAGE_E_WINNER_WEIGHTED_F1
        ),

    "negative_recall":
        float(
            STAGE_E_WINNER_NEG_RECALL
        ),

    "neutral_recall":
        float(
            STAGE_E_WINNER_NEU_RECALL
        ),

    "positive_recall":
        float(
            STAGE_E_WINNER_POS_RECALL
        ),

    "minority_recall_floor":
        float(
            STAGE_E_WINNER_MINORITY_FLOOR
        ),

    "eval_loss":
        float(
            STAGE_E_WINNER_EVAL_LOSS
        ),
}


# --------------------------------------------------------------------------------------------------
# 7. Historical Validation identity
#
# We identify the population here but DO NOT evaluate it.
# --------------------------------------------------------------------------------------------------

assert len(
    historical_validation
) == 5859, (
    "DAY 23 STOP: Historical Validation population mismatch."
)


historical_validation_start = (
    pd.to_datetime(
        historical_validation[
            "date"
        ],
        utc=True,
    )
    .min()
)


historical_validation_end = (
    pd.to_datetime(
        historical_validation[
            "date"
        ],
        utc=True,
    )
    .max()
)


historical_validation_label_counts = (
    historical_validation[
        "sentiment"
    ]
    .value_counts()
    .to_dict()
)


# --------------------------------------------------------------------------------------------------
# 8. Historical Validation ordered ID hash
# --------------------------------------------------------------------------------------------------

historical_validation_ordered_ids = (
    historical_validation[
        "observation_id"
    ]
    .astype(str)
    .tolist()
)


historical_validation_id_sha256 = (
    hashlib.sha256(
        "\n".join(
            historical_validation_ordered_ids
        ).encode(
            "utf-8"
        )
    )
    .hexdigest()
)


# --------------------------------------------------------------------------------------------------
# 9. Firewall declaration
#
# After this freeze:
#
#   Historical Validation:
#       may be evaluated ONCE for final model assessment.
#
#   Historical Validation may NOT:
#       - choose another checkpoint
#       - change LR
#       - change LoRA
#       - change prompt
#       - change sampling
#       - change threshold / label mapping
#       - trigger another optimization round
#
#   Future F1-F6 remain fully locked.
# --------------------------------------------------------------------------------------------------

FINAL_CONFIGURATION_FROZEN = True

HISTORICAL_VALIDATION_STATUS = (
    "LOCKED_UNTIL_ONE_SHOT_FINAL_EVALUATION"
)

FUTURE_F1_F6_STATUS = (
    "LOCKED"
)

POST_VALIDATION_RETUNING_ALLOWED = False


# --------------------------------------------------------------------------------------------------
# 10. Formal freeze checks
# --------------------------------------------------------------------------------------------------

final_freeze_checks = {

    "stage_e_complete":
        all_stage_e_selection_checks_pass,

    "winner_r16_a64":
        STAGE_E_WINNER
        ==
        "r16_a64",

    "final_sampling_natural":
        FINAL_SAMPLING
        ==
        "natural",

    "final_steps_500":
        FINAL_MAX_STEPS
        ==
        500,

    "final_lr_2e4":
        abs(
            FINAL_LEARNING_RATE
            -
            2e-4
        )
        <
        1e-12,

    "final_lora_r16":
        FINAL_LORA_R
        ==
        16,

    "final_lora_alpha64":
        FINAL_LORA_ALPHA
        ==
        64,

    "final_lora_dropout_0_1":
        abs(
            FINAL_LORA_DROPOUT
            -
            0.1
        )
        <
        1e-12,

    "final_seed_42":
        FINAL_SEED
        ==
        42,

    "final_effective_batch_8":
        FINAL_EFFECTIVE_BATCH
        ==
        8,

    "instruction_exact_match":
        FINAL_INSTRUCTION
        ==
        EXPECTED_FINAL_INSTRUCTION,

    "prompt_template_exact_match":
        FINAL_PROMPT_TEMPLATE
        ==
        EXPECTED_FINAL_PROMPT_TEMPLATE,

    "literal_backslash_n_preserved":
        (
            "\\n"
            in FINAL_PROMPT_TEMPLATE
        ),

    "tokenizer_eos_128001":
        int(
            tokenizer.eos_token_id
        )
        ==
        128001,

    "selected_adapter_exists":
        FINAL_SELECTED_ADAPTER_PATH.exists(),

    "adapter_config_exists":
        FINAL_ADAPTER_CONFIG_PATH.exists(),

    "adapter_weights_exist":
        FINAL_ADAPTER_WEIGHTS_PATH.exists(),

    "adapter_config_hash_created":
        len(
            FINAL_ADAPTER_CONFIG_SHA256
        )
        ==
        64,

    "adapter_weights_hash_created":
        len(
            FINAL_ADAPTER_WEIGHTS_SHA256
        )
        ==
        64,

    "opt_dev_accuracy_match":
        abs(
            FINAL_OPT_DEV_METRICS[
                "accuracy"
            ]
            -
            0.852659
        )
        <
        1e-6,

    "opt_dev_macro_f1_match":
        abs(
            FINAL_OPT_DEV_METRICS[
                "macro_f1"
            ]
            -
            0.820731
        )
        <
        1e-6,

    "historical_validation_n_5859":
        len(
            historical_validation
        )
        ==
        5859,

    "historical_validation_id_unique":
        historical_validation[
            "observation_id"
        ]
        .nunique()
        ==
        5859,

    "historical_validation_still_locked":
        HISTORICAL_VALIDATION_STATUS
        ==
        "LOCKED_UNTIL_ONE_SHOT_FINAL_EVALUATION",

    "post_validation_retuning_forbidden":
        POST_VALIDATION_RETUNING_ALLOWED
        is False,

    "future_f1_f6_locked":
        FUTURE_F1_F6_STATUS
        ==
        "LOCKED",
}


all_final_freeze_checks_pass = bool(
    all(
        final_freeze_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 11. Save final model configuration
# --------------------------------------------------------------------------------------------------

final_configuration_payload = {

    "day":
        23,

    "status":
        "FINAL_HISTORICAL_MODEL_CONFIGURATION_FROZEN",

    "model": {

        "base_model":
            FINAL_BASE_MODEL,

        "selected_adapter":
            str(
                FINAL_SELECTED_ADAPTER_PATH
            ),

        "adapter_config":
            str(
                FINAL_ADAPTER_CONFIG_PATH
            ),

        "adapter_weights":
            str(
                FINAL_ADAPTER_WEIGHTS_PATH
            ),

        "adapter_config_sha256":
            FINAL_ADAPTER_CONFIG_SHA256,

        "adapter_weights_sha256":
            FINAL_ADAPTER_WEIGHTS_SHA256,
    },

    "training_configuration": {

        "sampling":
            FINAL_SAMPLING,

        "max_steps":
            FINAL_MAX_STEPS,

        "learning_rate":
            FINAL_LEARNING_RATE,

        "scheduler":
            FINAL_SCHEDULER,

        "warmup_steps":
            FINAL_WARMUP_STEPS,

        "train_batch_size":
            FINAL_TRAIN_BATCH_SIZE,

        "gradient_accumulation_steps":
            FINAL_GRADIENT_ACCUMULATION,

        "effective_batch":
            FINAL_EFFECTIVE_BATCH,

        "seed":
            FINAL_SEED,

        "max_seq_length":
            FINAL_MAX_SEQ_LENGTH,

        "lora": {

            "r":
                FINAL_LORA_R,

            "alpha":
                FINAL_LORA_ALPHA,

            "dropout":
                FINAL_LORA_DROPOUT,

            "target_modules":
                FINAL_LORA_TARGET_MODULES,
        },

        "quantization":
            FINAL_QUANTIZATION,
    },

    "inference_protocol":
        FINAL_INFERENCE_PROTOCOL,

    "optimization_dev_selection_metrics":
        FINAL_OPT_DEV_METRICS,

    "historical_validation_population": {

        "n":
            int(
                len(
                    historical_validation
                )
            ),

        "date_start":
            historical_validation_start.isoformat(),

        "date_end":
            historical_validation_end.isoformat(),

        "label_counts": {

            str(k):
                int(v)

            for k, v
            in historical_validation_label_counts.items()
        },

        "ordered_observation_id_sha256":
            historical_validation_id_sha256,

        "status":
            HISTORICAL_VALIDATION_STATUS,
    },

    "future_population_status":
        FUTURE_F1_F6_STATUS,

    "post_validation_retuning_allowed":
        POST_VALIDATION_RETUNING_ALLOWED,

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in final_freeze_checks.items()
    },

    "all_checks_pass":
        all_final_freeze_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    FINAL_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        final_configuration_payload,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 12. Save firewall declaration
# --------------------------------------------------------------------------------------------------

historical_validation_firewall = {

    "historical_validation_status":
        HISTORICAL_VALIDATION_STATUS,

    "permitted_next_action":
        "ONE_SHOT_FINAL_EVALUATION",

    "prohibited_uses": [

        "checkpoint_selection",
        "learning_rate_selection",
        "lora_selection",
        "sampling_selection",
        "prompt_selection",
        "label_mapping_selection",
        "threshold_selection",
        "additional_hyperparameter_tuning",
        "model_retraining_based_on_validation_result",
    ],

    "post_evaluation_retuning_allowed":
        False,

    "future_f1_f6_status":
        FUTURE_F1_F6_STATUS,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    FINAL_FIREWALL_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        historical_validation_firewall,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 13. Completion marker
# --------------------------------------------------------------------------------------------------

if all_final_freeze_checks_pass:

    freeze_complete_payload = {

        "status":
            "PASS",

        "configuration_frozen":
            True,

        "selected_adapter":
            str(
                FINAL_SELECTED_ADAPTER_PATH
            ),

        "historical_validation_status":
            HISTORICAL_VALIDATION_STATUS,

        "next_action":
            "ONE_SHOT_HISTORICAL_VALIDATION_EVALUATION",

        "future_f1_f6_status":
            FUTURE_F1_F6_STATUS,

        "created_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


    with open(
        FINAL_FREEZE_COMPLETE_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            freeze_complete_payload,
            f,
            indent=2,
            ensure_ascii=False,
        )


# --------------------------------------------------------------------------------------------------
# 14. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — FINAL HISTORICAL MODEL CONFIGURATION FREEZE")
print("=" * 120)


print("\n[FINAL MODEL]")
print(
    f"Base model                  : "
    f"{FINAL_BASE_MODEL}"
)
print(
    f"Sampling                    : "
    f"{FINAL_SAMPLING}"
)
print(
    f"Training horizon            : "
    f"{FINAL_MAX_STEPS}"
)
print(
    f"Learning rate               : "
    f"{FINAL_LEARNING_RATE}"
)
print(
    f"LoRA                        : "
    f"r={FINAL_LORA_R}, "
    f"alpha={FINAL_LORA_ALPHA}, "
    f"dropout={FINAL_LORA_DROPOUT}"
)
print(
    f"Seed                        : "
    f"{FINAL_SEED}"
)


print("\n[OPTIMIZATION DEV — SELECTION PERFORMANCE]")
print(
    f"Accuracy                    : "
    f"{FINAL_OPT_DEV_METRICS['accuracy']:.6f}"
)
print(
    f"Macro-F1                    : "
    f"{FINAL_OPT_DEV_METRICS['macro_f1']:.6f}"
)
print(
    f"Weighted-F1                 : "
    f"{FINAL_OPT_DEV_METRICS['weighted_f1']:.6f}"
)
print(
    f"Minority recall floor       : "
    f"{FINAL_OPT_DEV_METRICS['minority_recall_floor']:.6f}"
)
print(
    f"Eval loss                   : "
    f"{FINAL_OPT_DEV_METRICS['eval_loss']:.6f}"
)


print("\n[FINAL ADAPTER]")
print(
    f"Directory                   : "
    f"{FINAL_SELECTED_ADAPTER_PATH}"
)
print(
    f"Config SHA256               : "
    f"{FINAL_ADAPTER_CONFIG_SHA256}"
)
print(
    f"Weights SHA256              : "
    f"{FINAL_ADAPTER_WEIGHTS_SHA256}"
)


print("\n[HISTORICAL VALIDATION POPULATION]")
print(
    f"N                           : "
    f"{len(historical_validation):,}"
)
print(
    f"Date start                  : "
    f"{historical_validation_start}"
)
print(
    f"Date end                    : "
    f"{historical_validation_end}"
)
print(
    f"Labels                      : "
    f"{historical_validation_label_counts}"
)
print(
    f"Ordered ID SHA256           : "
    f"{historical_validation_id_sha256}"
)


print("\n[FIREWALL]")
print(
    "Historical Validation       : LOCKED — ONE-SHOT FINAL EVALUATION ONLY"
)
print(
    "Retuning after validation   : FORBIDDEN"
)
print(
    "Future F1-F6                : LOCKED"
)


print("\n[FORMAL CHECKS]")

for name, passed in final_freeze_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Final configuration         : "
    f"{FINAL_CONFIG_PATH}"
)
print(
    f"Validation firewall         : "
    f"{FINAL_FIREWALL_PATH}"
)
print(
    f"Freeze completion marker    : "
    f"{FINAL_FREEZE_COMPLETE_PATH}"
)


print("\n" + "-" * 120)

if all_final_freeze_checks_pass:

    print(
        "DAY 23 CELL 21 STATUS: PASS — "
        "FINAL HISTORICAL MODEL CONFIGURATION FROZEN; "
        "READY FOR ONE-SHOT HISTORICAL VALIDATION EVALUATION"
    )

else:

    print(
        "DAY 23 CELL 21 STATUS: FAIL — "
        "DO NOT ACCESS HISTORICAL VALIDATION PERFORMANCE"
    )

print("=" * 120)

DAY 23 — FINAL HISTORICAL MODEL CONFIGURATION FREEZE

[FINAL MODEL]
Base model                  : meta-llama/Meta-Llama-3-8B
Sampling                    : natural
Training horizon            : 500
Learning rate               : 0.0002
LoRA                        : r=16, alpha=64, dropout=0.1
Seed                        : 42

[OPTIMIZATION DEV — SELECTION PERFORMANCE]
Accuracy                    : 0.852659
Macro-F1                    : 0.820731
Weighted-F1                 : 0.851846
Minority recall floor       : 0.735178
Eval loss                   : 0.196898

[FINAL ADAPTER]
Directory                   : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day23/stage_e_lora_capacity_audit/selected_adapter
Config SHA256               : b83cb88d966256d6250e64b0379f83280ff68a9028892cd4617b0919e9ea78ae
Weights SHA256              : 4e81774686aeeaa6fe7300ef7485bd19dad51f6e85af6b7effb832e597c9dc10

[HISTORICAL VALIDATION POPULATION]
N                           : 5,859
Date start             

In [32]:
# ==================================================================================================
# DAY 23 — CELL 22
# ONE-SHOT HISTORICAL VALIDATION FINAL EVALUATION
#
# IMPORTANT:
#   - final configuration was frozen in Cell 21
#   - this is the FIRST and ONLY model inference on Historical Validation
#   - no tuning / checkpoint selection / prompt modification is permitted
#   - all later Historical Validation analyses must use the saved predictions from this cell
# ==================================================================================================

import gc
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

from tqdm.auto import tqdm

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from transformers.utils import logging as hf_logging

from peft import PeftModel


# --------------------------------------------------------------------------------------------------
# 1. Final evaluation directory
# --------------------------------------------------------------------------------------------------

FINAL_HISTVAL_DIR = (
    DAY23_RESULTS_ROOT
    / "final_historical_validation"
)

FINAL_HISTVAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


FINAL_HISTVAL_PREDICTIONS_PATH = (
    FINAL_HISTVAL_DIR
    / "final_historical_validation_predictions.csv"
)

FINAL_HISTVAL_METRICS_PATH = (
    FINAL_HISTVAL_DIR
    / "final_historical_validation_metrics.json"
)

FINAL_HISTVAL_CLASS_METRICS_PATH = (
    FINAL_HISTVAL_DIR
    / "final_historical_validation_class_metrics.csv"
)

FINAL_HISTVAL_CONFUSION_PATH = (
    FINAL_HISTVAL_DIR
    / "final_historical_validation_confusion_matrix.csv"
)

FINAL_HISTVAL_COMPLETE_PATH = (
    FINAL_HISTVAL_DIR
    / "one_shot_historical_validation_complete.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Irreversibility / one-shot safety gate
#
# If a completed final evaluation already exists, STOP.
# --------------------------------------------------------------------------------------------------

assert FINAL_FREEZE_COMPLETE_PATH.exists(), (
    "DAY 23 STOP: final configuration freeze completion marker is missing."
)


assert all_final_freeze_checks_pass, (
    "DAY 23 STOP: final configuration freeze checks were not all PASS."
)


assert not FINAL_HISTVAL_COMPLETE_PATH.exists(), (
    "DAY 23 STOP: Historical Validation has already been formally evaluated. "
    "Do NOT rerun model inference."
)


assert not FINAL_HISTVAL_METRICS_PATH.exists(), (
    "DAY 23 STOP: Historical Validation metrics artifact already exists."
)


assert not FINAL_HISTVAL_PREDICTIONS_PATH.exists(), (
    "DAY 23 STOP: Historical Validation prediction artifact already exists."
)


# --------------------------------------------------------------------------------------------------
# 3. Historical Validation population identity re-check
# --------------------------------------------------------------------------------------------------

assert len(
    historical_validation
) == 5859


assert (
    historical_validation[
        "observation_id"
    ].nunique()
    ==
    5859
)


current_histval_id_hash = (
    __import__("hashlib")
    .sha256(
        "\n".join(
            historical_validation[
                "observation_id"
            ]
            .astype(str)
            .tolist()
        ).encode(
            "utf-8"
        )
    )
    .hexdigest()
)


assert (
    current_histval_id_hash
    ==
    historical_validation_id_sha256
), (
    "DAY 23 STOP: Historical Validation population identity changed."
)


assert (
    current_histval_id_hash
    ==
    "09ccb61d3ae6cc4e384c7fe5364043a5f7806ca31ee72830324d149f573748bf"
)


# --------------------------------------------------------------------------------------------------
# 4. Frozen adapter identity re-check
# --------------------------------------------------------------------------------------------------

current_adapter_config_hash = (
    sha256_file(
        FINAL_ADAPTER_CONFIG_PATH
    )
)

current_adapter_weights_hash = (
    sha256_file(
        FINAL_ADAPTER_WEIGHTS_PATH
    )
)


assert (
    current_adapter_config_hash
    ==
    FINAL_ADAPTER_CONFIG_SHA256
)


assert (
    current_adapter_weights_hash
    ==
    FINAL_ADAPTER_WEIGHTS_SHA256
)


# --------------------------------------------------------------------------------------------------
# 5. Release training-side model objects
#
# We intentionally reload the FROZEN selected adapter from disk.
# --------------------------------------------------------------------------------------------------

objects_to_release = [
    "stage_e_r16a64_trainer",
    "stage_e_r16a64_model",
    "stage_e_r16a64_base_model",
]


for obj_name in objects_to_release:

    if obj_name in globals():

        del globals()[
            obj_name
        ]


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()


gpu_after_cleanup_gb = (
    torch.cuda.memory_allocated()
    /
    (1024 ** 3)
)


# --------------------------------------------------------------------------------------------------
# 6. Reload frozen base model in 4-bit
# --------------------------------------------------------------------------------------------------

final_eval_bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_use_double_quant=True,

    bnb_4bit_compute_dtype=torch.float16,
)


final_eval_base_model = (
    AutoModelForCausalLM.from_pretrained(

        FINAL_BASE_MODEL,

        quantization_config=
            final_eval_bnb_config,

        device_map=
            "auto",

        trust_remote_code=
            True,
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Load EXACT frozen Stage E selected adapter
# --------------------------------------------------------------------------------------------------

final_eval_model = PeftModel.from_pretrained(

    final_eval_base_model,

    str(
        FINAL_SELECTED_ADAPTER_PATH
    ),

    is_trainable=False,
)


final_eval_model.eval()

final_eval_model.config.use_cache = True


# --------------------------------------------------------------------------------------------------
# 8. Frozen inference protocol
# --------------------------------------------------------------------------------------------------

FINAL_HISTVAL_INFERENCE_BATCH_SIZE = 8

VALID_LABELS = [
    "negative",
    "neutral",
    "positive",
]


assert (
    FINAL_INSTRUCTION
    ==
    FROZEN_INSTRUCTION
)


assert (
    FINAL_PROMPT_TEMPLATE
    ==
    FROZEN_PROMPT_TEMPLATE
)


assert (
    FINAL_PROMPT_TEMPLATE
    ==
    r"Instruction:{instruction}\nInput:{input}\nAnswer: "
)


original_padding_side = (
    tokenizer.padding_side
)

tokenizer.padding_side = "left"

tokenizer.pad_token = tokenizer.eos_token


assert int(
    tokenizer.pad_token_id
) == 128001

assert int(
    tokenizer.eos_token_id
) == 128001


# --------------------------------------------------------------------------------------------------
# 9. One-shot inference
# --------------------------------------------------------------------------------------------------

all_predictions = []
all_raw_generations = []


inference_start = time.time()


previous_hf_verbosity = (
    hf_logging.get_verbosity()
)

hf_logging.set_verbosity_error()


try:

    for batch_start in tqdm(
        range(
            0,
            len(historical_validation),
            FINAL_HISTVAL_INFERENCE_BATCH_SIZE,
        ),
        desc="FINAL Historical Validation inference",
    ):

        batch_end = min(
            batch_start
            +
            FINAL_HISTVAL_INFERENCE_BATCH_SIZE,

            len(
                historical_validation
            ),
        )


        batch_df = (
            historical_validation.iloc[
                batch_start:batch_end
            ]
        )


        batch_prompts = [
            build_inference_prompt(
                text
            )
            for text
            in batch_df[
                "text"
            ].tolist()
        ]


        encoded = tokenizer(

            batch_prompts,

            return_tensors=
                "pt",

            padding=
                True,

            truncation=
                True,

            max_length=
                FINAL_MAX_SEQ_LENGTH,

            add_special_tokens=
                False,
        )


        encoded = {

            key:
                value.to(
                    final_eval_model.device
                )

            for key, value
            in encoded.items()
        }


        input_length = (
            encoded[
                "input_ids"
            ].shape[1]
        )


        with torch.inference_mode():

            generated = (
                final_eval_model.generate(

                    **encoded,

                    do_sample=
                        False,

                    max_new_tokens=
                        4,

                    pad_token_id=
                        tokenizer.pad_token_id,

                    eos_token_id=
                        tokenizer.eos_token_id,

                    use_cache=
                        True,
                )
            )


        generated_only = (
            generated[
                :,
                input_length:
            ]
        )


        decoded = tokenizer.batch_decode(

            generated_only,

            skip_special_tokens=
                True,

            clean_up_tokenization_spaces=
                False,
        )


        for raw_text in decoded:

            normalized = (
                raw_text
                .strip()
                .lower()
            )


            all_raw_generations.append(
                raw_text
            )


            if normalized in VALID_LABELS:

                all_predictions.append(
                    normalized
                )

            else:

                all_predictions.append(
                    "__invalid__"
                )


finally:

    hf_logging.set_verbosity(
        previous_hf_verbosity
    )

    tokenizer.padding_side = (
        original_padding_side
    )


inference_runtime_seconds = (
    time.time()
    -
    inference_start
)


# --------------------------------------------------------------------------------------------------
# 10. Gold labels
# --------------------------------------------------------------------------------------------------

y_true = (
    historical_validation[
        "sentiment"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .tolist()
)


y_pred = (
    all_predictions
)


assert len(
    y_true
) == 5859

assert len(
    y_pred
) == 5859


invalid_prediction_count = int(
    sum(
        pred
        ==
        "__invalid__"

        for pred
        in y_pred
    )
)


invalid_prediction_rate = (
    invalid_prediction_count
    /
    len(
        y_pred
    )
)


# --------------------------------------------------------------------------------------------------
# 11. Final overall metrics
# --------------------------------------------------------------------------------------------------

accuracy = float(
    accuracy_score(
        y_true,
        y_pred,
    )
)


macro_f1 = float(
    f1_score(
        y_true,
        y_pred,

        labels=
            VALID_LABELS,

        average=
            "macro",

        zero_division=
            0,
    )
)


weighted_f1 = float(
    f1_score(
        y_true,
        y_pred,

        labels=
            VALID_LABELS,

        average=
            "weighted",

        zero_division=
            0,
    )
)


# --------------------------------------------------------------------------------------------------
# 12. Final class metrics
# --------------------------------------------------------------------------------------------------

(
    class_precision,
    class_recall,
    class_f1,
    class_support,
) = precision_recall_fscore_support(

    y_true,
    y_pred,

    labels=
        VALID_LABELS,

    zero_division=
        0,
)


class_metrics_df = pd.DataFrame(
    {
        "label":
            VALID_LABELS,

        "precision":
            class_precision,

        "recall":
            class_recall,

        "f1":
            class_f1,

        "support":
            class_support.astype(
                int
            ),
    }
)


negative_recall = float(
    class_metrics_df.loc[
        class_metrics_df[
            "label"
        ]
        ==
        "negative",

        "recall",
    ].iloc[0]
)


neutral_recall = float(
    class_metrics_df.loc[
        class_metrics_df[
            "label"
        ]
        ==
        "neutral",

        "recall",
    ].iloc[0]
)


positive_recall = float(
    class_metrics_df.loc[
        class_metrics_df[
            "label"
        ]
        ==
        "positive",

        "recall",
    ].iloc[0]
)


minority_recall_floor = float(
    min(
        negative_recall,
        positive_recall,
    )
)


# --------------------------------------------------------------------------------------------------
# 13. Final confusion matrix
# --------------------------------------------------------------------------------------------------

cm = confusion_matrix(

    y_true,
    y_pred,

    labels=
        VALID_LABELS,
)


confusion_df = pd.DataFrame(

    cm,

    index=[
        f"true_{label}"
        for label
        in VALID_LABELS
    ],

    columns=[
        f"pred_{label}"
        for label
        in VALID_LABELS
    ],
)


# --------------------------------------------------------------------------------------------------
# 14. Prediction artifact
#
# This becomes the canonical source for all later Historical Validation analyses.
# --------------------------------------------------------------------------------------------------

prediction_df = pd.DataFrame(
    {
        "observation_id":
            historical_validation[
                "observation_id"
            ].astype(str).tolist(),

        "date":
            historical_validation[
                "date"
            ].astype(str).tolist(),

        "source":
            historical_validation[
                "channel_username"
            ].astype(str).tolist(),

        "true_label":
            y_true,

        "predicted_label":
            y_pred,

        "raw_generation":
            all_raw_generations,
    }
)


prediction_df.to_csv(
    FINAL_HISTVAL_PREDICTIONS_PATH,
    index=False,
)


class_metrics_df.to_csv(
    FINAL_HISTVAL_CLASS_METRICS_PATH,
    index=False,
)


confusion_df.to_csv(
    FINAL_HISTVAL_CONFUSION_PATH
)


# --------------------------------------------------------------------------------------------------
# 15. Compare with frozen Optimization Dev
#
# This comparison is descriptive only.
# It cannot trigger retuning.
# --------------------------------------------------------------------------------------------------

accuracy_delta_vs_opt_dev = (
    accuracy
    -
    FINAL_OPT_DEV_METRICS[
        "accuracy"
    ]
)


macro_f1_delta_vs_opt_dev = (
    macro_f1
    -
    FINAL_OPT_DEV_METRICS[
        "macro_f1"
    ]
)


negative_recall_delta_vs_opt_dev = (
    negative_recall
    -
    FINAL_OPT_DEV_METRICS[
        "negative_recall"
    ]
)


neutral_recall_delta_vs_opt_dev = (
    neutral_recall
    -
    FINAL_OPT_DEV_METRICS[
        "neutral_recall"
    ]
)


positive_recall_delta_vs_opt_dev = (
    positive_recall
    -
    FINAL_OPT_DEV_METRICS[
        "positive_recall"
    ]
)


minority_floor_delta_vs_opt_dev = (
    minority_recall_floor
    -
    FINAL_OPT_DEV_METRICS[
        "minority_recall_floor"
    ]
)


# --------------------------------------------------------------------------------------------------
# 16. Prediction artifact hash
# --------------------------------------------------------------------------------------------------

FINAL_HISTVAL_PREDICTIONS_SHA256 = (
    sha256_file(
        FINAL_HISTVAL_PREDICTIONS_PATH
    )
)


# --------------------------------------------------------------------------------------------------
# 17. Formal checks
# --------------------------------------------------------------------------------------------------

final_histval_checks = {

    "configuration_freeze_complete":
        FINAL_FREEZE_COMPLETE_PATH.exists(),

    "frozen_adapter_hash_unchanged":
        (
            current_adapter_weights_hash
            ==
            FINAL_ADAPTER_WEIGHTS_SHA256
        ),

    "frozen_adapter_config_hash_unchanged":
        (
            current_adapter_config_hash
            ==
            FINAL_ADAPTER_CONFIG_SHA256
        ),

    "histval_population_5859":
        len(
            y_true
        )
        ==
        5859,

    "histval_prediction_count_5859":
        len(
            y_pred
        )
        ==
        5859,

    "histval_population_hash_unchanged":
        current_histval_id_hash
        ==
        historical_validation_id_sha256,

    "zero_invalid_predictions":
        invalid_prediction_count
        ==
        0,

    "accuracy_finite":
        np.isfinite(
            accuracy
        ),

    "macro_f1_finite":
        np.isfinite(
            macro_f1
        ),

    "weighted_f1_finite":
        np.isfinite(
            weighted_f1
        ),

    "negative_recall_finite":
        np.isfinite(
            negative_recall
        ),

    "neutral_recall_finite":
        np.isfinite(
            neutral_recall
        ),

    "positive_recall_finite":
        np.isfinite(
            positive_recall
        ),

    "confusion_total_5859":
        int(
            cm.sum()
        )
        ==
        5859,

    "prediction_artifact_saved":
        FINAL_HISTVAL_PREDICTIONS_PATH.exists(),

    "class_metrics_saved":
        FINAL_HISTVAL_CLASS_METRICS_PATH.exists(),

    "confusion_matrix_saved":
        FINAL_HISTVAL_CONFUSION_PATH.exists(),

    "prediction_hash_created":
        len(
            FINAL_HISTVAL_PREDICTIONS_SHA256
        )
        ==
        64,

    "retuning_remains_forbidden":
        POST_VALIDATION_RETUNING_ALLOWED
        is False,

    "future_f1_f6_remain_locked":
        FUTURE_F1_F6_STATUS
        ==
        "LOCKED",
}


all_final_histval_checks_pass = bool(
    all(
        final_histval_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 18. Save final metrics
# --------------------------------------------------------------------------------------------------

final_histval_metrics_payload = {

    "day":
        23,

    "status":
        "FINAL_ONE_SHOT_HISTORICAL_VALIDATION_EVALUATION",

    "model_configuration": {

        "base_model":
            FINAL_BASE_MODEL,

        "selected_adapter":
            str(
                FINAL_SELECTED_ADAPTER_PATH
            ),

        "sampling":
            FINAL_SAMPLING,

        "max_steps":
            FINAL_MAX_STEPS,

        "learning_rate":
            FINAL_LEARNING_RATE,

        "lora_r":
            FINAL_LORA_R,

        "lora_alpha":
            FINAL_LORA_ALPHA,

        "lora_dropout":
            FINAL_LORA_DROPOUT,

        "seed":
            FINAL_SEED,

        "adapter_weights_sha256":
            FINAL_ADAPTER_WEIGHTS_SHA256,
    },

    "evaluation_population": {

        "name":
            "historical_validation",

        "n":
            5859,

        "ordered_observation_id_sha256":
            current_histval_id_hash,
    },

    "metrics": {

        "accuracy":
            accuracy,

        "macro_f1":
            macro_f1,

        "weighted_f1":
            weighted_f1,

        "negative_recall":
            negative_recall,

        "neutral_recall":
            neutral_recall,

        "positive_recall":
            positive_recall,

        "minority_recall_floor":
            minority_recall_floor,

        "invalid_predictions":
            invalid_prediction_count,

        "invalid_prediction_rate":
            invalid_prediction_rate,
    },

    "descriptive_delta_vs_optimization_dev": {

        "accuracy":
            float(
                accuracy_delta_vs_opt_dev
            ),

        "macro_f1":
            float(
                macro_f1_delta_vs_opt_dev
            ),

        "negative_recall":
            float(
                negative_recall_delta_vs_opt_dev
            ),

        "neutral_recall":
            float(
                neutral_recall_delta_vs_opt_dev
            ),

        "positive_recall":
            float(
                positive_recall_delta_vs_opt_dev
            ),

        "minority_recall_floor":
            float(
                minority_floor_delta_vs_opt_dev
            ),
    },

    "inference": {

        "batch_size":
            FINAL_HISTVAL_INFERENCE_BATCH_SIZE,

        "do_sample":
            False,

        "max_new_tokens":
            4,

        "padding_side":
            "left",

        "runtime_seconds":
            float(
                inference_runtime_seconds
            ),

        "runtime_minutes":
            float(
                inference_runtime_seconds
                /
                60.0
            ),
    },

    "prediction_artifact": {

        "path":
            str(
                FINAL_HISTVAL_PREDICTIONS_PATH
            ),

        "sha256":
            FINAL_HISTVAL_PREDICTIONS_SHA256,
    },

    "firewall": {

        "post_validation_retuning_allowed":
            False,

        "future_f1_f6_status":
            FUTURE_F1_F6_STATUS,

        "all_future_historical_validation_analysis_must_use_saved_predictions":
            True,
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in final_histval_checks.items()
    },

    "all_checks_pass":
        all_final_histval_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    FINAL_HISTVAL_METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        final_histval_metrics_payload,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 19. Completion marker
#
# From this point onward, model inference on Historical Validation must NOT be rerun.
# --------------------------------------------------------------------------------------------------

if all_final_histval_checks_pass:

    final_histval_complete_payload = {

        "status":
            "PASS",

        "evaluation":
            "ONE_SHOT_HISTORICAL_VALIDATION",

        "model_inference_complete":
            True,

        "model_inference_must_not_be_repeated":
            True,

        "prediction_artifact":
            str(
                FINAL_HISTVAL_PREDICTIONS_PATH
            ),

        "prediction_sha256":
            FINAL_HISTVAL_PREDICTIONS_SHA256,

        "post_validation_retuning_allowed":
            False,

        "future_f1_f6_status":
            FUTURE_F1_F6_STATUS,

        "created_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


    with open(
        FINAL_HISTVAL_COMPLETE_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            final_histval_complete_payload,
            f,
            indent=2,
            ensure_ascii=False,
        )


# --------------------------------------------------------------------------------------------------
# 20. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — ONE-SHOT HISTORICAL VALIDATION FINAL EVALUATION")
print("=" * 120)


print("\n[FROZEN MODEL]")
print(
    f"Base model                  : "
    f"{FINAL_BASE_MODEL}"
)
print(
    f"Sampling                    : "
    f"{FINAL_SAMPLING}"
)
print(
    f"Training horizon            : "
    f"{FINAL_MAX_STEPS}"
)
print(
    f"Learning rate               : "
    f"{FINAL_LEARNING_RATE}"
)
print(
    f"LoRA                        : "
    f"r={FINAL_LORA_R}, "
    f"alpha={FINAL_LORA_ALPHA}"
)
print(
    f"Adapter weights SHA256      : "
    f"{FINAL_ADAPTER_WEIGHTS_SHA256}"
)


print("\n[EVALUATION POPULATION]")
print(
    "Split                       : Historical Validation"
)
print(
    f"N                           : "
    f"{len(y_true):,}"
)
print(
    f"Inference runtime           : "
    f"{inference_runtime_seconds / 60:.2f} minutes"
)
print(
    f"Invalid predictions         : "
    f"{invalid_prediction_count}"
)


print("\n[FINAL HISTORICAL VALIDATION METRICS]")
print(
    f"Accuracy                    : "
    f"{accuracy:.6f}"
)
print(
    f"Macro-F1                    : "
    f"{macro_f1:.6f}"
)
print(
    f"Weighted-F1                 : "
    f"{weighted_f1:.6f}"
)


print("\n[CLASS RECALL]")
print(
    f"Negative recall             : "
    f"{negative_recall:.6f}"
)
print(
    f"Neutral recall              : "
    f"{neutral_recall:.6f}"
)
print(
    f"Positive recall             : "
    f"{positive_recall:.6f}"
)
print(
    f"Minority recall floor       : "
    f"{minority_recall_floor:.6f}"
)


print("\n[CLASS METRICS]")
display(
    class_metrics_df
)


print("\n[CONFUSION MATRIX]")
display(
    confusion_df
)


print("\n[DESCRIPTIVE DELTA VS OPTIMIZATION DEV]")
print(
    f"Accuracy delta              : "
    f"{accuracy_delta_vs_opt_dev:+.6f} "
    f"({accuracy_delta_vs_opt_dev * 100:+.2f} pp)"
)
print(
    f"Macro-F1 delta              : "
    f"{macro_f1_delta_vs_opt_dev:+.6f}"
)
print(
    f"Negative recall delta       : "
    f"{negative_recall_delta_vs_opt_dev:+.6f}"
)
print(
    f"Neutral recall delta        : "
    f"{neutral_recall_delta_vs_opt_dev:+.6f}"
)
print(
    f"Positive recall delta       : "
    f"{positive_recall_delta_vs_opt_dev:+.6f}"
)
print(
    f"Minority floor delta        : "
    f"{minority_floor_delta_vs_opt_dev:+.6f}"
)


print("\n[FIREWALL AFTER EVALUATION]")
print(
    "Historical Validation       : EVALUATED ONCE"
)
print(
    "Model inference rerun       : FORBIDDEN"
)
print(
    "Retuning                    : FORBIDDEN"
)
print(
    "Future F1-F6                : LOCKED"
)
print(
    "Further HistVal analysis    : SAVED PREDICTIONS ONLY"
)


print("\n[FORMAL CHECKS]")

for name, passed in final_histval_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Predictions                 : "
    f"{FINAL_HISTVAL_PREDICTIONS_PATH}"
)
print(
    f"Predictions SHA256          : "
    f"{FINAL_HISTVAL_PREDICTIONS_SHA256}"
)
print(
    f"Metrics                     : "
    f"{FINAL_HISTVAL_METRICS_PATH}"
)
print(
    f"Class metrics               : "
    f"{FINAL_HISTVAL_CLASS_METRICS_PATH}"
)
print(
    f"Confusion matrix            : "
    f"{FINAL_HISTVAL_CONFUSION_PATH}"
)
print(
    f"Completion marker           : "
    f"{FINAL_HISTVAL_COMPLETE_PATH}"
)


print("\n" + "-" * 120)

if all_final_histval_checks_pass:

    print(
        "DAY 23 CELL 22 STATUS: PASS — "
        "ONE-SHOT HISTORICAL VALIDATION EVALUATION COMPLETE; "
        "FINAL PREDICTIONS FROZEN; RETUNING FORBIDDEN"
    )

else:

    print(
        "DAY 23 CELL 22 STATUS: FAIL — "
        "DO NOT INTERPRET HISTORICAL VALIDATION RESULTS"
    )

print("=" * 120)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

FINAL Historical Validation inference:   0%|          | 0/733 [00:00<?, ?it/s]

DAY 23 — ONE-SHOT HISTORICAL VALIDATION FINAL EVALUATION

[FROZEN MODEL]
Base model                  : meta-llama/Meta-Llama-3-8B
Sampling                    : natural
Training horizon            : 500
Learning rate               : 0.0002
LoRA                        : r=16, alpha=64
Adapter weights SHA256      : 4e81774686aeeaa6fe7300ef7485bd19dad51f6e85af6b7effb832e597c9dc10

[EVALUATION POPULATION]
Split                       : Historical Validation
N                           : 5,859
Inference runtime           : 12.83 minutes
Invalid predictions         : 0

[FINAL HISTORICAL VALIDATION METRICS]
Accuracy                    : 0.838710
Macro-F1                    : 0.804130
Weighted-F1                 : 0.839092

[CLASS RECALL]
Negative recall             : 0.739399
Neutral recall              : 0.877678
Positive recall             : 0.808654
Minority recall floor       : 0.739399

[CLASS METRICS]


,label,precision,recall,f1,support
0,negative,0.780784,0.739399,0.759528,1132
1,neutral,0.888523,0.877678,0.883067,3687
2,positive,0.734498,0.808654,0.769794,1040



[CONFUSION MATRIX]


,pred_negative,pred_neutral,pred_positive
true_negative,837,247,48
true_neutral,195,3236,256
true_positive,40,159,841



[DESCRIPTIVE DELTA VS OPTIMIZATION DEV]
Accuracy delta              : -0.013949 (-1.39 pp)
Macro-F1 delta              : -0.016601
Negative recall delta       : +0.004221
Neutral recall delta        : -0.025857
Positive recall delta       : +0.003776
Minority floor delta        : +0.004221

[FIREWALL AFTER EVALUATION]
Historical Validation       : EVALUATED ONCE
Model inference rerun       : FORBIDDEN
Retuning                    : FORBIDDEN
Future F1-F6                : LOCKED
Further HistVal analysis    : SAVED PREDICTIONS ONLY

[FORMAL CHECKS]
PASS   — configuration_freeze_complete
PASS   — frozen_adapter_hash_unchanged
PASS   — frozen_adapter_config_hash_unchanged
PASS   — histval_population_5859
PASS   — histval_prediction_count_5859
PASS   — histval_population_hash_unchanged
PASS   — zero_invalid_predictions
PASS   — accuracy_finite
PASS   — macro_f1_finite
PASS   — weighted_f1_finite
PASS   — negative_recall_finite
PASS   — neutral_recall_finite
PASS   — positive_recall_finite
P

In [33]:
# ==================================================================================================
# DAY 23 — CELL 23
# HISTORICAL VALIDATION BOOTSTRAP UNCERTAINTY ANALYSIS
#
# IMPORTANT:
# - NO model inference
# - uses ONLY the frozen prediction artifact from Cell 22
# - bootstrap uncertainty for final model metrics
# - Day15 comparison is descriptive only unless Day15 per-sample predictions are later recovered
# ==================================================================================================

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
)


# --------------------------------------------------------------------------------------------------
# 1. Bootstrap output directory
# --------------------------------------------------------------------------------------------------

FINAL_HISTVAL_BOOTSTRAP_DIR = (
    FINAL_HISTVAL_DIR
    / "bootstrap"
)

FINAL_HISTVAL_BOOTSTRAP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


FINAL_HISTVAL_BOOTSTRAP_DRAWS_PATH = (
    FINAL_HISTVAL_BOOTSTRAP_DIR
    / "historical_validation_bootstrap_draws.csv"
)

FINAL_HISTVAL_BOOTSTRAP_SUMMARY_PATH = (
    FINAL_HISTVAL_BOOTSTRAP_DIR
    / "historical_validation_bootstrap_summary.csv"
)

FINAL_HISTVAL_DAY15_COMPARISON_PATH = (
    FINAL_HISTVAL_BOOTSTRAP_DIR
    / "final_vs_day15_descriptive_comparison.csv"
)

FINAL_HISTVAL_BOOTSTRAP_MANIFEST_PATH = (
    FINAL_HISTVAL_BOOTSTRAP_DIR
    / "historical_validation_bootstrap_manifest.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Safety gate
# --------------------------------------------------------------------------------------------------

assert FINAL_HISTVAL_COMPLETE_PATH.exists(), (
    "DAY 23 STOP: one-shot Historical Validation completion marker missing."
)


assert FINAL_HISTVAL_PREDICTIONS_PATH.exists(), (
    "DAY 23 STOP: frozen Historical Validation predictions missing."
)


assert (
    sha256_file(
        FINAL_HISTVAL_PREDICTIONS_PATH
    )
    ==
    FINAL_HISTVAL_PREDICTIONS_SHA256
), (
    "DAY 23 STOP: frozen Historical Validation prediction artifact changed."
)


# --------------------------------------------------------------------------------------------------
# 3. Load ONLY frozen predictions
# --------------------------------------------------------------------------------------------------

histval_predictions = pd.read_csv(
    FINAL_HISTVAL_PREDICTIONS_PATH
)


assert len(
    histval_predictions
) == 5859


required_columns = {
    "observation_id",
    "true_label",
    "predicted_label",
}


assert required_columns.issubset(
    histval_predictions.columns
)


assert (
    histval_predictions[
        "observation_id"
    ].nunique()
    ==
    5859
)


# --------------------------------------------------------------------------------------------------
# 4. Frozen labels
# --------------------------------------------------------------------------------------------------

VALID_LABELS = [
    "negative",
    "neutral",
    "positive",
]


y_true = (
    histval_predictions[
        "true_label"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .to_numpy()
)


y_pred = (
    histval_predictions[
        "predicted_label"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .to_numpy()
)


assert set(
    np.unique(
        y_true
    )
).issubset(
    set(
        VALID_LABELS
    )
)


assert set(
    np.unique(
        y_pred
    )
).issubset(
    set(
        VALID_LABELS
    )
)


# --------------------------------------------------------------------------------------------------
# 5. Recompute point estimates from frozen predictions
# --------------------------------------------------------------------------------------------------

point_accuracy = float(
    accuracy_score(
        y_true,
        y_pred,
    )
)


point_macro_f1 = float(
    f1_score(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        average="macro",
        zero_division=0,
    )
)


point_recall = recall_score(
    y_true,
    y_pred,
    labels=VALID_LABELS,
    average=None,
    zero_division=0,
)


point_negative_recall = float(
    point_recall[0]
)

point_neutral_recall = float(
    point_recall[1]
)

point_positive_recall = float(
    point_recall[2]
)

point_minority_floor = float(
    min(
        point_negative_recall,
        point_positive_recall,
    )
)


# --------------------------------------------------------------------------------------------------
# 6. Identity check against Cell 22
# --------------------------------------------------------------------------------------------------

point_estimate_checks = {

    "accuracy_matches_cell22":
        abs(
            point_accuracy
            -
            0.838710
        )
        <
        1e-6,

    "macro_f1_matches_cell22":
        abs(
            point_macro_f1
            -
            0.804130
        )
        <
        1e-6,

    "negative_recall_matches_cell22":
        abs(
            point_negative_recall
            -
            0.739399
        )
        <
        1e-6,

    "neutral_recall_matches_cell22":
        abs(
            point_neutral_recall
            -
            0.877678
        )
        <
        1e-6,

    "positive_recall_matches_cell22":
        abs(
            point_positive_recall
            -
            0.808654
        )
        <
        1e-6,
}


assert all(
    point_estimate_checks.values()
), (
    "DAY 23 STOP: recomputed metrics do not reproduce Cell 22."
)


# --------------------------------------------------------------------------------------------------
# 7. Bootstrap protocol
# --------------------------------------------------------------------------------------------------

BOOTSTRAP_REPLICATES = 1000
BOOTSTRAP_SEED = 42
CI_LEVEL = 0.95

BOOTSTRAP_ALPHA = (
    1.0
    -
    CI_LEVEL
)

LOWER_Q = (
    BOOTSTRAP_ALPHA
    /
    2.0
)

UPPER_Q = (
    1.0
    -
    LOWER_Q
)


rng = np.random.default_rng(
    BOOTSTRAP_SEED
)


n = len(
    y_true
)


# --------------------------------------------------------------------------------------------------
# 8. Bootstrap
#
# Nonparametric row bootstrap.
# Historical Validation predictions remain fixed.
# --------------------------------------------------------------------------------------------------

bootstrap_records = []


for bootstrap_id in range(
    1,
    BOOTSTRAP_REPLICATES + 1,
):

    sampled_index = rng.integers(
        low=0,
        high=n,
        size=n,
    )


    boot_true = y_true[
        sampled_index
    ]

    boot_pred = y_pred[
        sampled_index
    ]


    boot_accuracy = float(
        accuracy_score(
            boot_true,
            boot_pred,
        )
    )


    boot_macro_f1 = float(
        f1_score(
            boot_true,
            boot_pred,
            labels=VALID_LABELS,
            average="macro",
            zero_division=0,
        )
    )


    boot_recall = recall_score(
        boot_true,
        boot_pred,
        labels=VALID_LABELS,
        average=None,
        zero_division=0,
    )


    boot_negative_recall = float(
        boot_recall[0]
    )

    boot_neutral_recall = float(
        boot_recall[1]
    )

    boot_positive_recall = float(
        boot_recall[2]
    )


    boot_minority_floor = float(
        min(
            boot_negative_recall,
            boot_positive_recall,
        )
    )


    bootstrap_records.append(
        {
            "bootstrap_id":
                bootstrap_id,

            "accuracy":
                boot_accuracy,

            "macro_f1":
                boot_macro_f1,

            "negative_recall":
                boot_negative_recall,

            "neutral_recall":
                boot_neutral_recall,

            "positive_recall":
                boot_positive_recall,

            "minority_recall_floor":
                boot_minority_floor,
        }
    )


bootstrap_df = pd.DataFrame(
    bootstrap_records
)


# --------------------------------------------------------------------------------------------------
# 9. Save bootstrap draws
# --------------------------------------------------------------------------------------------------

bootstrap_df.to_csv(
    FINAL_HISTVAL_BOOTSTRAP_DRAWS_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 10. Percentile confidence intervals
# --------------------------------------------------------------------------------------------------

metric_point_estimates = {

    "accuracy":
        point_accuracy,

    "macro_f1":
        point_macro_f1,

    "negative_recall":
        point_negative_recall,

    "neutral_recall":
        point_neutral_recall,

    "positive_recall":
        point_positive_recall,

    "minority_recall_floor":
        point_minority_floor,
}


bootstrap_summary_records = []


for metric_name, point_value in metric_point_estimates.items():

    values = (
        bootstrap_df[
            metric_name
        ]
        .to_numpy()
    )


    ci_lower = float(
        np.quantile(
            values,
            LOWER_Q,
        )
    )


    ci_upper = float(
        np.quantile(
            values,
            UPPER_Q,
        )
    )


    bootstrap_mean = float(
        np.mean(
            values
        )
    )


    bootstrap_sd = float(
        np.std(
            values,
            ddof=1,
        )
    )


    bootstrap_summary_records.append(
        {
            "metric":
                metric_name,

            "point_estimate":
                float(
                    point_value
                ),

            "bootstrap_mean":
                bootstrap_mean,

            "bootstrap_sd":
                bootstrap_sd,

            "ci_level":
                CI_LEVEL,

            "ci_lower":
                ci_lower,

            "ci_upper":
                ci_upper,

            "replicates":
                BOOTSTRAP_REPLICATES,

            "seed":
                BOOTSTRAP_SEED,
        }
    )


bootstrap_summary_df = pd.DataFrame(
    bootstrap_summary_records
)


bootstrap_summary_df.to_csv(
    FINAL_HISTVAL_BOOTSTRAP_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 11. Day15 baseline aggregate metrics
#
# NOTE:
# These are the already frozen Day15 Historical Validation point estimates.
# No Day15 per-sample prediction artifact is used here.
# Therefore:
#   - point-estimate deltas are valid descriptive summaries
#   - paired bootstrap CI for the improvement is NOT available in this cell
# --------------------------------------------------------------------------------------------------

DAY15_HISTVAL_METRICS = {

    "accuracy":
        0.7890424987,

    "macro_f1":
        0.7456777708,

    "negative_recall":
        0.639576,

    "neutral_recall":
        0.842148,

    "positive_recall":
        0.763462,
}


final_vs_day15_records = []


for metric_name in [
    "accuracy",
    "macro_f1",
    "negative_recall",
    "neutral_recall",
    "positive_recall",
]:

    final_value = float(
        metric_point_estimates[
            metric_name
        ]
    )

    baseline_value = float(
        DAY15_HISTVAL_METRICS[
            metric_name
        ]
    )

    delta = (
        final_value
        -
        baseline_value
    )


    final_vs_day15_records.append(
        {
            "metric":
                metric_name,

            "day15_baseline":
                baseline_value,

            "day23_final":
                final_value,

            "absolute_delta":
                float(
                    delta
                ),

            "delta_percentage_points":
                float(
                    delta
                    *
                    100.0
                ),

            "comparison_type":
                "descriptive_point_estimate_only",

            "paired_uncertainty_available":
                False,
        }
    )


final_vs_day15_df = pd.DataFrame(
    final_vs_day15_records
)


final_vs_day15_df.to_csv(
    FINAL_HISTVAL_DAY15_COMPARISON_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 12. Formal checks
# --------------------------------------------------------------------------------------------------

bootstrap_checks = {

    **point_estimate_checks,

    "prediction_artifact_hash_unchanged":
        (
            sha256_file(
                FINAL_HISTVAL_PREDICTIONS_PATH
            )
            ==
            FINAL_HISTVAL_PREDICTIONS_SHA256
        ),

    "bootstrap_replicates_1000":
        len(
            bootstrap_df
        )
        ==
        1000,

    "bootstrap_seed_42":
        BOOTSTRAP_SEED
        ==
        42,

    "all_bootstrap_metrics_finite":
        np.isfinite(
            bootstrap_df[
                [
                    "accuracy",
                    "macro_f1",
                    "negative_recall",
                    "neutral_recall",
                    "positive_recall",
                    "minority_recall_floor",
                ]
            ].to_numpy()
        ).all(),

    "all_ci_bounds_finite":
        np.isfinite(
            bootstrap_summary_df[
                [
                    "ci_lower",
                    "ci_upper",
                ]
            ].to_numpy()
        ).all(),

    "all_ci_lower_le_point":
        all(
            row[
                "ci_lower"
            ]
            <=
            row[
                "point_estimate"
            ]
            for _, row
            in bootstrap_summary_df.iterrows()
        ),

    "all_ci_upper_ge_point":
        all(
            row[
                "ci_upper"
            ]
            >=
            row[
                "point_estimate"
            ]
            for _, row
            in bootstrap_summary_df.iterrows()
        ),

    "day15_comparison_is_descriptive_only":
        (
            final_vs_day15_df[
                "comparison_type"
            ]
            ==
            "descriptive_point_estimate_only"
        ).all(),

    "no_paired_significance_claim":
        (
            final_vs_day15_df[
                "paired_uncertainty_available"
            ]
            ==
            False
        ).all(),

    "no_model_inference":
        True,

    "no_retuning":
        True,

    "future_not_used":
        True,
}


all_bootstrap_checks_pass = bool(
    all(
        bootstrap_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 13. Save bootstrap manifest
# --------------------------------------------------------------------------------------------------

bootstrap_manifest = {

    "day":
        23,

    "analysis":
        "historical_validation_bootstrap_uncertainty",

    "source_prediction_artifact":
        str(
            FINAL_HISTVAL_PREDICTIONS_PATH
        ),

    "source_prediction_sha256":
        FINAL_HISTVAL_PREDICTIONS_SHA256,

    "bootstrap_protocol": {

        "method":
            "nonparametric_row_bootstrap",

        "replicates":
            BOOTSTRAP_REPLICATES,

        "seed":
            BOOTSTRAP_SEED,

        "confidence_level":
            CI_LEVEL,

        "interval":
            "percentile",
    },

    "day15_comparison": {

        "status":
            "DESCRIPTIVE_POINT_ESTIMATE_ONLY",

        "reason":
            "Day15 per-sample Historical Validation predictions have not yet been recovered in the current formal workflow.",

        "paired_bootstrap_performed":
            False,

        "significance_claim_permitted":
            False,
    },

    "firewall": {

        "model_inference_performed":
            False,

        "historical_validation_predictions_reused_only":
            True,

        "retuning_performed":
            False,

        "future_f1_f6_used":
            False,
    },

    "checks": {

        key:
            bool(value)

        for key, value
        in bootstrap_checks.items()
    },

    "all_checks_pass":
        all_bootstrap_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    FINAL_HISTVAL_BOOTSTRAP_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        bootstrap_manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 14. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — HISTORICAL VALIDATION BOOTSTRAP UNCERTAINTY ANALYSIS")
print("=" * 120)


print("\n[SOURCE]")
print(
    "Historical Validation       : frozen Cell 22 predictions only"
)
print(
    f"N                           : {len(y_true):,}"
)
print(
    f"Bootstrap replicates        : {BOOTSTRAP_REPLICATES:,}"
)
print(
    f"Bootstrap seed              : {BOOTSTRAP_SEED}"
)
print(
    f"CI                           : {CI_LEVEL * 100:.0f}% percentile"
)


print("\n[FINAL MODEL BOOTSTRAP CI]")

display(
    bootstrap_summary_df[
        [
            "metric",
            "point_estimate",
            "ci_lower",
            "ci_upper",
            "bootstrap_sd",
        ]
    ]
)


print("\n[DESCRIPTIVE COMPARISON VS DAY15 BASELINE]")

display(
    final_vs_day15_df[
        [
            "metric",
            "day15_baseline",
            "day23_final",
            "absolute_delta",
            "delta_percentage_points",
        ]
    ]
)


print("\n[METHODOLOGICAL STATUS]")
print(
    "Final model bootstrap CI    : VALID"
)
print(
    "Day15 point-estimate delta  : VALID DESCRIPTIVE COMPARISON"
)
print(
    "Paired bootstrap delta CI   : NOT YET AVAILABLE"
)
print(
    "Significance claim vs Day15 : NOT PERMITTED YET"
)
print(
    "Model inference             : NOT PERFORMED"
)
print(
    "Retuning                    : NOT PERFORMED"
)
print(
    "Future F1-F6                : NOT USED"
)


print("\n[FORMAL CHECKS]")

for name, passed in bootstrap_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Bootstrap draws             : "
    f"{FINAL_HISTVAL_BOOTSTRAP_DRAWS_PATH}"
)
print(
    f"Bootstrap summary           : "
    f"{FINAL_HISTVAL_BOOTSTRAP_SUMMARY_PATH}"
)
print(
    f"Day15 comparison            : "
    f"{FINAL_HISTVAL_DAY15_COMPARISON_PATH}"
)
print(
    f"Bootstrap manifest          : "
    f"{FINAL_HISTVAL_BOOTSTRAP_MANIFEST_PATH}"
)


print("\n" + "-" * 120)

if all_bootstrap_checks_pass:

    print(
        "DAY 23 CELL 23 STATUS: PASS — "
        "HISTORICAL VALIDATION BOOTSTRAP CI COMPLETE; "
        "DAY15 IMPROVEMENT RECORDED DESCRIPTIVELY WITHOUT OVERCLAIMING SIGNIFICANCE"
    )

else:

    print(
        "DAY 23 CELL 23 STATUS: FAIL — "
        "DO NOT INTERPRET BOOTSTRAP RESULTS"
    )

print("=" * 120)

DAY 23 — HISTORICAL VALIDATION BOOTSTRAP UNCERTAINTY ANALYSIS

[SOURCE]
Historical Validation       : frozen Cell 22 predictions only
N                           : 5,859
Bootstrap replicates        : 1,000
Bootstrap seed              : 42
CI                           : 95% percentile

[FINAL MODEL BOOTSTRAP CI]


,metric,point_estimate,ci_lower,ci_upper,bootstrap_sd
0,accuracy,0.838710,0.829830,0.847585,0.004638
1,macro_f1,0.804130,0.793078,0.814513,0.005676
2,negative_recall,0.739399,0.712372,0.764470,0.013297
3,neutral_recall,0.877678,0.867205,0.887574,0.005340
4,positive_recall,0.808654,0.783965,0.832874,0.012100
5,minority_recall_floor,0.739399,0.712372,0.764470,0.013297



[DESCRIPTIVE COMPARISON VS DAY15 BASELINE]


,metric,day15_baseline,day23_final,absolute_delta,delta_percentage_points
0,accuracy,0.789042,0.838710,0.049667,4.966718
1,macro_f1,0.745678,0.804130,0.058452,5.845205
2,negative_recall,0.639576,0.739399,0.099823,9.982329
3,neutral_recall,0.842148,0.877678,0.035530,3.553033
4,positive_recall,0.763462,0.808654,0.045192,4.519185



[METHODOLOGICAL STATUS]
Final model bootstrap CI    : VALID
Day15 point-estimate delta  : VALID DESCRIPTIVE COMPARISON
Paired bootstrap delta CI   : NOT YET AVAILABLE
Significance claim vs Day15 : NOT PERMITTED YET
Model inference             : NOT PERFORMED
Retuning                    : NOT PERFORMED
Future F1-F6                : NOT USED

[FORMAL CHECKS]
PASS   — accuracy_matches_cell22
PASS   — macro_f1_matches_cell22
PASS   — negative_recall_matches_cell22
PASS   — neutral_recall_matches_cell22
PASS   — positive_recall_matches_cell22
PASS   — prediction_artifact_hash_unchanged
PASS   — bootstrap_replicates_1000
PASS   — bootstrap_seed_42
PASS   — all_bootstrap_metrics_finite
PASS   — all_ci_bounds_finite
PASS   — all_ci_lower_le_point
PASS   — all_ci_upper_ge_point
PASS   — day15_comparison_is_descriptive_only
PASS   — no_paired_significance_claim
PASS   — no_model_inference
PASS   — no_retuning
PASS   — future_not_used

[ARTIFACTS]
Bootstrap draws             : /content/gdrive_da

In [34]:
# ==================================================================================================
# DAY 23 — CELL 24
# DAY15 HISTORICAL VALIDATION PREDICTION ARTIFACT RECOVERY AUDIT
#
# PURPOSE:
#   Search existing Day15 artifacts for per-sample Historical Validation predictions.
#
# IMPORTANT:
#   - NO model inference
#   - NO retraining
#   - NO model loading
#   - NO performance-based selection
#   - only artifact discovery + structural validation
# ==================================================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Search roots
# --------------------------------------------------------------------------------------------------

DAY15_RESULTS_ROOT = (
    PROJECT_ROOT
    / "results"
    / "day15"
)


assert DAY15_RESULTS_ROOT.exists(), (
    f"DAY 23 STOP: Day15 results directory not found:\n"
    f"{DAY15_RESULTS_ROOT}"
)


DAY15_RECOVERY_DIR = (
    FINAL_HISTVAL_DIR
    / "day15_baseline_recovery"
)

DAY15_RECOVERY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


DAY15_RECOVERY_CANDIDATES_PATH = (
    DAY15_RECOVERY_DIR
    / "day15_prediction_candidate_audit.csv"
)

DAY15_RECOVERY_MANIFEST_PATH = (
    DAY15_RECOVERY_DIR
    / "day15_prediction_recovery_manifest.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Candidate filename keywords
# --------------------------------------------------------------------------------------------------

prediction_keywords = [
    "prediction",
    "predictions",
    "inference",
    "validation",
    "val",
]


# --------------------------------------------------------------------------------------------------
# 3. Discover candidate tabular files
# --------------------------------------------------------------------------------------------------

candidate_files = []


for suffix in [
    "*.csv",
    "*.parquet",
    "*.json",
    "*.jsonl",
]:

    for path in DAY15_RESULTS_ROOT.rglob(
        suffix
    ):

        path_lower = (
            path.name.lower()
        )


        if any(
            keyword in path_lower
            for keyword
            in prediction_keywords
        ):

            candidate_files.append(
                path
            )


candidate_files = sorted(
    set(
        candidate_files
    )
)


# --------------------------------------------------------------------------------------------------
# 4. Safe loader
# --------------------------------------------------------------------------------------------------

def safe_load_table(
    path,
):

    try:

        suffix = (
            path.suffix.lower()
        )


        if suffix == ".csv":

            df = pd.read_csv(
                path
            )


        elif suffix == ".parquet":

            df = pd.read_parquet(
                path
            )


        elif suffix == ".jsonl":

            df = pd.read_json(
                path,
                lines=True,
            )


        elif suffix == ".json":

            try:

                df = pd.read_json(
                    path
                )

            except Exception:

                with open(
                    path,
                    "r",
                    encoding="utf-8",
                ) as f:

                    payload = load_artifact_json(
                        f
                    )


                if isinstance(
                    payload,
                    list,
                ):

                    df = pd.DataFrame(
                        payload
                    )

                else:

                    return None


        else:

            return None


        return df


    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 5. Candidate structural audit
# --------------------------------------------------------------------------------------------------

audit_records = []


for path in candidate_files:

    df = safe_load_table(
        path
    )


    if df is None:

        audit_records.append(
            {
                "path":
                    str(
                        path
                    ),

                "loaded":
                    False,

                "n_rows":
                    None,

                "n_cols":
                    None,

                "has_5859_rows":
                    False,

                "has_observation_id":
                    False,

                "has_true_label_like":
                    False,

                "has_predicted_label_like":
                    False,

                "has_text":
                    False,

                "candidate_score":
                    0,

                "columns":
                    None,
            }
        )

        continue


    columns_lower = {
        str(col).lower():
            str(col)

        for col
        in df.columns
    }


    true_label_candidates = [
        "true_label",
        "label",
        "sentiment",
        "gold",
        "gold_label",
        "target",
        "y_true",
    ]


    pred_label_candidates = [
        "predicted_label",
        "prediction",
        "pred",
        "pred_label",
        "y_pred",
        "model_prediction",
    ]


    has_true_label_like = any(
        col in columns_lower
        for col
        in true_label_candidates
    )


    has_predicted_label_like = any(
        col in columns_lower
        for col
        in pred_label_candidates
    )


    has_observation_id = (
        "observation_id"
        in columns_lower
    )


    has_text = (
        "text"
        in columns_lower
    )


    has_5859_rows = (
        len(
            df
        )
        ==
        5859
    )


    candidate_score = int(
        has_5859_rows
    ) * 4 + int(
        has_predicted_label_like
    ) * 4 + int(
        has_true_label_like
    ) * 2 + int(
        has_observation_id
    ) * 2 + int(
        has_text
    )


    audit_records.append(
        {
            "path":
                str(
                    path
                ),

            "loaded":
                True,

            "n_rows":
                int(
                    len(
                        df
                    )
                ),

            "n_cols":
                int(
                    len(
                        df.columns
                    )
                ),

            "has_5859_rows":
                bool(
                    has_5859_rows
                ),

            "has_observation_id":
                bool(
                    has_observation_id
                ),

            "has_true_label_like":
                bool(
                    has_true_label_like
                ),

            "has_predicted_label_like":
                bool(
                    has_predicted_label_like
                ),

            "has_text":
                bool(
                    has_text
                ),

            "candidate_score":
                int(
                    candidate_score
                ),

            "columns":
                " | ".join(
                    [
                        str(
                            col
                        )
                        for col
                        in df.columns
                    ]
                ),
        }
    )


audit_df = pd.DataFrame(
    audit_records
)


# --------------------------------------------------------------------------------------------------
# 6. Rank candidate files
# --------------------------------------------------------------------------------------------------

if len(
    audit_df
) > 0:

    audit_df = (
        audit_df
        .sort_values(
            by=[
                "candidate_score",
                "has_5859_rows",
                "has_predicted_label_like",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )


audit_df.to_csv(
    DAY15_RECOVERY_CANDIDATES_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 7. Strict valid-candidate definition
#
# Minimum needed for paired bootstrap:
#   - 5,859 rows
#   - predicted-label-like column
#
# Better:
#   - true labels
#   - observation_id
#
# --------------------------------------------------------------------------------------------------

if len(
    audit_df
) > 0:

    strict_candidates_df = (
        audit_df.loc[
            (
                audit_df[
                    "loaded"
                ]
                ==
                True
            )
            &
            (
                audit_df[
                    "has_5859_rows"
                ]
                ==
                True
            )
            &
            (
                audit_df[
                    "has_predicted_label_like"
                ]
                ==
                True
            )
        ]
        .copy()
    )

else:

    strict_candidates_df = (
        pd.DataFrame()
    )


strict_candidate_count = int(
    len(
        strict_candidates_df
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Best candidate
# --------------------------------------------------------------------------------------------------

DAY15_PREDICTION_CANDIDATE_PATH = None


if strict_candidate_count > 0:

    DAY15_PREDICTION_CANDIDATE_PATH = Path(
        strict_candidates_df.iloc[
            0
        ][
            "path"
        ]
    )


# --------------------------------------------------------------------------------------------------
# 9. If candidate exists, inspect structure ONLY
# --------------------------------------------------------------------------------------------------

best_candidate_structure = None


if (
    DAY15_PREDICTION_CANDIDATE_PATH
    is not None
):

    best_candidate_df = safe_load_table(
        DAY15_PREDICTION_CANDIDATE_PATH
    )


    best_candidate_structure = {

        "path":
            str(
                DAY15_PREDICTION_CANDIDATE_PATH
            ),

        "n_rows":
            int(
                len(
                    best_candidate_df
                )
            ),

        "n_cols":
            int(
                len(
                    best_candidate_df.columns
                )
            ),

        "columns":
            [
                str(
                    col
                )
                for col
                in best_candidate_df.columns
            ],
    }


# --------------------------------------------------------------------------------------------------
# 10. Recovery status
# --------------------------------------------------------------------------------------------------

if strict_candidate_count == 0:

    recovery_status = (
        "NO_VALID_DAY15_PREDICTION_ARTIFACT_FOUND"
    )


elif strict_candidate_count == 1:

    recovery_status = (
        "ONE_VALID_DAY15_PREDICTION_CANDIDATE_FOUND"
    )


else:

    recovery_status = (
        "MULTIPLE_DAY15_PREDICTION_CANDIDATES_FOUND"
    )


# --------------------------------------------------------------------------------------------------
# 11. Formal checks
# --------------------------------------------------------------------------------------------------

recovery_checks = {

    "day15_results_root_exists":
        DAY15_RESULTS_ROOT.exists(),

    "candidate_audit_saved":
        DAY15_RECOVERY_CANDIDATES_PATH.exists(),

    "no_model_inference":
        True,

    "no_model_loading":
        True,

    "no_retraining":
        True,

    "no_retuning":
        True,

    "historical_validation_predictions_not_modified":
        (
            sha256_file(
                FINAL_HISTVAL_PREDICTIONS_PATH
            )
            ==
            FINAL_HISTVAL_PREDICTIONS_SHA256
        ),

    "future_not_used":
        True,
}


all_recovery_checks_pass = bool(
    all(
        recovery_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. Save recovery manifest
# --------------------------------------------------------------------------------------------------

recovery_manifest = {

    "day":
        23,

    "analysis":
        "day15_historical_validation_prediction_artifact_recovery",

    "search_root":
        str(
            DAY15_RESULTS_ROOT
        ),

    "candidate_files_discovered":
        int(
            len(
                candidate_files
            )
        ),

    "strict_candidate_count":
        strict_candidate_count,

    "recovery_status":
        recovery_status,

    "best_candidate":
        best_candidate_structure,

    "paired_bootstrap_ready":
        bool(
            strict_candidate_count
            ==
            1
        ),

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in recovery_checks.items()
    },

    "all_checks_pass":
        all_recovery_checks_pass,
}


with open(
    DAY15_RECOVERY_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        recovery_manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 13. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — DAY15 HISTORICAL VALIDATION PREDICTION ARTIFACT RECOVERY AUDIT")
print("=" * 120)


print("\n[SEARCH]")
print(
    f"Day15 results root          : "
    f"{DAY15_RESULTS_ROOT}"
)
print(
    f"Candidate files discovered  : "
    f"{len(candidate_files)}"
)


print("\n[CANDIDATE AUDIT]")

if len(
    audit_df
) > 0:

    display(
        audit_df[
            [
                "path",
                "n_rows",
                "n_cols",
                "has_5859_rows",
                "has_observation_id",
                "has_true_label_like",
                "has_predicted_label_like",
                "candidate_score",
            ]
        ].head(
            20
        )
    )

else:

    print(
        "No prediction-like tabular artifacts found."
    )


print("\n[RECOVERY STATUS]")
print(
    f"Strict candidate count      : "
    f"{strict_candidate_count}"
)
print(
    f"Status                      : "
    f"{recovery_status}"
)


if DAY15_PREDICTION_CANDIDATE_PATH is not None:

    print(
        f"Best candidate              : "
        f"{DAY15_PREDICTION_CANDIDATE_PATH}"
    )

    print(
        f"Rows                        : "
        f"{best_candidate_structure['n_rows']:,}"
    )

    print(
        f"Columns                     : "
        f"{best_candidate_structure['columns']}"
    )


print("\n[METHODOLOGICAL STATUS]")

if strict_candidate_count == 1:

    print(
        "Paired bootstrap readiness  : CANDIDATE FOUND — IDENTITY ALIGNMENT STILL REQUIRED"
    )

elif strict_candidate_count > 1:

    print(
        "Paired bootstrap readiness  : NOT YET — MULTIPLE CANDIDATES REQUIRE DISAMBIGUATION"
    )

else:

    print(
        "Paired bootstrap readiness  : NOT AVAILABLE FROM EXISTING DAY15 ARTIFACTS"
    )


print(
    "Model inference             : NOT PERFORMED"
)
print(
    "Retuning                    : NOT PERFORMED"
)
print(
    "Future F1-F6                : NOT USED"
)


print("\n[FORMAL CHECKS]")

for name, passed in recovery_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Candidate audit             : "
    f"{DAY15_RECOVERY_CANDIDATES_PATH}"
)
print(
    f"Recovery manifest           : "
    f"{DAY15_RECOVERY_MANIFEST_PATH}"
)


print("\n" + "-" * 120)

if all_recovery_checks_pass:

    if strict_candidate_count == 1:

        print(
            "DAY 23 CELL 24 STATUS: PASS — "
            "ONE DAY15 PREDICTION CANDIDATE RECOVERED; "
            "READY FOR SAMPLE-IDENTITY ALIGNMENT AUDIT"
        )

    elif strict_candidate_count > 1:

        print(
            "DAY 23 CELL 24 STATUS: PASS — "
            "MULTIPLE DAY15 PREDICTION CANDIDATES FOUND; "
            "DISAMBIGUATION REQUIRED BEFORE PAIRED BOOTSTRAP"
        )

    else:

        print(
            "DAY 23 CELL 24 STATUS: PASS — "
            "NO EXISTING DAY15 PER-SAMPLE PREDICTION ARTIFACT RECOVERED; "
            "PAIRED BOOTSTRAP NOT CURRENTLY AVAILABLE"
        )

else:

    print(
        "DAY 23 CELL 24 STATUS: FAIL — "
        "DO NOT USE ANY RECOVERED DAY15 ARTIFACT"
    )

print("=" * 120)

DAY 23 — DAY15 HISTORICAL VALIDATION PREDICTION ARTIFACT RECOVERY AUDIT

[SEARCH]
Day15 results root          : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day15
Candidate files discovered  : 7

[CANDIDATE AUDIT]


,path,n_rows,n_cols,has_5859_rows,has_observation_id,has_true_label_like,has_predicted_label_like,candidate_score
0,/content/gdrive_day23/MyDrive/FinGPT_Project/r...,5859,7,True,True,True,True,12
1,/content/gdrive_day23/MyDrive/FinGPT_Project/r...,3,3,False,False,False,True,4
2,/content/gdrive_day23/MyDrive/FinGPT_Project/r...,3,4,False,False,True,False,2
3,/content/gdrive_day23/MyDrive/FinGPT_Project/r...,3,4,False,False,False,False,0
4,/content/gdrive_day23/MyDrive/FinGPT_Project/r...,12,18,False,False,False,False,0
5,/content/gdrive_day23/MyDrive/FinGPT_Project/r...,6,2,False,False,False,False,0
6,/content/gdrive_day23/MyDrive/FinGPT_Project/r...,3,14,False,False,False,False,0



[RECOVERY STATUS]
Strict candidate count      : 1
Status                      : ONE_VALID_DAY15_PREDICTION_CANDIDATE_FOUND
Best candidate              : /content/gdrive_day23/MyDrive/FinGPT_Project/results/day15/historical_validation_predictions.csv
Rows                        : 5,859
Columns                     : ['observation_id', 'channel_username', 'message_id', 'date', 'true_label', 'predicted_label', 'prediction_valid']

[METHODOLOGICAL STATUS]
Paired bootstrap readiness  : CANDIDATE FOUND — IDENTITY ALIGNMENT STILL REQUIRED
Model inference             : NOT PERFORMED
Retuning                    : NOT PERFORMED
Future F1-F6                : NOT USED

[FORMAL CHECKS]
PASS   — day15_results_root_exists
PASS   — candidate_audit_saved
PASS   — no_model_inference
PASS   — no_model_loading
PASS   — no_retraining
PASS   — no_retuning
PASS   — historical_validation_predictions_not_modified
PASS   — future_not_used

[ARTIFACTS]
Candidate audit             : /content/gdrive_day23/MyDrive/

In [35]:
# ==================================================================================================
# DAY 23 — CELL 25
# DAY15 VS DAY23 HISTORICAL VALIDATION SAMPLE-IDENTITY ALIGNMENT AUDIT
#
# PURPOSE:
#   Verify that Day15 and Day23 predictions correspond to the exact same 5,859 Historical Validation
#   observations before any paired bootstrap comparison.
#
# IMPORTANT:
#   - NO model inference
#   - NO retraining
#   - NO retuning
#   - NO future data
#   - this cell performs identity/alignment validation only
# ==================================================================================================

import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
)


# --------------------------------------------------------------------------------------------------
# 1. Exact recovered Day15 prediction artifact
# --------------------------------------------------------------------------------------------------

DAY15_HISTVAL_PREDICTIONS_PATH = (PROJECT_ROOT / 'results' / 'day15' / 'historical_validation_predictions.csv')


assert DAY15_HISTVAL_PREDICTIONS_PATH.exists(), (
    f"DAY 23 STOP: Day15 Historical Validation predictions not found:\n"
    f"{DAY15_HISTVAL_PREDICTIONS_PATH}"
)


assert FINAL_HISTVAL_PREDICTIONS_PATH.exists(), (
    "DAY 23 STOP: frozen Day23 Historical Validation predictions missing."
)


# --------------------------------------------------------------------------------------------------
# 2. Output directory
# --------------------------------------------------------------------------------------------------

DAY15_ALIGNMENT_DIR = (
    DAY15_RECOVERY_DIR
    / "alignment"
)

DAY15_ALIGNMENT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


DAY15_ALIGNMENT_TABLE_PATH = (
    DAY15_ALIGNMENT_DIR
    / "day15_day23_historical_validation_alignment.csv"
)

DAY15_ALIGNMENT_SUMMARY_PATH = (
    DAY15_ALIGNMENT_DIR
    / "day15_day23_alignment_summary.json"
)


# --------------------------------------------------------------------------------------------------
# 3. Load Day15 and Day23 prediction artifacts
# --------------------------------------------------------------------------------------------------

day15_pred = pd.read_csv(
    DAY15_HISTVAL_PREDICTIONS_PATH
)


day23_pred = pd.read_csv(
    FINAL_HISTVAL_PREDICTIONS_PATH
)


# --------------------------------------------------------------------------------------------------
# 4. Required columns
# --------------------------------------------------------------------------------------------------

required_day15_columns = {
    "observation_id",
    "true_label",
    "predicted_label",
}


required_day23_columns = {
    "observation_id",
    "true_label",
    "predicted_label",
}


assert required_day15_columns.issubset(
    day15_pred.columns
), (
    "DAY 23 STOP: required Day15 prediction columns missing."
)


assert required_day23_columns.issubset(
    day23_pred.columns
), (
    "DAY 23 STOP: required Day23 prediction columns missing."
)


# --------------------------------------------------------------------------------------------------
# 5. Normalize identifiers / labels
# --------------------------------------------------------------------------------------------------

for df in [
    day15_pred,
    day23_pred,
]:

    df[
        "observation_id"
    ] = (
        df[
            "observation_id"
        ]
        .astype(str)
        .str.strip()
    )


    df[
        "true_label"
    ] = (
        df[
            "true_label"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
    )


    df[
        "predicted_label"
    ] = (
        df[
            "predicted_label"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
    )


# --------------------------------------------------------------------------------------------------
# 6. Basic population integrity
# --------------------------------------------------------------------------------------------------

day15_n = len(
    day15_pred
)

day23_n = len(
    day23_pred
)


day15_unique_ids = (
    day15_pred[
        "observation_id"
    ]
    .nunique()
)

day23_unique_ids = (
    day23_pred[
        "observation_id"
    ]
    .nunique()
)


# --------------------------------------------------------------------------------------------------
# 7. Ordered ID hashes
# --------------------------------------------------------------------------------------------------

def ordered_id_hash(
    df,
):

    ids = (
        df[
            "observation_id"
        ]
        .astype(str)
        .tolist()
    )


    return hashlib.sha256(
        "\n".join(
            ids
        ).encode(
            "utf-8"
        )
    ).hexdigest()


day15_ordered_id_sha256 = (
    ordered_id_hash(
        day15_pred
    )
)


day23_ordered_id_sha256 = (
    ordered_id_hash(
        day23_pred
    )
)


EXPECTED_HISTVAL_ID_SHA256 = (
    "09ccb61d3ae6cc4e384c7fe5364043a5f7806ca31ee72830324d149f573748bf"
)


# --------------------------------------------------------------------------------------------------
# 8. Set-level ID comparison
# --------------------------------------------------------------------------------------------------

day15_id_set = set(
    day15_pred[
        "observation_id"
    ]
)


day23_id_set = set(
    day23_pred[
        "observation_id"
    ]
)


ids_only_in_day15 = sorted(
    day15_id_set
    -
    day23_id_set
)


ids_only_in_day23 = sorted(
    day23_id_set
    -
    day15_id_set
)


same_id_set = (
    day15_id_set
    ==
    day23_id_set
)


same_id_order = (
    day15_pred[
        "observation_id"
    ].tolist()
    ==
    day23_pred[
        "observation_id"
    ].tolist()
)


# --------------------------------------------------------------------------------------------------
# 9. Build canonical aligned table by observation_id
#
# Use merge even if order already matches, so pairing is explicit.
# --------------------------------------------------------------------------------------------------

day15_core = (
    day15_pred[
        [
            "observation_id",
            "true_label",
            "predicted_label",
        ]
    ]
    .rename(
        columns={
            "true_label":
                "day15_true_label",

            "predicted_label":
                "day15_predicted_label",
        }
    )
)


day23_core = (
    day23_pred[
        [
            "observation_id",
            "true_label",
            "predicted_label",
        ]
    ]
    .rename(
        columns={
            "true_label":
                "day23_true_label",

            "predicted_label":
                "day23_predicted_label",
        }
    )
)


aligned_df = (
    day23_core
    .merge(
        day15_core,
        on="observation_id",
        how="inner",
        validate="one_to_one",
    )
)


# --------------------------------------------------------------------------------------------------
# 10. Gold-label alignment
# --------------------------------------------------------------------------------------------------

aligned_df[
    "gold_label_match"
] = (
    aligned_df[
        "day15_true_label"
    ]
    ==
    aligned_df[
        "day23_true_label"
    ]
)


gold_label_mismatch_count = int(
    (
        ~aligned_df[
            "gold_label_match"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 11. Prediction comparison fields
# --------------------------------------------------------------------------------------------------

aligned_df[
    "prediction_match"
] = (
    aligned_df[
        "day15_predicted_label"
    ]
    ==
    aligned_df[
        "day23_predicted_label"
    ]
)


aligned_df[
    "day15_correct"
] = (
    aligned_df[
        "day15_predicted_label"
    ]
    ==
    aligned_df[
        "day23_true_label"
    ]
)


aligned_df[
    "day23_correct"
] = (
    aligned_df[
        "day23_predicted_label"
    ]
    ==
    aligned_df[
        "day23_true_label"
    ]
)


prediction_match_count = int(
    aligned_df[
        "prediction_match"
    ].sum()
)


prediction_difference_count = int(
    (
        ~aligned_df[
            "prediction_match"
        ]
    ).sum()
)


day15_only_correct_count = int(
    (
        aligned_df[
            "day15_correct"
        ]
        &
        ~aligned_df[
            "day23_correct"
        ]
    ).sum()
)


day23_only_correct_count = int(
    (
        ~aligned_df[
            "day15_correct"
        ]
        &
        aligned_df[
            "day23_correct"
        ]
    ).sum()
)


both_correct_count = int(
    (
        aligned_df[
            "day15_correct"
        ]
        &
        aligned_df[
            "day23_correct"
        ]
    ).sum()
)


both_wrong_count = int(
    (
        ~aligned_df[
            "day15_correct"
        ]
        &
        ~aligned_df[
            "day23_correct"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 12. Reproduce Day15 aggregate metrics from recovered predictions
# --------------------------------------------------------------------------------------------------

VALID_LABELS = [
    "negative",
    "neutral",
    "positive",
]


day15_y_true = (
    aligned_df[
        "day23_true_label"
    ]
    .to_numpy()
)


day15_y_pred = (
    aligned_df[
        "day15_predicted_label"
    ]
    .to_numpy()
)


day15_recovered_accuracy = float(
    accuracy_score(
        day15_y_true,
        day15_y_pred,
    )
)


day15_recovered_macro_f1 = float(
    f1_score(
        day15_y_true,
        day15_y_pred,
        labels=VALID_LABELS,
        average="macro",
        zero_division=0,
    )
)


day15_recovered_recall = recall_score(
    day15_y_true,
    day15_y_pred,
    labels=VALID_LABELS,
    average=None,
    zero_division=0,
)


day15_recovered_negative_recall = float(
    day15_recovered_recall[0]
)

day15_recovered_neutral_recall = float(
    day15_recovered_recall[1]
)

day15_recovered_positive_recall = float(
    day15_recovered_recall[2]
)


# --------------------------------------------------------------------------------------------------
# 13. Reproduce Day23 aggregate metrics from aligned table
# --------------------------------------------------------------------------------------------------

day23_y_pred = (
    aligned_df[
        "day23_predicted_label"
    ]
    .to_numpy()
)


day23_recovered_accuracy = float(
    accuracy_score(
        day15_y_true,
        day23_y_pred,
    )
)


day23_recovered_macro_f1 = float(
    f1_score(
        day15_y_true,
        day23_y_pred,
        labels=VALID_LABELS,
        average="macro",
        zero_division=0,
    )
)


# --------------------------------------------------------------------------------------------------
# 14. Expected frozen metrics
# --------------------------------------------------------------------------------------------------

EXPECTED_DAY15 = {

    "accuracy":
        0.7890424987,

    "macro_f1":
        0.7456777708,

    "negative_recall":
        0.639576,

    "neutral_recall":
        0.842148,

    "positive_recall":
        0.763462,
}


EXPECTED_DAY23 = {

    "accuracy":
        0.838710,

    "macro_f1":
        0.804130,
}


# --------------------------------------------------------------------------------------------------
# 15. Save canonical paired alignment artifact
# --------------------------------------------------------------------------------------------------

aligned_df.to_csv(
    DAY15_ALIGNMENT_TABLE_PATH,
    index=False,
)


DAY15_ALIGNMENT_SHA256 = (
    sha256_file(
        DAY15_ALIGNMENT_TABLE_PATH
    )
)


# --------------------------------------------------------------------------------------------------
# 16. Formal alignment checks
# --------------------------------------------------------------------------------------------------

alignment_checks = {

    "day15_n_5859":
        day15_n
        ==
        5859,

    "day23_n_5859":
        day23_n
        ==
        5859,

    "day15_ids_unique":
        day15_unique_ids
        ==
        5859,

    "day23_ids_unique":
        day23_unique_ids
        ==
        5859,

    "same_id_set":
        same_id_set,

    "no_ids_only_in_day15":
        len(
            ids_only_in_day15
        )
        ==
        0,

    "no_ids_only_in_day23":
        len(
            ids_only_in_day23
        )
        ==
        0,

    "same_id_order":
        same_id_order,

    "day15_ordered_hash_match":
        day15_ordered_id_sha256
        ==
        EXPECTED_HISTVAL_ID_SHA256,

    "day23_ordered_hash_match":
        day23_ordered_id_sha256
        ==
        EXPECTED_HISTVAL_ID_SHA256,

    "aligned_n_5859":
        len(
            aligned_df
        )
        ==
        5859,

    "gold_labels_identical":
        gold_label_mismatch_count
        ==
        0,

    "day15_accuracy_reproduced":
        abs(
            day15_recovered_accuracy
            -
            EXPECTED_DAY15[
                "accuracy"
            ]
        )
        <
        1e-9,

    "day15_macro_f1_reproduced":
        abs(
            day15_recovered_macro_f1
            -
            EXPECTED_DAY15[
                "macro_f1"
            ]
        )
        <
        1e-9,

    "day15_negative_recall_reproduced":
        abs(
            day15_recovered_negative_recall
            -
            EXPECTED_DAY15[
                "negative_recall"
            ]
        )
        <
        1e-6,

    "day15_neutral_recall_reproduced":
        abs(
            day15_recovered_neutral_recall
            -
            EXPECTED_DAY15[
                "neutral_recall"
            ]
        )
        <
        1e-6,

    "day15_positive_recall_reproduced":
        abs(
            day15_recovered_positive_recall
            -
            EXPECTED_DAY15[
                "positive_recall"
            ]
        )
        <
        1e-6,

    "day23_accuracy_reproduced":
        abs(
            day23_recovered_accuracy
            -
            EXPECTED_DAY23[
                "accuracy"
            ]
        )
        <
        1e-6,

    "day23_macro_f1_reproduced":
        abs(
            day23_recovered_macro_f1
            -
            EXPECTED_DAY23[
                "macro_f1"
            ]
        )
        <
        1e-6,

    "alignment_artifact_saved":
        DAY15_ALIGNMENT_TABLE_PATH.exists(),

    "alignment_hash_created":
        len(
            DAY15_ALIGNMENT_SHA256
        )
        ==
        64,

    "no_model_inference":
        True,

    "no_retraining":
        True,

    "no_retuning":
        True,

    "future_not_used":
        True,
}


all_alignment_checks_pass = bool(
    all(
        alignment_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 17. Paired-bootstrap readiness
# --------------------------------------------------------------------------------------------------

PAIRED_BOOTSTRAP_READY = bool(
    all_alignment_checks_pass
)


# --------------------------------------------------------------------------------------------------
# 18. Save alignment summary
# --------------------------------------------------------------------------------------------------

alignment_summary = {

    "day":
        23,

    "analysis":
        "day15_day23_historical_validation_sample_identity_alignment",

    "day15_prediction_artifact":
        str(
            DAY15_HISTVAL_PREDICTIONS_PATH
        ),

    "day23_prediction_artifact":
        str(
            FINAL_HISTVAL_PREDICTIONS_PATH
        ),

    "population": {

        "n":
            5859,

        "ordered_observation_id_sha256":
            EXPECTED_HISTVAL_ID_SHA256,

        "same_id_set":
            bool(
                same_id_set
            ),

        "same_id_order":
            bool(
                same_id_order
            ),

        "gold_label_mismatch_count":
            gold_label_mismatch_count,
    },

    "prediction_comparison": {

        "same_prediction_count":
            prediction_match_count,

        "different_prediction_count":
            prediction_difference_count,

        "both_correct":
            both_correct_count,

        "both_wrong":
            both_wrong_count,

        "day15_only_correct":
            day15_only_correct_count,

        "day23_only_correct":
            day23_only_correct_count,
    },

    "reproduced_day15_metrics": {

        "accuracy":
            day15_recovered_accuracy,

        "macro_f1":
            day15_recovered_macro_f1,

        "negative_recall":
            day15_recovered_negative_recall,

        "neutral_recall":
            day15_recovered_neutral_recall,

        "positive_recall":
            day15_recovered_positive_recall,
    },

    "reproduced_day23_metrics": {

        "accuracy":
            day23_recovered_accuracy,

        "macro_f1":
            day23_recovered_macro_f1,
    },

    "alignment_artifact": {

        "path":
            str(
                DAY15_ALIGNMENT_TABLE_PATH
            ),

        "sha256":
            DAY15_ALIGNMENT_SHA256,
    },

    "paired_bootstrap_ready":
        PAIRED_BOOTSTRAP_READY,

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in alignment_checks.items()
    },

    "all_checks_pass":
        all_alignment_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    DAY15_ALIGNMENT_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        alignment_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 19. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — DAY15 VS DAY23 HISTORICAL VALIDATION SAMPLE-IDENTITY ALIGNMENT")
print("=" * 120)


print("\n[POPULATION ALIGNMENT]")
print(
    f"Day15 N                     : "
    f"{day15_n:,}"
)
print(
    f"Day23 N                     : "
    f"{day23_n:,}"
)
print(
    f"Same ID set                 : "
    f"{same_id_set}"
)
print(
    f"Same ID order               : "
    f"{same_id_order}"
)
print(
    f"IDs only in Day15           : "
    f"{len(ids_only_in_day15)}"
)
print(
    f"IDs only in Day23           : "
    f"{len(ids_only_in_day23)}"
)
print(
    f"Gold-label mismatches       : "
    f"{gold_label_mismatch_count}"
)
print(
    f"Ordered ID SHA256           : "
    f"{EXPECTED_HISTVAL_ID_SHA256}"
)


print("\n[DAY15 METRIC REPRODUCTION]")
print(
    f"Accuracy                    : "
    f"{day15_recovered_accuracy:.10f}"
)
print(
    f"Macro-F1                    : "
    f"{day15_recovered_macro_f1:.10f}"
)
print(
    f"Negative recall             : "
    f"{day15_recovered_negative_recall:.6f}"
)
print(
    f"Neutral recall              : "
    f"{day15_recovered_neutral_recall:.6f}"
)
print(
    f"Positive recall             : "
    f"{day15_recovered_positive_recall:.6f}"
)


print("\n[DAY23 METRIC REPRODUCTION]")
print(
    f"Accuracy                    : "
    f"{day23_recovered_accuracy:.6f}"
)
print(
    f"Macro-F1                    : "
    f"{day23_recovered_macro_f1:.6f}"
)


print("\n[PAIRWISE PREDICTION STRUCTURE]")
print(
    f"Same prediction             : "
    f"{prediction_match_count:,}"
)
print(
    f"Different prediction        : "
    f"{prediction_difference_count:,}"
)
print(
    f"Both correct                : "
    f"{both_correct_count:,}"
)
print(
    f"Both wrong                  : "
    f"{both_wrong_count:,}"
)
print(
    f"Day15 only correct          : "
    f"{day15_only_correct_count:,}"
)
print(
    f"Day23 only correct          : "
    f"{day23_only_correct_count:,}"
)


print("\n[METHODOLOGICAL STATUS]")
print(
    f"Paired bootstrap ready      : "
    f"{PAIRED_BOOTSTRAP_READY}"
)
print(
    "Model inference             : NOT PERFORMED"
)
print(
    "Retraining                  : NOT PERFORMED"
)
print(
    "Retuning                    : NOT PERFORMED"
)
print(
    "Future F1-F6                : NOT USED"
)


print("\n[FORMAL CHECKS]")

for name, passed in alignment_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Aligned paired table        : "
    f"{DAY15_ALIGNMENT_TABLE_PATH}"
)
print(
    f"Aligned table SHA256        : "
    f"{DAY15_ALIGNMENT_SHA256}"
)
print(
    f"Alignment summary           : "
    f"{DAY15_ALIGNMENT_SUMMARY_PATH}"
)


print("\n" + "-" * 120)

if all_alignment_checks_pass:

    print(
        "DAY 23 CELL 25 STATUS: PASS — "
        "DAY15 AND DAY23 HISTORICAL VALIDATION PREDICTIONS ARE EXACTLY SAMPLE-ALIGNED; "
        "PAIRED BOOTSTRAP READY"
    )

else:

    print(
        "DAY 23 CELL 25 STATUS: FAIL — "
        "DO NOT PERFORM PAIRED BOOTSTRAP"
    )

print("=" * 120)

DAY 23 — DAY15 VS DAY23 HISTORICAL VALIDATION SAMPLE-IDENTITY ALIGNMENT

[POPULATION ALIGNMENT]
Day15 N                     : 5,859
Day23 N                     : 5,859
Same ID set                 : True
Same ID order               : True
IDs only in Day15           : 0
IDs only in Day23           : 0
Gold-label mismatches       : 0
Ordered ID SHA256           : 09ccb61d3ae6cc4e384c7fe5364043a5f7806ca31ee72830324d149f573748bf

[DAY15 METRIC REPRODUCTION]
Accuracy                    : 0.7890424987
Macro-F1                    : 0.7456777708
Negative recall             : 0.639576
Neutral recall              : 0.842148
Positive recall             : 0.763462

[DAY23 METRIC REPRODUCTION]
Accuracy                    : 0.838710
Macro-F1                    : 0.804130

[PAIRWISE PREDICTION STRUCTURE]
Same prediction             : 4,968
Different prediction        : 891
Both correct                : 4,346
Both wrong                  : 668
Day15 only correct          : 277
Day23 only correct       

In [3]:
# ==================================================================================================
# DAY 23 — CELL 26
# PAIRED BOOTSTRAP OF DAY23 - DAY15 HISTORICAL VALIDATION IMPROVEMENT
#
# PURPOSE:
#   Quantify uncertainty in the performance improvement of the frozen Day23 final model
#   relative to the Day15 baseline on the exact same 5,859 Historical Validation samples.
#
# IMPORTANT:
#   - NO model inference
#   - NO retraining
#   - NO retuning
#   - NO future data
#   - paired bootstrap uses the SAME resampled observation indices for both models
# ==================================================================================================

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
)


# --------------------------------------------------------------------------------------------------
# 1. Output directory
# --------------------------------------------------------------------------------------------------

PAIRED_BOOTSTRAP_DIR = (
    FINAL_HISTVAL_DIR
    / "paired_bootstrap_day23_vs_day15"
)

PAIRED_BOOTSTRAP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


PAIRED_BOOTSTRAP_DRAWS_PATH = (
    PAIRED_BOOTSTRAP_DIR
    / "day23_minus_day15_paired_bootstrap_draws.csv"
)

PAIRED_BOOTSTRAP_SUMMARY_PATH = (
    PAIRED_BOOTSTRAP_DIR
    / "day23_minus_day15_paired_bootstrap_summary.csv"
)

PAIRED_BOOTSTRAP_MANIFEST_PATH = (
    PAIRED_BOOTSTRAP_DIR
    / "day23_minus_day15_paired_bootstrap_manifest.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Safety gate
# --------------------------------------------------------------------------------------------------

assert PAIRED_BOOTSTRAP_READY, (
    "DAY 23 STOP: paired bootstrap readiness flag is False."
)


assert all_alignment_checks_pass, (
    "DAY 23 STOP: Day15-Day23 sample alignment did not fully pass."
)


assert DAY15_ALIGNMENT_TABLE_PATH.exists(), (
    "DAY 23 STOP: canonical paired alignment artifact missing."
)


assert (
    sha256_file(
        DAY15_ALIGNMENT_TABLE_PATH
    )
    ==
    DAY15_ALIGNMENT_SHA256
), (
    "DAY 23 STOP: paired alignment artifact hash changed."
)


assert (
    sha256_file(
        FINAL_HISTVAL_PREDICTIONS_PATH
    )
    ==
    FINAL_HISTVAL_PREDICTIONS_SHA256
), (
    "DAY 23 STOP: frozen Day23 Historical Validation predictions changed."
)


# --------------------------------------------------------------------------------------------------
# 3. Load canonical paired table ONLY
# --------------------------------------------------------------------------------------------------

paired_df = pd.read_csv(
    DAY15_ALIGNMENT_TABLE_PATH
)


assert len(
    paired_df
) == 5859


assert (
    paired_df[
        "observation_id"
    ].nunique()
    ==
    5859
)


assert (
    paired_df[
        "gold_label_match"
    ]
    .astype(bool)
    .all()
)


# --------------------------------------------------------------------------------------------------
# 4. Extract aligned vectors
# --------------------------------------------------------------------------------------------------

VALID_LABELS = [
    "negative",
    "neutral",
    "positive",
]


y_true = (
    paired_df[
        "day23_true_label"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .to_numpy()
)


day15_pred = (
    paired_df[
        "day15_predicted_label"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .to_numpy()
)


day23_pred = (
    paired_df[
        "day23_predicted_label"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .to_numpy()
)


assert len(
    y_true
) == 5859

assert len(
    day15_pred
) == 5859

assert len(
    day23_pred
) == 5859


# --------------------------------------------------------------------------------------------------
# 5. Metric helper
# --------------------------------------------------------------------------------------------------

def compute_classification_metrics(
    gold,
    pred,
):

    accuracy = float(
        accuracy_score(
            gold,
            pred,
        )
    )


    macro_f1 = float(
        f1_score(
            gold,
            pred,
            labels=VALID_LABELS,
            average="macro",
            zero_division=0,
        )
    )


    recalls = recall_score(
        gold,
        pred,
        labels=VALID_LABELS,
        average=None,
        zero_division=0,
    )


    negative_recall = float(
        recalls[0]
    )

    neutral_recall = float(
        recalls[1]
    )

    positive_recall = float(
        recalls[2]
    )


    minority_recall_floor = float(
        min(
            negative_recall,
            positive_recall,
        )
    )


    return {
        "accuracy":
            accuracy,

        "macro_f1":
            macro_f1,

        "negative_recall":
            negative_recall,

        "neutral_recall":
            neutral_recall,

        "positive_recall":
            positive_recall,

        "minority_recall_floor":
            minority_recall_floor,
    }


# --------------------------------------------------------------------------------------------------
# 6. Point estimates
# --------------------------------------------------------------------------------------------------

day15_point = (
    compute_classification_metrics(
        y_true,
        day15_pred,
    )
)


day23_point = (
    compute_classification_metrics(
        y_true,
        day23_pred,
    )
)


metric_names = [
    "accuracy",
    "macro_f1",
    "negative_recall",
    "neutral_recall",
    "positive_recall",
    "minority_recall_floor",
]


point_delta = {

    metric:
        float(
            day23_point[
                metric
            ]
            -
            day15_point[
                metric
            ]
        )

    for metric
    in metric_names
}


# --------------------------------------------------------------------------------------------------
# 7. Point-estimate identity checks
# --------------------------------------------------------------------------------------------------

point_checks = {

    "day15_accuracy_match":
        abs(
            day15_point[
                "accuracy"
            ]
            -
            0.7890424987
        )
        <
        1e-9,

    "day15_macro_f1_match":
        abs(
            day15_point[
                "macro_f1"
            ]
            -
            0.7456777708
        )
        <
        1e-9,

    "day23_accuracy_match":
        abs(
            day23_point[
                "accuracy"
            ]
            -
            0.838710
        )
        <
        1e-6,

    "day23_macro_f1_match":
        abs(
            day23_point[
                "macro_f1"
            ]
            -
            0.804130
        )
        <
        1e-6,

    "accuracy_delta_positive":
        point_delta[
            "accuracy"
        ]
        >
        0,

    "macro_f1_delta_positive":
        point_delta[
            "macro_f1"
        ]
        >
        0,
}


assert all(
    point_checks.values()
), (
    "DAY 23 STOP: paired point estimates do not reproduce frozen results."
)


# --------------------------------------------------------------------------------------------------
# 8. Paired bootstrap protocol
#
# SAME sampled indices are applied to:
#   - gold labels
#   - Day15 predictions
#   - Day23 predictions
#
# This preserves observation-level pairing.
# --------------------------------------------------------------------------------------------------

PAIRED_BOOTSTRAP_REPLICATES = 1000
PAIRED_BOOTSTRAP_SEED = 42
PAIRED_CI_LEVEL = 0.95

alpha = (
    1.0
    -
    PAIRED_CI_LEVEL
)

lower_q = (
    alpha
    /
    2.0
)

upper_q = (
    1.0
    -
    lower_q
)


rng = np.random.default_rng(
    PAIRED_BOOTSTRAP_SEED
)


n = len(
    y_true
)


bootstrap_records = []


for bootstrap_id in range(
    1,
    PAIRED_BOOTSTRAP_REPLICATES + 1,
):

    sampled_index = rng.integers(
        low=0,
        high=n,
        size=n,
    )


    boot_true = (
        y_true[
            sampled_index
        ]
    )

    boot_day15_pred = (
        day15_pred[
            sampled_index
        ]
    )

    boot_day23_pred = (
        day23_pred[
            sampled_index
        ]
    )


    boot_day15 = (
        compute_classification_metrics(
            boot_true,
            boot_day15_pred,
        )
    )


    boot_day23 = (
        compute_classification_metrics(
            boot_true,
            boot_day23_pred,
        )
    )


    record = {
        "bootstrap_id":
            bootstrap_id
    }


    for metric in metric_names:

        record[
            f"day15_{metric}"
        ] = (
            boot_day15[
                metric
            ]
        )

        record[
            f"day23_{metric}"
        ] = (
            boot_day23[
                metric
            ]
        )

        record[
            f"delta_{metric}"
        ] = (
            boot_day23[
                metric
            ]
            -
            boot_day15[
                metric
            ]
        )


    bootstrap_records.append(
        record
    )


paired_bootstrap_df = pd.DataFrame(
    bootstrap_records
)


paired_bootstrap_df.to_csv(
    PAIRED_BOOTSTRAP_DRAWS_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 9. Bootstrap summary
# --------------------------------------------------------------------------------------------------

summary_records = []


for metric in metric_names:

    delta_values = (
        paired_bootstrap_df[
            f"delta_{metric}"
        ]
        .to_numpy()
    )


    ci_lower = float(
        np.quantile(
            delta_values,
            lower_q,
        )
    )


    ci_upper = float(
        np.quantile(
            delta_values,
            upper_q,
        )
    )


    bootstrap_mean = float(
        np.mean(
            delta_values
        )
    )


    bootstrap_sd = float(
        np.std(
            delta_values,
            ddof=1,
        )
    )


    probability_positive = float(
        np.mean(
            delta_values
            >
            0
        )
    )


    probability_negative = float(
        np.mean(
            delta_values
            <
            0
        )
    )


    ci_excludes_zero = bool(
        (
            ci_lower
            >
            0
        )
        or
        (
            ci_upper
            <
            0
        )
    )


    direction = (
        "positive"
        if point_delta[
            metric
        ]
        >
        0

        else
        (
            "negative"
            if point_delta[
                metric
            ]
            <
            0

            else
            "zero"
        )
    )


    summary_records.append(
        {
            "metric":
                metric,

            "day15_point":
                float(
                    day15_point[
                        metric
                    ]
                ),

            "day23_point":
                float(
                    day23_point[
                        metric
                    ]
                ),

            "point_delta":
                float(
                    point_delta[
                        metric
                    ]
                ),

            "delta_percentage_points":
                float(
                    point_delta[
                        metric
                    ]
                    *
                    100.0
                ),

            "bootstrap_mean_delta":
                bootstrap_mean,

            "bootstrap_sd_delta":
                bootstrap_sd,

            "ci_level":
                PAIRED_CI_LEVEL,

            "ci_lower":
                ci_lower,

            "ci_upper":
                ci_upper,

            "ci_excludes_zero":
                ci_excludes_zero,

            "bootstrap_probability_delta_gt_0":
                probability_positive,

            "bootstrap_probability_delta_lt_0":
                probability_negative,

            "point_direction":
                direction,
        }
    )


paired_bootstrap_summary_df = pd.DataFrame(
    summary_records
)


paired_bootstrap_summary_df.to_csv(
    PAIRED_BOOTSTRAP_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 10. Key inference flags
#
# These are CI-based paired uncertainty statements.
# They are NOT classical hypothesis-test p-values.
# --------------------------------------------------------------------------------------------------

def get_summary_row(
    metric,
):

    return (
        paired_bootstrap_summary_df.loc[
            paired_bootstrap_summary_df[
                "metric"
            ]
            ==
            metric
        ]
        .iloc[0]
    )


accuracy_summary = get_summary_row(
    "accuracy"
)

macro_f1_summary = get_summary_row(
    "macro_f1"
)

negative_recall_summary = get_summary_row(
    "negative_recall"
)

neutral_recall_summary = get_summary_row(
    "neutral_recall"
)

positive_recall_summary = get_summary_row(
    "positive_recall"
)


ACCURACY_IMPROVEMENT_CI_EXCLUDES_ZERO = bool(
    accuracy_summary[
        "ci_lower"
    ]
    >
    0
)


MACRO_F1_IMPROVEMENT_CI_EXCLUDES_ZERO = bool(
    macro_f1_summary[
        "ci_lower"
    ]
    >
    0
)


# --------------------------------------------------------------------------------------------------
# 11. Formal checks
# --------------------------------------------------------------------------------------------------

paired_bootstrap_checks = {

    **point_checks,

    "alignment_artifact_hash_unchanged":
        (
            sha256_file(
                DAY15_ALIGNMENT_TABLE_PATH
            )
            ==
            DAY15_ALIGNMENT_SHA256
        ),

    "paired_population_5859":
        len(
            paired_df
        )
        ==
        5859,

    "paired_ids_unique":
        paired_df[
            "observation_id"
        ]
        .nunique()
        ==
        5859,

    "bootstrap_replicates_1000":
        len(
            paired_bootstrap_df
        )
        ==
        1000,

    "bootstrap_seed_42":
        PAIRED_BOOTSTRAP_SEED
        ==
        42,

    "same_indices_used_for_both_models":
        True,

    "all_delta_values_finite":
        np.isfinite(
            paired_bootstrap_df[
                [
                    f"delta_{metric}"
                    for metric
                    in metric_names
                ]
            ]
            .to_numpy()
        )
        .all(),

    "all_ci_bounds_finite":
        np.isfinite(
            paired_bootstrap_summary_df[
                [
                    "ci_lower",
                    "ci_upper",
                ]
            ]
            .to_numpy()
        )
        .all(),

    "accuracy_ci_contains_point_delta":
        (
            accuracy_summary[
                "ci_lower"
            ]
            <=
            point_delta[
                "accuracy"
            ]
            <=
            accuracy_summary[
                "ci_upper"
            ]
        ),

    "macro_f1_ci_contains_point_delta":
        (
            macro_f1_summary[
                "ci_lower"
            ]
            <=
            point_delta[
                "macro_f1"
            ]
            <=
            macro_f1_summary[
                "ci_upper"
            ]
        ),

    "no_model_inference":
        True,

    "no_retraining":
        True,

    "no_retuning":
        True,

    "future_not_used":
        True,
}


all_paired_bootstrap_checks_pass = bool(
    all(
        paired_bootstrap_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. Save manifest
# --------------------------------------------------------------------------------------------------

paired_bootstrap_manifest = {

    "day":
        23,

    "analysis":
        "paired_bootstrap_day23_minus_day15_historical_validation",

    "population": {

        "n":
            5859,

        "alignment_artifact":
            str(
                DAY15_ALIGNMENT_TABLE_PATH
            ),

        "alignment_sha256":
            DAY15_ALIGNMENT_SHA256,
    },

    "protocol": {

        "method":
            "paired_nonparametric_row_bootstrap",

        "pairing_unit":
            "observation_id",

        "same_resampled_indices_for_both_models":
            True,

        "replicates":
            PAIRED_BOOTSTRAP_REPLICATES,

        "seed":
            PAIRED_BOOTSTRAP_SEED,

        "confidence_level":
            PAIRED_CI_LEVEL,

        "interval":
            "percentile",
    },

    "primary_results": {

        "accuracy": {

            "day15":
                float(
                    day15_point[
                        "accuracy"
                    ]
                ),

            "day23":
                float(
                    day23_point[
                        "accuracy"
                    ]
                ),

            "delta":
                float(
                    point_delta[
                        "accuracy"
                    ]
                ),

            "ci_lower":
                float(
                    accuracy_summary[
                        "ci_lower"
                    ]
                ),

            "ci_upper":
                float(
                    accuracy_summary[
                        "ci_upper"
                    ]
                ),

            "ci_excludes_zero":
                ACCURACY_IMPROVEMENT_CI_EXCLUDES_ZERO,
        },

        "macro_f1": {

            "day15":
                float(
                    day15_point[
                        "macro_f1"
                    ]
                ),

            "day23":
                float(
                    day23_point[
                        "macro_f1"
                    ]
                ),

            "delta":
                float(
                    point_delta[
                        "macro_f1"
                    ]
                ),

            "ci_lower":
                float(
                    macro_f1_summary[
                        "ci_lower"
                    ]
                ),

            "ci_upper":
                float(
                    macro_f1_summary[
                        "ci_upper"
                    ]
                ),

            "ci_excludes_zero":
                MACRO_F1_IMPROVEMENT_CI_EXCLUDES_ZERO,
        },
    },

    "interpretation_rule": {

        "allowed_statement":
            (
                "A paired 95% bootstrap confidence interval that lies entirely above zero "
                "supports a positive performance difference on this fixed Historical Validation population."
            ),

        "classical_p_value_computed":
            False,

        "do_not_report_as":
            "classical hypothesis-test significance unless a separate paired test is performed",
    },

    "firewall": {

        "model_inference_performed":
            False,

        "retraining_performed":
            False,

        "retuning_performed":
            False,

        "future_f1_f6_used":
            False,
    },

    "checks": {

        key:
            bool(
                value
            )

        for key, value
        in paired_bootstrap_checks.items()
    },

    "all_checks_pass":
        all_paired_bootstrap_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    PAIRED_BOOTSTRAP_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        paired_bootstrap_manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 13. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — PAIRED BOOTSTRAP: DAY23 FINAL MODEL VS DAY15 BASELINE")
print("=" * 120)


print("\n[PAIRING]")
print(
    "Population                  : Same Historical Validation observations"
)
print(
    f"N                           : {len(paired_df):,}"
)
print(
    "Pairing key                 : observation_id"
)
print(
    "Gold labels                 : IDENTICAL"
)
print(
    "Bootstrap type              : Paired nonparametric row bootstrap"
)
print(
    f"Replicates                  : {PAIRED_BOOTSTRAP_REPLICATES:,}"
)
print(
    f"Seed                        : {PAIRED_BOOTSTRAP_SEED}"
)
print(
    f"CI                          : {PAIRED_CI_LEVEL * 100:.0f}% percentile"
)


print("\n[PAIRED PERFORMANCE DIFFERENCE: DAY23 - DAY15]")

display(
    paired_bootstrap_summary_df[
        [
            "metric",
            "day15_point",
            "day23_point",
            "point_delta",
            "delta_percentage_points",
            "ci_lower",
            "ci_upper",
            "ci_excludes_zero",
            "bootstrap_probability_delta_gt_0",
        ]
    ]
)


print("\n[PRIMARY RESULTS]")

print(
    f"Accuracy Day15              : "
    f"{day15_point['accuracy']:.6f}"
)
print(
    f"Accuracy Day23              : "
    f"{day23_point['accuracy']:.6f}"
)
print(
    f"Accuracy improvement        : "
    f"{point_delta['accuracy']:+.6f} "
    f"({point_delta['accuracy'] * 100:+.2f} pp)"
)
print(
    f"Accuracy improvement 95% CI: "
    f"[{accuracy_summary['ci_lower']:+.6f}, "
    f"{accuracy_summary['ci_upper']:+.6f}]"
)
print(
    f"Accuracy CI excludes zero   : "
    f"{ACCURACY_IMPROVEMENT_CI_EXCLUDES_ZERO}"
)


print()

print(
    f"Macro-F1 Day15              : "
    f"{day15_point['macro_f1']:.6f}"
)
print(
    f"Macro-F1 Day23              : "
    f"{day23_point['macro_f1']:.6f}"
)
print(
    f"Macro-F1 improvement        : "
    f"{point_delta['macro_f1']:+.6f} "
    f"({point_delta['macro_f1'] * 100:+.2f} pp)"
)
print(
    f"Macro-F1 improvement 95% CI: "
    f"[{macro_f1_summary['ci_lower']:+.6f}, "
    f"{macro_f1_summary['ci_upper']:+.6f}]"
)
print(
    f"Macro-F1 CI excludes zero   : "
    f"{MACRO_F1_IMPROVEMENT_CI_EXCLUDES_ZERO}"
)


print("\n[CLASS RECALL DIFFERENCES]")

print(
    f"Negative recall delta       : "
    f"{point_delta['negative_recall']:+.6f} "
    f"95% CI "
    f"[{negative_recall_summary['ci_lower']:+.6f}, "
    f"{negative_recall_summary['ci_upper']:+.6f}]"
)

print(
    f"Neutral recall delta        : "
    f"{point_delta['neutral_recall']:+.6f} "
    f"95% CI "
    f"[{neutral_recall_summary['ci_lower']:+.6f}, "
    f"{neutral_recall_summary['ci_upper']:+.6f}]"
)

print(
    f"Positive recall delta       : "
    f"{point_delta['positive_recall']:+.6f} "
    f"95% CI "
    f"[{positive_recall_summary['ci_lower']:+.6f}, "
    f"{positive_recall_summary['ci_upper']:+.6f}]"
)


print("\n[METHODOLOGICAL INTERPRETATION]")
print(
    "Paired bootstrap           : VALID"
)
print(
    "Accuracy delta CI           : "
    + (
        "ENTIRELY ABOVE ZERO"
        if ACCURACY_IMPROVEMENT_CI_EXCLUDES_ZERO
        else
        "INCLUDES ZERO"
    )
)
print(
    "Macro-F1 delta CI           : "
    + (
        "ENTIRELY ABOVE ZERO"
        if MACRO_F1_IMPROVEMENT_CI_EXCLUDES_ZERO
        else
        "INCLUDES ZERO"
    )
)
print(
    "Classical p-value           : NOT COMPUTED"
)
print(
    "Model inference             : NOT PERFORMED"
)
print(
    "Retuning                    : NOT PERFORMED"
)
print(
    "Future F1-F6                : NOT USED"
)


print("\n[FORMAL CHECKS]")

for name, passed in paired_bootstrap_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Bootstrap draws             : "
    f"{PAIRED_BOOTSTRAP_DRAWS_PATH}"
)
print(
    f"Bootstrap summary           : "
    f"{PAIRED_BOOTSTRAP_SUMMARY_PATH}"
)
print(
    f"Manifest                    : "
    f"{PAIRED_BOOTSTRAP_MANIFEST_PATH}"
)


print("\n" + "-" * 120)

if all_paired_bootstrap_checks_pass:

    print(
        "DAY 23 CELL 26 STATUS: PASS — "
        "PAIRED BOOTSTRAP OF DAY23 VS DAY15 COMPLETE; "
        "PERFORMANCE-IMPROVEMENT UNCERTAINTY QUANTIFIED"
    )

else:

    print(
        "DAY 23 CELL 26 STATUS: FAIL — "
        "DO NOT INTERPRET PAIRED IMPROVEMENT RESULTS"
    )

print("=" * 120)

NameError: name 'FINAL_HISTVAL_DIR' is not defined

In [16]:
# ==================================================================================================
# DAY 23 — CELL 27R
# RESTART-SAFE FINAL SCIENTIFIC INTEGRATION
#
# PURPOSE:
#   Recover the already-completed Day23 terminal scientific state directly from frozen disk artifacts
#   after a Colab runtime reset.
#
# IMPORTANT:
#   - NO training
#   - NO model loading
#   - NO inference
#   - NO retuning
#   - NO Future F1-F6 access
# ==================================================================================================

import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Reconstruct roots directly
# --------------------------------------------------------------------------------------------------

# PROJECT_ROOT is discovered in the project-path setup cell.

DAY23_RESULTS_ROOT = (
    PROJECT_ROOT
    / "results"
    / "day23"
)

assert PROJECT_ROOT.exists(), (
    f"DAY 23 STOP: PROJECT_ROOT missing:\n{PROJECT_ROOT}"
)

assert DAY23_RESULTS_ROOT.exists(), (
    f"DAY 23 STOP: DAY23_RESULTS_ROOT missing:\n{DAY23_RESULTS_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 2. Define FINAL artifact paths directly
# --------------------------------------------------------------------------------------------------

STAGE_D_DIR = (
    DAY23_RESULTS_ROOT
    / "stage_d_learning_rate_audit"
)

STAGE_E_DIR = (
    DAY23_RESULTS_ROOT
    / "stage_e_lora_capacity_audit"
)

STAGE_E_ARTIFACT_DIR = (
    STAGE_E_DIR
    / "artifacts"
)

STAGE_E_SELECTED_ADAPTER_DIR = (
    STAGE_E_DIR
    / "selected_adapter"
)

STAGE_E_SELECTION_SUMMARY_PATH = (
    STAGE_E_ARTIFACT_DIR
    / "stage_e_selection_summary.json"
)


FINAL_FREEZE_DIR = (
    DAY23_RESULTS_ROOT
    / "final_configuration_freeze"
)

FINAL_CONFIG_PATH = (
    FINAL_FREEZE_DIR
    / "final_historical_model_configuration.json"
)

FINAL_FIREWALL_PATH = (
    FINAL_FREEZE_DIR
    / "historical_validation_firewall.json"
)

FINAL_FREEZE_COMPLETE_PATH = (
    FINAL_FREEZE_DIR
    / "final_configuration_freeze_complete.json"
)


FINAL_HISTVAL_DIR = (
    DAY23_RESULTS_ROOT
    / "final_historical_validation"
)

FINAL_HISTVAL_PREDICTIONS_PATH = (
    FINAL_HISTVAL_DIR
    / "final_historical_validation_predictions.csv"
)

FINAL_HISTVAL_METRICS_PATH = (
    FINAL_HISTVAL_DIR
    / "final_historical_validation_metrics.json"
)

FINAL_HISTVAL_CLASS_METRICS_PATH = (
    FINAL_HISTVAL_DIR
    / "final_historical_validation_class_metrics.csv"
)

FINAL_HISTVAL_CONFUSION_PATH = (
    FINAL_HISTVAL_DIR
    / "final_historical_validation_confusion_matrix.csv"
)

FINAL_HISTVAL_COMPLETE_PATH = (
    FINAL_HISTVAL_DIR
    / "one_shot_historical_validation_complete.json"
)


FINAL_HISTVAL_BOOTSTRAP_DIR = (
    FINAL_HISTVAL_DIR
    / "bootstrap"
)

FINAL_HISTVAL_BOOTSTRAP_SUMMARY_PATH = (
    FINAL_HISTVAL_BOOTSTRAP_DIR
    / "historical_validation_bootstrap_summary.csv"
)


DAY15_HISTVAL_PREDICTIONS_PATH = (
    PROJECT_ROOT
    / "results"
    / "day15"
    / "historical_validation_predictions.csv"
)


DAY15_ALIGNMENT_DIR = (
    FINAL_HISTVAL_DIR
    / "day15_baseline_recovery"
    / "alignment"
)

DAY15_ALIGNMENT_TABLE_PATH = (
    DAY15_ALIGNMENT_DIR
    / "day15_day23_historical_validation_alignment.csv"
)

DAY15_ALIGNMENT_SUMMARY_PATH = (
    DAY15_ALIGNMENT_DIR
    / "day15_day23_alignment_summary.json"
)


PAIRED_BOOTSTRAP_DIR = (
    FINAL_HISTVAL_DIR
    / "paired_bootstrap_day23_vs_day15"
)

PAIRED_BOOTSTRAP_SUMMARY_PATH = (
    PAIRED_BOOTSTRAP_DIR
    / "day23_minus_day15_paired_bootstrap_summary.csv"
)

PAIRED_BOOTSTRAP_MANIFEST_PATH = (
    PAIRED_BOOTSTRAP_DIR
    / "day23_minus_day15_paired_bootstrap_manifest.json"
)


# --------------------------------------------------------------------------------------------------
# 3. Required terminal artifacts
# --------------------------------------------------------------------------------------------------

required_terminal_artifacts = {

    "stage_e_selection_summary":
        STAGE_E_SELECTION_SUMMARY_PATH,

    "stage_e_selected_adapter":
        STAGE_E_SELECTED_ADAPTER_DIR,

    "final_configuration":
        FINAL_CONFIG_PATH,

    "final_firewall":
        FINAL_FIREWALL_PATH,

    "final_freeze_complete":
        FINAL_FREEZE_COMPLETE_PATH,

    "final_histval_predictions":
        FINAL_HISTVAL_PREDICTIONS_PATH,

    "final_histval_metrics":
        FINAL_HISTVAL_METRICS_PATH,

    "final_histval_complete":
        FINAL_HISTVAL_COMPLETE_PATH,

    "final_histval_bootstrap_summary":
        FINAL_HISTVAL_BOOTSTRAP_SUMMARY_PATH,

    "day15_predictions":
        DAY15_HISTVAL_PREDICTIONS_PATH,

    "day15_day23_alignment":
        DAY15_ALIGNMENT_TABLE_PATH,

    "day15_day23_alignment_summary":
        DAY15_ALIGNMENT_SUMMARY_PATH,

    "paired_bootstrap_summary":
        PAIRED_BOOTSTRAP_SUMMARY_PATH,

    "paired_bootstrap_manifest":
        PAIRED_BOOTSTRAP_MANIFEST_PATH,
}


terminal_artifact_checks = {

    name:
        Path(path).exists()

    for name, path
    in required_terminal_artifacts.items()
}


assert all(
    terminal_artifact_checks.values()
), (
    "DAY 23 STOP: one or more required completed Day23 terminal artifacts are missing."
)


# --------------------------------------------------------------------------------------------------
# 4. SHA256 helper
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


# --------------------------------------------------------------------------------------------------
# 5. Load frozen terminal JSON artifacts
# --------------------------------------------------------------------------------------------------

with open(
    STAGE_E_SELECTION_SUMMARY_PATH,
    "r",
    encoding="utf-8",
) as f:

    stage_e_summary = load_artifact_json(
        f
    )


with open(
    FINAL_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as f:

    final_config = load_artifact_json(
        f
    )


with open(
    FINAL_FIREWALL_PATH,
    "r",
    encoding="utf-8",
) as f:

    final_firewall = load_artifact_json(
        f
    )


with open(
    FINAL_FREEZE_COMPLETE_PATH,
    "r",
    encoding="utf-8",
) as f:

    final_freeze_complete = load_artifact_json(
        f
    )


with open(
    FINAL_HISTVAL_METRICS_PATH,
    "r",
    encoding="utf-8",
) as f:

    histval_metrics_payload = load_artifact_json(
        f
    )


with open(
    FINAL_HISTVAL_COMPLETE_PATH,
    "r",
    encoding="utf-8",
) as f:

    histval_complete = load_artifact_json(
        f
    )


with open(
    DAY15_ALIGNMENT_SUMMARY_PATH,
    "r",
    encoding="utf-8",
) as f:

    alignment_summary = load_artifact_json(
        f
    )


with open(
    PAIRED_BOOTSTRAP_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as f:

    paired_manifest = load_artifact_json(
        f
    )


# --------------------------------------------------------------------------------------------------
# 6. Load CSV summaries
# --------------------------------------------------------------------------------------------------

histval_bootstrap_summary_df = pd.read_csv(
    FINAL_HISTVAL_BOOTSTRAP_SUMMARY_PATH
)


paired_bootstrap_summary_df = pd.read_csv(
    PAIRED_BOOTSTRAP_SUMMARY_PATH
)


histval_predictions_df = pd.read_csv(
    FINAL_HISTVAL_PREDICTIONS_PATH
)


alignment_df = pd.read_csv(
    DAY15_ALIGNMENT_TABLE_PATH
)


# --------------------------------------------------------------------------------------------------
# 7. Recover final model configuration FROM DISK
# --------------------------------------------------------------------------------------------------

final_training_config = (
    final_config[
        "training_configuration"
    ]
)

final_lora_config = (
    final_training_config[
        "lora"
    ]
)


FINAL_BASE_MODEL = (
    final_config[
        "model"
    ][
        "base_model"
    ]
)

FINAL_SAMPLING = (
    final_training_config[
        "sampling"
    ]
)

FINAL_MAX_STEPS = int(
    final_training_config[
        "max_steps"
    ]
)

FINAL_LEARNING_RATE = float(
    final_training_config[
        "learning_rate"
    ]
)

FINAL_LORA_R = int(
    final_lora_config[
        "r"
    ]
)

FINAL_LORA_ALPHA = int(
    final_lora_config[
        "alpha"
    ]
)

FINAL_LORA_DROPOUT = float(
    final_lora_config[
        "dropout"
    ]
)

FINAL_SEED = int(
    final_training_config[
        "seed"
    ]
)

FINAL_SELECTED_ADAPTER_PATH = Path(
    final_config[
        "model"
    ][
        "selected_adapter"
    ]
)

FINAL_ADAPTER_WEIGHTS_SHA256 = (
    final_config[
        "model"
    ][
        "adapter_weights_sha256"
    ]
)


# --------------------------------------------------------------------------------------------------
# 8. Recover Optimization Dev final winner FROM DISK
# --------------------------------------------------------------------------------------------------

FINAL_OPT_DEV_METRICS = (
    final_config[
        "optimization_dev_selection_metrics"
    ]
)


# --------------------------------------------------------------------------------------------------
# 9. Recover Historical Validation results FROM DISK
# --------------------------------------------------------------------------------------------------

histval_metrics = (
    histval_metrics_payload[
        "metrics"
    ]
)


FINAL_HISTVAL_ACCURACY = float(
    histval_metrics[
        "accuracy"
    ]
)

FINAL_HISTVAL_MACRO_F1 = float(
    histval_metrics[
        "macro_f1"
    ]
)

FINAL_HISTVAL_WEIGHTED_F1 = float(
    histval_metrics[
        "weighted_f1"
    ]
)

FINAL_HISTVAL_NEG_RECALL = float(
    histval_metrics[
        "negative_recall"
    ]
)

FINAL_HISTVAL_NEU_RECALL = float(
    histval_metrics[
        "neutral_recall"
    ]
)

FINAL_HISTVAL_POS_RECALL = float(
    histval_metrics[
        "positive_recall"
    ]
)


# --------------------------------------------------------------------------------------------------
# 10. Recover bootstrap CIs FROM DISK
# --------------------------------------------------------------------------------------------------

def get_bootstrap_row(
    df,
    metric,
):

    rows = df.loc[
        df[
            "metric"
        ]
        ==
        metric
    ]

    assert len(
        rows
    ) == 1

    return rows.iloc[
        0
    ]


histval_accuracy_boot = get_bootstrap_row(
    histval_bootstrap_summary_df,
    "accuracy",
)

histval_macro_boot = get_bootstrap_row(
    histval_bootstrap_summary_df,
    "macro_f1",
)


FINAL_HISTVAL_ACCURACY_CI = [
    float(
        histval_accuracy_boot[
            "ci_lower"
        ]
    ),
    float(
        histval_accuracy_boot[
            "ci_upper"
        ]
    ),
]


FINAL_HISTVAL_MACRO_F1_CI = [
    float(
        histval_macro_boot[
            "ci_lower"
        ]
    ),
    float(
        histval_macro_boot[
            "ci_upper"
        ]
    ),
]


# --------------------------------------------------------------------------------------------------
# 11. Recover paired Day23 - Day15 results FROM DISK
# --------------------------------------------------------------------------------------------------

paired_accuracy = get_bootstrap_row(
    paired_bootstrap_summary_df,
    "accuracy",
)

paired_macro_f1 = get_bootstrap_row(
    paired_bootstrap_summary_df,
    "macro_f1",
)

paired_negative = get_bootstrap_row(
    paired_bootstrap_summary_df,
    "negative_recall",
)

paired_neutral = get_bootstrap_row(
    paired_bootstrap_summary_df,
    "neutral_recall",
)

paired_positive = get_bootstrap_row(
    paired_bootstrap_summary_df,
    "positive_recall",
)


DAY15_ACCURACY = float(
    paired_accuracy[
        "day15_point"
    ]
)

DAY23_ACCURACY = float(
    paired_accuracy[
        "day23_point"
    ]
)

ACCURACY_DELTA = float(
    paired_accuracy[
        "point_delta"
    ]
)

ACCURACY_DELTA_CI = [
    float(
        paired_accuracy[
            "ci_lower"
        ]
    ),
    float(
        paired_accuracy[
            "ci_upper"
        ]
    ),
]


DAY15_MACRO_F1 = float(
    paired_macro_f1[
        "day15_point"
    ]
)

DAY23_MACRO_F1 = float(
    paired_macro_f1[
        "day23_point"
    ]
)

MACRO_F1_DELTA = float(
    paired_macro_f1[
        "point_delta"
    ]
)

MACRO_F1_DELTA_CI = [
    float(
        paired_macro_f1[
            "ci_lower"
        ]
    ),
    float(
        paired_macro_f1[
            "ci_upper"
        ]
    ),
]


# --------------------------------------------------------------------------------------------------
# 12. Verify frozen prediction artifact integrity
# --------------------------------------------------------------------------------------------------

current_prediction_sha256 = (
    sha256_file(
        FINAL_HISTVAL_PREDICTIONS_PATH
    )
)


expected_prediction_sha256 = (
    histval_metrics_payload[
        "prediction_artifact"
    ][
        "sha256"
    ]
)


# --------------------------------------------------------------------------------------------------
# 13. Terminal scientific integrity checks
# --------------------------------------------------------------------------------------------------

terminal_scientific_checks = {

    "stage_e_winner_r16_a64":
        (
            stage_e_summary[
                "winner"
            ][
                "condition"
            ]
            ==
            "r16_a64"
        ),

    "final_sampling_natural":
        FINAL_SAMPLING
        ==
        "natural",

    "final_horizon_500":
        FINAL_MAX_STEPS
        ==
        500,

    "final_lr_2e4":
        abs(
            FINAL_LEARNING_RATE
            -
            2e-4
        )
        <
        1e-12,

    "final_lora_r16":
        FINAL_LORA_R
        ==
        16,

    "final_lora_alpha64":
        FINAL_LORA_ALPHA
        ==
        64,

    "final_seed_42":
        FINAL_SEED
        ==
        42,

    "selected_adapter_exists":
        FINAL_SELECTED_ADAPTER_PATH.exists(),

    "final_freeze_pass":
        final_freeze_complete.get(
            "status"
        )
        ==
        "PASS",

    "histval_evaluated_once":
        histval_complete.get(
            "status"
        )
        ==
        "PASS",

    "histval_predictions_n_5859":
        len(
            histval_predictions_df
        )
        ==
        5859,

    "histval_prediction_hash_unchanged":
        current_prediction_sha256
        ==
        expected_prediction_sha256,

    "alignment_n_5859":
        len(
            alignment_df
        )
        ==
        5859,

    "alignment_pass":
        bool(
            alignment_summary[
                "all_checks_pass"
            ]
        ),

    "paired_bootstrap_pass":
        bool(
            paired_manifest[
                "all_checks_pass"
            ]
        ),

    "accuracy_delta_positive":
        ACCURACY_DELTA
        >
        0,

    "accuracy_delta_ci_above_zero":
        ACCURACY_DELTA_CI[
            0
        ]
        >
        0,

    "macro_f1_delta_positive":
        MACRO_F1_DELTA
        >
        0,

    "macro_f1_delta_ci_above_zero":
        MACRO_F1_DELTA_CI[
            0
        ]
        >
        0,

    "historical_validation_retuning_forbidden":
        (
            final_firewall[
                "post_evaluation_retuning_allowed"
            ]
            is False
        ),

    "future_f1_f6_locked":
        (
            final_firewall[
                "future_f1_f6_status"
            ]
            ==
            "LOCKED"
        ),

    "all_terminal_artifacts_exist":
        all(
            terminal_artifact_checks.values()
        ),

    "no_training":
        True,

    "no_model_loading":
        True,

    "no_inference":
        True,

    "no_retuning":
        True,

    "future_not_accessed":
        True,
}


all_terminal_scientific_checks_pass = bool(
    all(
        terminal_scientific_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 14. Reconstruct historical optimization trajectory
#
# These values were already established by completed Stage A-E experiments.
# This table is integration-only; it performs no selection.
# --------------------------------------------------------------------------------------------------

optimization_trajectory_df = pd.DataFrame(
    [
        {
            "stage":
                "A",

            "intervention":
                "baseline reproduction",

            "sampling":
                "natural",

            "max_steps":
                200,

            "learning_rate":
                1e-4,

            "lora_r":
                8,

            "lora_alpha":
                32,

            "optimization_dev_accuracy":
                0.779542,

            "optimization_dev_macro_f1":
                0.726037,
        },

        {
            "stage":
                "B",

            "intervention":
                "training horizon",

            "sampling":
                "natural",

            "max_steps":
                500,

            "learning_rate":
                1e-4,

            "lora_r":
                8,

            "lora_alpha":
                32,

            "optimization_dev_accuracy":
                0.840842,

            "optimization_dev_macro_f1":
                0.808201,
        },

        {
            "stage":
                "C",

            "intervention":
                "sampling audit",

            "sampling":
                "natural",

            "max_steps":
                500,

            "learning_rate":
                1e-4,

            "lora_r":
                8,

            "lora_alpha":
                32,

            "optimization_dev_accuracy":
                0.840842,

            "optimization_dev_macro_f1":
                0.808201,
        },

        {
            "stage":
                "D",

            "intervention":
                "learning-rate audit",

            "sampling":
                "natural",

            "max_steps":
                500,

            "learning_rate":
                2e-4,

            "lora_r":
                8,

            "lora_alpha":
                32,

            "optimization_dev_accuracy":
                0.843427,

            "optimization_dev_macro_f1":
                0.813516,
        },

        {
            "stage":
                "E",

            "intervention":
                "LoRA capacity audit",

            "sampling":
                "natural",

            "max_steps":
                500,

            "learning_rate":
                2e-4,

            "lora_r":
                16,

            "lora_alpha":
                64,

            "optimization_dev_accuracy":
                float(
                    FINAL_OPT_DEV_METRICS[
                        "accuracy"
                    ]
                ),

            "optimization_dev_macro_f1":
                float(
                    FINAL_OPT_DEV_METRICS[
                        "macro_f1"
                    ]
                ),
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 15. Final integration directory
# --------------------------------------------------------------------------------------------------

DAY23_FINAL_DIR = (
    DAY23_RESULTS_ROOT
    / "final_integration"
)

DAY23_FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


DAY23_FINAL_TRAJECTORY_PATH = (
    DAY23_FINAL_DIR
    / "day23_historical_optimization_trajectory.csv"
)

DAY23_FINAL_SUMMARY_PATH = (
    DAY23_FINAL_DIR
    / "day23_final_scientific_summary.json"
)

DAY23_FINAL_COMPLETION_PATH = (
    DAY23_FINAL_DIR
    / "day23_completion_checkpoint.json"
)


optimization_trajectory_df.to_csv(
    DAY23_FINAL_TRAJECTORY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 16. Final scientific summary
# --------------------------------------------------------------------------------------------------

final_summary = {

    "day":
        23,

    "status":
        "HISTORICAL_MODEL_OPTIMIZATION_AND_VALIDATION_COMPLETE",

    "runtime_recovery_mode":
        True,

    "final_model": {

        "base_model":
            FINAL_BASE_MODEL,

        "sampling":
            FINAL_SAMPLING,

        "training_horizon":
            FINAL_MAX_STEPS,

        "learning_rate":
            FINAL_LEARNING_RATE,

        "lora_r":
            FINAL_LORA_R,

        "lora_alpha":
            FINAL_LORA_ALPHA,

        "lora_dropout":
            FINAL_LORA_DROPOUT,

        "seed":
            FINAL_SEED,

        "selected_adapter":
            str(
                FINAL_SELECTED_ADAPTER_PATH
            ),

        "adapter_weights_sha256":
            FINAL_ADAPTER_WEIGHTS_SHA256,
    },

    "optimization_dev": {

        key:
            value

        for key, value
        in FINAL_OPT_DEV_METRICS.items()
    },

    "historical_validation": {

        "n":
            5859,

        "accuracy":
            FINAL_HISTVAL_ACCURACY,

        "accuracy_ci_95":
            FINAL_HISTVAL_ACCURACY_CI,

        "macro_f1":
            FINAL_HISTVAL_MACRO_F1,

        "macro_f1_ci_95":
            FINAL_HISTVAL_MACRO_F1_CI,

        "weighted_f1":
            FINAL_HISTVAL_WEIGHTED_F1,

        "negative_recall":
            FINAL_HISTVAL_NEG_RECALL,

        "neutral_recall":
            FINAL_HISTVAL_NEU_RECALL,

        "positive_recall":
            FINAL_HISTVAL_POS_RECALL,
    },

    "day15_vs_day23_paired_improvement": {

        "n":
            5859,

        "accuracy_day15":
            DAY15_ACCURACY,

        "accuracy_day23":
            DAY23_ACCURACY,

        "accuracy_delta":
            ACCURACY_DELTA,

        "accuracy_delta_ci_95":
            ACCURACY_DELTA_CI,

        "macro_f1_day15":
            DAY15_MACRO_F1,

        "macro_f1_day23":
            DAY23_MACRO_F1,

        "macro_f1_delta":
            MACRO_F1_DELTA,

        "macro_f1_delta_ci_95":
            MACRO_F1_DELTA_CI,
    },

    "generalization_gap": {

        "histval_minus_optdev_accuracy":
            float(
                FINAL_HISTVAL_ACCURACY
                -
                float(
                    FINAL_OPT_DEV_METRICS[
                        "accuracy"
                    ]
                )
            ),

        "histval_minus_optdev_macro_f1":
            float(
                FINAL_HISTVAL_MACRO_F1
                -
                float(
                    FINAL_OPT_DEV_METRICS[
                        "macro_f1"
                    ]
                )
            ),
    },

    "scientific_interpretation": {

        "historical_adequacy":
            (
                "Historical model adequacy improved materially relative to the Day15 baseline."
            ),

        "heldout_validation":
            (
                "The improvement persisted on the untouched Historical Validation population."
            ),

        "paired_uncertainty":
            (
                "Paired bootstrap confidence intervals for Accuracy and Macro-F1 improvement "
                "are entirely above zero."
            ),

        "future_scope":
            (
                "Future F1-F6 have not been re-evaluated with the optimized model. "
                "Therefore improved future temporal generalization has not yet been established."
            ),
    },

    "firewall": {

        "historical_validation":
            "EVALUATED_ONCE_AND_FROZEN",

        "post_validation_retuning":
            "FORBIDDEN",

        "future_f1_f6":
            "LOCKED",
    },

    "terminal_checks": {

        key:
            bool(
                value
            )

        for key, value
        in terminal_scientific_checks.items()
    },

    "all_checks_pass":
        all_terminal_scientific_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    DAY23_FINAL_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        final_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 17. Completion checkpoint
# --------------------------------------------------------------------------------------------------

if all_terminal_scientific_checks_pass:

    completion_payload = {

        "day":
            23,

        "status":
            "PASS",

        "runtime_recovery":
            True,

        "historical_optimization_complete":
            True,

        "historical_validation_complete":
            True,

        "paired_baseline_comparison_complete":
            True,

        "historical_validation_rerun_allowed":
            False,

        "retuning_allowed":
            False,

        "future_f1_f6_status":
            "LOCKED",

        "next_action":
            "SAVE_NOTEBOOK_AND_CLOSE_DAY23",

        "created_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


    with open(
        DAY23_FINAL_COMPLETION_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            completion_payload,
            f,
            indent=2,
            ensure_ascii=False,
        )


# --------------------------------------------------------------------------------------------------
# 18. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 23 — RESTART-SAFE FINAL SCIENTIFIC INTEGRATION")
print("=" * 120)


print("\n[RECOVERY MODE]")
print(
    "Runtime state               : RESET"
)
print(
    "Scientific state            : RECOVERED FROM FROZEN DISK ARTIFACTS"
)
print(
    "Training rerun              : NO"
)
print(
    "Inference rerun             : NO"
)


print("\n[FINAL MODEL]")
print(
    f"Base model                  : {FINAL_BASE_MODEL}"
)
print(
    f"Sampling                    : {FINAL_SAMPLING}"
)
print(
    f"Training horizon            : {FINAL_MAX_STEPS}"
)
print(
    f"Learning rate               : {FINAL_LEARNING_RATE}"
)
print(
    f"LoRA                        : r={FINAL_LORA_R}, alpha={FINAL_LORA_ALPHA}"
)
print(
    f"Seed                        : {FINAL_SEED}"
)


print("\n[OPTIMIZATION TRAJECTORY]")
display(
    optimization_trajectory_df
)


print("\n[FINAL HISTORICAL VALIDATION]")
print(
    f"Accuracy                    : "
    f"{FINAL_HISTVAL_ACCURACY:.6f} "
    f"[{FINAL_HISTVAL_ACCURACY_CI[0]:.6f}, "
    f"{FINAL_HISTVAL_ACCURACY_CI[1]:.6f}]"
)
print(
    f"Macro-F1                    : "
    f"{FINAL_HISTVAL_MACRO_F1:.6f} "
    f"[{FINAL_HISTVAL_MACRO_F1_CI[0]:.6f}, "
    f"{FINAL_HISTVAL_MACRO_F1_CI[1]:.6f}]"
)


print("\n[PAIRED IMPROVEMENT VS DAY15]")
print(
    f"Accuracy                    : "
    f"{DAY15_ACCURACY:.6f} -> {DAY23_ACCURACY:.6f}"
)
print(
    f"Accuracy delta              : "
    f"{ACCURACY_DELTA:+.6f} "
    f"({ACCURACY_DELTA * 100:+.2f} pp)"
)
print(
    f"Accuracy delta 95% CI       : "
    f"[{ACCURACY_DELTA_CI[0]:+.6f}, "
    f"{ACCURACY_DELTA_CI[1]:+.6f}]"
)

print(
    f"Macro-F1                    : "
    f"{DAY15_MACRO_F1:.6f} -> {DAY23_MACRO_F1:.6f}"
)
print(
    f"Macro-F1 delta              : "
    f"{MACRO_F1_DELTA:+.6f} "
    f"({MACRO_F1_DELTA * 100:+.2f} pp)"
)
print(
    f"Macro-F1 delta 95% CI       : "
    f"[{MACRO_F1_DELTA_CI[0]:+.6f}, "
    f"{MACRO_F1_DELTA_CI[1]:+.6f}]"
)


print("\n[GENERALIZATION GAP]")
print(
    f"HistVal - OptDev Accuracy   : "
    f"{FINAL_HISTVAL_ACCURACY - float(FINAL_OPT_DEV_METRICS['accuracy']):+.6f}"
)
print(
    f"HistVal - OptDev Macro-F1   : "
    f"{FINAL_HISTVAL_MACRO_F1 - float(FINAL_OPT_DEV_METRICS['macro_f1']):+.6f}"
)


print("\n[FIREWALL]")
print(
    "Historical Validation       : EVALUATED ONCE AND FROZEN"
)
print(
    "Historical inference rerun  : FORBIDDEN"
)
print(
    "Post-validation retuning    : FORBIDDEN"
)
print(
    "Future F1-F6                : LOCKED"
)


print("\n[TERMINAL ARTIFACT CHECKS]")

for name, passed in terminal_artifact_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[SCIENTIFIC CHECKS]")

for name, passed in terminal_scientific_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Optimization trajectory     : "
    f"{DAY23_FINAL_TRAJECTORY_PATH}"
)
print(
    f"Final scientific summary    : "
    f"{DAY23_FINAL_SUMMARY_PATH}"
)
print(
    f"Completion checkpoint       : "
    f"{DAY23_FINAL_COMPLETION_PATH}"
)


print("\n" + "-" * 120)

if all_terminal_scientific_checks_pass:

    print(
        "DAY 23 CELL 27R STATUS: PASS — "
        "FINAL DAY23 SCIENTIFIC STATE RECOVERED FROM FROZEN ARTIFACTS; "
        "NO EXPENSIVE EXPERIMENTS RERUN; READY FOR NOTEBOOK SAVE"
    )

else:

    print(
        "DAY 23 CELL 27R STATUS: FAIL — "
        "DO NOT CLOSE DAY23"
    )

print("=" * 120)

DAY 23 — RESTART-SAFE FINAL SCIENTIFIC INTEGRATION

[RECOVERY MODE]
Runtime state               : RESET
Scientific state            : RECOVERED FROM FROZEN DISK ARTIFACTS
Training rerun              : NO
Inference rerun             : NO

[FINAL MODEL]
Base model                  : meta-llama/Meta-Llama-3-8B
Sampling                    : natural
Training horizon            : 500
Learning rate               : 0.0002
LoRA                        : r=16, alpha=64
Seed                        : 42

[OPTIMIZATION TRAJECTORY]


,stage,intervention,sampling,max_steps,learning_rate,lora_r,lora_alpha,optimization_dev_accuracy,optimization_dev_macro_f1
0,A,baseline reproduction,natural,200,0.0001,8,32,0.779542,0.726037
1,B,training horizon,natural,500,0.0001,8,32,0.840842,0.808201
2,C,sampling audit,natural,500,0.0001,8,32,0.840842,0.808201
3,D,learning-rate audit,natural,500,0.0002,8,32,0.843427,0.813516
4,E,LoRA capacity audit,natural,500,0.0002,16,64,0.852659,0.820731



[FINAL HISTORICAL VALIDATION]
Accuracy                    : 0.838710 [0.829830, 0.847585]
Macro-F1                    : 0.804130 [0.793078, 0.814513]

[PAIRED IMPROVEMENT VS DAY15]
Accuracy                    : 0.789042 -> 0.838710
Accuracy delta              : +0.049667 (+4.97 pp)
Accuracy delta 95% CI       : [+0.040621, +0.059054]
Macro-F1                    : 0.745678 -> 0.804130
Macro-F1 delta              : +0.058452 (+5.85 pp)
Macro-F1 delta 95% CI       : [+0.047362, +0.069502]

[GENERALIZATION GAP]
HistVal - OptDev Accuracy   : -0.013949
HistVal - OptDev Macro-F1   : -0.016601

[FIREWALL]
Historical Validation       : EVALUATED ONCE AND FROZEN
Historical inference rerun  : FORBIDDEN
Post-validation retuning    : FORBIDDEN
Future F1-F6                : LOCKED

[TERMINAL ARTIFACT CHECKS]
PASS   — stage_e_selection_summary
PASS   — stage_e_selected_adapter
PASS   — final_configuration
PASS   — final_firewall
PASS   — final_freeze_complete
PASS   — final_histval_predictions
PASS 